# HERMES stage attribution — recent construction versus history readout

Upload **this notebook alone**, preferably into your existing AutoDL container, then **Restart Kernel → Run All**. Defaults need no edits. It reuses verified assets under `/root/autodl-tmp/hermes_causal_pilot` and writes to `/root/autodl-tmp/hermes_stage_attribution_v2`, preserving the first run. A fresh container is supported with extra setup/download/feature preparation.

**Main experiment: 40 questions × five conditions = 200 answers.** Each input uses the last up to 20 original sampled frames: older H (up to 16) plus recent R (last up to four). All answers reset state and use no eviction, past-answer guidance or gold-dependent policy.

| Arm | H visible while R is encoded? | H directly visible during question/answer? |
|---|---|---|
| A | Yes | Yes |
| B | No | Yes |
| C | Yes | No |
| D | No | No |
| Compact recent-4 | No H present | No H present; ordinary compact positions |

A–D retain identical physical H/R cache slots and logical positions. B/D hide H during R encoding, validated against physically independent R construction at the same RoPE positions. C/D mask H at **every layer for question prefill and every generated token**. C can still carry history indirectly through R. Compact recent-4 checks the position/sink-distance difference. Questions with no H are null controls, not extra independent history interventions.

The default also runs **four separate A–D answers for 204/q3's longer prefix**, kept separate from the main table. Set `RUN_LONG_CASE=False` before a new run if you want only the 200-answer matrix. This is a bounded diagnostic/existing-mechanism comparator, not a new method or independent confirmation. B is not a complete DSCache reproduction.

**Mandatory gates:** feature/native-attention preflight; explicit-mask versus native logits; independent-construction K/V agreement at fixed positions; masked readout versus physically removed H through prefill and two decode steps; zero hidden-H/future attention at all layers; source-cache preservation. A and compact must replay their earlier choices on every question or the experiment stops and exports failure evidence. Do not bypass a mismatch by rerunning.

The two-frame stage preflight checks mask and position equivalence with FP32 math attention, restores the original model precision, then repeats the comparisons in FP16. The evidence ZIP includes per-layer K/V errors in `exports/stage_preflight_diagnostics.json`.

Recommended: **RTX 4090 / 24 GB**, CUDA 12.8-compatible driver, 18 GiB free for fresh setup. Same pinned 0.5B model, source sampling, prompt/parser and greedy 16-token policy as before. The kernel never imports torch. Shared A/B/compact cache construction is recorded separately; held control snapshots and transient buffers are included in observed allocation, so per-answer times are not standalone algorithm efficiency.

Setup and inference have stage output, per-arm results, progress JSON and 30-second liveness messages. `MAX_HOURS=1.0` is a soft post-model budget checked between question blocks, excluding setup/download/model loading/final packaging; it is not a runtime estimate. Rerun with the same WORK/settings to resume validated checkpoints; change WORK for changed code or long-case settings.

At completion download **`HERMES_stage_attribution_evidence.zip`** and save the executed notebook. It includes source/config/feature provenance, all masks/positions/groups, preflight metrics, checkpoints, paired summaries and closed logs. Weights, videos and feature tensors are excluded. Keep those assets. Local CPU tests cannot certify a complete GPU execution; failed gates stop QA.


In [1]:
!source /etc/network_turbo

设置成功!
注意：
1. 仅限学术用途和加速访问github/huggingface，不承诺稳定性
2. 开启加速后对访问其他资源如pip源等会*更慢*


In [2]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_stage_attribution_v2')
REUSE = Path('/root/autodl-tmp/hermes_causal_pilot')
RUN_LONG_CASE = True
MAX_HOURS = 1.0                 # soft post-model budget; 0 disables; rerun resumes
REPO_URL = 'https://github.com/md-hameem/HERMES-reproduction.git'
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
MODEL_SHA = '74dd0bf867a4cda7950c17663794267c60cf4b40'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_STAGE_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
assert MAX_HOURS >= 0
REPO = WORK/'repository'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'stage_runner.py'
print('Plan: 200 main stage answers plus four separate long-case answers if enabled')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')


Plan: 200 main stage answers plus four separate long-case answers if enabled
WORK: /root/autodl-tmp/hermes_stage_attribution_v2 
Reuse candidate: /root/autodl-tmp/hermes_causal_pilot
Mode: AutoDL execution


## 1. Pinned source, network, environment and model
Existing assets are verified before reuse. Missing assets are fetched/prepared separately.


In [3]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    env.update(HF_HOME=str((REUSE if (REUSE/'cache/huggingface').is_dir() else WORK)/'cache/huggingface'), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    assert free >= 18, f'Need at least 18 GiB free on data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 15000, 'Use a GPU with at least 15 GiB VRAM'
    # Never print the environment or proxy values.


GPU: NVIDIA GeForce RTX 4090, 24564 MiB, 580.105.08 
Free GiB: 23.5


In [4]:
SOURCE_HASHES = {'inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'video_qa/base.py': '795cf98e8d1125ea934dc2c9744ba1bbd3b29701f17aae2d935f382c7343b2d5', 'video_qa/hermes_vqa.py': 'f25bac6cd19db7148e7fa39cc9d2e96fb40d1f14709778f2490023a65dddc95b', 'scripts/build_failure_pilot.py': 'c81681e7f148702d07d1a2cc193a1ec32cf4f89778a911d92283088050ed7835', 'data/streamingbench/streamingbench_realtime.json': '6a2969953e11db44f5f8584d0d9fd3ec47c41dacf63807ef0003ccf446869f73'}
if not VALIDATE_ONLY:
    if not (REPO/'.git').is_dir():
        assert not REPO.exists() or not list(REPO.iterdir()), 'Non-Git source directory has files; use a new WORK'
        REPO.mkdir(parents=True, exist_ok=True)
        logged(['git', 'init', REPO], 'clone.log', ENV)
        logged(['git', 'remote', 'add', 'origin', REPO_URL], 'clone.log', ENV, REPO)
    remote = probe(['git', 'remote', 'get-url', 'origin'], ENV, REPO)
    assert remote.returncode == 0 and remote.stdout.strip() == REPO_URL, 'Unexpected source repository origin'
    dirty = probe(['git', 'status', '--porcelain', '--untracked-files=no'], ENV, REPO)
    assert dirty.returncode == 0 and not dirty.stdout.strip(), 'Pinned checkout has local edits; preserve them and use a new WORK'
    if probe(['git', 'cat-file', '-e', BASE_REVISION+'^{commit}'], ENV, REPO).returncode:
        for attempt in range(4):
            try:
                logged(['git', 'fetch', '--filter=blob:none', '--depth', '1', 'origin', BASE_REVISION], 'clone.log', ENV, REPO)
                break
            except RuntimeError:
                if attempt == 3:
                    raise
                print('GITHUB_FETCH_RETRY', attempt+1, flush=True)
                time.sleep(3*(attempt+1))
    logged(['git', 'sparse-checkout', 'init', '--cone'], 'clone.log', ENV, REPO)
    logged(['git', 'sparse-checkout', 'set', 'inference', 'video_qa', 'scripts', 'data/streamingbench'], 'clone.log', ENV, REPO)
    for attempt in range(4):
        try:
            logged(['git', 'checkout', '--detach', BASE_REVISION], 'clone.log', ENV, REPO)
            break
        except RuntimeError:
            if attempt == 3:
                raise
            print('GITHUB_CHECKOUT_RETRY', attempt+1, flush=True)
            time.sleep(3*(attempt+1))
    assert probe(['git', 'rev-parse', 'HEAD'], ENV, REPO).stdout.strip() == BASE_REVISION
    for rel, expected in SOURCE_HASHES.items():
        assert hashlib.sha256((REPO/rel).read_bytes().replace(b'\r\n', b'\n')).hexdigest() == expected, 'Pinned source mismatch: '+rel
    print('PINNED_SOURCE_PASS', BASE_REVISION, flush=True)


SETUP_STAGE clone.log
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /root/autodl-tmp/hermes_stage_attribution_v2/repository/.git/
SETUP_STAGE clone.log
SETUP_STAGE clone.log
From https://github.com/md-hameem/HERMES-reproduction
 * branch            2a1385e0abc94debb64761b0465713c799a81c1c -> FETCH_HEAD
SETUP_STAGE clone.log
SETUP_STAGE clone.log
SETUP_STAGE clone.log
HEAD is now at 2a1385e Keep generated pilot artifacts consistent across platforms
PINNED_SOURCE_PASS 2a1385e0abc94debb64761b0465713c799a81c1c


In [5]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
if not VALIDATE_ONLY:
    candidate_python = REUSE/'venv/bin/python'
    if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
        PYTHON = candidate_python
        print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)
    PIP_ENV = package_environment(ENV)
    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                             'https://pypi.tuna.tsinghua.edu.cn/simple'], PIP_ENV)
            if attempt.returncode:
                logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                        'https://mirrors.aliyun.com/pypi/simple'], 'setup.log', PIP_ENV)
        UV = [sys.executable, '-m', 'uv']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)
        def install(packages, extra=()):
            install_packages(UV, PYTHON, packages, ENV, extra)
        install(['torch @ https://download-r2.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl'])
        install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
                 'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
                 'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
                 'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops', 'psutil', 'pandas'])
        # Download GitHub source with turbo, then install the local archive using
        # direct package networking for any isolated build dependencies.
        tf_archive = WORK/'downloads'/f'transformers-{TF_SHA}.zip'
        tf_archive.parent.mkdir(parents=True, exist_ok=True)
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path
dest = Path(sys.argv[2]); temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3: raise
        time.sleep(3 * (attempt + 1))
"""
            logged([sys.executable, '-c', fetch_tf,
                    f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}', tf_archive],
                   'source_download.log', ENV)
        install([str(tf_archive)], ['--no-deps'])
    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO/'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)


REUSING_VERIFIED_PINNED_ENVIRONMENT /root/autodl-tmp/hermes_causal_pilot/venv/bin/python
SETUP_STAGE environment.log
PINNED_ENV_PASS 2.10.0+cu128 4.45.0.dev0 NVIDIA GeForce RTX 4090
SETUP_STAGE environment_freeze.txt
accelerate==0.34.2
certifi==2026.10.23
charset-normalizer==3.5.2
cuda-bindings==12.9.4
cuda-pathfinder==1.9.1
decord==0.6.0
einops==0.8.2
filelock==4.2.12
fsspec==2026.9.0
hf-xet==1.8.0
huggingface-hub==0.34.4
idna==3.20
Jinja2==3.1.6
logzero==1.7.0
MarkupSafe==3.0.4
mpmath==1.3.0
networkx==3.7
numpy==1.26.4
nvidia-cublas-cu12==12.8.4.1
nvidia-cuda-cupti-cu12==12.8.90
nvidia-cuda-nvrtc-cu12==12.8.93
nvidia-cuda-runtime-cu12==12.8.90
nvidia-cudnn-cu12==9.10.2.21
nvidia-cufft-cu12==11.3.3.83
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12==10.3.9.90
nvidia-cusolver-cu12==11.7.3.90
nvidia-cusparse-cu12==12.5.8.93
nvidia-cusparselt-cu12==0.7.1
nvidia-nccl-cu12==2.27.5
nvidia-nvjitlink-cu12==12.8.93
nvidia-nvshmem-cu12==3.4.5
nvidia-nvtx-cu12==12.8.90
packaging==26.3
pandas==3.

In [6]:
MODEL_DOWNLOAD = r"""
import json, sys
from pathlib import Path
from huggingface_hub import snapshot_download
from safetensors import safe_open
path = Path(snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], cache_dir=sys.argv[3],
    allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.vocab', 'tokenizer.*', 'merges.txt'],
    ignore_patterns=['onnx/*', '*.onnx', '*.bin', '*.msgpack'], max_workers=4))
assert path.name == sys.argv[2] and (path/'config.json').is_file()
files = list(path.glob('*.safetensors'))
assert files
for file in files:
    with safe_open(str(file), framework='pt', device='cpu') as handle:
        assert list(handle.keys())
Path(sys.argv[4]).write_text(json.dumps(str(path)), encoding='utf-8')
print('MODEL_REVISION_PASS', path.name)
"""
if not VALIDATE_ONLY:
    pointer = WORK/'model_path.json'
    failures = []
    for host in ('https://hf-mirror.com', 'https://huggingface.co'):
        try:
            logged([PYTHON, '-c', MODEL_DOWNLOAD, MODEL_ID, MODEL_SHA,
                    Path(ENV['HF_HOME'])/'hub', pointer], 'model_download.log', {**ENV, 'HF_ENDPOINT': host})
            break
        except RuntimeError as exc:
            failures.append(type(exc).__name__)
            print('Model endpoint failed; trying fallback', flush=True)
    else:
        raise RuntimeError('Model download failed on both endpoints; rerun to resume cached files')
    MODEL_PATH = Path(json.loads(pointer.read_text(encoding='utf-8')))
    assert MODEL_PATH.name == MODEL_SHA


SETUP_STAGE model_download.log

Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 1577.82it/s]
MODEL_REVISION_PASS 74dd0bf867a4cda7950c17663794267c60cf4b40


## 2. Embedded runner and immutable references
Source modules come from this tested checkout; uploaded evidence supplies comparison data only.


In [7]:
RUNNER_SOURCE = r'''"""Factor history visibility during recent encoding and question/answer readout.

No production inference files are changed. Masking H during R construction is
validated against physically independent R construction at identical RoPE positions.
"""
import argparse
import contextlib
import copy
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZIP_DEFLATED,ZipFile

try:
    import stage_context_utils as c
except ModuleNotFoundError as exc:
    if exc.name != "stage_context_utils": raise
    import autodl_context_diagnostic as c

d,p=c.d,c.p
ARMS=("A_history_both","B_independent_recent","C_hide_history_readout","D_independent_hide_readout","compact_recent4")
ARTIFACT="HERMES_stage_attribution_evidence.zip"


def split_frames(video,qi,long_prefix=False):
    end=d.prefix_end(video,qi)
    selected=list(range(0 if long_prefix else max(0,end-20),end))
    recent=selected[-4:]
    history=selected[:-len(recent)] if recent else selected
    d.validate_frames(selected,video,qi,video["timestamps"])
    return history,recent


def cache_length(cache):
    if cache is None:return 0
    if hasattr(cache,"get_seq_length"):return int(cache.get_seq_length())
    return int(cache[0][0].shape[2])


def causal_mask(batch,q,past,dtype,device,blocked_start,blocked_stop):
    import torch
    if not 0<=blocked_start<=blocked_stop<=past:
        raise ValueError("History mask span must lie entirely in physical past cache")
    mask=torch.zeros((batch,1,q,past+q),dtype=dtype,device=device)
    minimum=torch.finfo(dtype).min
    mask[:,:,:,blocked_start:blocked_stop]=minimum
    future=torch.ones((q,q),dtype=torch.bool,device=device).triu(1)
    mask[:,:,:,past:].masked_fill_(future,minimum)
    return mask


class HistoryMask:
    """Apply one explicit causal+H mask to every LM call, including all decode steps."""
    def __init__(self,model,start,stop,force=False):
        self.model,self.start,self.stop,self.force=model,start,stop,force
        self.events=[];self.handle=None

    def hook(self,module,args,kwargs):
        if args:raise ValueError("Expected named LM inputs for audited mask")
        if kwargs.get("attention_mask") is not None:raise ValueError("Refuse to overwrite another attention mask")
        value=kwargs.get("inputs_embeds")
        if value is None:value=kwargs["input_ids"]
        batch,q=value.shape[:2]
        past=cache_length(kwargs.get("past_key_values"))
        mask=causal_mask(batch,q,past,self.model.dtype,self.model.device,self.start,self.stop)
        kwargs["attention_mask"]=mask
        self.events.append(dict(query_tokens=q,past_tokens=past,blocked_start=self.start,blocked_stop=self.stop,
                               shape=list(mask.shape),policy="causal plus hidden historical physical span"))
        return args,kwargs

    def __enter__(self):
        if self.stop>self.start or self.force:
            self.handle=self.model.language_model.register_forward_pre_hook(self.hook,with_kwargs=True)
        return self

    def __exit__(self,*exc):
        if self.handle is not None:self.handle.remove()
        return False


def snapshot(model):
    return dict(kv=[(k.clone(),v.clone()) for k,v in model.kv_cache],positions=[x.clone() for x in model._position_ids_cache],
                prompt=model.visual_start_idx)


def restore(model,state):
    model._layer_position_ids.clear()
    model.kv_cache=[(k.clone(),v.clone()) for k,v in state["kv"]]
    model._position_ids_cache=[x.clone() for x in state["positions"]]
    model.visual_start_idx=state["prompt"]
    model.conv_history=[]
    model.token_activity_cache=[None]*model.num_layers


def prefix_state(state,length):
    return dict(kv=[(k[:,:,:length].contiguous().clone(),v[:,:,:length].contiguous().clone()) for k,v in state["kv"]],
                positions=[x[:length].clone() for x in state["positions"]],prompt=state["prompt"])


def drop_history(state,start,stop):
    import torch
    n=state["kv"][0][0].shape[2]
    keep=torch.cat((torch.arange(start,device=state["positions"][0].device),torch.arange(stop,n,device=state["positions"][0].device)))
    return dict(kv=[(k.index_select(2,keep),v.index_select(2,keep)) for k,v in state["kv"]],
                positions=[x.index_select(0,keep) for x in state["positions"]],prompt=state["prompt"])


def assert_state(model,state):
    import torch
    for (k,v),(expected_k,expected_v),pos,expected_pos in zip(model.kv_cache,state["kv"],model._position_ids_cache,state["positions"]):
        torch.testing.assert_close(k,expected_k,atol=0,rtol=0)
        torch.testing.assert_close(v,expected_v,atol=0,rtol=0)
        torch.testing.assert_close(pos,expected_pos,atol=0,rtol=0)


def ingest(model,bank,indices,meta,logical_start=None):
    groups=c.input_groups(indices,"original",meta)
    for group in groups:
        if logical_start is None:
            p.ingest_features(model,bank.get(group),len(group))
        else:
            # Native independent reference: physical cache omits H while R's logical positions retain its gap.
            original=model._get_next_start_pos_per_layer
            try:
                model._get_next_start_pos_per_layer=lambda:[logical_start]*model.num_layers
                p.ingest_features(model,bank.get(group),len(group))
            finally:model._get_next_start_pos_per_layer=original
            logical_start+=len(group)*196
    return groups


def prepare_states(model,bank,meta,history,recent):
    import torch
    p.reset(model)
    full_groups=ingest(model,bank,history+recent,meta)
    a=snapshot(model)
    d.assert_fresh_cache(model,len(history)+len(recent))
    prompt=model.visual_start_idx
    stop=prompt+len(history)*196
    if history:
        restore(model,prefix_state(a,stop))
        with HistoryMask(model,prompt,stop) as mask:
            recent_groups=ingest(model,bank,recent,meta)
        b=snapshot(model)
        d.assert_fresh_cache(model,len(history)+len(recent))
        for (ak,av),(bk,bv) in zip(a["kv"],b["kv"]):
            torch.testing.assert_close(ak[:,:,:stop],bk[:,:,:stop],atol=0,rtol=0)
            torch.testing.assert_close(av[:,:,:stop],bv[:,:,:stop],atol=0,rtol=0)
        events=mask.events
    else:
        b=a;recent_groups=full_groups;events=[]
    p.reset(model)
    compact_groups=ingest(model,bank,recent,meta)
    compact=snapshot(model)
    d.assert_fresh_cache(model,len(recent))
    return dict(A=a,B=b,compact=compact),dict(history_stop=stop,prompt_tokens=prompt,full_groups=full_groups,
        independent_recent_groups=recent_groups,independent_encoding_mask_trace=events,compact_groups=compact_groups,
        state_bytes={name:sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in state["kv"]) for name,state in [("A",a),("B",b),("compact",compact)]},
        note="A/B/compact control snapshots are prepared once per question and held simultaneously; answer-arm timing is not standalone system cost")


def probe(model,state,ids,start,stop,force=False,steps=2,attentions=False):
    import torch
    restore(model,state)
    cache=model.kv_cache
    logical=int(state["positions"][0][-1])+1
    logits=[];weights=[]
    with HistoryMask(model,start,stop,force=force) as mask:
        try:
            for step in range(steps+1):
                current=ids if step==0 else ids[:,:1]
                n=current.shape[1]
                model._set_rotary_required_len(logical+n)
                for layer in range(model.num_layers):model._layer_position_ids[layer]=model._build_position_ids(logical,n,current.shape[0])
                output=model.language_model(input_ids=current,past_key_values=cache,position_ids=model._layer_position_ids[0],
                                            use_cache=True,output_attentions=attentions,return_dict=True)
                cache=output.past_key_values
                logits.append(output.logits.detach().clone())
                if attentions:weights.append(output.attentions)
                logical+=n
        finally:model._layer_position_ids.clear()
    return logits,weights,mask.events


def error_statistics(actual,expected,atol,rtol):
    import torch
    a,b=actual.float(),expected.float()
    if a.shape!=b.shape or not torch.isfinite(a).all() or not torch.isfinite(b).all():
        raise ValueError("Nonfinite or shape-mismatched preflight tensors")
    delta=(a-b).abs()
    return dict(elements=a.numel(),max_abs=float(delta.max()),rms=float(delta.square().mean().sqrt()),
                relative_l2=float(torch.linalg.vector_norm(a-b)/torch.linalg.vector_norm(b).clamp_min(1e-12)),
                exceeds_original_005_bound=int((delta>(.005+.005*b.abs())).sum()),
                exceeds_active_bound=int((delta>(atol+rtol*b.abs())).sum()))


class PrecisionProxy:
    """Override only public dtype; all state/methods stay on the original model."""
    def __init__(self,model,dtype):
        object.__setattr__(self,"_wrapped",model);object.__setattr__(self,"_dtype",dtype)
    def __getattr__(self,name):
        return object.__getattribute__(self,"_dtype") if name=="dtype" else getattr(object.__getattribute__(self,"_wrapped"),name)
    def __setattr__(self,name,value):setattr(object.__getattribute__(self,"_wrapped"),name,value)


@contextlib.contextmanager
def fp32_structural_reference(model):
    """Temporarily promote LM parameters, preserving original mixed buffer dtypes/values.

    Vision weights are never cast. Cached rotary buffers are restored exactly,
    rather than blindly calling half() on float32 frequency buffers afterward.
    """
    import torch
    from torch.nn.attention import sdpa_kernel,SDPBackend
    lm=model.language_model
    parameter_dtypes={name:value.dtype for name,value in lm.named_parameters()}
    buffers={name:value.clone() for name,value in lm.named_buffers()}
    rotary=[(layer.self_attn.rotary_emb,layer.self_attn.rotary_emb.max_seq_len_cached,
             getattr(layer.self_attn.rotary_emb,"_required_seq_len",0)) for layer in lm.model.layers]
    try:
        for value in lm.parameters():
            if value.is_floating_point() and value.dtype not in (torch.float16,torch.bfloat16,torch.float32):
                raise ValueError("Unexpected parameter dtype for reversible FP32 reference")
        model.clear_cache()
        for value in lm.parameters():
            if value.is_floating_point():value.data=value.data.float()
        # Both reference paths use the same high-precision math implementation.
        with sdpa_kernel(SDPBackend.MATH):yield PrecisionProxy(model,torch.float32)
    finally:
        for name,value in lm.named_parameters():value.data=value.data.to(parameter_dtypes[name])
        for name,value in buffers.items():
            parent,_,leaf=name.rpartition(".")
            setattr(lm.get_submodule(parent) if parent else lm,leaf,value)
        for module,length,required in rotary:
            module.max_seq_len_cached=length;module._required_seq_len=required
        model.clear_cache();model.kv_cache=None
        model._layer_position_ids.clear();model._position_ids_cache=[None]*model.num_layers
        model.token_activity_cache=[None]*model.num_layers


def _preflight_pass(model,bank,meta,ids,atol=.02,rtol=.005,kv_atol=.005,kv_rtol=.005,diagnostics=None,save=None):
    """Check physical masking against independent construction and compact readout."""
    import torch
    p.reset(model);sink=snapshot(model);s=sink["prompt"]
    states,detail=prepare_states(model,bank,meta,[0],[1])
    stop=s+196
    restore(model,sink)
    ingest(model,bank,[1],meta,logical_start=stop)
    independent=snapshot(model)
    construction_error=0.0
    comparisons=[]
    for layer,((fk,fv),(ik,iv)) in enumerate(zip(states["B"]["kv"],independent["kv"])):
        for kind,actual,expected in (("K",fk[:,:,stop:],ik[:,:,s:]),("V",fv[:,:,stop:],iv[:,:,s:])):
            comparisons.append(dict(layer=layer,kind=kind,**error_statistics(actual,expected,kv_atol,kv_rtol)))
    if diagnostics is not None:
        diagnostics.update(dtype=str(model.dtype),kv_atol=kv_atol,kv_rtol=kv_rtol,construction_by_layer=comparisons)
        if save:save()
    for layer,((fk,fv),(ik,iv)) in enumerate(zip(states["B"]["kv"],independent["kv"])):
        for kind,actual,expected in (("K",fk[:,:,stop:],ik[:,:,s:]),("V",fv[:,:,stop:],iv[:,:,s:])):
            torch.testing.assert_close(actual,expected,atol=kv_atol,rtol=kv_rtol,
                                       msg=lambda msg: f"Construction preflight layer {layer} {kind}: {msg}")
            construction_error=max(construction_error,float((actual-expected).abs().max()))
    records=[]
    plain,_,_=probe(model,states["A"],ids,0,0)
    explicit,_,all_events=probe(model,states["A"],ids,0,0,force=True)
    for name,left,right in [("all_visible_explicit_causal_matches_native",plain,explicit)]:
        maximum=0.0
        for a,b in zip(left,right):
            torch.testing.assert_close(a,b,atol=atol,rtol=rtol)
            assert torch.equal(a[:,-1].argmax(-1),b[:,-1].argmax(-1)),"Preflight argmax changed"
            maximum=max(maximum,float((a-b).abs().max()))
        records.append(dict(check=name,max_logit_error=maximum,argmax_equal=True))
    for label,state in (("contextual_recent",states["A"]),("independent_recent",states["B"])):
        masked,_,events=probe(model,state,ids,s,stop)
        compact,_,_=probe(model,drop_history(state,s,stop),ids,0,0)
        maximum=0.0
        for a,b in zip(masked,compact):
            torch.testing.assert_close(a,b,atol=atol,rtol=rtol)
            assert torch.equal(a[:,-1].argmax(-1),b[:,-1].argmax(-1)),"Masked/compact preflight argmax changed"
            maximum=max(maximum,float((a-b).abs().max()))
        assert len(events)==3 and [e["query_tokens"] for e in events][1:]==[1,1]
        records.append(dict(check=label+"_masked_matches_physically_removed_history",max_logit_error=maximum,
                            argmax_equal=True,query_and_decode_mask_calls=len(events)))
    _,weights,events=probe(model,states["B"],ids,s,stop,attentions=True)
    for step,layers in enumerate(weights):
        for attention in layers:
            assert not torch.any(attention[...,s:stop]),"Historical attention leaked"
            q=attention.shape[-2]
            assert not torch.any(torch.triu(attention[...,-q:],diagonal=1)),"Future attention leaked"
    # Query path must leave the stored control source unchanged.
    restore(model,states["A"])
    source=snapshot(model)
    probe(model,states["A"],ids,s,stop)
    for (a,b),(x,y) in zip(states["A"]["kv"],source["kv"]):
        torch.testing.assert_close(a,x,atol=0,rtol=0);torch.testing.assert_close(b,y,atol=0,rtol=0)
    model.clear_cache()
    return dict(status="pass",history_tokens=196,recent_tokens=196,prompt_tokens=s,
        independent_construction_max_kv_error=construction_error,comparisons=records,
        all_layer_hidden_history_attention_zero=True,all_layer_future_attention_zero=True,
        source_snapshots_unchanged=True,attention_checks_steps=len(events),layers=model.num_layers,
        tolerances=dict(kv_atol=kv_atol,kv_rtol=kv_rtol,logits_atol=atol,logits_rtol=rtol),
        scope="One two-frame visual prefix; explicit history mask at question prefill and two forced decode steps; not every production prefix")


def numerical_preflight(model,bank,meta,ids,atol=.02,rtol=.005,diagnostic_path=None):
    import torch
    diagnostics=dict(policy="v2_strict_fp32_structure_then_fp16_production",fp32={},production={})
    def save():
        if diagnostic_path is not None:p.atomic_json(diagnostic_path,diagnostics)
    if model.dtype in (torch.float16,torch.bfloat16):
        print("STAGE_PREFLIGHT strict FP32 structural reference (math attention)",flush=True)
        with fp32_structural_reference(model) as reference:
            structural=_preflight_pass(reference,bank,meta,ids,atol=.001,rtol=.001,kv_atol=.0001,kv_rtol=.0001,
                                       diagnostics=diagnostics["fp32"],save=save)
        diagnostics["fp32"]["result"]=structural;save()
        print("STAGE_PREFLIGHT restored production precision; checking FP16 numerical bounds",flush=True)
        result=_preflight_pass(model,bank,meta,ids,atol=atol,rtol=rtol,kv_atol=.01,kv_rtol=.01,
                               diagnostics=diagnostics["production"],save=save)
        result["fp32_structural_reference"]=structural
        result["production_precision_restored"]=True
    else:
        result=_preflight_pass(model,bank,meta,ids,atol=atol,rtol=rtol,diagnostics=diagnostics["production"],save=save)
    result["validation_policy"]=diagnostics["policy"]
    result["numerical_limits"]="FP32 checks mask and position equivalence. Production precision has a separate K/V tolerance. Zero hidden-history/future attention, argmax and per-question replay gates are mandatory."
    diagnostics["production"]["result"]=result;save()
    return result


def checkpoint_valid(value,key,identity,arm,history,recent):
    try:
        row=value["row"]
        return (value["identity"]==key and value["row_sha256"]==d.fingerprint(row) and all(row[k]==v for k,v in identity.items())
            and row["method"]==arm and row["history_indices"]==history and row["recent_indices"]==recent
            and ((row["status"]=="answered" and row["correct"]==int(row["pred_choice"]==row["gold_choice"]))
                 or (row["status"]=="skipped" and row["correct"] is None and bool(row["skip_reason"]))))
    except (KeyError,ValueError,TypeError):return False


def run_answer(model,analyzer,video,qi,arm,scope,states,detail,history,recent,feature_hash,config_hash,args,old_choice=None):
    import torch
    identity=d.question_identity(video,qi,arm)
    key=d.fingerprint(dict(config=config_hash,scope=scope,question=identity,history=history,recent=recent,features=feature_hash))
    dest=args.work/"checkpoints"/scope/arm/f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value=json.loads(dest.read_text(encoding="utf-8"))
        if not checkpoint_valid(value,key,identity,arm,history,recent):raise d.RunConflict("Stage checkpoint differs; preserve and use new WORK")
        if value["row"].get("reference_choice_matches") is False:
            raise ValueError("Previously saved replay mismatch remains unresolved; do not bypass the attribution gate")
        print(f"REUSE {scope} {arm} sample={video['sample_id']} q={qi}",flush=True)
        return value["row"]
    construction=arm in (ARMS[1],ARMS[3])
    hidden=arm in (ARMS[2],ARMS[3])
    state=states["compact" if arm==ARMS[4] else "B" if construction else "A"]
    s=state["prompt"];stop=s+len(history)*196
    p.report_progress("stage_answer",sample_id=video["sample_id"],qa_index=qi,arm=arm,scope=scope)
    row=dict(**identity,scope=scope,history_indices=history,recent_indices=recent,history_empty=not history,
        independent_construction=construction,history_readout_hidden=hidden,factor_cell=arm!=ARMS[4],
        history_physically_present=bool(history) and arm!=ARMS[4],status="answered",skip_reason="",
        feature_fingerprint=feature_hash,cache_lengths=[x[0].shape[2] for x in state["kv"]],
        recent_logical_start=s+(0 if arm==ARMS[4] else len(history)*196),
        recent_physical_start=s+(0 if arm==ARMS[4] else len(history)*196),
        actual_visual_indices=recent if arm==ARMS[4] else history+recent,mask_trace=[],
        query_logical_start=int(state["positions"][0][-1])+1,
        pred_choice="",pred_answer="",correct=None,answer_s=0.0,answer_wall_s=0.0,peak_allocated_gib=0.0,
        reference_choice=old_choice if old_choice is not None else "",reference_choice_matches="")
    started=time.perf_counter()
    restore(model,state)
    if model.conv_history:raise ValueError("Previous answer history leaked")
    random.seed(p.SEED);torch.manual_seed(p.SEED)
    torch.cuda.reset_peak_memory_stats()
    try:
        prompt=analyzer.format_mcqa_prompt(identity["question"],identity["choices"])
        qlen=len(model.processor.tokenizer(prompt["prompt"]).input_ids)
        context=model.language_model.config.max_position_embeddings
        if int(state["positions"][0][-1])+1+qlen+48>context:
            if scope=="main":raise ValueError("Main context limit exceeded")
            row.update(status="skipped",skip_reason="context_limit",correct=None)
        else:
            p.sync();before=time.perf_counter()
            with HistoryMask(model,s,stop if hidden else s) as mask:
                answer=model.question_answering(prompt,max_new_tokens=16)
            p.sync();row["answer_s"]=time.perf_counter()-before
            row["mask_trace"]=mask.events
            if hidden and history:
                assert mask.events and all(e["blocked_start"]==s and e["blocked_stop"]==stop for e in mask.events)
                assert all(e["query_tokens"]==1 for e in mask.events[1:])
            assert_state(model,state)
            choice=analyzer.extract_characters_regex(answer)
            row.update(pred_choice=choice,pred_answer=answer.replace("\n",""),correct=int(choice==identity["gold_choice"]),
                       peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
            if old_choice is not None:row["reference_choice_matches"]=choice==old_choice
    finally:model._layer_position_ids.clear();model.clear_cache()
    row["answer_wall_s"]=time.perf_counter()-started
    value=dict(identity=key,row=row,row_sha256=d.fingerprint(row))
    if not checkpoint_valid(value,key,identity,arm,history,recent):raise ValueError("Stage checkpoint failed validation")
    p.atomic_json(dest,value)
    print(f"STAGE_DONE {scope} {arm} sample={video['sample_id']} q={qi} correct={row['correct']} replay={row['reference_choice_matches']}",flush=True)
    if row["reference_choice_matches"] is False:
        raise ValueError("No-op/compact historical replay mismatch: evidence saved; stop attribution and audit numerics")
    return row


def export(rows,work):
    fields=["scope","sample_id","qa_index","method","question","choices","gold_choice","pred_choice","correct","status","skip_reason",
        "history_indices","recent_indices","actual_visual_indices","history_empty","factor_cell","history_physically_present",
        "independent_construction","history_readout_hidden","cache_lengths","query_logical_start",
        "recent_logical_start","mask_trace","reference_choice","reference_choice_matches","answer_s","answer_wall_s","peak_allocated_gib"]
    p.write_csv(work/"exports/predictions.csv",[{k:r.get(k,"") for k in fields} for r in rows])
    summary=[]
    for scope in ("main","long_204_q3"):
        for arm in (ARMS if scope=="main" else ARMS[:4]):
            subset=[r for r in rows if r["scope"]==scope and r["method"]==arm]
            if not subset:continue
            answers=[r for r in subset if r["status"]=="answered"]
            summary.append(dict(scope=scope,arm=arm,recorded=len(subset),answered=len(answers),correct=sum(r["correct"] for r in answers),
                history_empty_questions=sum(r["history_empty"] for r in answers)))
    p.write_csv(work/"exports/summary.csv",summary)
    pairs=[]
    keys={(r["scope"],r["sample_id"],r["qa_index"]):{} for r in rows}
    for r in rows:keys[r["scope"],r["sample_id"],r["qa_index"]][r["method"]]=r
    for a,b in [(ARMS[0],ARMS[1]),(ARMS[0],ARMS[2]),(ARMS[1],ARMS[3]),(ARMS[2],ARMS[3]),(ARMS[3],ARMS[4])]:
        for scope in ("main","long_204_q3"):
            matched=[(v[a],v[b]) for k,v in keys.items() if k[0]==scope and a in v and b in v and v[a]["status"]==v[b]["status"]=="answered"]
            if matched:pairs.append(dict(scope=scope,arm_a=a,arm_b=b,questions=len(matched),a_only_correct=sum(bool(x["correct"]) and not y["correct"] for x,y in matched),
                b_only_correct=sum(not x["correct"] and bool(y["correct"]) for x,y in matched),choice_changes=sum(x["pred_choice"]!=y["pred_choice"] for x,y in matched),
                history_empty_pairs=sum(x["history_empty"] for x,y in matched)))
    if pairs:p.write_csv(work/"exports/paired.csv",pairs)
    text=["# Stage attribution screen","","Inspected development questions; no original-method or population claim.",
        "","| Scope | Arm | Answered | Correct |","|---|---|---:|---:|"]
    text += [f"| {x['scope']} | {x['arm']} | {x['answered']} | {x['correct']} |" for x in summary]
    text += ["","A: H visible in encoding/readout. B: H masked during R encoding only. C: H masked at every question/decode call only. D: H masked at both stages. Compact: fresh recent-4 at compact positions.",
        "B/D use a physical H mask validated against independent cache construction at preserved logical positions; they are not complete DSCache implementations.",
        "H remains physically stored in A-D. C may retain historical information mixed indirectly into R. History-empty questions are null controls.",
        "Long 204/q3 results are separate from the main 40-question matrix. A and compact must replay earlier choices or execution stops.",
        "Shared cache preparation/snapshots are logged separately; per-answer timing/peak allocation includes shared control snapshots and is not standalone algorithm efficiency."]
    (work/"exports/report.md").write_text("\n".join(text)+"\n",encoding="utf-8")


def package(work,status):
    captured=work/"captured_logs";captured.mkdir(parents=True,exist_ok=True)
    for x in (work/"logs").glob("*"):
        if x.is_file():shutil.copy2(x,captured/x.name)
    if (work/"progress.json").exists():shutil.copy2(work/"progress.json",captured/"progress_snapshot.json")
    files=[]
    for area in ("exports","checkpoints","provenance","preparation","captured_logs"):
        files += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("stage_runner.py","stage_context_utils.py","context_input_utils.py","diagnostic_pilot_utils.py","reference.json","launch.json","asset_manifest.json","preflight.json","stage_preflight.json"):
        if (work/name).exists():files.append(work/name)
    manifest=[dict(path=x.relative_to(work).as_posix(),bytes=x.stat().st_size,sha256=p.digest(x)) for x in sorted(set(files))]
    p.atomic_json(work/"evidence_manifest.json",dict(status=status,files=manifest))
    target=work/ARTIFACT
    with ZipFile(target.with_suffix(".tmp"),"w",compression=ZIP_DEFLATED) as z:
        for x in manifest:z.write(work/x["path"],x["path"])
        z.write(work/"evidence_manifest.json","evidence_manifest.json")
    os.replace(target.with_suffix(".tmp"),target)
    print(f"EVIDENCE_ZIP={target}\nSHA256={p.digest(target)}",flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    reference=json.loads((args.work/"reference.json").read_text(encoding="utf-8"))
    d.check_reference(reference,args.repo)
    if args.model.name!=p.MODEL_SHA or torch.__version__!="2.10.0+cu128" or transformers.__version__!="4.45.0.dev0" or not torch.cuda.is_available():raise ValueError("Pinned model/CUDA environment required")
    config=dict(reference_sha256=p.digest(args.work/"reference.json"),sources={n:p.digest(args.work/n) for n in
        ("stage_runner.py","stage_context_utils.py","context_input_utils.py","diagnostic_pilot_utils.py")},arms=ARMS,
        model_revision=p.MODEL_SHA,torch=torch.__version__,transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),
        seed=p.SEED,main_window=20,recent_window=4,long_case=args.long_case,mask_policy="all layers, prefill and every decode; physical H retained",shared_control_snapshots=True)
    config_hash=d.fingerprint(config);path=args.work/"exports/run_config.json"
    if path.exists() and json.loads(path.read_text())["config_hash"]!=config_hash:raise d.RunConflict("Stage configuration changed; use new WORK")
    p.atomic_json(path,dict(**config,config_hash=config_hash))
    failure=args.work/"exports/failure.json"
    if failure.exists():
        dest=args.work/"logs"/f"previous_failure_{time.time_ns()}.json";dest.parent.mkdir(parents=True,exist_ok=True);os.replace(failure,dest)
    videos=copy.deepcopy(reference["videos"]);d.acquire_videos(videos,args.reuse,args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json",videos)
    sys.path.insert(0,str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8,os.cpu_count() or 1));torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    random.seed(p.SEED);np.random.seed(p.SEED);torch.manual_seed(p.SEED)
    p.report_progress("loading_stage_model")
    model,processor=load_model(model_path=str(args.model),kv_size=4000,streaming=True,sample_fps=p.FPS);model.eval()
    analyzer=HermesVQA(anno=[],save_dir=str(args.work),sample_fps=p.FPS,qa_model=model,qa_processor=processor)
    old={(r["sample_id"],r["qa_index"],r["method"]):r for r in reference["input_rows"]}
    rows=[];done=0;started=time.perf_counter();preflight_done=False;long_done=not args.long_case
    with torch.inference_mode():
        # Start with the already inspected 204 case, then preserve a deterministic video order.
        videos.sort(key=lambda v:(v["sample_id"]!=204,v["sample_id"]))
        for video in videos:
            directory,meta,_=d.acquire_features(model,video,args.reuse,args.work,d.fingerprint(dict(config=config_hash,video=video["video_sha256"])))
            bank=c.FeatureBank(directory,meta);feature_hash=d.fingerprint(meta)
            if not preflight_done:
                p.preflight(model,video,args.work)
                ids=processor.tokenizer("What is visible?",return_tensors="pt").input_ids.to(model.device)
                p.report_progress("stage_mask_position_preflight")
                p.atomic_json(args.work/"stage_preflight.json",numerical_preflight(model,bank,meta,ids,
                    diagnostic_path=args.work/"exports/stage_preflight_diagnostics.json"))
                preflight_done=True
            for qi in range(len(video["conversations"])):
                if done and args.max_hours and time.perf_counter()-started>args.max_hours*3600:break
                h,r=split_frames(video,qi)
                torch.cuda.reset_peak_memory_stats()
                before=time.perf_counter();states,detail=prepare_states(model,bank,meta,h,r);p.sync()
                detail.update(shared_preparation_s=time.perf_counter()-before,history_indices=h,recent_indices=r,
                              preparation_peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
                p.atomic_json(args.work/"preparation/main"/f"sample_{video['sample_id']}_q{qi}.json",detail)
                # Replay gates run first; no scientific factor comparison after a mismatch.
                for arm in (ARMS[0],ARMS[4],ARMS[1],ARMS[2],ARMS[3]):
                    previous=old[video["sample_id"],qi,"recent_20" if arm==ARMS[0] else "recent_4"] if arm in (ARMS[0],ARMS[4]) else None
                    rows.append(run_answer(model,analyzer,video,qi,arm,"main",states,detail,h,r,feature_hash,config_hash,args,
                                           previous["pred_choice"] if previous else None))
                del states
                if args.long_case and (video["sample_id"],qi)==(204,3):
                    h,r=split_frames(video,qi,True)
                    torch.cuda.reset_peak_memory_stats()
                    before=time.perf_counter();states,detail=prepare_states(model,bank,meta,h,r);p.sync()
                    detail.update(shared_preparation_s=time.perf_counter()-before,history_indices=h,recent_indices=r,
                                  preparation_peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
                    p.atomic_json(args.work/"preparation/long_204_q3.json",detail)
                    for arm in ARMS[:4]:
                        rows.append(run_answer(model,analyzer,video,qi,arm,"long_204_q3",states,detail,h,r,feature_hash,config_hash,args,
                                               reference["long_reference_choice"] if arm==ARMS[0] else None))
                    del states;long_done=True
                done+=1;export(rows,args.work)
                elapsed=time.perf_counter()-started
                print(f"QUESTION_BLOCK_DONE {done}/40 elapsed_min={elapsed/60:.2f} rough_remaining_min={(40-done)*elapsed/done/60:.2f}",flush=True)
            del bank
            if done<40 and args.max_hours and time.perf_counter()-started>args.max_hours*3600:break
    status="complete" if done==40 and long_done else "partial_time_budget"
    if status=="complete" and (sum(x["scope"]=="main" for x in rows)!=200 or any(x["status"]!="answered" for x in rows if x["scope"]=="main")):raise ValueError("Main stage matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json",dict(status=status,main_questions=done,main_planned=40,main_answers=sum(x["scope"]=="main" and x["status"]=="answered" for x in rows),
        long_case_done=long_done,long_case_enabled=bool(args.long_case),rows=len(rows),config_hash=config_hash,elapsed_post_model_s=time.perf_counter()-started,
        timing_scope="Includes preflight, verified feature work, shared control cache construction, answers and exports; excludes setup/download/model load/final packaging"))
    p.report_progress(status,main_questions=done);package(args.work,status)
    print("STAGE_ATTRIBUTION_COMPLETE" if status=="complete" else "STAGE_ATTRIBUTION_PARTIAL",flush=True)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    for n in ("repo","work","model","reuse"):parser.add_argument("--"+n,type=Path,required=True)
    parser.add_argument("--long-case",type=int,choices=(0,1),default=1)
    parser.add_argument("--max-hours",type=float,default=1.0)
    args=parser.parse_args()
    for n in ("repo","work","model","reuse"):setattr(args,n,getattr(args,n).resolve())
    d.validate_workspaces(args.work,args.reuse)
    if args.max_hours<0:parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:execute(args)
        except d.RunConflict:raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json",dict(status="failed",type=type(exc).__name__,detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json",dict(status="failed"));package(args.work,"failed_partial");raise


if __name__=="__main__":main()
'''
ast.parse(RUNNER_SOURCE)
assert hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest() == 'd677ec07a0f0244cb219da0d049c2fb4b9d82d9658b5039a9ec6e4767a89c408'
CONTEXT_SOURCE = r'''"""Small fresh-state context/content and chunk-partition interventions.

All replacements are declared counterfactuals using original pre-query features.
Candidate-support probes are analyst-selected, not certified human oracles.
"""
import argparse
import copy
import hashlib
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

try:
    import context_input_utils as d
except ModuleNotFoundError as exc:
    if exc.name != "context_input_utils":
        raise
    import autodl_input_diagnostic as d

p = d.p
ARTIFACT = "HERMES_context_diagnostic_evidence.zip"
CASES = ((204,0),(203,2),(204,3),(209,0),(208,4),(202,0),(208,0))
ARMS = ("question_only", "recent_4_original", "recent_20_original", "full_prefix_original",
        "full_prefix_framewise", "recent_20_fixed4", "recent_20_opening_fill", "recent_20_local_fill",
        "full_prefix_fixed4", "full_prefix_opening_fill", "full_prefix_local_fill", "candidate_support")
SUPPORT_CANDIDATES = {(204,0):[6,7], (204,3):[69]}


def validate_indices(indices, video, qi):
    end = d.prefix_end(video,qi)
    if any(type(i) is not int or i < 0 or i >= end for i in indices):
        raise ValueError("Counterfactual contains an invalid/future sampled index")
    if indices != sorted(indices):
        raise ValueError("Input mapping must be nondecreasing; repeated fillers are explicit")
    query = float(video["conversations"][qi]["end_time"])
    if any(float(video["timestamps"][i]["start_s"]) > query+1e-6 for i in indices):
        raise ValueError("Source frame starts after query")


def arm_plan(video, qi, arm):
    if arm not in ARMS:
        raise ValueError("Unknown intervention arm")
    end = d.prefix_end(video,qi)
    recent = list(range(max(0,end-20),end))
    tail = list(range(max(0,end-4),end))
    if arm == "question_only":
        indices,partition = [],"original"
    elif arm == "candidate_support":
        indices,partition = list(SUPPORT_CANDIDATES.get((video["sample_id"],qi),[])),"fixed4"
    elif arm == "recent_4_original":
        indices,partition = tail,"original"
    else:
        indices = list(range(end)) if arm.startswith("full_prefix_") else recent
        partition = "original" if arm.endswith("_original") else "framewise" if arm.endswith("_framewise") else "fixed4"
        if arm.endswith("_fill"):
            filler = 0 if arm.endswith("_opening_fill") else tail[0]
            indices = [filler]*(len(indices)-len(tail)) + tail
    validate_indices(indices,video,qi)
    replacement = arm.endswith("_fill")
    return dict(arm=arm,sampled_indices=indices,partition=partition,counterfactual=replacement,
                protected_tail_slots=list(range(max(0,len(indices)-4),len(indices))) if replacement else [],
                candidate_only=arm == "candidate_support",
                skip_reason="candidate_not_defined" if arm == "candidate_support" and not indices else "",
                intervention_effective=replacement and indices != (list(range(end)) if arm.startswith("full_prefix_") else recent))


def input_groups(indices, partition, meta):
    if partition in ("fixed4","framewise"):
        size = 4 if partition == "fixed4" else 1
        return [indices[i:i+size] for i in range(0,len(indices),size)]
    slices = d.feature_slices(indices,meta["chunks"])
    by_file = {c["file"]:c for c in meta["chunks"]}
    return [list(range(by_file[c["file"]]["start"]+c["start"],by_file[c["file"]]["start"]+c["stop"])) for c in slices]


class FeatureBank:
    def __init__(self,directory,meta):
        import torch
        self.frames = {}
        for chunk in meta["chunks"]:
            value = torch.load(directory/chunk["file"],map_location="cpu",weights_only=True)
            count = chunk["stop"]-chunk["start"]
            if value.ndim != 3 or value.shape[:2] != (1,count*196) or value.dtype != torch.float16:
                raise ValueError("Feature layout differs from original per-frame token blocks")
            for offset,index in enumerate(range(chunk["start"],chunk["stop"])):
                if index in self.frames:
                    raise ValueError("Overlapping feature chunks")
                self.frames[index] = value[:,offset*196:(offset+1)*196]

    def get(self,indices):
        import torch
        return torch.cat([self.frames[i] for i in indices],dim=1).contiguous()


def valid_checkpoint(value,key,identity,plan):
    try:
        row = value["row"]
        return (value["identity"] == key and value["row_sha256"] == d.fingerprint(row)
            and all(row[k] == v for k,v in identity.items()) and row["plan"] == plan
            and ((row["status"] == "answered" and row["correct"] == int(row["pred_choice"] == identity["gold_choice"]))
              or (row["status"] == "skipped" and row["correct"] is None and bool(row["skip_reason"]))))
    except (KeyError,TypeError,ValueError):
        return False


def run_arm(model,analyzer,video,qi,arm,bank,meta,config_hash,args):
    import torch
    plan = arm_plan(video,qi,arm)
    identity = d.question_identity(video,qi,arm)
    key = d.fingerprint(dict(config=config_hash,question=identity,plan=plan,features=d.fingerprint(meta)))
    dest = args.work/"checkpoints"/arm/f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value = json.loads(dest.read_text(encoding="utf-8"))
        if not valid_checkpoint(value,key,identity,plan):
            raise d.RunConflict("Checkpoint identity/correctness mismatch; preserve and choose new WORK")
        print(f"REUSE {arm} sample={video['sample_id']} q={qi}",flush=True)
        return value["row"]
    indices = plan["sampled_indices"]
    p.report_progress("context_arm",sample_id=video["sample_id"],qa_index=qi,arm=arm,input_slots=len(indices))
    row = dict(**identity,plan=plan,status="skipped",skip_reason=plan["skip_reason"],pred_answer="",pred_choice="",correct=None,
               visual_tokens=len(indices)*196,source_frame_indices=[int(video["timestamps"][i]["source_frame"]) for i in indices],
               source_start_seconds=[float(video["timestamps"][i]["start_s"]) for i in indices],
               input_identity=d.fingerprint(dict(features=d.fingerprint(meta),indices=indices)),
               ingest_s=0.0,answer_s=0.0,wall_s=0.0,peak_allocated_gib=0.0,cache_lengths_before_answer=[])
    started = time.perf_counter()
    if not row["skip_reason"]:
        random.seed(p.SEED)
        torch.manual_seed(p.SEED)
        p.reset(model)
        try:
            d.assert_fresh_cache(model,0)
            prompt = analyzer.format_mcqa_prompt(identity["question"],identity["choices"])
            qtokens = len(model.processor.tokenizer(prompt["prompt"]).input_ids)
            row["prompt_tokens"] = model.visual_start_idx
            row["question_tokens"] = qtokens
            row["context_limit"] = model.language_model.config.max_position_embeddings
            reason = d.context_reason(len(indices),model.visual_start_idx,qtokens,row["context_limit"],args.max_visual_tokens,
                                      "native_prefix" if arm.startswith("full_prefix_") else arm)
            if not reason and arm.startswith("full_prefix_"):
                base = sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in model.kv_cache)
                needed = base/model.visual_start_idx*(len(indices)*196+qtokens+48)
                if torch.cuda.mem_get_info()[0] < needed*1.5+2*1024**3:
                    reason = "estimated_vram_limit"
            if reason:
                row["skip_reason"] = reason
            else:
                groups = input_groups(indices,plan["partition"],meta)
                row["ingestion_groups"] = groups
                p.sync()
                torch.cuda.reset_peak_memory_stats()
                before = time.perf_counter()
                for group in groups:
                    p.ingest_features(model,bank.get(group),len(group))
                p.sync()
                row["ingest_s"] = time.perf_counter()-before
                d.assert_fresh_cache(model,len(indices))
                row["cache_lengths_before_answer"] = model._get_cache_seq_len_per_layer()
                row["protected_tail_token_starts"] = [model.visual_start_idx+i*196 for i in plan["protected_tail_slots"]]
                before = time.perf_counter()
                answer = model.question_answering(prompt,max_new_tokens=16)
                p.sync()
                prediction = analyzer.extract_characters_regex(answer)
                row.update(status="answered",pred_answer=answer.replace("\n",""),pred_choice=prediction,
                           correct=int(prediction == identity["gold_choice"]),answer_s=time.perf_counter()-before,
                           peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
        except torch.cuda.OutOfMemoryError:
            if not arm.startswith("full_prefix_"):
                raise
            row.update(status="skipped",skip_reason="cuda_oom",correct=None)
        finally:
            model._layer_position_ids.clear()
            model.clear_cache()
    row["wall_s"] = time.perf_counter()-started
    value = dict(identity=key,row=row,row_sha256=d.fingerprint(row))
    if not valid_checkpoint(value,key,identity,plan):
        raise ValueError("New arm checkpoint failed validation")
    p.atomic_json(dest,value)
    print(f"ARM_DONE {arm} sample={video['sample_id']} q={qi} status={row['status']} correct={row['correct']} reason={row['skip_reason']}",flush=True)
    return row


def export(rows,reference,work):
    fields = ["sample_id","qa_index","method","question","choices","gold_answer","gold_choice","status","skip_reason","pred_choice","correct",
              "plan","input_identity","cache_lengths_before_answer","protected_tail_token_starts","ingest_s","answer_s","wall_s","peak_allocated_gib"]
    p.write_csv(work/"exports/predictions.csv",[{k:r.get(k,"") for k in fields} for r in rows])
    summary = []
    for arm in ARMS:
        cases = [r for r in rows if r["method"] == arm]
        answers = [r for r in cases if r["status"] == "answered"]
        summary.append(dict(arm=arm,recorded=len(cases),answered=len(answers),skipped=len(cases)-len(answers),correct=sum(r["correct"] for r in answers)))
    p.write_csv(work/"exports/summary.csv",summary)
    old = {(r["sample_id"],r["qa_index"],r["method"]):r for r in reference["input_rows"]}
    replay = []
    arm_mapping = {"question_only":"question_only","recent_4_original":"recent_4","recent_20_original":"recent_20","full_prefix_original":"native_prefix"}
    for row in rows:
        if row["method"] in arm_mapping:
            prior = old[row["sample_id"],row["qa_index"],arm_mapping[row["method"]]]
            comparable = row["status"] == prior["status"] == "answered"
            replay.append(dict(sample_id=row["sample_id"],qa_index=row["qa_index"],arm=row["method"],comparable=comparable,
                               previous_choice=prior["pred_choice"],new_choice=row["pred_choice"],
                               choice_matches=row["pred_choice"] == prior["pred_choice"] if comparable else ""))
    p.write_csv(work/"exports/replay_checks.csv",replay)
    by_key = {(r["sample_id"],r["qa_index"],r["method"]):r for r in rows}
    pairs = []
    contrasts = [("full_prefix_original","full_prefix_framewise"),("full_prefix_original","full_prefix_fixed4"),
                 ("recent_20_original","recent_20_fixed4"),("recent_20_fixed4","recent_20_opening_fill"),
                 ("recent_20_fixed4","recent_20_local_fill"),("full_prefix_fixed4","full_prefix_opening_fill"),
                 ("full_prefix_fixed4","full_prefix_local_fill")]
    for sid,qi in CASES:
        for a,b in contrasts:
            if (sid,qi,a) not in by_key or (sid,qi,b) not in by_key:
                continue
            x,y = by_key[sid,qi,a],by_key[sid,qi,b]
            answered = x["status"] == y["status"] == "answered"
            fixed_content = x["plan"]["sampled_indices"] == y["plan"]["sampled_indices"]
            fixed_positions = len(x["plan"]["sampled_indices"]) == len(y["plan"]["sampled_indices"])
            same_tail = x["plan"]["sampled_indices"][-4:] == y["plan"]["sampled_indices"][-4:]
            if b.endswith("_fill") and (not fixed_positions or not same_tail):
                raise ValueError("Counterfactual changed protected tail length/source positions")
            pairs.append(dict(sample_id=sid,qa_index=qi,arm_a=a,arm_b=b,both_answered=answered,
                same_content=fixed_content,same_length=fixed_positions,choice_changed=x["pred_choice"] != y["pred_choice"] if answered else "",
                same_tail_sources=same_tail,replacement_effective=y["plan"]["intervention_effective"],
                a_correct=x["correct"],b_correct=y["correct"],
                interpretation=("Identical-input null control" if fixed_content and x["plan"]["partition"] == y["plan"]["partition"]
                    else "Chunking/numerical control" if fixed_content else "Counterfactual earlier-content replacement; repeated filler is a confound")))
    if pairs:
        p.write_csv(work/"exports/contrasts.csv",pairs)
    text = ["# Context/partition diagnostic", "", "Seven selected development cases; not an algorithm evaluation or population accuracy estimate.",
            "", "| Arm | Answered | Skipped | Correct |", "|---|---:|---:|---:|"]
    text += [f"| {r['arm']} | {r['answered']} | {r['skipped']} | {r['correct']} |" for r in summary]
    mismatches = sum(r["choice_matches"] is False for r in replay if r["comparable"])
    text += ["",f"Historical replay choice mismatches: {mismatches}. Inspect mismatches before interpreting interventions.",
        "Inspect exact-content chunking controls before attributing differences to semantics.",
        "Replacement arms keep last-four source frames and their token positions fixed at equal input length; they repeat pre-query fillers, which are not certified neutral.",
        "When there are no earlier slots, replacement arms are identical-input controls, not independent evidence.",
        "Candidate support at 204/q0 [6,7] and 204/q3 [69] comes from analyst-reviewed saved sheets; no human oracle certification is claimed.",
        "Gold is used only for scoring. All answers reset state and use no compression/past answers. Resource-limited full prefixes are skipped without truncation.",
        "Question wall times exclude model/setup/shared preparation. No publication-ready improvement is implied."]
    (work/"exports/report.md").write_text("\n".join(text)+"\n",encoding="utf-8")


def export_review(video,qi,work):
    from decord import VideoReader
    from PIL import Image,ImageDraw
    reader = VideoReader(video["video_path"],num_threads=1)
    directory = work/"review"/f"sample_{video['sample_id']}_q{qi}"
    directory.mkdir(parents=True,exist_ok=True)
    end = d.prefix_end(video,qi)
    for start in range(0,end,12):
        target = directory/f"page_{start//12+1:02d}.jpg"
        if target.exists():
            continue
        chosen = list(range(start,min(start+12,end)))
        frames = reader.get_batch([int(video["timestamps"][i]["source_frame"]) for i in chosen]).asnumpy()
        canvas = Image.new("RGB",(1024,520),"white")
        draw = ImageDraw.Draw(canvas)
        draw.text((4,4),f"sample={video['sample_id']} q={qi} | ORIGINAL PREFIX: no filler frames | support review pending",fill="black")
        for n,(frame,index) in enumerate(zip(frames,chosen)):
            picture = Image.fromarray(frame)
            picture.thumbnail((256,140))
            x,y = n%4*256,25+n//4*164
            canvas.paste(picture,(x,y))
            draw.text((x+3,y+143),f"sampled {index} | {video['timestamps'][index]['start_s']}s",fill="black")
        canvas.save(target,quality=85)
    for index in SUPPORT_CANDIDATES.get((video["sample_id"],qi),[]):
        target = directory/f"candidate_sampled_{index:04d}_original.png"
        if not target.exists():
            source = int(video["timestamps"][index]["source_frame"])
            Image.fromarray(reader[source].asnumpy()).save(target)


def package(work,status):
    captured = work/"captured_logs"
    captured.mkdir(exist_ok=True,parents=True)
    for source in (work/"logs").glob("*"):
        if source.is_file():
            shutil.copy2(source,captured/source.name)
    if (work/"progress.json").is_file():
        shutil.copy2(work/"progress.json",captured/"progress_snapshot.json")
    files = []
    for area in ("exports","checkpoints","provenance","review","captured_logs"):
        files += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("context_runner.py","context_input_utils.py","diagnostic_pilot_utils.py","reference.json","launch.json","asset_manifest.json","preflight.json"):
        if (work/name).exists():
            files.append(work/name)
    manifest = [dict(path=x.relative_to(work).as_posix(),bytes=x.stat().st_size,sha256=p.digest(x)) for x in sorted(set(files))]
    p.atomic_json(work/"evidence_manifest.json",dict(status=status,files=manifest))
    target = work/ARTIFACT
    with ZipFile(target.with_suffix(".tmp"),"w",compression=ZIP_DEFLATED) as z:
        for row in manifest:
            z.write(work/row["path"],row["path"])
        z.write(work/"evidence_manifest.json","evidence_manifest.json")
    os.replace(target.with_suffix(".tmp"),target)
    print(f"EVIDENCE_ZIP={target}\nSHA256={p.digest(target)}",flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    reference = json.loads((args.work/"reference.json").read_text(encoding="utf-8"))
    d.check_reference(reference,args.repo)
    if args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128" or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available():
        raise ValueError("Pinned model/CUDA environment required")
    config = dict(reference_sha256=p.digest(args.work/"reference.json"),sources={name:p.digest(args.work/name) for name in
        ("context_runner.py","context_input_utils.py","diagnostic_pilot_utils.py")},cases=CASES,arms=ARMS,
        max_visual_tokens=args.max_visual_tokens,seed=p.SEED,model_revision=p.MODEL_SHA,
        torch=torch.__version__,transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),
        no_compression=True,fresh_state=True,candidate_support="Analyst sheet candidates, not certified oracle")
    config_hash = d.fingerprint(config)
    path = args.work/"exports/run_config.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8"))["config_hash"] != config_hash:
        raise d.RunConflict("Run configuration changed; use a new WORK directory")
    p.atomic_json(path,dict(**config,config_hash=config_hash))
    previous_failure = args.work/"exports/failure.json"
    if previous_failure.exists():
        destination = args.work/"logs"/f"previous_failure_{time.time_ns()}.json"
        destination.parent.mkdir(exist_ok=True,parents=True)
        os.replace(previous_failure,destination)
    videos = copy.deepcopy(reference["videos"])
    d.acquire_videos(videos,args.reuse,args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json",videos)
    sys.path.insert(0,str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8,os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_model")
    model,processor = load_model(model_path=str(args.model),kv_size=4000,streaming=True,sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[],save_dir=str(args.work),sample_fps=p.FPS,qa_model=model,qa_processor=processor)
    by_id = {v["sample_id"]:v for v in videos}
    rows,assets = [],{}
    started = time.perf_counter()
    completed = 0
    with torch.inference_mode():
        p.preflight(model,by_id[204],args.work)
        for index,(sid,qi) in enumerate(CASES):
            if index and args.max_hours and time.perf_counter()-started > args.max_hours*3600:
                break
            video = by_id[sid]
            if sid not in assets:
                directory,meta,_ = d.acquire_features(model,video,args.reuse,args.work,d.fingerprint(dict(config=config_hash,video=video["video_sha256"])))
                assets[sid] = (FeatureBank(directory,meta),meta)
            bank,meta = assets[sid]
            order = list(ARMS)
            random.Random(p.SEED+sid*10+qi).shuffle(order)
            for arm in order:
                rows.append(run_arm(model,analyzer,video,qi,arm,bank,meta,config_hash,args))
            export_review(video,qi,args.work)
            export(rows,reference,args.work)
            completed += 1
            elapsed = time.perf_counter()-started
            print(f"CASE_DONE {completed}/{len(CASES)} elapsed_min={elapsed/60:.2f} rough_remaining_min={(len(CASES)-completed)*elapsed/completed/60:.2f}",flush=True)
    status = "complete" if completed == len(CASES) else "partial_time_budget"
    if status == "complete":
        if len(rows) != len(CASES)*len(ARMS) or any(r["status"] != "answered" for r in rows if r["method"] in ARMS[:3]):
            raise ValueError("Core replay control matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json",dict(status=status,case_blocks=completed,planned_blocks=len(CASES),
        rows=len(rows),answered=sum(r["status"] == "answered" for r in rows),config_hash=config_hash,
        elapsed_post_model_s=time.perf_counter()-started,timing_scope="Post-model preflight, verified feature work, arms, review/export; excludes setup/download/model load/final packaging"))
    p.report_progress(status,case_blocks=completed)
    package(args.work,status)
    print("CONTEXT_COMPLETE" if status == "complete" else "CONTEXT_PARTIAL",flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo","work","model","reuse"):
        parser.add_argument("--"+name,type=Path,required=True)
    parser.add_argument("--max-visual-tokens",type=int,default=16384)
    parser.add_argument("--max-hours",type=float,default=.75)
    args = parser.parse_args()
    for name in ("repo","work","model","reuse"):
        setattr(args,name,getattr(args,name).resolve())
    d.validate_workspaces(args.work,args.reuse)
    if args.max_hours < 0 or args.max_visual_tokens < 3920:
        parser.error("Invalid budget/context cap")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json",dict(status="failed",type=type(exc).__name__,detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json",dict(status="failed"))
            package(args.work,"failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(CONTEXT_SOURCE)
assert hashlib.sha256(CONTEXT_SOURCE.encode()).hexdigest() == '5f75057770b517add6a7d735474d671875bfc3a8f2d14899c82f4dfdacfc6d31'
INPUT_SOURCE = r'''"""Matched fresh-state input controls. Heavy libraries are imported only at runtime.

The companion notebook embeds this runner, the tested pilot utilities and the
audited historical reference. Historical artifacts are data, never executed.
"""
import argparse
import copy
import hashlib
import html
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

try:
    import diagnostic_pilot_utils as p
except ModuleNotFoundError as exc:
    if exc.name != "diagnostic_pilot_utils":
        raise
    import autodl_failure_pilot as p

METHODS = ("question_only", "recent_4", "recent_20", "native_prefix", "support_oracle")
ARTIFACT = "HERMES_input_diagnostic_evidence.zip"
PRIORITY = {(204, 0), (202, 0), (208, 0), (212, 3), (218, 3), (204, 3), (219, 3)}


class RunConflict(ValueError):
    """Existing scientific outputs must not be replaced by a different run."""


def fingerprint(value):
    return hashlib.sha256(p.stable(value).encode()).hexdigest()


def validate_workspaces(work, reuse):
    work, reuse = Path(work).resolve(), Path(reuse).resolve()
    if work == reuse or work.is_relative_to(reuse) or reuse.is_relative_to(work):
        raise ValueError("WORK and reusable pilot directory must be separate non-nested locations")


def check_reference(reference, repo):
    for name, expected in reference["source_sha256"].items():
        actual = hashlib.sha256((repo / name).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
        if actual != expected:
            raise ValueError(f"Pinned source differs: {name}")
    annotation = json.loads((repo / "data/streamingbench/streamingbench_realtime.json").read_text(encoding="utf-8"))
    by_id = {str(v["video_id"]): v for v in annotation}
    for video in reference["videos"]:
        original = by_id[str(video["video_id"])]
        if original["conversations"] != video["conversations"] or p.sample_id(original) != video["sample_id"]:
            raise ValueError("Historical annotation identity mismatch")


def prefix_end(video, qi):
    return math.ceil(float(video["conversations"][qi]["end_time"]) * p.FPS)


def validate_frames(indices, video, qi, timestamps):
    end = prefix_end(video, qi)
    if any(type(i) is not int for i in indices) or indices != sorted(set(indices)):
        raise ValueError("Frame indices must be sorted unique integers")
    if any(i < 0 or i >= end or i >= len(timestamps) for i in indices):
        raise ValueError("Selected frame outside original query prefix")
    query = float(video["conversations"][qi]["end_time"])
    if any(float(timestamps[i]["start_s"]) > query + 1e-6 for i in indices):
        raise ValueError("Sampled frame starts after query time")
    return indices


def validate_oracles(items, videos, max_frames=20):
    if not isinstance(items, list):
        raise ValueError("Support JSON must be a list")
    by_id = {v["sample_id"]: v for v in videos}
    result = {}
    for item in items:
        sid, qi = item["sample_id"], item["qa_index"]
        if type(sid) is not int or type(qi) is not int or sid not in by_id:
            raise ValueError("Unknown oracle sample or invalid index type")
        video = by_id[sid]
        if not 0 <= qi < len(video["conversations"]) or (sid, qi) in result:
            raise ValueError("Invalid/duplicate oracle question")
        if item.get("verified") is not True or not str(item.get("notes", "")).strip():
            raise ValueError("Oracle requires verified=true and support review notes")
        indices = item["sampled_indices"]
        if not isinstance(indices, list) or not 1 <= len(indices) <= max_frames:
            raise ValueError("Oracle must contain 1..20 sampled frame indices")
        validate_frames(indices, video, qi, video["timestamps"])
        result[(sid, qi)] = item
    return result


def selected_frames(method, video, qi, oracles, native_enabled):
    end = prefix_end(video, qi)
    if method == "question_only":
        return [], None
    if method in ("recent_4", "recent_20"):
        count = 4 if method == "recent_4" else 20
        return list(range(max(0, end - count), end)), None
    if method == "native_prefix":
        return (list(range(end)), None) if native_enabled else ([], "disabled")
    oracle = oracles.get((video["sample_id"], qi))
    return (oracle["sampled_indices"], None) if oracle else ([], "not_requested")


def context_reason(frame_count, prompt_tokens, question_tokens, context_limit, native_cap, method):
    # question_answering uses greedy max_new_tokens=16; 32 extra positions are a safety reserve.
    needed = prompt_tokens + frame_count * 196 + question_tokens + 16 + 32
    if needed > context_limit:
        return "context_limit"
    if method == "native_prefix" and frame_count * 196 > native_cap:
        return "configured_native_cap"
    return None


def feature_slices(indices, chunks):
    """Return chunk-local runs without mixing or duplicating original frame identities."""
    covered, slices = [], []
    for item in chunks:
        chosen = [i for i in indices if item["start"] <= i < item["stop"]]
        for i in chosen:
            covered.append(i)
            if slices and slices[-1]["file"] == item["file"] and slices[-1]["stop"] == i - item["start"]:
                slices[-1]["stop"] += 1
            else:
                slices.append(dict(file=item["file"], start=i-item["start"], stop=i-item["start"]+1))
    if covered != indices:
        raise ValueError("Feature chunks do not cover selected frames exactly once")
    return slices


def read_features(directory, item):
    import torch
    value = torch.load(directory / item["file"], map_location="cpu", weights_only=True)
    if value.ndim != 3 or value.shape[0] != 1 or value.shape[1] % 196 or value.dtype != torch.float16:
        raise ValueError("Unexpected feature layout/dtype; never guess frame boundaries")
    return value[:, item["start"]*196:item["stop"]*196].contiguous()


def acquire_videos(videos, reuse, work):
    # Only exact trusted historical video hashes can seed resumption/reuse.
    entries, missing = [], []
    for v in videos:
        sid = v["sample_id"]
        candidates = [work / "videos" / f"sample_{sid}_real.mp4", reuse / "videos" / f"sample_{sid}_real.mp4"]
        found = next((x for x in candidates if x.is_file() and p.digest(x) == v["video_sha256"]), None)
        if found is None:
            missing.append(v)
        else:
            entries.append(dict(sample_id=sid, path=str(found.resolve()), bytes=found.stat().st_size,
                                sha256=v["video_sha256"]))
    if not missing:
        assets = dict(mode="verified_existing_videos", videos=entries)
    else:
        downloaded = p.prepare_assets(missing, work)
        assets = dict(mode="verified_existing_plus_downloaded", videos=entries+downloaded["videos"],
                      download_details={k:v for k,v in downloaded.items() if k != "videos"})
    by_id = {x["sample_id"]: x for x in assets["videos"]}
    for v in videos:
        item = by_id[v["sample_id"]]
        if item["sha256"] != v["video_sha256"] or p.digest(item["path"]) != v["video_sha256"]:
            raise ValueError("Video differs from audited source")
        v["video_path"] = item["path"]
    p.atomic_json(work / "asset_manifest.json", assets)


def verify_feature_files(directory, meta):
    for c in meta["chunks"]:
        if Path(c["file"]).name != c["file"] or c["tokens"] != (c["stop"]-c["start"])*196:
            raise ValueError("Invalid feature chunk metadata")
        path = directory / c["file"]
        if not path.is_file() or p.digest(path) != c["sha256"]:
            raise ValueError(f"Missing/corrupt feature: {path}")


def acquire_features(model, video, reuse, work, identity):
    sid = video["sample_id"]
    old_dir = reuse / "features" / f"sample_{sid}"
    expected = video["feature_meta"]
    if (old_dir / "complete.json").is_file():
        try:
            old = json.loads((old_dir / "complete.json").read_text(encoding="utf-8"))
            if old != expected or p.read_csv(old_dir / "sampled_timestamps.csv") != video["timestamps"]:
                raise ValueError("Old feature metadata differs from audited archive")
            verify_feature_files(old_dir, expected)
            meta, directory, origin = old, old_dir, "verified_historical_cache"
            print(f"FEATURE_REUSE_VERIFIED sample={sid}", flush=True)
        except (OSError, ValueError, KeyError) as exc:
            print(f"FEATURE_REUSE_REJECTED sample={sid} reason={type(exc).__name__}; regenerating separately", flush=True)
            directory = None
    else:
        directory = None
    if directory is None:
        directory = work / "features" / f"sample_{sid}"
        meta = p.prepare_features(model, video, work, identity)
        verify_feature_files(directory, meta)
        origin = "diagnostic_cache_verified"
        if p.read_csv(directory / "sampled_timestamps.csv") != video["timestamps"]:
            raise ValueError("Regenerated source sampling differs from original pilot")
        original_layout = [{k:c[k] for k in ("qa_index", "start", "stop", "tokens")} for c in expected["chunks"]]
        if [{k:c[k] for k in ("qa_index", "start", "stop", "tokens")} for c in meta["chunks"]] != original_layout:
            raise ValueError("Regenerated feature chunk boundaries differ")
    saved = dict(origin=origin, metadata=meta, directory=str(directory.resolve()),
                 timestamp_rows=video["timestamps"], feature_fingerprint=fingerprint(meta))
    p.atomic_json(work / "provenance" / f"features_{sid}.json", saved)
    return directory, meta, saved


def question_identity(video, qi, method):
    qa = video["conversations"][qi]
    return dict(sample_id=video["sample_id"], video_id=str(video["video_id"]), qa_index=qi,
                method=method, question=qa["question"], choices=qa["choices"],
                gold_answer=qa["answer"], gold_choice="ABCDEFGH"[qa["choices"].index(qa["answer"])],
                query_time_s=qa["end_time"], task=qa.get("task", ""))


def checkpoint_valid(value, expected, identity, frames):
    try:
        if value["checkpoint_identity"] != expected or value["row_sha256"] != fingerprint(value["row"]):
            return False
        row = value["row"]
        if any(row[k] != v for k,v in identity.items()) or row["sampled_indices"] != frames:
            return False
        if row["status"] == "answered":
            return row["correct"] == int(row["pred_choice"] == row["gold_choice"])
        return row["status"] == "skipped" and bool(row["skip_reason"]) and row["correct"] is None
    except (KeyError, TypeError, ValueError):
        return False


def assert_fresh_cache(model, frames):
    import torch
    expected = model.visual_start_idx + frames * 196
    if model.conv_history or model._get_cache_seq_len_per_layer() != [expected]*model.num_layers:
        raise ValueError("Fresh control retained history or has an unexpected cache length")
    for positions in model._position_ids_cache:
        torch.testing.assert_close(positions, torch.arange(expected, device=model.device), atol=0, rtol=0)


def run_question(model, analyzer, video, qi, method, directory, meta, config_hash, oracles, args):
    import torch
    identity = question_identity(video, qi, method)
    frames, reason = selected_frames(method, video, qi, oracles, args.native)
    validate_frames(frames, video, qi, video["timestamps"])
    check_id = fingerprint(dict(config=config_hash, feature=fingerprint(meta), question=identity, frames=frames))
    dest = args.work / "checkpoints" / method / f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value = json.loads(dest.read_text(encoding="utf-8"))
        if not checkpoint_valid(value, check_id, identity, frames):
            raise RunConflict(f"Existing checkpoint mismatched/corrupt; preserve and use new WORK: {dest}")
        print(f"REUSE {method} sample={video['sample_id']} q={qi}", flush=True)
        return value["row"]
    p.report_progress("question", sample_id=video["sample_id"], qa_index=qi, method=method, selected_frames=len(frames))
    row = dict(**identity, status="skipped", skip_reason=reason or "", sampled_indices=frames,
               source_frame_indices=[int(video["timestamps"][i]["source_frame"]) for i in frames],
               sampled_start_seconds=[float(video["timestamps"][i]["start_s"]) for i in frames],
               pred_answer="", pred_choice="", correct=None, visual_tokens=len(frames)*196,
               ingest_s=0.0, answer_s=0.0, wall_s=0.0, peak_allocated_gib=0.0,
               fresh_state=True, no_compression=True, feature_fingerprint=fingerprint(meta))
    start = time.perf_counter()
    if not reason:
        random.seed(p.SEED)
        torch.manual_seed(p.SEED)
        p.reset(model)
        assert_fresh_cache(model, 0)
        prompt = analyzer.format_mcqa_prompt(identity["question"], identity["choices"])
        qlen = len(model.processor.tokenizer(prompt["prompt"]).input_ids)
        reason = context_reason(len(frames), model.visual_start_idx, qlen,
                                model.language_model.config.max_position_embeddings, args.native_max_tokens, method)
        if not reason and method == "native_prefix":
            # Conservative KV estimate plus 2 GiB workspace; hard cap remains separate.
            base_bytes = sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in model.kv_cache)
            kv_estimate = base_bytes / model.visual_start_idx * (len(frames)*196+qlen+48)
            free, _ = torch.cuda.mem_get_info()
            if free < kv_estimate*1.5 + 2*1024**3:
                reason = "estimated_vram_limit"
        if reason:
            row["skip_reason"] = reason
        else:
            torch.cuda.reset_peak_memory_stats()
            try:
                p.sync()
                before = time.perf_counter()
                for item in feature_slices(frames, meta["chunks"]):
                    value = read_features(directory, item)
                    p.ingest_features(model, value, item["stop"]-item["start"])
                    del value
                p.sync()
                row["ingest_s"] = time.perf_counter()-before
                assert_fresh_cache(model, len(frames))
                row["cache_lengths_before_answer"] = model._get_cache_seq_len_per_layer()
                before = time.perf_counter()
                # Gold is used only AFTER generation. Same prompt/parser/16-token greedy policy as pilot.
                answer = model.question_answering(prompt, max_new_tokens=16)
                p.sync()
                pred = analyzer.extract_characters_regex(answer)
                row.update(status="answered", pred_answer=answer.replace("\n", ""), pred_choice=pred,
                           correct=int(pred == identity["gold_choice"]), answer_s=time.perf_counter()-before,
                           peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
            except torch.cuda.OutOfMemoryError:
                if method != "native_prefix":
                    raise
                row.update(status="skipped", skip_reason="cuda_oom", correct=None)
                print("NATIVE_PREFIX_OOM: case skipped explicitly; other controls continue", flush=True)
            finally:
                p.reset(model)
        # Release even a cap/context-skipped fresh state.
        model.clear_cache()
    row["wall_s"] = time.perf_counter()-start
    value = dict(checkpoint_identity=check_id, row=row, row_sha256=fingerprint(row))
    if not checkpoint_valid(value, check_id, identity, frames):
        raise ValueError("New checkpoint failed identity validation")
    p.atomic_json(dest, value)
    print(f"QUESTION_DONE {method} sample={video['sample_id']} q={qi} status={row['status']} correct={row['correct']} reason={row['skip_reason']}", flush=True)
    return row


def export(rows, reference, work):
    fields = list(question_identity(reference["videos"][0], 0, "question_only")) + [
        "status", "skip_reason", "sampled_indices", "source_frame_indices", "sampled_start_seconds", "pred_answer", "pred_choice", "correct",
        "visual_tokens", "ingest_s", "answer_s", "wall_s", "peak_allocated_gib", "fresh_state", "no_compression", "feature_fingerprint", "cache_lengths_before_answer"]
    p.write_csv(work / "exports/predictions.csv", [{k:r.get(k, "") for k in fields} for r in rows])
    summary = []
    for method in METHODS:
        subset = [r for r in rows if r["method"] == method]
        answered = [r for r in subset if r["status"] == "answered"]
        summary.append(dict(method=method, planned=40, recorded=len(subset), answered=len(answered),
                            skipped=len(subset)-len(answered), correct=sum(r["correct"] for r in answered),
                            accuracy_pct=100*sum(r["correct"] for r in answered)/len(answered) if answered else "",
                            measured_wall_s=sum(r["wall_s"] for r in answered)))
    p.write_csv(work / "exports/summary.csv", summary)
    old = {(int(r["sample_id"]), int(r["qa_index"]), r["method"]): r for r in reference["predictions"]}
    paired = []
    for method in METHODS:
        new = [r for r in rows if r["method"] == method and r["status"] == "answered"]
        for baseline in ("fifo", "hermes", "causal"):
            pairs = [(r, old[(r["sample_id"], r["qa_index"], baseline)]) for r in new]
            paired.append(dict(method=method, historical_baseline=baseline, matched_questions=len(pairs),
                diagnostic_correct=sum(r["correct"] for r,b in pairs),
                historical_correct=sum(int(b["correct"]) for r,b in pairs),
                diagnostic_only_correct=sum(r["correct"] and not int(b["correct"]) for r,b in pairs),
                historical_only_correct=sum(not r["correct"] and int(b["correct"]) for r,b in pairs),
                comparison_scope="Exact shared identities; historical session; changed input/history; descriptive development comparison"))
    p.write_csv(work / "exports/paired_historical.csv", paired)
    text = ["# Fresh-state input diagnostic", "", "Selected development cases; no population or original-method claim.", "",
            "| Method | Answered | Skipped | Correct | Accuracy on answered |", "|---|---:|---:|---:|---:|"]
    for s in summary:
        accuracy = f"{s['accuracy_pct']:.1f}%" if s["answered"] else "N/A"
        text.append(f"| {s['method']} | {s['answered']} | {s['skipped']} | {s['correct']} | {accuracy} |")
    text += ["", "Native-prefix/oracle rows may cover different subsets: use paired_historical.csv for exact-subset comparisons.",
        "All controls reset per question; no eviction or past-answer history. Recent/oracle inputs compact selected frames chronologically.",
        "Historical comparisons reuse old predictions; they do not establish same-session efficiency or isolate memory policy alone.",
        "Oracle is privileged human support, not a deployable method. No oracle runs without verified annotations.",
        "Question wall time includes reset/feature reading/ingestion/answer; excludes setup and shared feature preparation.",
        "Review sheets show original sampled frames, not proof of support annotation. Source videos and feature tensors are excluded from ZIP."]
    (work / "exports/report.md").write_text("\n".join(text)+"\n", encoding="utf-8")
    return summary


def initialize_review(videos, work):
    sheet = work / "review/support_review.csv"
    if not sheet.exists():
        rows = []
        for video in videos:
            for qi, qa in enumerate(video["conversations"]):
                rows.append(dict(sample_id=video["sample_id"], qa_index=qi,
                    priority="first" if (video["sample_id"], qi) in PRIORITY else "later",
                    question=qa["question"], query_time_s=qa["end_time"], review_status="pending",
                    support_intervals_seconds_json="", verified_sampled_indices_json="",
                    sampled_support_sufficient="", ambiguity="", cause="unclassified", notes=""))
        p.write_csv(sheet, rows)
    content = ["<!doctype html><meta charset='utf-8'><title>Priority support review</title>",
               "<h1>Original sampled prefixes: support still needs review</h1>",
               "<p>Indices are sampled-frame indices. All pages stop at the original query prefix. "
               "Gold answers are review labels, never provided as model inputs. Fill support_review.csv separately.</p>"]
    for video in videos:
        for qi, qa in enumerate(video["conversations"]):
            if (video["sample_id"], qi) not in PRIORITY:
                continue
            content += [f"<h2>sample {video['sample_id']} / q{qi}</h2>",
                        f"<p>{html.escape(qa['question'])} | query {qa['end_time']} s</p>",
                        f"<p>Choices: {html.escape(str(qa['choices']))} | gold: {html.escape(str(qa['answer']))}</p>"]
            for page in range(math.ceil(prefix_end(video, qi)/12)):
                name = f"sample_{video['sample_id']}_q{qi}/page_{page+1:02d}.jpg"
                content.append(f"<p><a href='{name}'>Page {page+1}</a></p>")
    (work/"review/index.html").write_text("\n".join(content), encoding="utf-8")


def review_pages(video, work):
    from decord import VideoReader
    from PIL import Image, ImageDraw
    reader = None
    for qi, qa in enumerate(video["conversations"]):
        if (video["sample_id"], qi) not in PRIORITY:
            continue
        directory = work / "review" / f"sample_{video['sample_id']}_q{qi}"
        expected = math.ceil(prefix_end(video, qi)/12)
        if all((directory/f"page_{i+1:02d}.jpg").is_file() for i in range(expected)):
            continue
        directory.mkdir(parents=True, exist_ok=True)
        if reader is None:
            reader = VideoReader(video["video_path"], num_threads=1)
        for start in range(0, prefix_end(video, qi), 12):
            p.report_progress("priority_review_frames", sample_id=video["sample_id"], qa_index=qi, sampled_start=start)
            indices = list(range(start, min(start+12, prefix_end(video, qi))))
            source = [int(video["timestamps"][i]["source_frame"]) for i in indices]
            frames = reader.get_batch(source).asnumpy()
            canvas = Image.new("RGB", (1024, 520), "white")
            draw = ImageDraw.Draw(canvas)
            draw.text((4, 4), f"sample={video['sample_id']} q={qi} query={qa['end_time']}s | ORIGINAL SAMPLED PREFIX | support unreviewed", fill="black")
            for n, (frame, i) in enumerate(zip(frames, indices)):
                image = Image.fromarray(frame)
                image.thumbnail((256, 140))
                x, y = n%4*256, 25+n//4*164
                canvas.paste(image, (x,y))
                draw.text((x+3,y+143), f"sampled {i} | {video['timestamps'][i]['start_s']}s", fill="black")
            canvas.save(directory/f"page_{start//12+1:02d}.jpg", quality=85)


def package(work, status):
    captured = work / "captured_logs"
    captured.mkdir(parents=True, exist_ok=True)
    for path in (work / "logs").glob("*"):
        if path.is_file():
            shutil.copy2(path, captured/path.name)
    if (work/"progress.json").is_file():
        shutil.copy2(work/"progress.json", captured/"progress_snapshot.json")
    paths = []
    for area in ("exports", "checkpoints", "provenance", "review", "captured_logs"):
        paths += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("diagnostic_runner.py", "diagnostic_pilot_utils.py", "reference.json", "launch.json",
                 "asset_manifest.json", "preflight.json", "support_oracle.json"):
        if (work/name).is_file():
            paths.append(work/name)
    manifest = [dict(path=x.relative_to(work).as_posix(), bytes=x.stat().st_size, sha256=p.digest(x)) for x in sorted(set(paths))]
    p.atomic_json(work/"evidence_manifest.json", dict(status=status, files=manifest))
    output = work / ARTIFACT
    with ZipFile(output.with_suffix(".tmp"), "w", compression=ZIP_DEFLATED) as z:
        for item in manifest:
            z.write(work/item["path"], item["path"])
        z.write(work/"evidence_manifest.json", "evidence_manifest.json")
    os.replace(output.with_suffix(".tmp"), output)
    print(f"EVIDENCE_ZIP={output}\nSHA256={p.digest(output)}", flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    args.work, args.repo, args.reuse = args.work.resolve(), args.repo.resolve(), args.reuse.resolve()
    validate_workspaces(args.work, args.reuse)
    if args.model.name != p.MODEL_SHA:
        raise ValueError("Model snapshot revision differs")
    reference = json.loads((args.work / "reference.json").read_text(encoding="utf-8"))
    check_reference(reference, args.repo)
    videos = copy.deepcopy(reference["videos"])
    initialize_review(videos, args.work)
    oracles = validate_oracles(json.loads((args.work/"support_oracle.json").read_text(encoding="utf-8")), videos)
    if torch.__version__ != "2.10.0+cu128" or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available():
        raise RuntimeError("Pinned GPU environment required")
    config = dict(reference_sha256=p.digest(args.work/"reference.json"), runner_sha256=p.digest(Path(__file__)),
                  utils_sha256=p.digest(Path(p.__file__)), model_revision=p.MODEL_SHA, sample_fps=p.FPS,
                  native=args.native, native_max_tokens=args.native_max_tokens,
                  support=sorted(oracles.values(), key=lambda x:(x["sample_id"],x["qa_index"])),
                  torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0),
                  methods=METHODS, seed=p.SEED, fresh_state=True, max_new_tokens=16)
    config_hash = fingerprint(config)
    previous = args.work/"exports/run_config.json"
    if previous.is_file() and json.loads(previous.read_text(encoding="utf-8"))["config_hash"] != config_hash:
        raise RunConflict("Configuration differs from existing run; preserve it and choose a new WORK directory")
    p.atomic_json(previous, dict(**config, config_hash=config_hash))
    failure = args.work/"exports/failure.json"
    if failure.exists():
        target = args.work/"logs"/f"previous_failure_{time.time_ns()}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        os.replace(failure, target)
    p.report_progress("verifying_or_downloading_videos")
    acquire_videos(videos, args.reuse, args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json", videos)
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows = []
    started = time.perf_counter()
    complete = True
    with torch.inference_mode():
        p.preflight(model, videos[0], args.work)
        for vi, video in enumerate(videos):
            if vi and args.max_hours and time.perf_counter()-started > args.max_hours*3600:
                complete = False
                break
            feature_id = fingerprint(dict(config=config_hash, video=video["video_sha256"]))
            directory, meta, provenance = acquire_features(model, video, args.reuse, args.work, feature_id)
            for qi in range(len(video["conversations"])):
                order = list(METHODS)
                random.Random(p.SEED+video["sample_id"]*10+qi).shuffle(order)
                for method in order:
                    rows.append(run_question(model, analyzer, video, qi, method, directory, meta, config_hash, oracles, args))
            if args.review:
                review_pages(video, args.work)
            summary = export(rows, reference, args.work)
            elapsed = time.perf_counter()-started
            print(f"VIDEO_DONE {vi+1}/8 elapsed_min={elapsed/60:.2f} rough_remaining_min={(7-vi)*elapsed/(vi+1)/60:.2f}", flush=True)
    status = "complete" if complete else "partial_time_budget"
    mandatory = [r for r in rows if r["method"] in METHODS[:3]]
    if complete and (len(mandatory) != 120 or any(r["status"] != "answered" for r in mandatory)):
        raise ValueError("Mandatory 120-answer control matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json", dict(status=status, rows=len(rows),
                  mandatory_answered=sum(r["status"] == "answered" for r in mandatory),
                  config_hash=config_hash, elapsed_post_model_s=time.perf_counter()-started,
                  timing_scope="Includes preflight, shared feature work, per-question controls and review export; excludes setup/download/model loading"))
    p.report_progress(status, recorded=len(rows))
    package(args.work, status)
    print("DIAGNOSTIC_COMPLETE" if complete else "DIAGNOSTIC_PARTIAL", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--"+name, type=Path, required=True)
    parser.add_argument("--native", type=int, default=1, choices=(0,1))
    parser.add_argument("--native-max-tokens", type=int, default=8192)
    parser.add_argument("--review", type=int, default=1, choices=(0,1))
    parser.add_argument("--max-hours", type=float, default=1.0)
    args = parser.parse_args()
    if args.max_hours < 0 or args.native_max_tokens < 196:
        parser.error("Invalid budget/context cap")
    # Reject dangerous output locations before the monitor or failure packager can write there.
    validate_workspaces(args.work, args.reuse)
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except RunConflict:
            # Do not replace an older completion/config with this rejected attempt.
            raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json", dict(status="failed", type=type(exc).__name__, detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(INPUT_SOURCE)
assert hashlib.sha256(INPUT_SOURCE.encode()).hexdigest() == 'c152696ec0401a0afe08c7406a559aa37b3914b77fd6a7c1f25ed782cff3a521'
UTILS_SOURCE = r'''"""Resumable AutoDL scorer screen. Bootstrap is embedded in the companion notebook.

Only experimental instances are patched. Official inference files remain intact.
Heavy imports are deferred so selection, checkpoint and download logic can be
verified on a CPU host without installing the GPU environment.
"""

import argparse
import ast
import copy
import csv
import hashlib
import html
import io
import json
import math
import os
import random
import re
import subprocess
import sys
import threading
import time
import types
from pathlib import Path
from zipfile import ZipFile


MODEL_ID = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
MODEL_SHA = "74dd0bf867a4cda7950c17663794267c60cf4b40"
SHARD_NAME = "Real-Time Visual Understanding_201-250.zip"
SHARD_BYTES = 2230277873
SHARD_SHA = "b7946e8b678fbf704b9816fa708e6febb77f5c37a7ad884084adf858f5d20dbf"
METHODS = ("fifo", "hermes", "causal")
FPS = 0.5
SEED = 2024
_PROGRESS = None


def stable(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
    os.replace(temp, path)


def write_csv(path, rows):
    if not rows:
        raise ValueError("Cannot publish an empty result table")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp, path)


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


class ProgressMonitor:
    """Report liveness separately from completed work, including blocked I/O."""
    def __init__(self, work, interval=30):
        self.work, self.interval = Path(work), interval
        self.work.mkdir(parents=True, exist_ok=True)
        self.started = time.monotonic()
        self.changed = self.started
        self.lock = threading.Lock()
        self.stop = threading.Event()
        self.state = {"stage": "starting", "detail": "No completed work yet"}
        self.thread = threading.Thread(target=self._heartbeat, daemon=True)

    def _write(self, heartbeat=False):
        # Caller holds the lock, so update and heartbeat cannot race the temp file.
        value = {**self.state, "elapsed_s": round(time.monotonic() - self.started, 1),
                 "seconds_since_stage_update": round(time.monotonic() - self.changed, 1),
                 "heartbeat_unix_time": time.time(),
                 "note": "Heartbeat shows process liveness; it does not prove download or GPU progress"}
        try:
            atomic_json(self.work / "progress.json", value)
        except OSError as exc:
            # A Windows reader can temporarily prevent atomic replacement.
            # Progress reporting must not terminate a scientific computation;
            # a subsequent heartbeat retries while stdout remains available.
            print(f"PROGRESS_FILE_RETRY error_type={type(exc).__name__}", flush=True)
        label = "HEARTBEAT" if heartbeat else "STAGE"
        print(label + " " + stable(value), flush=True)

    def update(self, stage, **detail):
        with self.lock:
            self.state = {"stage": stage, **detail}
            self.changed = time.monotonic()
            self._write()

    def _heartbeat(self):
        while not self.stop.wait(self.interval):
            with self.lock:
                self._write(heartbeat=True)

    def __enter__(self):
        global _PROGRESS
        _PROGRESS = self
        self.update("starting", detail="Selecting diagnostic videos")
        self.thread.start()
        return self

    def __exit__(self, kind, error, traceback):
        global _PROGRESS
        self.stop.set()
        self.thread.join(timeout=5)
        if kind is not None:
            self.update("failed", error_type=kind.__name__)
        _PROGRESS = None


def report_progress(stage, **detail):
    if _PROGRESS is not None:
        _PROGRESS.update(stage, **detail)


def sample_id(video):
    return int(re.search(r"sample_(\d+)", video["video_path"])[1])


def choose_screen(repo, per_outcome=2):
    from build_failure_pilot import select_cases
    rows = read_csv(repo / "results/failure_pilot/paired_inventory.csv")
    pool = []
    for row in rows:
        if 201 <= int(row["sample_id"]) <= 250:
            pool.append({**row, "duplicate_gold_options": row["duplicate_gold_options"] == "True"})
    quotas = {k: per_outcome for k in ("hermes_only", "fifo_only", "both_wrong", "both_correct")}
    selected = select_cases(pool, seed=SEED, quotas=quotas)
    annotations = json.loads((repo / "data/streamingbench/streamingbench_realtime.json").read_text(encoding="utf-8"))
    ids = {r["video_id"] for r in selected}
    by_id = {str(v["video_id"]): v for v in annotations if str(v["video_id"]) in ids}
    # Calibrate on one HERMES-only and one FIFO-only target before other cases.
    ordered = [selected[group * per_outcome + index]["video_id"]
               for index in range(per_outcome) for group in range(4)]
    videos = [by_id[vid] for vid in ordered]
    if len(videos) != per_outcome * 4:
        raise ValueError("Screen selection and annotation IDs differ")
    return selected, videos


class RangeUnsupported(RuntimeError):
    pass


class HTTPRangeFile(io.RawIOBase):
    """Seekable bounded HTTP reader for ZIP/ZIP64 central directories and members.

    A server returning 200 to a Range request is rejected BEFORE its body is read.
    ZipFile verifies each extracted member's CRC. Whole-archive SHA is only
    verified when the full-download fallback is used.
    """
    def __init__(self, url, size, session, block_size=8 * 1024**2):
        self.url, self.size, self.session = url, size, session
        self.pos, self.block_size = 0, block_size
        self.buffer_start, self.buffer = -1, b""
        self.bytes_received = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        if pos < 0:
            raise ValueError("Negative seek")
        self.pos = pos
        return pos

    def read(self, count=-1):
        count = max(0, self.size - self.pos) if count is None or count < 0 else min(count, max(0, self.size - self.pos))
        chunks = []
        while count:
            if not self.buffer_start <= self.pos < self.buffer_start + len(self.buffer):
                start = self.pos
                end = min(self.size - 1, start + max(self.block_size, min(count, self.block_size)) - 1)
                for attempt in range(3):
                    try:
                        from urllib.parse import urlsplit
                        report_progress("dataset_range_request", endpoint=urlsplit(self.url).hostname,
                                        request_bytes=end - start + 1, range_start=start,
                                        received_mib=round(self.bytes_received / 1024**2, 2), attempt=attempt + 1)
                        with self.session.get(self.url, headers={"Range": f"bytes={start}-{end}",
                                                               "Accept-Encoding": "identity"},
                                              stream=True, timeout=(30, 120)) as response:
                            expected = f"bytes {start}-{end}/{self.size}"
                            if response.status_code != 206 or response.headers.get("Content-Range") != expected:
                                raise RangeUnsupported(f"Range unavailable or size mismatch, HTTP {response.status_code}")
                            data = response.content
                            if len(data) != end - start + 1:
                                raise OSError("Truncated HTTP range")
                            self.bytes_received += len(data)
                            report_progress("dataset_range_received", received_mib=round(self.bytes_received / 1024**2, 2))
                            self.buffer_start, self.buffer = start, data
                        break
                    except RangeUnsupported:
                        raise
                    except Exception as exc:
                        report_progress("dataset_range_retry", attempt=attempt + 1, error_type=type(exc).__name__)
                        if attempt == 2:
                            raise
                        time.sleep(2 * (attempt + 1))
            offset = self.pos - self.buffer_start
            take = min(count, len(self.buffer) - offset)
            chunks.append(self.buffer[offset:offset + take])
            self.pos += take
            count -= take
        return b"".join(chunks)


def extract_members(archive, videos, directory):
    directory.mkdir(parents=True, exist_ok=True)
    members = {}
    for item in archive.infolist():
        if item.is_dir() or Path(item.filename).name.startswith("._"):
            continue
        match = re.search(r"sample_(\d+)", item.filename)
        if match and item.filename.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            sid = int(match[1])
            if sid in members:
                raise ValueError(f"Ambiguous source members for sample {sid}")
            members[sid] = item
    metadata = []
    for video in videos:
        sid = sample_id(video)
        item = members[sid]
        report_progress("extracting_video", sample_id=sid, expected_mib=round(item.file_size / 1024**2, 2))
        # The archive name is never used as an extraction path.
        dest = directory / f"sample_{sid}_real{Path(item.filename).suffix.lower()}"
        temp = dest.with_suffix(dest.suffix + ".tmp")
        with archive.open(item) as source, temp.open("wb") as target:
            while True:
                block = source.read(4 * 1024**2)
                if not block:
                    break
                target.write(block)
        if temp.stat().st_size != item.file_size:
            raise ValueError(f"Truncated extracted video {sid}")
        os.replace(temp, dest)
        metadata.append(dict(sample_id=sid, path=str(dest.resolve()), bytes=item.file_size,
                             member=item.filename, zip_crc32=item.CRC, sha256=digest(dest)))
        print(f"VIDEO_READY sample={sid} MiB={item.file_size/1024**2:.1f}", flush=True)
    return metadata


def download_full(urls, path):
    import requests
    if path.is_file() and path.stat().st_size == SHARD_BYTES and digest(path) == SHARD_SHA:
        return
    for url in urls:
        for attempt in range(3):
            try:
                start = path.stat().st_size if path.exists() else 0
                if start >= SHARD_BYTES:
                    path.unlink()
                    start = 0
                report_progress("dataset_full_download", cached_mib=round(start / 1024**2, 2))
                with requests.get(url, headers={"Range": f"bytes={start}-", "Accept-Encoding": "identity"},
                                  stream=True, timeout=(30, 180)) as response:
                    response.raise_for_status()
                    if response.status_code == 206:
                        if not response.headers.get("Content-Range", "").startswith(f"bytes {start}-"):
                            raise ValueError("Wrong resume offset")
                        mode = "ab"
                    else:
                        mode = "wb"
                    last = time.monotonic()
                    with path.open(mode) as handle:
                        for block in response.iter_content(4 * 1024**2):
                            handle.write(block)
                            if time.monotonic() - last > 30:
                                report_progress("dataset_full_download", downloaded_mib=round(path.stat().st_size / 1024**2, 2))
                                print(f"Archive {path.stat().st_size/1024**3:.2f}/2.08 GiB", flush=True)
                                last = time.monotonic()
                if path.stat().st_size == SHARD_BYTES and digest(path) == SHARD_SHA:
                    return
                raise ValueError("Full archive size/hash mismatch")
            except Exception as exc:
                print(f"Download retry {attempt+1}: {type(exc).__name__}", flush=True)
                time.sleep(3 * (attempt + 1))
    raise RuntimeError("Dataset download failed on mirror and official endpoint; rerun to resume")


def prepare_assets(videos, work):
    import requests
    directory = work / "videos"
    marker = work / "asset_manifest.json"
    if marker.exists():
        report_progress("verifying_cached_videos", videos=len(videos))
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if {x["sample_id"] for x in saved["videos"]} == {sample_id(v) for v in videos}:
            if all(Path(x["path"]).is_file() and Path(x["path"]).stat().st_size == x["bytes"]
                   and digest(x["path"]) == x["sha256"] for x in saved["videos"]):
                print("Reusing SHA-verified selected videos", flush=True)
                return saved
    from urllib.parse import quote
    urls = [f"https://{host}/datasets/mjuicem/StreamingBench/resolve/main/{quote(SHARD_NAME)}?download=true"
            for host in ("hf-mirror.com", "huggingface.co")]
    for url in urls:
        try:
            from urllib.parse import urlsplit
            report_progress("opening_remote_zip", endpoint=urlsplit(url).hostname, selected_videos=len(videos))
            with requests.Session() as session:
                remote = HTTPRangeFile(url, SHARD_BYTES, session)
                with ZipFile(remote) as archive:
                    metadata = extract_members(archive, videos, directory)
            saved = dict(mode="range_selected_members", videos=metadata, transferred_bytes=remote.bytes_received,
                         archive_expected_sha256=SHARD_SHA, whole_archive_sha256_verified=False,
                         integrity="ZIP member CRC verified; observed video SHA256 saved for resumption",
                         source=url)
            atomic_json(marker, saved)
            return saved
        except Exception as exc:
            print(f"Selective ZIP unavailable ({type(exc).__name__}); trying next endpoint", flush=True)
    print("Falling back to the single 2.08 GiB archive; extracting only selected videos", flush=True)
    archive_path = work / "downloads" / SHARD_NAME
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    download_full(urls, archive_path)
    with ZipFile(archive_path) as archive:
        metadata = extract_members(archive, videos, directory)
    saved = dict(mode="full_archive", videos=metadata, archive_expected_sha256=SHARD_SHA,
                 whole_archive_sha256_verified=True, source=urls)
    atomic_json(marker, saved)
    return saved


def chunk_plan(video, sampled_frames):
    cursor, chunks = 0, []
    for i, qa in enumerate(video["conversations"]):
        end = math.ceil(float(qa["end_time"]) * FPS)
        if end < cursor or end > sampled_frames:
            raise ValueError(f"Invalid prefix bound for sample {sample_id(video)}, question {i}")
        while cursor < end:
            stop = min(cursor + 16, end)
            chunks.append(dict(qa_index=i, start=cursor, stop=stop))
            cursor = stop
    return chunks


def causal_scores(model, input_ids, past_key_values=None):
    """Fresh layerwise causal propagation through actual Qwen2 decoder layers.

    The temporary DynamicCache has separate lists; Qwen2 appends via torch.cat
    and never writes original K/V tensors. The source cache remains unchanged.
    Explicit masks allow different physical cache lengths per layer.
    """
    import torch
    from transformers import DynamicCache
    past = model.kv_cache if past_key_values is None else past_key_values
    cache = DynamicCache.from_legacy_cache(tuple((k, v) for k, v in past))
    hidden = model.get_input_embeddings()(input_ids)
    batch, q_len = input_ids.shape
    starts = model._get_next_start_pos_per_layer()
    model._set_rotary_required_len(max(starts) + q_len)
    original_hooks = dict(model._layer_position_ids)
    weights = []
    try:
        for i, layer in enumerate(model.language_model.model.layers):
            pos = model._build_position_ids(starts[i], q_len, batch)
            model._layer_position_ids[i] = pos
            past_len = past[i][0].shape[2]
            mask = torch.zeros((batch, 1, q_len, past_len + q_len), device=hidden.device, dtype=hidden.dtype)
            future = torch.ones((q_len, q_len), device=hidden.device, dtype=torch.bool).triu(1)
            mask[:, :, :, past_len:].masked_fill_(future, torch.finfo(hidden.dtype).min)
            result = layer(hidden, attention_mask=mask, position_ids=pos, past_key_value=cache,
                           output_attentions=True, use_cache=False)
            hidden = result[0]
            weights.append(result[1])
            if result[1] is None or result[1].shape[-1] != past_len + q_len:
                raise ValueError("Native decoder did not return expected attention")
    finally:
        model._layer_position_ids.clear()
        model._layer_position_ids.update(original_hooks)
    return weights


def reference_scores(model, input_ids):
    import torch
    from transformers import DynamicCache
    starts = model._get_next_start_pos_per_layer()
    q_len, batch = input_ids.shape[1], input_ids.shape[0]
    previous = dict(model._layer_position_ids)
    try:
        for i in range(model.num_layers):
            model._layer_position_ids[i] = model._build_position_ids(starts[i], q_len, batch)
        model._set_rotary_required_len(max(starts) + q_len)
        cache = DynamicCache.from_legacy_cache(tuple((k.clone(), v.clone()) for k, v in model.kv_cache))
        output = model.language_model.model(input_ids=input_ids, past_key_values=cache,
                                            position_ids=model._layer_position_ids[0],
                                            use_cache=False, output_attentions=True,
                                            attention_mask=None, return_dict=True)
        return list(output.attentions)
    finally:
        model._layer_position_ids.clear()
        model._layer_position_ids.update(previous)


def ingest_features(model, features, frames):
    import torch
    from inference.reindex_1d import contiguous_kv
    features = features.to(device=model.device, dtype=model.dtype)
    starts = model._get_next_start_pos_per_layer()
    batch, length = features.shape[:2]
    model._set_rotary_required_len(max(starts) + length)
    try:
        for i in range(model.num_layers):
            model._layer_position_ids[i] = model._build_position_ids(starts[i], length, batch)
        output = model.language_model(inputs_embeds=features, past_key_values=model.kv_cache,
                                      use_cache=True, return_dict=True,
                                      position_ids=model._layer_position_ids[0],
                                      cache_position=None, attention_mask=None)
        model.kv_cache = contiguous_kv(output.past_key_values)
        model._append_position_ids(starts, length)
        model.last_encoded_frames = frames
        model.total_processed_frames += frames
    finally:
        model._layer_position_ids.clear()


def fifo_compress(model):
    import torch
    from inference.reindex_1d import compute_cos_sin_for_positions, rotary_delta, apply_rotary_delta_to_keys_only
    lengths = model._get_cache_seq_len_per_layer()
    if max(lengths) <= model.kv_size:
        return
    threshold = model.language_model.config.max_position_embeddings - 1024
    compact = max(int(p[-1]) for p in model._position_ids_cache) > threshold
    new = []
    for i, (keys, values) in enumerate(model.kv_cache):
        keep = torch.cat((torch.arange(model.visual_start_idx, device=model.device),
                          torch.arange(max(model.visual_start_idx, lengths[i] - model.kv_size), lengths[i], device=model.device)))
        old = model._position_ids_cache[i][keep]
        pos = torch.arange(len(keep), device=model.device) if compact else old
        k = keys.index_select(2, keep)
        if compact:
            co, si = compute_cos_sin_for_positions(model.language_model, len(old), old, keys.dtype, model.device)
            cn, sn = compute_cos_sin_for_positions(model.language_model, len(pos), pos, keys.dtype, model.device)
            cd, sd = rotary_delta(co, si, cn, sn)
            k = apply_rotary_delta_to_keys_only(k, cd, sd)
        new.append((k.contiguous(), values.index_select(2, keep).contiguous()))
        model._position_ids_cache[i] = pos.clone()
    model.kv_cache = new
    model.token_activity_cache = [None] * model.num_layers


def reset(model):
    model.clear_cache()
    model._layer_position_ids.clear()
    model._position_ids_cache = [None] * model.num_layers
    model.token_activity_cache = [None] * model.num_layers
    model.total_processed_frames = 0
    model.encode_init_prompt()


def sync():
    import torch
    torch.cuda.synchronize()


def prepare_features(model, video, work, identity):
    import numpy as np
    import torch
    from decord import VideoReader
    from PIL import Image, ImageDraw
    directory = work / "features" / f"sample_{sample_id(video)}"
    report_progress("preparing_video_features", sample_id=sample_id(video))
    marker = directory / "complete.json"
    if marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if saved["identity"] == identity and all((directory / x["file"]).is_file()
                and digest(directory / x["file"]) == x["sha256"] for x in saved["chunks"]):
            print(f"FEATURE_CACHE reuse sample={sample_id(video)}", flush=True)
            return saved
    directory.mkdir(parents=True, exist_ok=True)
    reader = VideoReader(video["video_path"], num_threads=1)
    fps = round(reader.get_avg_fps())
    step = int(fps / FPS)
    if step < 1:
        raise ValueError("Invalid source fps")
    indices = list(range(0, len(reader), step))
    plan = chunk_plan(video, len(indices))
    timestamps = reader.get_frame_timestamp(indices).tolist()
    frame_rows = [dict(sampled_index=i, source_frame=frame, start_s=float(ts[0]), end_s=float(ts[1]))
                  for i, (frame, ts) in enumerate(zip(indices, timestamps))]
    write_csv(directory / "sampled_timestamps.csv", frame_rows)
    chunks, total_encoder_s = [], 0.0
    for index, item in enumerate(plan):
        sync()
        start = time.perf_counter()
        frames = torch.from_numpy(reader.get_batch(indices[item["start"]:item["stop"]]).asnumpy())
        pixels = model.processor.video_processor(frames.permute(0, 3, 1, 2), return_tensors="pt").pixel_values_videos
        features = model.get_video_features(pixels.to(model.device, model.dtype)).detach().cpu().contiguous()
        sync()
        total_encoder_s += time.perf_counter() - start
        dest = directory / f"chunk_{index:04d}.pt"
        temp = dest.with_suffix(".tmp")
        torch.save(features, temp)
        os.replace(temp, dest)
        chunks.append(dict(**item, file=dest.name, sha256=digest(dest), tokens=features.shape[1]))
        report_progress("preparing_video_features", sample_id=sample_id(video), chunks_done=index + 1,
                        chunks_total=len(plan))
    # Contact sheets are sparse review aids, not evidence annotations.
    review_dir = work / "review"
    review_dir.mkdir(exist_ok=True)
    for qi, qa in enumerate(video["conversations"]):
        report_progress("making_contact_sheets", sample_id=sample_id(video), qa_index=qi)
        end = math.ceil(float(qa["end_time"]) * FPS)
        chosen = sorted(set(np.linspace(0, end - 1, min(12, end), dtype=int).tolist()))
        thumbnails = reader.get_batch([indices[i] for i in chosen]).asnumpy()
        canvas = Image.new("RGB", (4 * 256, 3 * 164), "white")
        draw = ImageDraw.Draw(canvas)
        for n, (frame, si) in enumerate(zip(thumbnails, chosen)):
            im = Image.fromarray(frame)
            im.thumbnail((256, 142))
            x, y = (n % 4) * 256, (n // 4) * 164
            canvas.paste(im, (x, y))
            draw.text((x + 3, y + 144), f"sampled {si} | {timestamps[si][0]:.2f}s", fill="black")
        canvas.save(review_dir / f"sample_{sample_id(video)}_q{qi}.jpg", quality=80)
    saved = dict(identity=identity, sample_id=sample_id(video), source_video_sha256=digest(video["video_path"]),
                 source_fps_rounded=fps, sampling_step=step, chunks=chunks,
                 preparation_s=total_encoder_s, sampled_frames_available=len(indices),
                 note="Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization")
    atomic_json(marker, saved)
    return saved


def checkpoint_valid(path, expected_hash, video, expected_method=None):
    if not path.exists():
        return False
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
        if value.get("config_hash") != expected_hash or value.get("status") != "complete":
            return False
        if expected_method is not None and value.get("method") != expected_method:
            return False
        rows = value["predictions"]
        if len(rows) != len(video["conversations"]):
            return False
        for i, (row, qa) in enumerate(zip(rows, video["conversations"])):
            gold = "ABCDEFGH"[qa["choices"].index(qa["answer"])]
            if not (row["qa_index"] == i and row["video_id"] == str(video["video_id"])
                    and row["question"] == qa["question"] and row["choices"] == qa["choices"]
                    and row["gold_choice"] == gold and row["gold_answer"] == qa["answer"]
                    and row["correct"] == int(row["pred_choice"] == gold)):
                return False
        return sha_predictions(rows) == value["prediction_sha256"]
    except (OSError, ValueError, KeyError, TypeError):
        return False


def sha_predictions(rows):
    return hashlib.sha256(stable(rows).encode()).hexdigest()


def run_method(model, analyzer, video, method, feature_meta, work, config_hash):
    import torch
    output = work / "checkpoints" / method / f"sample_{sample_id(video)}.json"
    report_progress("method_replay", method=method, sample_id=sample_id(video))
    if checkpoint_valid(output, config_hash, video, method):
        print(f"REUSE method={method} sample={sample_id(video)}", flush=True)
        return json.loads(output.read_text(encoding="utf-8"))
    random.seed(SEED)
    torch.manual_seed(SEED)
    reset(model)
    original = model._compute_attention_scores_manually
    if method == "causal":
        model._compute_attention_scores_manually = types.MethodType(causal_scores, model)
    feature_dir = work / "features" / f"sample_{sample_id(video)}"
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    events, rows = [], []
    try:
        for i, qa in enumerate(video["conversations"]):
            report_progress("method_replay", method=method, sample_id=sample_id(video), qa_index=i)
            for item in feature_meta["chunks"]:
                if item["qa_index"] != i:
                    continue
                features = torch.load(feature_dir / item["file"], map_location="cpu", weights_only=True)
                sync()
                before = time.perf_counter()
                ingest_features(model, features, item["stop"] - item["start"])
                sync()
                ingest_s = time.perf_counter() - before
                before = time.perf_counter()
                if method == "fifo":
                    fifo_compress(model)
                else:
                    model.predict_and_compress()
                sync()
                events.append(dict(qa_index=i, sampled_start=item["start"], sampled_stop=item["stop"],
                                   ingest_s=ingest_s, compression_s=time.perf_counter() - before,
                                   cache_lengths=model._get_cache_seq_len_per_layer()))
            gold = "ABCDEFGH"[qa["choices"].index(qa["answer"])]
            sync()
            before = time.perf_counter()
            answer = analyzer.video_close_qa(qa["question"], qa["choices"], gold)
            sync()
            rows.append(dict(video_id=str(video["video_id"]), sample_id=sample_id(video), qa_index=i,
                             query_time_s=qa["end_time"], task=qa["task"], question=qa["question"],
                             choices=qa["choices"], gold_answer=qa["answer"], gold_choice=gold,
                             pred_answer=answer["pred_answer"], pred_choice=answer["pred_choice"],
                             correct=int(answer["acc"]), answer_s=time.perf_counter() - before))
        sync()
        value = dict(status="complete", config_hash=config_hash, method=method,
                     predictions=rows, prediction_sha256=sha_predictions(rows), events=events,
                     replay_wall_s=time.perf_counter() - start,
                     peak_allocated_gib=torch.cuda.max_memory_allocated() / 1024**3,
                     peak_reserved_gib=torch.cuda.max_memory_reserved() / 1024**3)
        atomic_json(output, value)
        if not checkpoint_valid(output, config_hash, video, method):
            raise ValueError("Newly written checkpoint failed independent identity check")
        print(f"METHOD_DONE {method} sample={sample_id(video)} correct={sum(r['correct'] for r in rows)}/{len(rows)} replay_s={value['replay_wall_s']:.1f}", flush=True)
        return value
    finally:
        model._compute_attention_scores_manually = original
        model.clear_cache()


def preflight(model, video, work):
    """Prove feature ingestion and fresh scores agree with original/native paths."""
    import torch
    report_progress("gpu_preflight", sample_id=sample_id(video))
    from decord import VideoReader
    reader = VideoReader(video["video_path"], num_threads=1)
    step = int(round(reader.get_avg_fps()) / FPS)
    indices = list(range(0, len(reader), step))[:16]
    if len(indices) != 16:
        raise ValueError("Preflight needs 16 sampled frames")
    frames = torch.from_numpy(reader.get_batch(indices).asnumpy())
    reset(model)
    original = model.get_video_features
    captured = []
    def capture(pixels):
        features = original(pixels)
        captured.append(features.detach().cpu())
        return features
    model.get_video_features = capture
    try:
        model.encode_video_chunk(frames)
    finally:
        model.get_video_features = original
    cache = tuple((k.clone(), v.clone()) for k, v in model.kv_cache)
    positions = [p.clone() for p in model._position_ids_cache]
    reset(model)
    ingest_features(model, captured[0], len(frames))
    for (ka, va), (kb, vb), pa, pb in zip(cache, model.kv_cache, positions, model._position_ids_cache):
        torch.testing.assert_close(ka, kb, atol=0, rtol=0)
        torch.testing.assert_close(va, vb, atol=0, rtol=0)
        torch.testing.assert_close(pa, pb, atol=0, rtol=0)
    del cache
    comparisons = []
    for stage in ("before_pruning", "after_position_sensitive_pruning"):
        if stage != "before_pruning":
            # Constrain selection to force real pruning and summary positions.
            keep = []
            for i, length in enumerate(model._get_cache_seq_len_per_layer()):
                n = 255 if i >= model.long_term_threshold else 256
                keep.append(torch.cat((torch.arange(model.visual_start_idx, device=model.device),
                                       torch.arange(length - n, length, device=model.device))))
            model.apply_kv_cache_pruning_strict(keep)
        ids = model.processor.tokenizer("Describe objects and earlier events in the scene.", return_tensors="pt").input_ids.to(model.device)
        previous = tuple((k.clone(), v.clone()) for k, v in model.kv_cache)
        before_positions = [p.clone() for p in model._position_ids_cache]
        before_activity = [p.clone() if p is not None else None for p in model.token_activity_cache]
        actual = causal_scores(model, ids)
        expected = reference_scores(model, ids)
        max_error = 0.0
        for a, b in zip(actual, expected):
            torch.testing.assert_close(a, b, atol=2e-3, rtol=2e-3)
            max_error = max(max_error, float((a - b).abs().max()))
            # Explicit causality must give future probe tokens no mass.
            q = ids.shape[1]
            assert not torch.any(torch.triu(a[..., -q:], diagonal=1))
        for (ka, va), (kb, vb), pa, pb in zip(previous, model.kv_cache, before_positions, model._position_ids_cache):
            torch.testing.assert_close(ka, kb, atol=0, rtol=0)
            torch.testing.assert_close(va, vb, atol=0, rtol=0)
            torch.testing.assert_close(pa, pb, atol=0, rtol=0)
        assert all(a is None and b is None or a is not None and torch.equal(a, b)
                   for a, b in zip(before_activity, model.token_activity_cache))
        comparisons.append(dict(stage=stage, max_attention_error=max_error,
                                cache_lengths=model._get_cache_seq_len_per_layer()))
        del actual, expected, previous
    result = dict(status="pass", feature_ingestion="bitwise_equal_KV_and_positions",
                  causal_scorer_comparisons=comparisons, source_cache_unchanged=True)
    atomic_json(work / "preflight.json", result)
    model.clear_cache()
    print("PREFLIGHT_PASS: feature reuse, native attention, causality and cache immutability", flush=True)
    return result


def export_results(results, selected, work, config):
    targets = {r["case_id"] for r in selected}
    flat = [dict(method=x["method"], **r) for x in results for r in x["predictions"]]
    write_csv(work / "exports/predictions.csv", flat)
    summary, pairs = [], []
    for method in METHODS:
        runs = [x for x in results if x["method"] == method]
        rows = [r for x in runs for r in x["predictions"]]
        focus = [r for r in rows if f"sample_{r['sample_id']:03d}_q{r['qa_index']}" in targets]
        summary.append(dict(method=method, videos=len(runs), questions=len(rows),
                            correct=sum(r["correct"] for r in rows),
                            accuracy_pct=100 * sum(r["correct"] for r in rows) / len(rows),
                            selected_targets=len(focus), target_correct=sum(r["correct"] for r in focus),
                            total_replay_s=sum(x["replay_wall_s"] for x in runs),
                            peak_allocated_gib=max(x["peak_allocated_gib"] for x in runs)))
    indexed = {(r["method"], r["video_id"], r["qa_index"]): r for r in flat}
    for row in flat:
        if row["method"] != "hermes":
            continue
        f, c = [indexed[(m, row["video_id"], row["qa_index"])] for m in ("fifo", "causal")]
        pairs.append(dict(sample_id=row["sample_id"], video_id=row["video_id"], qa_index=row["qa_index"],
                          task=row["task"], hermes_correct=row["correct"], fifo_correct=f["correct"],
                          causal_correct=c["correct"], hermes_choice=row["pred_choice"],
                          fifo_choice=f["pred_choice"], causal_choice=c["pred_choice"],
                          is_selected_target=f"sample_{row['sample_id']:03d}_q{row['qa_index']}" in targets))
    write_csv(work / "exports/summary.csv", summary)
    write_csv(work / "exports/paired_questions.csv", pairs)
    atomic_json(work / "exports/run_config.json", config)
    lines = ["# AutoDL causal-scorer screen", "", "Exploratory outcome-selected development reuse. No confirmatory accuracy claim.", "",
             "Shared decode/vision feature preparation is outside method replay timings. These are not full-system streaming latencies.", "",
             "| Method | Correct / QA | Target correct | Replay seconds |", "|---|---:|---:|---:|"]
    for r in summary:
        lines.append(f"| {r['method']} | {r['correct']}/{r['questions']} | {r['target_correct']}/{r['selected_targets']} | {r['total_replay_s']:.1f} |")
    lines += ["", "Evidence intervals, token survival and visual causes remain unannotated. Contact sheets are sparse review aids.",
              "A fresh causal scorer is a reproduction diagnostic. This run does not establish an original algorithm."]
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return summary


def package(work, status):
    import shutil
    from zipfile import ZIP_DEFLATED
    if (work / "progress.json").exists():
        atomic_json(work / "exports/progress_snapshot.json", json.loads((work / "progress.json").read_text(encoding="utf-8")))
    # The notebook writes the live subprocess log concurrently. Freeze copies
    # before hashing, otherwise archive entries can race their manifest hashes.
    for source in (work / "logs").rglob("*"):
        if source.is_file():
            target = work / "captured_logs" / source.relative_to(work / "logs")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    files = []
    for area in ("exports", "checkpoints", "review", "captured_logs"):
        files.extend(p for p in (work / area).rglob("*") if p.is_file())
    files.extend(p for name in ("preflight.json", "asset_manifest.json", "selection.json", "pilot_runner.py", "launch.json")
                 if (p := work / name).is_file())
    files.extend(p for p in (work / "features").rglob("*.csv"))
    files.extend(p for p in (work / "features").rglob("complete.json"))
    manifest = [dict(path=p.relative_to(work).as_posix(), bytes=p.stat().st_size, sha256=digest(p))
                for p in sorted(set(files))]
    atomic_json(work / "evidence_manifest.json", dict(status=status, files=manifest))
    output = work / "HERMES_causal_scorer_pilot_evidence.zip"
    temp = output.with_suffix(".tmp")
    with ZipFile(temp, "w", compression=ZIP_DEFLATED) as archive:
        for item in manifest:
            archive.write(work / item["path"], item["path"])
        archive.write(work / "evidence_manifest.json", "evidence_manifest.json")
    os.replace(temp, output)
    print(f"EVIDENCE_ZIP={output}\nSHA256={digest(output)}", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo", type=Path, required=True)
    parser.add_argument("--work", type=Path, required=True)
    parser.add_argument("--model", type=Path, required=True)
    parser.add_argument("--per-outcome", type=int, default=2, choices=[1, 2, 3])
    parser.add_argument("--max-hours", type=float, default=2.0)
    args = parser.parse_args()
    with ProgressMonitor(args.work):
        execute(args)


def execute(args):
    if args.model.name != MODEL_SHA:
        raise ValueError("Use the pinned model snapshot directory named by MODEL_SHA")
    work, repo = args.work.resolve(), args.repo.resolve()
    sys.path.insert(0, str(repo / "scripts"))
    sys.path.insert(0, str(repo))
    selected, videos = choose_screen(repo, args.per_outcome)
    report_progress("dataset_preparation", selected_videos=len(videos))
    work.mkdir(parents=True, exist_ok=True)
    review_sheet = work / "review/evidence_review.csv"
    if review_sheet.exists() and [r["case_id"] for r in read_csv(review_sheet)] != [r["case_id"] for r in selected]:
        raise ValueError("Existing review sheet has different cases; use a new WORK directory for a changed screen size")
    previous_failure = work / "exports/failure.json"
    if previous_failure.exists():
        history = work / "logs" / f"previous_failure_{time.time_ns()}.json"
        history.parent.mkdir(parents=True, exist_ok=True)
        os.replace(previous_failure, history)
    atomic_json(work / "selection.json", dict(cases=selected, source="201-250 inspected development shard",
                                             per_outcome=args.per_outcome, seed=SEED))
    assets = prepare_assets(videos, work)
    asset_by_id = {x["sample_id"]: x for x in assets["videos"]}
    for v in videos:
        v["video_path"] = asset_by_id[sample_id(v)]["path"]
    atomic_json(work / "exports/replay_annotations.json", videos)
    if not review_sheet.exists():
        write_csv(review_sheet, [dict(case_id=r["case_id"], review_status="pending",
                                    support_intervals_seconds_json="", support_visible_at_original_sampling="",
                                    failure_cause="unclassified", notes="") for r in selected])
    # Pin inference sources and assets in every checkpoint identity.
    report_progress("importing_gpu_libraries")
    import numpy as np
    import torch
    import transformers
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required")
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    source_files = ["inference/llavaov_hermes.py", "inference/abstract_hermes.py", "inference/reindex_1d.py",
                    "video_qa/base.py", "video_qa/hermes_vqa.py", "scripts/build_failure_pilot.py"]
    config = dict(model_id=MODEL_ID, model_revision=MODEL_SHA, kv_size=4000, sample_fps=FPS, seed=SEED,
                  source_sha256={p: digest(repo / p) for p in source_files},
                  runner_sha256=hashlib.sha256(Path(__file__).read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
                  torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0),
                  methods=METHODS, shared_features=True, feature_cache_dtype="float16",
                  assets={str(x["sample_id"]): x["sha256"] for x in assets["videos"]},
                  conversations=[dict(video_id=v["video_id"], conversations=v["conversations"]) for v in videos],
                  selected_case_ids=[r["case_id"] for r in selected],
                  note="Outcome-selected development screen; full-history replay; no population inference")
    config_hash = hashlib.sha256(stable(config).encode()).hexdigest()
    config["config_hash"] = config_hash
    config["soft_budget_hours_this_invocation"] = args.max_hours
    atomic_json(work / "exports/run_config.json", config)
    report_progress("loading_model", model_revision=MODEL_SHA)
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=FPS)
    if model.language_model.config._attn_implementation != "sdpa":
        raise RuntimeError("Current HERMES control must use its original SDPA/manual-score path")
    analyzer = HermesVQA(anno=[], save_dir=str(work), sample_fps=FPS, qa_model=model, qa_processor=processor)
    results, completed_videos = [], 0
    started = time.perf_counter()
    summary = []
    try:
        with torch.inference_mode():
            preflight(model, videos[0], work)
            for index, video in enumerate(videos):
                # Stop only between complete three-method video blocks.
                if index >= 2 and args.max_hours > 0 and time.perf_counter() - started >= args.max_hours * 3600:
                    print("TIME_BUDGET_STOP: valid partial screen; rerun to resume", flush=True)
                    break
                feature_identity = hashlib.sha256(stable(dict(
                    model_revision=MODEL_SHA, source=config["source_sha256"],
                    source_video=asset_by_id[sample_id(video)]["sha256"],
                    conversations=video["conversations"], sample_fps=FPS,
                    runner=config["runner_sha256"], torch=torch.__version__)).encode()).hexdigest()
                meta = prepare_features(model, video, work, feature_identity)
                order = list(METHODS)
                random.Random(SEED + index).shuffle(order)
                for method in order:
                    results.append(run_method(model, analyzer, video, method, meta, work, config_hash))
                completed_videos += 1
                report_progress("video_block_complete", completed_videos=completed_videos, planned_videos=len(videos))
                summary = export_results(results, selected, work, config)
                elapsed = time.perf_counter() - started
                print(f"BLOCK_DONE {completed_videos}/{len(videos)} elapsed_min={elapsed/60:.1f} remaining_estimate_min={(len(videos)-completed_videos)*elapsed/completed_videos/60:.1f}", flush=True)
        status = "complete" if completed_videos == len(videos) else "partial_time_budget"
        atomic_json(work / "exports/completion.json", dict(status=status, videos=completed_videos,
                    planned_videos=len(videos), methods=list(METHODS), summary=summary,
                    config_hash=config_hash, elapsed_s=time.perf_counter() - started))
        report_progress(status, completed_videos=completed_videos, planned_videos=len(videos))
        package(work, status)
        print("PILOT_COMPLETE" if status == "complete" else "PILOT_PARTIAL", flush=True)
    except Exception as exc:
        atomic_json(work / "exports/failure.json", dict(status="failed", type=type(exc).__name__,
                    detail=str(exc)[-1500:], completed_video_blocks=completed_videos))
        atomic_json(work / "exports/completion.json", dict(status="failed", config_hash=config_hash,
                    completed_video_blocks=completed_videos, planned_videos=len(videos)))
        package(work, "failed_partial")
        raise


if __name__ == "__main__":
    main()
'''
ast.parse(UTILS_SOURCE)
assert hashlib.sha256(UTILS_SOURCE.encode()).hexdigest() == 'f91d6d839198e8ebfcf597a8fff1738bcccd89f575c9eb18677711c310977db3'
REFERENCE_JSON = '{"archive_sha256":"af4b7392e90bd3f433ad20182a9058d8c8027a8ede936ed5d759572be04e0361","context_archive_sha256":"fee3165e323c0f7106688a37cb185ec390d4f831e9debdfbf0352bd72881b774","input_archive_sha256":"f184271bd8929f2ca5dead71a1ec60e4e1b555f33f2ed66d9909e3d7e0ffa1e8","input_rows":[{"answer_s":0.13301409408450127,"cache_lengths_before_answer":[1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189],"choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"Yellow and Red.","gold_choice":"C","ingest_s":0.025045201182365417,"method":"native_prefix","no_compression":true,"peak_allocated_gib":2.8779053688049316,"pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":0,"query_time_s":12.0,"question":"What colors are the wheels of the race car?","sample_id":202,"sampled_indices":[0,1,2,3,4,5],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":1176,"wall_s":0.22635002247989178},{"answer_s":0.13568025827407837,"cache_lengths_before_answer":[7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069,7069],"choices":["A sad face.","A happy face.","A star.","A heart."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A happy face.","gold_choice":"B","ingest_s":0.12837261706590652,"method":"native_prefix","no_compression":true,"peak_allocated_gib":4.599101543426514,"pred_answer":"B) A happy face.","pred_choice":"B","qa_index":1,"query_time_s":72.0,"question":"What feature is present on the front of the toy?","sample_id":202,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":7056,"wall_s":0.3688268642872572},{"answer_s":0.0,"choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"In a straight line.","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":202,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780],"status":"skipped","task":"Spatial Understanding","video_id":"459","visual_tokens":12544,"wall_s":0.018911749124526978},{"answer_s":0.0,"choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":202.0,"question":"What is the new item introduced in the video?","sample_id":202,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000],"status":"skipped","task":"Object Recognition","video_id":"459","visual_tokens":19796,"wall_s":0.01882094331085682},{"answer_s":0.0,"choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":227.0,"question":"What happened to the toy vehicle just now?","sample_id":202,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0,212.0,214.0,216.0,218.0,220.0,222.0,224.0,226.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780],"status":"skipped","task":"Event Understanding","video_id":"459","visual_tokens":22344,"wall_s":0.020449623465538025},{"answer_s":0.0966744814068079,"cache_lengths_before_answer":[1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581],"choices":["Eight.","Sixteen.","Four.","Twelve."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Eight.","gold_choice":"A","ingest_s":0.03261445090174675,"method":"native_prefix","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"A) Eight.","pred_choice":"A","qa_index":0,"query_time_s":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420],"status":"answered","task":"Counting","video_id":"355","visual_tokens":1568,"wall_s":0.203896289691329},{"answer_s":0.10129381157457829,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["Three.","Five.","Seven.","Four."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Four.","gold_choice":"D","ingest_s":0.05634419806301594,"method":"native_prefix","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"D) Four.","pred_choice":"D","qa_index":1,"query_time_s":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Counting","video_id":"355","visual_tokens":1960,"wall_s":0.22737760841846466},{"answer_s":0.09545655362308025,"cache_lengths_before_answer":[7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461,7461],"choices":["Yellow.","Orange.","Green.","Blue."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"C","ingest_s":0.15345196053385735,"method":"native_prefix","no_compression":true,"peak_allocated_gib":4.617183208465576,"pred_answer":"A) Yellow.","pred_choice":"A","qa_index":2,"query_time_s":76.0,"question":"What is the color of the block that the hand is reaching for right now?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220],"status":"answered","task":"Attribute Recognition","video_id":"355","visual_tokens":7448,"wall_s":0.346147870644927},{"answer_s":0.0,"choices":["Green.","Yellow.","Orange.","Red."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960],"status":"skipped","task":"Event Understanding","video_id":"355","visual_tokens":13132,"wall_s":0.01862090453505516},{"answer_s":0.0,"choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green, yellow, orange.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200],"status":"skipped","task":"Event Understanding","video_id":"355","visual_tokens":13916,"wall_s":0.01880013383924961},{"answer_s":0.09551026485860348,"cache_lengths_before_answer":[1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581],"choices":["Red.","Blue.","Yellow.","Purple."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Yellow.","gold_choice":"C","ingest_s":0.0273384228348732,"method":"native_prefix","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"A) Red.","pred_choice":"A","qa_index":0,"query_time_s":16.0,"question":"What color is the plastic block added on top of the green blocks?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":1568,"wall_s":0.1986919678747654},{"answer_s":0.1818887311965227,"cache_lengths_before_answer":[6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089,6089],"choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Red, green, and yellow.","gold_choice":"B","ingest_s":0.1061326190829277,"method":"native_prefix","no_compression":true,"peak_allocated_gib":4.608142375946045,"pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":1,"query_time_s":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":6076,"wall_s":0.3842499256134033},{"answer_s":0.0,"choices":["A white block.","A green block.","An yellow block.","A blue block."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"An yellow block.","gold_choice":"C","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520],"status":"skipped","task":"Event Understanding","video_id":"335","visual_tokens":8428,"wall_s":0.01912441849708557},{"answer_s":0.0,"choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":141.0,"question":"What was the white text on the purple animated overlay read just now?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200],"status":"skipped","task":"Text-Rich Understanding","video_id":"335","visual_tokens":13916,"wall_s":0.01875043287873268},{"answer_s":0.0,"choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880],"status":"skipped","task":"Event Understanding","video_id":"335","visual_tokens":19404,"wall_s":0.019366499036550522},{"answer_s":0.16368762776255608,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Yellow, green.","gold_choice":"A","ingest_s":0.022658951580524445,"method":"native_prefix","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":0,"query_time_s":7.0,"question":"What are the colors of the crane arm?","sample_id":208,"sampled_indices":[0,1,2,3],"sampled_start_seconds":[0.0,2.0,4.0,6.0],"skip_reason":"","source_frame_indices":[0,60,120,180],"status":"answered","task":"Attribute Recognition","video_id":"448","visual_tokens":784,"wall_s":0.2395964302122593},{"answer_s":0.18304254300892353,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It lifts and swings gently.","gold_choice":"C","ingest_s":0.046508664265275,"method":"native_prefix","no_compression":true,"peak_allocated_gib":2.895987033843994,"pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":1,"query_time_s":19.0,"question":"What happens to the orange block as the video progresses?","sample_id":208,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":1960,"wall_s":0.29223909229040146},{"answer_s":0.0,"choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":96.0,"question":"What was the sequence of actions performed with the blocks just now?","sample_id":208,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820],"status":"skipped","task":"Event Understanding","video_id":"448","visual_tokens":9408,"wall_s":0.01916978694498539},{"answer_s":0.0,"choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","sample_id":208,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100],"status":"skipped","task":"Event Understanding","video_id":"448","visual_tokens":16856,"wall_s":0.018971703946590424},{"answer_s":0.0,"choices":["Ten.","Three.","Nine.","Eight."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Nine.","gold_choice":"C","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":265.0,"question":"How many green blocks are stacked now?","sample_id":208,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0,212.0,214.0,216.0,218.0,220.0,222.0,224.0,226.0,228.0,230.0,232.0,234.0,236.0,238.0,240.0,242.0,244.0,246.0,248.0,250.0,252.0,254.0,256.0,258.0,260.0,262.0,264.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920],"status":"skipped","task":"Counting","video_id":"448","visual_tokens":26068,"wall_s":0.018789120018482208},{"answer_s":0.1955207847058773,"cache_lengths_before_answer":[2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561],"choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"correct":1,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The hand adds another green block.","gold_choice":"D","ingest_s":0.03814482130110264,"method":"native_prefix","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"D) The hand adds another green block.","pred_choice":"D","qa_index":0,"query_time_s":25.0,"question":"What happens after the first green block is placed on the yellow block?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":2548,"wall_s":0.3208392057567835},{"answer_s":0.0,"choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Pressing down the green blocks.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":118.0,"question":"What action do the hands perform just before moving away from the structure?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480],"status":"skipped","task":"Action Recognition","video_id":"118","visual_tokens":11564,"wall_s":0.018969601020216942},{"answer_s":0.0,"choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240],"status":"skipped","task":"Event Understanding","video_id":"118","visual_tokens":20580,"wall_s":0.01959080994129181},{"answer_s":0.0,"choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"In front of the original structure.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300],"status":"skipped","task":"Spatial Understanding","video_id":"118","visual_tokens":20776,"wall_s":0.018643638119101524},{"answer_s":0.0,"choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":315.0,"question":"What did the person do with the yellow block just now?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0,212.0,214.0,216.0,218.0,220.0,222.0,224.0,226.0,228.0,230.0,232.0,234.0,236.0,238.0,240.0,242.0,244.0,246.0,248.0,250.0,252.0,254.0,256.0,258.0,260.0,262.0,264.0,266.0,268.0,270.0,272.0,274.0,276.0,278.0,280.0,282.0,284.0,286.0,288.0,290.0,292.0,294.0,296.0,298.0,300.0,302.0,304.0,306.0,308.0,310.0,312.0,314.0],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420],"status":"skipped","task":"Action Recognition","video_id":"118","visual_tokens":30968,"wall_s":0.018549129366874695},{"answer_s":0.16758546605706215,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add mixed numbers.","gold_choice":"A","ingest_s":0.030477210879325867,"method":"native_prefix","no_compression":true,"peak_allocated_gib":3.555534839630127,"pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":0,"query_time_s":20.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":1960,"wall_s":0.27328724414110184},{"answer_s":0.0,"choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":130.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":12740,"wall_s":0.01924736611545086},{"answer_s":0.0,"choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":248.0,"question":"What might the speaker do next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":24304,"wall_s":0.018697604537010193},{"answer_s":0.0,"choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two fractions numbers.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":396.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656,332.3320007324219,334.3340148925781,336.33599853515625,338.3380126953125,340.3399963378906,342.3420104980469,344.343994140625,346.34600830078125,348.3479919433594,350.3500061035156,352.35198974609375,354.35400390625,356.3559875488281,358.3580017089844,360.3599853515625,362.36199951171875,364.364013671875,366.3659973144531,368.3680114746094,370.3699951171875,372.37200927734375,374.3739929199219,376.3760070800781,378.37799072265625,380.3800048828125,382.3819885253906,384.3840026855469,386.385986328125,388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719],"skip_reason":"context_limit","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900,9960,10020,10080,10140,10200,10260,10320,10380,10440,10500,10560,10620,10680,10740,10800,10860,10920,10980,11040,11100,11160,11220,11280,11340,11400,11460,11520,11580,11640,11700,11760,11820],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":38808,"wall_s":0.01988065056502819},{"answer_s":0.0,"choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"Adding the fractions together.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":478.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656,332.3320007324219,334.3340148925781,336.33599853515625,338.3380126953125,340.3399963378906,342.3420104980469,344.343994140625,346.34600830078125,348.3479919433594,350.3500061035156,352.35198974609375,354.35400390625,356.3559875488281,358.3580017089844,360.3599853515625,362.36199951171875,364.364013671875,366.3659973144531,368.3680114746094,370.3699951171875,372.37200927734375,374.3739929199219,376.3760070800781,378.37799072265625,380.3800048828125,382.3819885253906,384.3840026855469,386.385986328125,388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719,396.39599609375,398.39801025390625,400.3999938964844,402.4020080566406,404.40399169921875,406.406005859375,408.4079895019531,410.4100036621094,412.4119873046875,414.41400146484375,416.4159851074219,418.4179992675781,420.4200134277344,422.4219970703125,424.42401123046875,426.4259948730469,428.4280090332031,430.42999267578125,432.4320068359375,434.4339904785156,436.4360046386719,438.43798828125,440.44000244140625,442.4419860839844,444.4440002441406,446.4460144042969,448.447998046875,450.45001220703125,452.4519958496094,454.4540100097656,456.45599365234375,458.4580078125,460.4599914550781,462.4620056152344,464.4639892578125,466.46600341796875,468.4679870605469,470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375],"skip_reason":"context_limit","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900,9960,10020,10080,10140,10200,10260,10320,10380,10440,10500,10560,10620,10680,10740,10800,10860,10920,10980,11040,11100,11160,11220,11280,11340,11400,11460,11520,11580,11640,11700,11760,11820,11880,11940,12000,12060,12120,12180,12240,12300,12360,12420,12480,12540,12600,12660,12720,12780,12840,12900,12960,13020,13080,13140,13200,13260,13320,13380,13440,13500,13560,13620,13680,13740,13800,13860,13920,13980,14040,14100,14160,14220,14280],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":46844,"wall_s":0.01946260593831539},{"answer_s":0.16587380319833755,"cache_lengths_before_answer":[1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189],"choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","ingest_s":0.024747123941779137,"method":"native_prefix","no_compression":true,"peak_allocated_gib":2.8779053688049316,"pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":0,"query_time_s":11.0,"question":"What topic might the teacher explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":1176,"wall_s":0.24963464960455894},{"answer_s":0.0,"choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":190.0,"question":"What might the teacher explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":18620,"wall_s":0.027171339839696884},{"answer_s":0.0,"choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":331.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":32536,"wall_s":0.022030776366591454},{"answer_s":0.0,"choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":500.0,"question":"What concept will the speaker likely explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656,332.3320007324219,334.3340148925781,336.33599853515625,338.3380126953125,340.3399963378906,342.3420104980469,344.343994140625,346.34600830078125,348.3479919433594,350.3500061035156,352.35198974609375,354.35400390625,356.3559875488281,358.3580017089844,360.3599853515625,362.36199951171875,364.364013671875,366.3659973144531,368.3680114746094,370.3699951171875,372.37200927734375,374.3739929199219,376.3760070800781,378.37799072265625,380.3800048828125,382.3819885253906,384.3840026855469,386.385986328125,388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719,396.39599609375,398.39801025390625,400.3999938964844,402.4020080566406,404.40399169921875,406.406005859375,408.4079895019531,410.4100036621094,412.4119873046875,414.41400146484375,416.4159851074219,418.4179992675781,420.4200134277344,422.4219970703125,424.42401123046875,426.4259948730469,428.4280090332031,430.42999267578125,432.4320068359375,434.4339904785156,436.4360046386719,438.43798828125,440.44000244140625,442.4419860839844,444.4440002441406,446.4460144042969,448.447998046875,450.45001220703125,452.4519958496094,454.4540100097656,456.45599365234375,458.4580078125,460.4599914550781,462.4620056152344,464.4639892578125,466.46600341796875,468.4679870605469,470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375,478.4779968261719,480.4800109863281,482.48199462890625,484.4840087890625,486.4859924316406,488.4880065917969,490.489990234375,492.49200439453125,494.4939880371094,496.4960021972656,498.49798583984375],"skip_reason":"context_limit","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900,9960,10020,10080,10140,10200,10260,10320,10380,10440,10500,10560,10620,10680,10740,10800,10860,10920,10980,11040,11100,11160,11220,11280,11340,11400,11460,11520,11580,11640,11700,11760,11820,11880,11940,12000,12060,12120,12180,12240,12300,12360,12420,12480,12540,12600,12660,12720,12780,12840,12900,12960,13020,13080,13140,13200,13260,13320,13380,13440,13500,13560,13620,13680,13740,13800,13860,13920,13980,14040,14100,14160,14220,14280,14340,14400,14460,14520,14580,14640,14700,14760,14820,14880,14940],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":49000,"wall_s":0.018674595281481743},{"answer_s":0.0,"choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":646.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656,332.3320007324219,334.3340148925781,336.33599853515625,338.3380126953125,340.3399963378906,342.3420104980469,344.343994140625,346.34600830078125,348.3479919433594,350.3500061035156,352.35198974609375,354.35400390625,356.3559875488281,358.3580017089844,360.3599853515625,362.36199951171875,364.364013671875,366.3659973144531,368.3680114746094,370.3699951171875,372.37200927734375,374.3739929199219,376.3760070800781,378.37799072265625,380.3800048828125,382.3819885253906,384.3840026855469,386.385986328125,388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719,396.39599609375,398.39801025390625,400.3999938964844,402.4020080566406,404.40399169921875,406.406005859375,408.4079895019531,410.4100036621094,412.4119873046875,414.41400146484375,416.4159851074219,418.4179992675781,420.4200134277344,422.4219970703125,424.42401123046875,426.4259948730469,428.4280090332031,430.42999267578125,432.4320068359375,434.4339904785156,436.4360046386719,438.43798828125,440.44000244140625,442.4419860839844,444.4440002441406,446.4460144042969,448.447998046875,450.45001220703125,452.4519958496094,454.4540100097656,456.45599365234375,458.4580078125,460.4599914550781,462.4620056152344,464.4639892578125,466.46600341796875,468.4679870605469,470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375,478.4779968261719,480.4800109863281,482.48199462890625,484.4840087890625,486.4859924316406,488.4880065917969,490.489990234375,492.49200439453125,494.4939880371094,496.4960021972656,498.49798583984375,500.5,502.50201416015625,504.5039978027344,506.5060119628906,508.50799560546875,510.510009765625,512.5120239257812,514.5139770507812,516.5159912109375,518.5180053710938,520.52001953125,522.52197265625,524.5239868164062,526.5260009765625,528.5280151367188,530.530029296875,532.531982421875,534.5339965820312,536.5360107421875,538.5380249023438,540.5399780273438,542.5419921875,544.5440063476562,546.5460205078125,548.5479736328125,550.5499877929688,552.552001953125,554.5540161132812,556.5560302734375,558.5579833984375,560.5599975585938,562.56201171875,564.5640258789062,566.5659790039062,568.5679931640625,570.5700073242188,572.572021484375,574.573974609375,576.5759887695312,578.5780029296875,580.5800170898438,582.5819702148438,584.583984375,586.5859985351562,588.5880126953125,590.5900268554688,592.5919799804688,594.593994140625,596.5960083007812,598.5980224609375,600.5999755859375,602.6019897460938,604.60400390625,606.6060180664062,608.6079711914062,610.6099853515625,612.6119995117188,614.614013671875,616.6160278320312,618.6179809570312,620.6199951171875,622.6220092773438,624.6240234375,626.6259765625,628.6279907226562,630.6300048828125,632.6320190429688,634.6339721679688,636.635986328125,638.6380004882812,640.6400146484375,642.6420288085938,644.6439819335938],"skip_reason":"context_limit","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900,9960,10020,10080,10140,10200,10260,10320,10380,10440,10500,10560,10620,10680,10740,10800,10860,10920,10980,11040,11100,11160,11220,11280,11340,11400,11460,11520,11580,11640,11700,11760,11820,11880,11940,12000,12060,12120,12180,12240,12300,12360,12420,12480,12540,12600,12660,12720,12780,12840,12900,12960,13020,13080,13140,13200,13260,13320,13380,13440,13500,13560,13620,13680,13740,13800,13860,13920,13980,14040,14100,14160,14220,14280,14340,14400,14460,14520,14580,14640,14700,14760,14820,14880,14940,15000,15060,15120,15180,15240,15300,15360,15420,15480,15540,15600,15660,15720,15780,15840,15900,15960,16020,16080,16140,16200,16260,16320,16380,16440,16500,16560,16620,16680,16740,16800,16860,16920,16980,17040,17100,17160,17220,17280,17340,17400,17460,17520,17580,17640,17700,17760,17820,17880,17940,18000,18060,18120,18180,18240,18300,18360,18420,18480,18540,18600,18660,18720,18780,18840,18900,18960,19020,19080,19140,19200,19260,19320],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":63308,"wall_s":0.018624020740389824},{"answer_s":0.15558875538408756,"cache_lengths_before_answer":[993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993],"choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Multiplying Fractions.","gold_choice":"D","ingest_s":0.024517681449651718,"method":"native_prefix","no_compression":true,"peak_allocated_gib":2.708498001098633,"pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":0,"query_time_s":10.0,"question":"What mathematical topic might the speaker focus on next?","sample_id":219,"sampled_indices":[0,1,2,3,4],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332],"skip_reason":"","source_frame_indices":[0,60,120,180,240],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":980,"wall_s":0.24338097870349884},{"answer_s":0.2054725605994463,"cache_lengths_before_answer":[5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501,5501],"choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.09756186045706272,"method":"native_prefix","no_compression":true,"peak_allocated_gib":4.594581127166748,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":1,"query_time_s":55.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":5488,"wall_s":0.39873453602194786},{"answer_s":0.0,"choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":149.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":14700,"wall_s":0.019910821691155434},{"answer_s":0.0,"choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Another explanation to fractions.","gold_choice":"D","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":231.0,"question":"What mathematical concept might the speaker move on to explain next?","sample_id":219,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":22736,"wall_s":0.01976478286087513},{"answer_s":0.0,"choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"How to multiply the denominators.","gold_choice":"C","ingest_s":0.0,"method":"native_prefix","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":310.0,"question":"What might the speaker discuss next?","sample_id":219,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219,56.055999755859375,58.05799865722656,60.060001373291016,62.0620002746582,64.06400299072266,66.06600189208984,68.06800079345703,70.06999969482422,72.0719985961914,74.0739974975586,76.07599639892578,78.0780029296875,80.08000183105469,82.08200073242188,84.08399963378906,86.08599853515625,88.08799743652344,90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172,150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906,190.19000244140625,192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562,248.2480010986328,250.25,252.2519989013672,254.25399780273438,256.2560119628906,258.25799560546875,260.260009765625,262.2619934082031,264.2640075683594,266.2659912109375,268.26800537109375,270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156],"skip_reason":"configured_native_cap","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920,7980,8040,8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":30380,"wall_s":0.023538773879408836},{"answer_s":0.12725329212844372,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"Yellow and Red.","gold_choice":"C","ingest_s":1.1939555406570435e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9055233001708984,"pred_answer":"B) Red and blue.","pred_choice":"B","qa_index":0,"query_time_s":12.0,"question":"What colors are the wheels of the race car?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":0,"wall_s":0.1669453363865614},{"answer_s":0.12690467573702335,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["A sad face.","A happy face.","A star.","A heart."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A happy face.","gold_choice":"B","ingest_s":1.1011958122253418e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.906397819519043,"pred_answer":"A) A sad face.","pred_choice":"A","qa_index":1,"query_time_s":72.0,"question":"What feature is present on the front of the toy?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":0,"wall_s":0.16484346613287926},{"answer_s":0.144328648224473,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"In a straight line.","gold_choice":"B","ingest_s":1.0745599865913391e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9150395393371582,"pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":2,"query_time_s":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Spatial Understanding","video_id":"459","visual_tokens":0,"wall_s":0.18262770771980286},{"answer_s":0.3030245751142502,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","ingest_s":1.1302530765533447e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9280033111572266,"pred_answer":"B) A small structure made of yellow and red blocks with wheels.Explanation:","pred_choice":"B","qa_index":3,"query_time_s":202.0,"question":"What is the new item introduced in the video?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Object Recognition","video_id":"459","visual_tokens":0,"wall_s":0.34194877557456493},{"answer_s":0.22233289293944836,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","ingest_s":1.1004507541656494e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.922818660736084,"pred_answer":"D) The toy vehicle is lifted off the surface.","pred_choice":"D","qa_index":4,"query_time_s":227.0,"question":"What happened to the toy vehicle just now?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"459","visual_tokens":0,"wall_s":0.2612690404057503},{"answer_s":0.09996414370834827,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Eight.","Sixteen.","Four.","Twelve."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Eight.","gold_choice":"A","ingest_s":1.1783093214035034e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9055233001708984,"pred_answer":"D) Twelve.","pred_choice":"D","qa_index":0,"query_time_s":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Counting","video_id":"355","visual_tokens":0,"wall_s":0.13914062082767487},{"answer_s":0.0985669307410717,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Three.","Five.","Seven.","Four."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Four.","gold_choice":"D","ingest_s":1.1712312698364258e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9046697616577148,"pred_answer":"D) Four.","pred_choice":"D","qa_index":1,"query_time_s":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Counting","video_id":"355","visual_tokens":0,"wall_s":0.13768601045012474},{"answer_s":0.09720836766064167,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Yellow.","Orange.","Green.","Blue."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"C","ingest_s":1.1280179023742676e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9055233001708984,"pred_answer":"A) Yellow.","pred_choice":"A","qa_index":2,"query_time_s":76.0,"question":"What is the color of the block that the hand is reaching for right now?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"355","visual_tokens":0,"wall_s":0.14072953537106514},{"answer_s":0.05811888165771961,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Green.","Yellow.","Orange.","Red."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"A","ingest_s":9.82917845249176e-06,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9081263542175293,"pred_answer":"B).","pred_choice":"B","qa_index":3,"query_time_s":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":0,"wall_s":0.09606309421360493},{"answer_s":0.16554323211312294,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green, yellow, orange.","gold_choice":"A","ingest_s":1.0399147868156433e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.918508529663086,"pred_answer":"D) Green, orange, yellow.","pred_choice":"D","qa_index":4,"query_time_s":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":0,"wall_s":0.20426158234477043},{"answer_s":0.09101265668869019,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Red.","Blue.","Yellow.","Purple."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Yellow.","gold_choice":"C","ingest_s":1.0631978511810303e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.903794765472412,"pred_answer":"A) Red.","pred_choice":"A","qa_index":0,"query_time_s":16.0,"question":"What color is the plastic block added on top of the green blocks?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":0,"wall_s":0.12922984920442104},{"answer_s":0.056074462831020355,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Red, green, and yellow.","gold_choice":"B","ingest_s":1.050904393196106e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9210901260375977,"pred_answer":"C).","pred_choice":"C","qa_index":1,"query_time_s":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":0,"wall_s":0.09464294835925102},{"answer_s":0.1272735372185707,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["A white block.","A green block.","An yellow block.","A blue block."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"An yellow block.","gold_choice":"C","ingest_s":1.0488554835319519e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9133110046386719,"pred_answer":"A) A white block.","pred_choice":"A","qa_index":2,"query_time_s":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":0,"wall_s":0.16618774645030499},{"answer_s":0.3045847490429878,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","ingest_s":1.0456889867782593e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.924546718597412,"pred_answer":"A) \\"Alphabet A transform to Box\\".Explanation: The image shows a","pred_choice":"A","qa_index":3,"query_time_s":141.0,"question":"What was the white text on the purple animated overlay read just now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Text-Rich Understanding","video_id":"335","visual_tokens":0,"wall_s":0.34438324347138405},{"answer_s":0.26173833198845387,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","ingest_s":1.1224299669265747e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9254217147827148,"pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":4,"query_time_s":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":0,"wall_s":0.3025311604142189},{"answer_s":0.1621512956917286,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Yellow, green.","gold_choice":"A","ingest_s":1.0458752512931824e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9098544120788574,"pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":0,"query_time_s":7.0,"question":"What are the colors of the crane arm?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Attribute Recognition","video_id":"448","visual_tokens":0,"wall_s":0.20046929083764553},{"answer_s":0.16169427707791328,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It lifts and swings gently.","gold_choice":"C","ingest_s":1.0877847671508789e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9115829467773438,"pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":1,"query_time_s":19.0,"question":"What happens to the orange block as the video progresses?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":0.20011178776621819},{"answer_s":0.3173997960984707,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","ingest_s":1.0825693607330322e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9461612701416016,"pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":2,"query_time_s":96.0,"question":"What was the sequence of actions performed with the blocks just now?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":0.35689757764339447},{"answer_s":0.22608243115246296,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","ingest_s":1.0928139090538025e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9254217147827148,"pred_answer":"D) It is placed back beside the larger structure.","pred_choice":"D","qa_index":3,"query_time_s":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":0.26574813574552536},{"answer_s":0.09097013808786869,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Ten.","Three.","Nine.","Eight."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Nine.","gold_choice":"C","ingest_s":1.0740011930465698e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.8986101150512695,"pred_answer":"C) Nine.","pred_choice":"C","qa_index":4,"query_time_s":265.0,"question":"How many green blocks are stacked now?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Counting","video_id":"448","visual_tokens":0,"wall_s":0.12899166718125343},{"answer_s":0.18896359018981457,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The hand adds another green block.","gold_choice":"D","ingest_s":1.1565163731575012e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9219651222229004,"pred_answer":"A) The hand removes the green block.","pred_choice":"A","qa_index":0,"query_time_s":25.0,"question":"What happens after the first green block is placed on the yellow block?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":0,"wall_s":0.22787775099277496},{"answer_s":0.31351756490767,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"correct":1,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Pressing down the green blocks.","gold_choice":"D","ingest_s":1.2407079339027405e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9167675971984863,"pred_answer":"D) Pressing down the green blocks.The action you\'re referring to is","pred_choice":"D","qa_index":1,"query_time_s":118.0,"question":"What action do the hands perform just before moving away from the structure?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":0,"wall_s":0.35377438738942146},{"answer_s":0.32146716490387917,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","ingest_s":1.2435019016265869e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9418296813964844,"pred_answer":"B) The yellow block is placed first, followed by the green block.The","pred_choice":"B","qa_index":2,"query_time_s":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":0,"wall_s":0.36399987526237965},{"answer_s":0.16304456628859043,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"In front of the original structure.","gold_choice":"D","ingest_s":1.1924654245376587e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9193620681762695,"pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":3,"query_time_s":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Spatial Understanding","video_id":"118","visual_tokens":0,"wall_s":0.20092887617647648},{"answer_s":0.27748999185860157,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","ingest_s":1.1267140507698059e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.929731845855713,"pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":4,"query_time_s":315.0,"question":"What did the person do with the yellow block just now?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":0,"wall_s":0.31590158864855766},{"answer_s":0.16501556523144245,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add mixed numbers.","gold_choice":"A","ingest_s":1.272745430469513e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.910707950592041,"pred_answer":"B) The subtraction of mixed numbers.","pred_choice":"B","qa_index":0,"query_time_s":20.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":0.20412010699510574},{"answer_s":0.23633763939142227,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":1.2854114174842834e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9158930778503418,"pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":1,"query_time_s":130.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":0.2755314391106367},{"answer_s":0.125291146337986,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":1.1926516890525818e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9081263542175293,"pred_answer":"D) Divide the fractions.","pred_choice":"D","qa_index":2,"query_time_s":248.0,"question":"What might the speaker do next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":0.16356948018074036},{"answer_s":0.18622241728007793,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two fractions numbers.","gold_choice":"D","ingest_s":1.4331191778182983e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9150395393371582,"pred_answer":"C) The process of subtracting fractions.","pred_choice":"C","qa_index":3,"query_time_s":396.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":0.22638450004160404},{"answer_s":0.16727451048791409,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"Adding the fractions together.","gold_choice":"A","ingest_s":1.4627352356910706e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9124364852905273,"pred_answer":"C) Finding a different common denominator.","pred_choice":"C","qa_index":4,"query_time_s":478.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":0.20710114389657974},{"answer_s":0.16444887220859528,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","ingest_s":1.2874603271484375e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.906397819519043,"pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":0,"query_time_s":11.0,"question":"What topic might the teacher explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":0.20184610784053802},{"answer_s":0.2936119195073843,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","ingest_s":1.3452023267745972e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9210901260375977,"pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":1,"query_time_s":190.0,"question":"What might the teacher explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":0.3320704959332943},{"answer_s":0.15122120641171932,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","ingest_s":1.2814998626708984e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9176335334777832,"pred_answer":"C) Simplifying improper fractions.","pred_choice":"C","qa_index":2,"query_time_s":331.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":0.1901126690208912},{"answer_s":0.14857028052210808,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","ingest_s":1.270882785320282e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9158930778503418,"pred_answer":"D) Simplifying mixed fractions.","pred_choice":"D","qa_index":3,"query_time_s":500.0,"question":"What concept will the speaker likely explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":0.18868791311979294},{"answer_s":0.3123299665749073,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","ingest_s":1.3668090105056763e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9210901260375977,"pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":4,"query_time_s":646.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":0.3507693037390709},{"answer_s":0.3305916413664818,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"correct":0,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Multiplying Fractions.","gold_choice":"D","ingest_s":1.1790543794631958e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9098544120788574,"pred_answer":"C) Simplifying Fractions.The speaker might focus on simplifying fractions next","pred_choice":"C","qa_index":0,"query_time_s":10.0,"question":"What mathematical topic might the speaker focus on next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":0.37459009885787964},{"answer_s":0.2057629693299532,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":1.171603798866272e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9176335334777832,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":1,"query_time_s":55.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":0.24470274709165096},{"answer_s":0.2169785089790821,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":1.272745430469513e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.908979892730713,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":2,"query_time_s":149.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":0.2572668120265007},{"answer_s":0.1432184111326933,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"correct":0,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Another explanation to fractions.","gold_choice":"D","ingest_s":1.1939555406570435e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.910707950592041,"pred_answer":"C) Simplifying fractions.","pred_choice":"C","qa_index":3,"query_time_s":231.0,"question":"What mathematical concept might the speaker move on to explain next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":0.18401109986007214},{"answer_s":0.14652915857732296,"cache_lengths_before_answer":[13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13],"choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"correct":0,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"How to multiply the denominators.","gold_choice":"C","ingest_s":1.1917203664779663e-05,"method":"question_only","no_compression":true,"peak_allocated_gib":1.9098544120788574,"pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":4,"query_time_s":310.0,"question":"What might the speaker discuss next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"","source_frame_indices":[],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":0.1848748829215765},{"answer_s":0.13005930557847023,"cache_lengths_before_answer":[1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189],"choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"Yellow and Red.","gold_choice":"C","ingest_s":0.025070028379559517,"method":"recent_20","no_compression":true,"peak_allocated_gib":2.8779053688049316,"pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":0,"query_time_s":12.0,"question":"What colors are the wheels of the race car?","sample_id":202,"sampled_indices":[0,1,2,3,4,5],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":1176,"wall_s":0.21655205264687538},{"answer_s":0.13968031853437424,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["A sad face.","A happy face.","A star.","A heart."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A happy face.","gold_choice":"B","ingest_s":0.07264471612870693,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.260286808013916,"pred_answer":"B) A happy face.","pred_choice":"B","qa_index":1,"query_time_s":72.0,"question":"What feature is present on the front of the toy?","sample_id":202,"sampled_indices":[16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35],"sampled_start_seconds":[32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0],"skip_reason":"","source_frame_indices":[960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":3920,"wall_s":0.31442240439355373},{"answer_s":0.14990956895053387,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"In a straight line.","gold_choice":"B","ingest_s":0.06825409270823002,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.9305129051208496,"pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":2,"query_time_s":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":202,"sampled_indices":[44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63],"sampled_start_seconds":[88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0],"skip_reason":"","source_frame_indices":[2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780],"status":"answered","task":"Spatial Understanding","video_id":"459","visual_tokens":3920,"wall_s":0.3018726725131273},{"answer_s":0.2178208213299513,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","ingest_s":0.06575651280581951,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.402571678161621,"pred_answer":"C) A toy vehicle with blue and yellow wheels.","pred_choice":"C","qa_index":3,"query_time_s":202.0,"question":"What is the new item introduced in the video?","sample_id":202,"sampled_indices":[81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100],"sampled_start_seconds":[162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0],"skip_reason":"","source_frame_indices":[4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000],"status":"answered","task":"Object Recognition","video_id":"459","visual_tokens":3920,"wall_s":0.37771587260067463},{"answer_s":0.23818853311240673,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","ingest_s":0.08370101638138294,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.095399856567383,"pred_answer":"D) The toy vehicle is lifted off the surface.","pred_choice":"D","qa_index":4,"query_time_s":227.0,"question":"What happened to the toy vehicle just now?","sample_id":202,"sampled_indices":[94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113],"sampled_start_seconds":[188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0,212.0,214.0,216.0,218.0,220.0,222.0,224.0,226.0],"skip_reason":"","source_frame_indices":[5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780],"status":"answered","task":"Event Understanding","video_id":"459","visual_tokens":3920,"wall_s":0.4185637943446636},{"answer_s":0.09884278289973736,"cache_lengths_before_answer":[1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581],"choices":["Eight.","Sixteen.","Four.","Twelve."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Eight.","gold_choice":"A","ingest_s":0.028308607637882233,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"A) Eight.","pred_choice":"A","qa_index":0,"query_time_s":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420],"status":"answered","task":"Counting","video_id":"355","visual_tokens":1568,"wall_s":0.1970246620476246},{"answer_s":0.1024393830448389,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["Three.","Five.","Seven.","Four."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Four.","gold_choice":"D","ingest_s":0.049551473930478096,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"D) Four.","pred_choice":"D","qa_index":1,"query_time_s":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":203,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Counting","video_id":"355","visual_tokens":1960,"wall_s":0.22440307214856148},{"answer_s":0.100685715675354,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Yellow.","Orange.","Green.","Blue."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"C","ingest_s":0.0690364483743906,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.9305129051208496,"pred_answer":"A) Yellow.","pred_choice":"A","qa_index":2,"query_time_s":76.0,"question":"What is the color of the block that the hand is reaching for right now?","sample_id":203,"sampled_indices":[18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37],"sampled_start_seconds":[36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0],"skip_reason":"","source_frame_indices":[1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220],"status":"answered","task":"Attribute Recognition","video_id":"355","visual_tokens":3920,"wall_s":0.2559079919010401},{"answer_s":0.10460159927606583,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Green.","Yellow.","Orange.","Red."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"A","ingest_s":0.06621393002569675,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.095399856567383,"pred_answer":"B) Yellow.","pred_choice":"B","qa_index":3,"query_time_s":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":203,"sampled_indices":[47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66],"sampled_start_seconds":[94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0],"skip_reason":"","source_frame_indices":[2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":3920,"wall_s":0.26246814988553524},{"answer_s":0.1693048495799303,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"correct":0,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green, yellow, orange.","gold_choice":"A","ingest_s":0.08343605324625969,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.07731819152832,"pred_answer":"B) Yellow, green, orange.","pred_choice":"B","qa_index":4,"query_time_s":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":203,"sampled_indices":[51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70],"sampled_start_seconds":[102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0],"skip_reason":"","source_frame_indices":[3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":3920,"wall_s":0.3425279427319765},{"answer_s":0.09475401975214481,"cache_lengths_before_answer":[1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581,1581],"choices":["Red.","Blue.","Yellow.","Purple."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Yellow.","gold_choice":"C","ingest_s":0.02692808397114277,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.2167201042175293,"pred_answer":"A) Red.","pred_choice":"A","qa_index":0,"query_time_s":16.0,"question":"What color is the plastic block added on top of the green blocks?","sample_id":204,"sampled_indices":[0,1,2,3,4,5,6,7],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":1568,"wall_s":0.1921789702028036},{"answer_s":0.18521905690431595,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Red, green, and yellow.","gold_choice":"B","ingest_s":0.06364995241165161,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":1,"query_time_s":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":204,"sampled_indices":[11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30],"sampled_start_seconds":[22.0,24.0,26.0,28.0,30.0,32.0,34.0,36.0,38.0,40.0,42.0,44.0,46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0],"skip_reason":"","source_frame_indices":[660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620,1680,1740,1800],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":3920,"wall_s":0.3367957826703787},{"answer_s":0.13312358409166336,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["A white block.","A green block.","An yellow block.","A blue block."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"An yellow block.","gold_choice":"C","ingest_s":0.08660302869975567,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.9305129051208496,"pred_answer":"B) A green block.","pred_choice":"B","qa_index":2,"query_time_s":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","sample_id":204,"sampled_indices":[23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42],"sampled_start_seconds":[46.0,48.0,50.0,52.0,54.0,56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0],"skip_reason":"","source_frame_indices":[1380,1440,1500,1560,1620,1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":3920,"wall_s":0.3048722557723522},{"answer_s":0.20850951224565506,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","ingest_s":0.06571501307189465,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.9305129051208496,"pred_answer":"B) \\"Alphabet B transform to Box\\".","pred_choice":"B","qa_index":3,"query_time_s":141.0,"question":"What was the white text on the purple animated overlay read just now?","sample_id":204,"sampled_indices":[51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70],"sampled_start_seconds":[102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0,118.0,120.0,122.0,124.0,126.0,128.0,130.0,132.0,134.0,136.0,138.0,140.0],"skip_reason":"","source_frame_indices":[3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200],"status":"answered","task":"Text-Rich Understanding","video_id":"335","visual_tokens":3920,"wall_s":0.3586174864321947},{"answer_s":0.26476045697927475,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","ingest_s":0.06665783189237118,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.9305129051208496,"pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":4,"query_time_s":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","sample_id":204,"sampled_indices":[79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98],"sampled_start_seconds":[158.0,160.0,162.0,164.0,166.0,168.0,170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0],"skip_reason":"","source_frame_indices":[4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":3920,"wall_s":0.41833440586924553},{"answer_s":0.16866429522633553,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Yellow, green.","gold_choice":"A","ingest_s":0.02258857525885105,"method":"recent_20","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":0,"query_time_s":7.0,"question":"What are the colors of the crane arm?","sample_id":208,"sampled_indices":[0,1,2,3],"sampled_start_seconds":[0.0,2.0,4.0,6.0],"skip_reason":"","source_frame_indices":[0,60,120,180],"status":"answered","task":"Attribute Recognition","video_id":"448","visual_tokens":784,"wall_s":0.2445373386144638},{"answer_s":0.16733673959970474,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It lifts and swings gently.","gold_choice":"C","ingest_s":0.0468192920088768,"method":"recent_20","no_compression":true,"peak_allocated_gib":2.895987033843994,"pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":1,"query_time_s":19.0,"question":"What happens to the orange block as the video progresses?","sample_id":208,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":1960,"wall_s":0.2765303310006857},{"answer_s":0.3145172540098429,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","ingest_s":0.07750491052865982,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.233164310455322,"pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":2,"query_time_s":96.0,"question":"What was the sequence of actions performed with the blocks just now?","sample_id":208,"sampled_indices":[28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47],"sampled_start_seconds":[56.0,58.0,60.0,62.0,64.0,66.0,68.0,70.0,72.0,74.0,76.0,78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0],"skip_reason":"","source_frame_indices":[1680,1740,1800,1860,1920,1980,2040,2100,2160,2220,2280,2340,2400,2460,2520,2580,2640,2700,2760,2820],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":3920,"wall_s":0.4876213502138853},{"answer_s":0.18374746665358543,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","ingest_s":0.07054147124290466,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.233164310455322,"pred_answer":"A) It is disassembled.","pred_choice":"A","qa_index":3,"query_time_s":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","sample_id":208,"sampled_indices":[66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85],"sampled_start_seconds":[132.0,134.0,136.0,138.0,140.0,142.0,144.0,146.0,148.0,150.0,152.0,154.0,156.0,158.0,160.0,162.0,164.0,166.0,168.0,170.0],"skip_reason":"","source_frame_indices":[3960,4020,4080,4140,4200,4260,4320,4380,4440,4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":3920,"wall_s":0.35279758647084236},{"answer_s":0.10977571085095406,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Ten.","Three.","Nine.","Eight."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Nine.","gold_choice":"C","ingest_s":0.08390908688306808,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.425173759460449,"pred_answer":"D) Eight.","pred_choice":"D","qa_index":4,"query_time_s":265.0,"question":"How many green blocks are stacked now?","sample_id":208,"sampled_indices":[113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132],"sampled_start_seconds":[226.0,228.0,230.0,232.0,234.0,236.0,238.0,240.0,242.0,244.0,246.0,248.0,250.0,252.0,254.0,256.0,258.0,260.0,262.0,264.0],"skip_reason":"","source_frame_indices":[6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380,7440,7500,7560,7620,7680,7740,7800,7860,7920],"status":"answered","task":"Counting","video_id":"448","visual_tokens":3920,"wall_s":0.28979786671698093},{"answer_s":0.18955794908106327,"cache_lengths_before_answer":[2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561,2561],"choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"correct":1,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The hand adds another green block.","gold_choice":"D","ingest_s":0.03883855789899826,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"D) The hand adds another green block.","pred_choice":"D","qa_index":0,"query_time_s":25.0,"question":"What happens after the first green block is placed on the yellow block?","sample_id":209,"sampled_indices":[0,1,2,3,4,5,6,7,8,9,10,11,12],"sampled_start_seconds":[0.0,2.0,4.0,6.0,8.0,10.0,12.0,14.0,16.0,18.0,20.0,22.0,24.0],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540,600,660,720],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":2548,"wall_s":0.3188165854662657},{"answer_s":0.19323278963565826,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"correct":1,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Pressing down the green blocks.","gold_choice":"D","ingest_s":0.06821686029434204,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.260286808013916,"pred_answer":"D) Pressing down the green blocks.","pred_choice":"D","qa_index":1,"query_time_s":118.0,"question":"What action do the hands perform just before moving away from the structure?","sample_id":209,"sampled_indices":[39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58],"sampled_start_seconds":[78.0,80.0,82.0,84.0,86.0,88.0,90.0,92.0,94.0,96.0,98.0,100.0,102.0,104.0,106.0,108.0,110.0,112.0,114.0,116.0],"skip_reason":"","source_frame_indices":[2340,2400,2460,2520,2580,2640,2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":3920,"wall_s":0.3522478435188532},{"answer_s":0.3236278295516968,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","ingest_s":0.07346663065254688,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.260286808013916,"pred_answer":"D) The yellow block is adjusted, then the green block is placed.","pred_choice":"D","qa_index":2,"query_time_s":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":209,"sampled_indices":[85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104],"sampled_start_seconds":[170.0,172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0],"skip_reason":"","source_frame_indices":[5100,5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":3920,"wall_s":0.49278497509658337},{"answer_s":0.1764000542461872,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"In front of the original structure.","gold_choice":"D","ingest_s":0.09723353758454323,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.25576639175415,"pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":3,"query_time_s":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":209,"sampled_indices":[86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105],"sampled_start_seconds":[172.0,174.0,176.0,178.0,180.0,182.0,184.0,186.0,188.0,190.0,192.0,194.0,196.0,198.0,200.0,202.0,204.0,206.0,208.0,210.0],"skip_reason":"","source_frame_indices":[5160,5220,5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,5940,6000,6060,6120,6180,6240,6300],"status":"answered","task":"Spatial Understanding","video_id":"118","visual_tokens":3920,"wall_s":0.3707905299961567},{"answer_s":0.27393357641994953,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","ingest_s":0.06884306855499744,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.57197904586792,"pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":4,"query_time_s":315.0,"question":"What did the person do with the yellow block just now?","sample_id":209,"sampled_indices":[138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157],"sampled_start_seconds":[276.0,278.0,280.0,282.0,284.0,286.0,288.0,290.0,292.0,294.0,296.0,298.0,300.0,302.0,304.0,306.0,308.0,310.0,312.0,314.0],"skip_reason":"","source_frame_indices":[8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":3920,"wall_s":0.4413786642253399},{"answer_s":0.17013559862971306,"cache_lengths_before_answer":[1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973,1973],"choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add mixed numbers.","gold_choice":"A","ingest_s":0.029945380985736847,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.555534839630127,"pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":0,"query_time_s":20.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[0,1,2,3,4,5,6,7,8,9],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836,12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300,360,420,480,540],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":1960,"wall_s":0.2754353452473879},{"answer_s":0.24040810018777847,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.06773550622165203,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":1,"query_time_s":130.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64],"sampled_start_seconds":[90.08999633789062,92.09200286865234,94.09400177001953,96.09600067138672,98.0979995727539,100.0999984741211,102.10199737548828,104.10399627685547,106.10600280761719,108.10800170898438,110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453],"skip_reason":"","source_frame_indices":[2700,2760,2820,2880,2940,3000,3060,3120,3180,3240,3300,3360,3420,3480,3540,3600,3660,3720,3780,3840],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":3920,"wall_s":0.39638838171958923},{"answer_s":0.2057383768260479,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.062340445816516876,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.7656259536743164,"pred_answer":"A) How to add those two mixed numbers.","pred_choice":"A","qa_index":2,"query_time_s":248.0,"question":"What might the speaker do next?","sample_id":212,"sampled_indices":[104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123],"sampled_start_seconds":[208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906,232.23199462890625,234.23399353027344,236.23599243164062,238.23800659179688,240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562],"skip_reason":"","source_frame_indices":[6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900,6960,7020,7080,7140,7200,7260,7320,7380],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":3920,"wall_s":0.34894340857863426},{"answer_s":0.215764582157135,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two fractions numbers.","gold_choice":"D","ingest_s":0.061869675293564796,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.600739002227783,"pred_answer":"D) How to add those two fractions numbers.","pred_choice":"D","qa_index":3,"query_time_s":396.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197],"sampled_start_seconds":[356.3559875488281,358.3580017089844,360.3599853515625,362.36199951171875,364.364013671875,366.3659973144531,368.3680114746094,370.3699951171875,372.37200927734375,374.3739929199219,376.3760070800781,378.37799072265625,380.3800048828125,382.3819885253906,384.3840026855469,386.385986328125,388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719],"skip_reason":"","source_frame_indices":[10680,10740,10800,10860,10920,10980,11040,11100,11160,11220,11280,11340,11400,11460,11520,11580,11640,11700,11760,11820],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":3920,"wall_s":0.35721301287412643},{"answer_s":0.17332817241549492,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"Adding the fractions together.","gold_choice":"A","ingest_s":0.062485843896865845,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.724942207336426,"pred_answer":"C) Finding a different common denominator.","pred_choice":"C","qa_index":4,"query_time_s":478.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238],"sampled_start_seconds":[438.43798828125,440.44000244140625,442.4419860839844,444.4440002441406,446.4460144042969,448.447998046875,450.45001220703125,452.4519958496094,454.4540100097656,456.45599365234375,458.4580078125,460.4599914550781,462.4620056152344,464.4639892578125,466.46600341796875,468.4679870605469,470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375],"skip_reason":"","source_frame_indices":[13140,13200,13260,13320,13380,13440,13500,13560,13620,13680,13740,13800,13860,13920,13980,14040,14100,14160,14220,14280],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":3920,"wall_s":0.3186086304485798},{"answer_s":0.17471948079764843,"cache_lengths_before_answer":[1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189,1189],"choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","ingest_s":0.024767635390162468,"method":"recent_20","no_compression":true,"peak_allocated_gib":2.8779053688049316,"pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":0,"query_time_s":11.0,"question":"What topic might the teacher explain next?","sample_id":218,"sampled_indices":[0,1,2,3,4,5],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836],"skip_reason":"","source_frame_indices":[0,60,120,180,240,300],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":1176,"wall_s":0.2585426867008209},{"answer_s":0.30428336560726166,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","ingest_s":0.062563331797719,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.724942207336426,"pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":1,"query_time_s":190.0,"question":"What might the teacher explain next?","sample_id":218,"sampled_indices":[75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94],"sampled_start_seconds":[150.14999389648438,152.15199279785156,154.1540069580078,156.156005859375,158.1580047607422,160.16000366210938,162.16200256347656,164.16400146484375,166.16600036621094,168.16799926757812,170.1699981689453,172.1719970703125,174.1739959716797,176.17599487304688,178.17799377441406,180.17999267578125,182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906],"skip_reason":"","source_frame_indices":[4500,4560,4620,4680,4740,4800,4860,4920,4980,5040,5100,5160,5220,5280,5340,5400,5460,5520,5580,5640],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":3920,"wall_s":0.45685155130922794},{"answer_s":0.24601147696375847,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","ingest_s":0.06238394416868687,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"B) The method for adding mixed numbers and improper fractions.","pred_choice":"B","qa_index":2,"query_time_s":331.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165],"sampled_start_seconds":[292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156,310.30999755859375,312.31201171875,314.3139953613281,316.3160095214844,318.3179931640625,320.32000732421875,322.3219909667969,324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656],"skip_reason":"","source_frame_indices":[8760,8820,8880,8940,9000,9060,9120,9180,9240,9300,9360,9420,9480,9540,9600,9660,9720,9780,9840,9900],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":3920,"wall_s":0.3966048266738653},{"answer_s":0.15173640474677086,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","ingest_s":0.07186080329120159,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.57197904586792,"pred_answer":"D) Simplifying mixed fractions.","pred_choice":"D","qa_index":3,"query_time_s":500.0,"question":"What concept will the speaker likely explain next?","sample_id":218,"sampled_indices":[230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249],"sampled_start_seconds":[460.4599914550781,462.4620056152344,464.4639892578125,466.46600341796875,468.4679870605469,470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375,478.4779968261719,480.4800109863281,482.48199462890625,484.4840087890625,486.4859924316406,488.4880065917969,490.489990234375,492.49200439453125,494.4939880371094,496.4960021972656,498.49798583984375],"skip_reason":"","source_frame_indices":[13800,13860,13920,13980,14040,14100,14160,14220,14280,14340,14400,14460,14520,14580,14640,14700,14760,14820,14880,14940],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":3920,"wall_s":0.3250497840344906},{"answer_s":0.31992613337934017,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","ingest_s":0.06233755499124527,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.724942207336426,"pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":4,"query_time_s":646.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322],"sampled_start_seconds":[606.6060180664062,608.6079711914062,610.6099853515625,612.6119995117188,614.614013671875,616.6160278320312,618.6179809570312,620.6199951171875,622.6220092773438,624.6240234375,626.6259765625,628.6279907226562,630.6300048828125,632.6320190429688,634.6339721679688,636.635986328125,638.6380004882812,640.6400146484375,642.6420288085938,644.6439819335938],"skip_reason":"","source_frame_indices":[18180,18240,18300,18360,18420,18480,18540,18600,18660,18720,18780,18840,18900,18960,19020,19080,19140,19200,19260,19320],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":3920,"wall_s":0.4651927463710308},{"answer_s":0.15570159442722797,"cache_lengths_before_answer":[993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993,993],"choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Multiplying Fractions.","gold_choice":"D","ingest_s":0.024841075763106346,"method":"recent_20","no_compression":true,"peak_allocated_gib":2.708498001098633,"pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":0,"query_time_s":10.0,"question":"What mathematical topic might the speaker focus on next?","sample_id":219,"sampled_indices":[0,1,2,3,4],"sampled_start_seconds":[0.0,2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332],"skip_reason":"","source_frame_indices":[0,60,120,180,240],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":980,"wall_s":0.2503045666962862},{"answer_s":0.20556827448308468,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.0651400089263916,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":1,"query_time_s":55.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27],"sampled_start_seconds":[16.016000747680664,18.01799964904785,20.020000457763672,22.02199935913086,24.02400016784668,26.025999069213867,28.027999877929688,30.030000686645508,32.03200149536133,34.034000396728516,36.0359992980957,38.03799819946289,40.040000915527344,42.04199981689453,44.04399871826172,46.04600143432617,48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219],"skip_reason":"","source_frame_indices":[480,540,600,660,720,780,840,900,960,1020,1080,1140,1200,1260,1320,1380,1440,1500,1560,1620],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":3920,"wall_s":0.3583711329847574},{"answer_s":0.22226965241134167,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.07175165973603725,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.425173759460449,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":2,"query_time_s":149.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"sampled_start_seconds":[110.11000061035156,112.11199951171875,114.11399841308594,116.11599731445312,118.11799621582031,120.12000274658203,122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453,130.1300048828125,132.1320037841797,134.13400268554688,136.13600158691406,138.13800048828125,140.13999938964844,142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172],"skip_reason":"","source_frame_indices":[3300,3360,3420,3480,3540,3600,3660,3720,3780,3840,3900,3960,4020,4080,4140,4200,4260,4320,4380,4440],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":3920,"wall_s":0.39533030800521374},{"answer_s":0.1518267821520567,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Another explanation to fractions.","gold_choice":"D","ingest_s":0.06203294172883034,"method":"recent_20","no_compression":true,"peak_allocated_gib":3.724942207336426,"pred_answer":"D) Another explanation to fractions.","pred_choice":"D","qa_index":3,"query_time_s":231.0,"question":"What mathematical concept might the speaker move on to explain next?","sample_id":219,"sampled_indices":[96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115],"sampled_start_seconds":[192.19200134277344,194.19400024414062,196.1959991455078,198.197998046875,200.1999969482422,202.20199584960938,204.20399475097656,206.20599365234375,208.20799255371094,210.2100067138672,212.21200561523438,214.21400451660156,216.21600341796875,218.21800231933594,220.22000122070312,222.2220001220703,224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906],"skip_reason":"","source_frame_indices":[5760,5820,5880,5940,6000,6060,6120,6180,6240,6300,6360,6420,6480,6540,6600,6660,6720,6780,6840,6900],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":3920,"wall_s":0.2938617244362831},{"answer_s":0.1566462703049183,"cache_lengths_before_answer":[3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933,3933],"choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"correct":0,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"How to multiply the denominators.","gold_choice":"C","ingest_s":0.06502029672265053,"method":"recent_20","no_compression":true,"peak_allocated_gib":4.063756942749023,"pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":4,"query_time_s":310.0,"question":"What might the speaker discuss next?","sample_id":219,"sampled_indices":[135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154],"sampled_start_seconds":[270.2699890136719,272.2720031738281,274.27398681640625,276.2760009765625,278.27801513671875,280.2799987792969,282.2820129394531,284.28399658203125,286.2860107421875,288.2879943847656,290.2900085449219,292.2919921875,294.29400634765625,296.2959899902344,298.2980041503906,300.29998779296875,302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156],"skip_reason":"","source_frame_indices":[8100,8160,8220,8280,8340,8400,8460,8520,8580,8640,8700,8760,8820,8880,8940,9000,9060,9120,9180,9240],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":3920,"wall_s":0.3089817725121975},{"answer_s":0.1291318889707327,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"Yellow and Red.","gold_choice":"C","ingest_s":0.023038089275360107,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":0,"query_time_s":12.0,"question":"What colors are the wheels of the race car?","sample_id":202,"sampled_indices":[2,3,4,5],"sampled_start_seconds":[4.0,6.0,8.0,10.0],"skip_reason":"","source_frame_indices":[120,180,240,300],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":784,"wall_s":0.2056125160306692},{"answer_s":0.13540862314403057,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["A sad face.","A happy face.","A star.","A heart."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A happy face.","gold_choice":"B","ingest_s":0.026614151895046234,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) A happy face.","pred_choice":"B","qa_index":1,"query_time_s":72.0,"question":"What feature is present on the front of the toy?","sample_id":202,"sampled_indices":[32,33,34,35],"sampled_start_seconds":[64.0,66.0,68.0,70.0],"skip_reason":"","source_frame_indices":[1920,1980,2040,2100],"status":"answered","task":"Attribute Recognition","video_id":"459","visual_tokens":784,"wall_s":0.21582546271383762},{"answer_s":0.14605681784451008,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"In a straight line.","gold_choice":"B","ingest_s":0.02329343743622303,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":2,"query_time_s":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":202,"sampled_indices":[60,61,62,63],"sampled_start_seconds":[120.0,122.0,124.0,126.0],"skip_reason":"","source_frame_indices":[3600,3660,3720,3780],"status":"answered","task":"Spatial Understanding","video_id":"459","visual_tokens":784,"wall_s":0.22203806601464748},{"answer_s":0.2895253002643585,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"correct":1,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","ingest_s":0.022490547969937325,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) A small structure made of yellow and red blocks with wheels.","pred_choice":"B","qa_index":3,"query_time_s":202.0,"question":"What is the new item introduced in the video?","sample_id":202,"sampled_indices":[97,98,99,100],"sampled_start_seconds":[194.0,196.0,198.0,200.0],"skip_reason":"","source_frame_indices":[5820,5880,5940,6000],"status":"answered","task":"Object Recognition","video_id":"459","visual_tokens":784,"wall_s":0.36417773738503456},{"answer_s":0.20918339490890503,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"correct":0,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","ingest_s":0.026291266083717346,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) The toy vehicle is disassembled.","pred_choice":"A","qa_index":4,"query_time_s":227.0,"question":"What happened to the toy vehicle just now?","sample_id":202,"sampled_indices":[110,111,112,113],"sampled_start_seconds":[220.0,222.0,224.0,226.0],"skip_reason":"","source_frame_indices":[6600,6660,6720,6780],"status":"answered","task":"Event Understanding","video_id":"459","visual_tokens":784,"wall_s":0.2889930158853531},{"answer_s":0.09898963198065758,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Eight.","Sixteen.","Four.","Twelve."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Eight.","gold_choice":"A","ingest_s":0.02384336106479168,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Eight.","pred_choice":"A","qa_index":0,"query_time_s":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":203,"sampled_indices":[4,5,6,7],"sampled_start_seconds":[8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[240,300,360,420],"status":"answered","task":"Counting","video_id":"355","visual_tokens":784,"wall_s":0.18269622512161732},{"answer_s":0.09491012617945671,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Three.","Five.","Seven.","Four."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Four.","gold_choice":"D","ingest_s":0.04821963422000408,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.2093167304992676,"pred_answer":"D) Four.","pred_choice":"D","qa_index":1,"query_time_s":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":203,"sampled_indices":[6,7,8,9],"sampled_start_seconds":[12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[360,420,480,540],"status":"answered","task":"Counting","video_id":"355","visual_tokens":784,"wall_s":0.18990103900432587},{"answer_s":0.0927924644201994,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Yellow.","Orange.","Green.","Blue."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"C","ingest_s":0.02331377938389778,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) Green.","pred_choice":"C","qa_index":2,"query_time_s":76.0,"question":"What is the color of the block that the hand is reaching for right now?","sample_id":203,"sampled_indices":[34,35,36,37],"sampled_start_seconds":[68.0,70.0,72.0,74.0],"skip_reason":"","source_frame_indices":[2040,2100,2160,2220],"status":"answered","task":"Attribute Recognition","video_id":"355","visual_tokens":784,"wall_s":0.16818242706358433},{"answer_s":0.09543978422880173,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Green.","Yellow.","Orange.","Red."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"A","ingest_s":0.023478444665670395,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Green.","pred_choice":"A","qa_index":3,"query_time_s":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":203,"sampled_indices":[63,64,65,66],"sampled_start_seconds":[126.0,128.0,130.0,132.0],"skip_reason":"","source_frame_indices":[3780,3840,3900,3960],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":784,"wall_s":0.1709833275526762},{"answer_s":0.1676542181521654,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"correct":1,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green, yellow, orange.","gold_choice":"A","ingest_s":0.022907674312591553,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Green, yellow, orange.","pred_choice":"A","qa_index":4,"query_time_s":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":203,"sampled_indices":[67,68,69,70],"sampled_start_seconds":[134.0,136.0,138.0,140.0],"skip_reason":"","source_frame_indices":[4020,4080,4140,4200],"status":"answered","task":"Event Understanding","video_id":"355","visual_tokens":784,"wall_s":0.2432203870266676},{"answer_s":0.09335401654243469,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Red.","Blue.","Yellow.","Purple."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Yellow.","gold_choice":"C","ingest_s":0.023198604583740234,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) Yellow.","pred_choice":"C","qa_index":0,"query_time_s":16.0,"question":"What color is the plastic block added on top of the green blocks?","sample_id":204,"sampled_indices":[4,5,6,7],"sampled_start_seconds":[8.0,10.0,12.0,14.0],"skip_reason":"","source_frame_indices":[240,300,360,420],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":784,"wall_s":0.17024646140635014},{"answer_s":0.18330045975744724,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Red, green, and yellow.","gold_choice":"B","ingest_s":0.023161059245467186,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":1,"query_time_s":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":204,"sampled_indices":[27,28,29,30],"sampled_start_seconds":[54.0,56.0,58.0,60.0],"skip_reason":"","source_frame_indices":[1620,1680,1740,1800],"status":"answered","task":"Attribute Recognition","video_id":"335","visual_tokens":784,"wall_s":0.25942910090088844},{"answer_s":0.1295844055712223,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["A white block.","A green block.","An yellow block.","A blue block."],"correct":0,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"An yellow block.","gold_choice":"C","ingest_s":0.025299936532974243,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) A green block.","pred_choice":"B","qa_index":2,"query_time_s":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","sample_id":204,"sampled_indices":[39,40,41,42],"sampled_start_seconds":[78.0,80.0,82.0,84.0],"skip_reason":"","source_frame_indices":[2340,2400,2460,2520],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":784,"wall_s":0.20765829645097256},{"answer_s":0.20107362605631351,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","ingest_s":0.023429464548826218,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) \\"Alphabet B transform to Box\\".","pred_choice":"B","qa_index":3,"query_time_s":141.0,"question":"What was the white text on the purple animated overlay read just now?","sample_id":204,"sampled_indices":[67,68,69,70],"sampled_start_seconds":[134.0,136.0,138.0,140.0],"skip_reason":"","source_frame_indices":[4020,4080,4140,4200],"status":"answered","task":"Text-Rich Understanding","video_id":"335","visual_tokens":784,"wall_s":0.27756204828619957},{"answer_s":0.26213269867002964,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"correct":1,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","ingest_s":0.024221530184149742,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":4,"query_time_s":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","sample_id":204,"sampled_indices":[95,96,97,98],"sampled_start_seconds":[190.0,192.0,194.0,196.0],"skip_reason":"","source_frame_indices":[5700,5760,5820,5880],"status":"answered","task":"Event Understanding","video_id":"335","visual_tokens":784,"wall_s":0.34059423580765724},{"answer_s":0.1644231602549553,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Yellow, green.","gold_choice":"A","ingest_s":0.02301565743982792,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":0,"query_time_s":7.0,"question":"What are the colors of the crane arm?","sample_id":208,"sampled_indices":[0,1,2,3],"sampled_start_seconds":[0.0,2.0,4.0,6.0],"skip_reason":"","source_frame_indices":[0,60,120,180],"status":"answered","task":"Attribute Recognition","video_id":"448","visual_tokens":784,"wall_s":0.24090391024947166},{"answer_s":0.1645109560340643,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It lifts and swings gently.","gold_choice":"C","ingest_s":0.022843237966299057,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":1,"query_time_s":19.0,"question":"What happens to the orange block as the video progresses?","sample_id":208,"sampled_indices":[6,7,8,9],"sampled_start_seconds":[12.0,14.0,16.0,18.0],"skip_reason":"","source_frame_indices":[360,420,480,540],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":784,"wall_s":0.2440314032137394},{"answer_s":0.3086579814553261,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"correct":1,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","ingest_s":0.022835057228803635,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":2,"query_time_s":96.0,"question":"What was the sequence of actions performed with the blocks just now?","sample_id":208,"sampled_indices":[44,45,46,47],"sampled_start_seconds":[88.0,90.0,92.0,94.0],"skip_reason":"","source_frame_indices":[2640,2700,2760,2820],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":784,"wall_s":0.3895093146711588},{"answer_s":0.1713037919253111,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","ingest_s":0.02295282483100891,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) It is disassembled.","pred_choice":"A","qa_index":3,"query_time_s":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","sample_id":208,"sampled_indices":[82,83,84,85],"sampled_start_seconds":[164.0,166.0,168.0,170.0],"skip_reason":"","source_frame_indices":[4920,4980,5040,5100],"status":"answered","task":"Event Understanding","video_id":"448","visual_tokens":784,"wall_s":0.2488532643765211},{"answer_s":0.09590780735015869,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Ten.","Three.","Nine.","Eight."],"correct":0,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Nine.","gold_choice":"C","ingest_s":0.02490084432065487,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Ten.","pred_choice":"A","qa_index":4,"query_time_s":265.0,"question":"How many green blocks are stacked now?","sample_id":208,"sampled_indices":[129,130,131,132],"sampled_start_seconds":[258.0,260.0,262.0,264.0],"skip_reason":"","source_frame_indices":[7740,7800,7860,7920],"status":"answered","task":"Counting","video_id":"448","visual_tokens":784,"wall_s":0.17848345451056957},{"answer_s":0.18860137090086937,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The hand adds another green block.","gold_choice":"D","ingest_s":0.024478623643517494,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) The hand removes the green block.","pred_choice":"A","qa_index":0,"query_time_s":25.0,"question":"What happens after the first green block is placed on the yellow block?","sample_id":209,"sampled_indices":[9,10,11,12],"sampled_start_seconds":[18.0,20.0,22.0,24.0],"skip_reason":"","source_frame_indices":[540,600,660,720],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":784,"wall_s":0.2749940734356642},{"answer_s":0.2200890351086855,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"correct":1,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Pressing down the green blocks.","gold_choice":"D","ingest_s":0.02509269118309021,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Pressing down the green blocks.","pred_choice":"D","qa_index":1,"query_time_s":118.0,"question":"What action do the hands perform just before moving away from the structure?","sample_id":209,"sampled_indices":[55,56,57,58],"sampled_start_seconds":[110.0,112.0,114.0,116.0],"skip_reason":"","source_frame_indices":[3300,3360,3420,3480],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":784,"wall_s":0.30309905856847763},{"answer_s":0.32300661131739616,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","ingest_s":0.026303593069314957,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) The yellow block is placed first, followed by the green block.","pred_choice":"B","qa_index":2,"query_time_s":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":209,"sampled_indices":[101,102,103,104],"sampled_start_seconds":[202.0,204.0,206.0,208.0],"skip_reason":"","source_frame_indices":[6060,6120,6180,6240],"status":"answered","task":"Event Understanding","video_id":"118","visual_tokens":784,"wall_s":0.40710020065307617},{"answer_s":0.16296080872416496,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"In front of the original structure.","gold_choice":"D","ingest_s":0.0492394994944334,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.369683265686035,"pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":3,"query_time_s":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":209,"sampled_indices":[102,103,104,105],"sampled_start_seconds":[204.0,206.0,208.0,210.0],"skip_reason":"","source_frame_indices":[6120,6180,6240,6300],"status":"answered","task":"Spatial Understanding","video_id":"118","visual_tokens":784,"wall_s":0.2608053684234619},{"answer_s":0.2854783535003662,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"correct":0,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","ingest_s":0.022440986707806587,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":4,"query_time_s":315.0,"question":"What did the person do with the yellow block just now?","sample_id":209,"sampled_indices":[154,155,156,157],"sampled_start_seconds":[308.0,310.0,312.0,314.0],"skip_reason":"","source_frame_indices":[9240,9300,9360,9420],"status":"answered","task":"Action Recognition","video_id":"118","visual_tokens":784,"wall_s":0.35998618975281715},{"answer_s":0.1942371241748333,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add mixed numbers.","gold_choice":"A","ingest_s":0.028484681621193886,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":0,"query_time_s":20.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[6,7,8,9],"sampled_start_seconds":[12.01200008392334,14.013999938964844,16.016000747680664,18.01799964904785],"skip_reason":"","source_frame_indices":[360,420,480,540],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":784,"wall_s":0.27793169766664505},{"answer_s":0.2392342910170555,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.023467471823096275,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":1,"query_time_s":130.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[61,62,63,64],"sampled_start_seconds":[122.12200164794922,124.1240005493164,126.1259994506836,128.1280059814453],"skip_reason":"","source_frame_indices":[3660,3720,3780,3840],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":784,"wall_s":0.31629366613924503},{"answer_s":0.20148450322449207,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.023393649607896805,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) How to add those two mixed numbers.","pred_choice":"A","qa_index":2,"query_time_s":248.0,"question":"What might the speaker do next?","sample_id":212,"sampled_indices":[120,121,122,123],"sampled_start_seconds":[240.24000549316406,242.24200439453125,244.24400329589844,246.24600219726562],"skip_reason":"","source_frame_indices":[7200,7260,7320,7380],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":784,"wall_s":0.27777252346277237},{"answer_s":0.19385393150150776,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"correct":0,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two fractions numbers.","gold_choice":"D","ingest_s":0.023210309445858,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) The process of subtracting fractions.","pred_choice":"C","qa_index":3,"query_time_s":396.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[194,195,196,197],"sampled_start_seconds":[388.38800048828125,390.3900146484375,392.3919982910156,394.3940124511719],"skip_reason":"","source_frame_indices":[11640,11700,11760,11820],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":784,"wall_s":0.2752739395946264},{"answer_s":0.15264977142214775,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"correct":1,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"Adding the fractions together.","gold_choice":"A","ingest_s":0.024002624675631523,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) Adding the fractions together.","pred_choice":"A","qa_index":4,"query_time_s":478.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[235,236,237,238],"sampled_start_seconds":[470.4700012207031,472.47198486328125,474.4739990234375,476.47601318359375],"skip_reason":"","source_frame_indices":[14100,14160,14220,14280],"status":"answered","task":"Prospective Reasoning","video_id":"125","visual_tokens":784,"wall_s":0.23093157075345516},{"answer_s":0.16775531135499477,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","ingest_s":0.023355916142463684,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":0,"query_time_s":11.0,"question":"What topic might the teacher explain next?","sample_id":218,"sampled_indices":[2,3,4,5],"sampled_start_seconds":[4.004000186920166,6.00600004196167,8.008000373840332,10.010000228881836],"skip_reason":"","source_frame_indices":[120,180,240,300],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":784,"wall_s":0.24802863225340843},{"answer_s":0.29252662509679794,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","ingest_s":0.026664214208722115,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":1,"query_time_s":190.0,"question":"What might the teacher explain next?","sample_id":218,"sampled_indices":[91,92,93,94],"sampled_start_seconds":[182.1820068359375,184.1840057373047,186.18600463867188,188.18800354003906],"skip_reason":"","source_frame_indices":[5460,5520,5580,5640],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":784,"wall_s":0.39665644615888596},{"answer_s":0.24662981182336807,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","ingest_s":0.0230318121612072,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) The method for adding mixed numbers and improper fractions.","pred_choice":"B","qa_index":2,"query_time_s":331.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[162,163,164,165],"sampled_start_seconds":[324.3240051269531,326.32598876953125,328.3280029296875,330.3299865722656],"skip_reason":"","source_frame_indices":[9720,9780,9840,9900],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":784,"wall_s":0.32209604792296886},{"answer_s":0.1492472141981125,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"correct":0,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","ingest_s":0.022795652970671654,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Simplifying mixed fractions.","pred_choice":"D","qa_index":3,"query_time_s":500.0,"question":"What concept will the speaker likely explain next?","sample_id":218,"sampled_indices":[246,247,248,249],"sampled_start_seconds":[492.49200439453125,494.4939880371094,496.4960021972656,498.49798583984375],"skip_reason":"","source_frame_indices":[14760,14820,14880,14940],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":784,"wall_s":0.23195300810039043},{"answer_s":0.31838888488709927,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"correct":1,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","ingest_s":0.0231422558426857,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":4,"query_time_s":646.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[319,320,321,322],"sampled_start_seconds":[638.6380004882812,640.6400146484375,642.6420288085938,644.6439819335938],"skip_reason":"","source_frame_indices":[19140,19200,19260,19320],"status":"answered","task":"Prospective Reasoning","video_id":"492","visual_tokens":784,"wall_s":0.3990254085510969},{"answer_s":0.15857111290097237,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Multiplying Fractions.","gold_choice":"D","ingest_s":0.02388351783156395,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":0,"query_time_s":10.0,"question":"What mathematical topic might the speaker focus on next?","sample_id":219,"sampled_indices":[1,2,3,4],"sampled_start_seconds":[2.002000093460083,4.004000186920166,6.00600004196167,8.008000373840332],"skip_reason":"","source_frame_indices":[60,120,180,240],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":784,"wall_s":0.23841130547225475},{"answer_s":0.20172547362744808,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.023204445838928223,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":1,"query_time_s":55.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[24,25,26,27],"sampled_start_seconds":[48.04800033569336,50.04999923706055,52.051998138427734,54.05400085449219],"skip_reason":"","source_frame_indices":[1440,1500,1560,1620],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":784,"wall_s":0.2774018235504627},{"answer_s":0.2305435985326767,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.025446586310863495,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":2,"query_time_s":149.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[71,72,73,74],"sampled_start_seconds":[142.14199829101562,144.1439971923828,146.14599609375,148.1479949951172],"skip_reason":"","source_frame_indices":[4260,4320,4380,4440],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":784,"wall_s":0.31095454283058643},{"answer_s":0.14884832687675953,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"correct":1,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Another explanation to fractions.","gold_choice":"D","ingest_s":0.033826082944869995,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"D) Another explanation to fractions.","pred_choice":"D","qa_index":3,"query_time_s":231.0,"question":"What mathematical concept might the speaker move on to explain next?","sample_id":219,"sampled_indices":[112,113,114,115],"sampled_start_seconds":[224.2239990234375,226.2259979248047,228.22799682617188,230.22999572753906],"skip_reason":"","source_frame_indices":[6720,6780,6840,6900],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":784,"wall_s":0.23721254616975784},{"answer_s":0.15346265025436878,"cache_lengths_before_answer":[797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797,797],"choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"correct":0,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"How to multiply the denominators.","gold_choice":"C","ingest_s":0.02297801524400711,"method":"recent_4","no_compression":true,"peak_allocated_gib":2.539090633392334,"pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":4,"query_time_s":310.0,"question":"What might the speaker discuss next?","sample_id":219,"sampled_indices":[151,152,153,154],"sampled_start_seconds":[302.302001953125,304.3039855957031,306.3059997558594,308.3080139160156],"skip_reason":"","source_frame_indices":[9060,9120,9180,9240],"status":"answered","task":"Prospective Reasoning","video_id":"456","visual_tokens":784,"wall_s":0.22863688319921494},{"answer_s":0.0,"choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"Yellow and Red.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":12.0,"question":"What colors are the wheels of the race car?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"459","visual_tokens":0,"wall_s":2.514570951461792e-07},{"answer_s":0.0,"choices":["A sad face.","A happy face.","A star.","A heart."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A happy face.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":72.0,"question":"What feature is present on the front of the toy?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"459","visual_tokens":0,"wall_s":2.9243528842926025e-07},{"answer_s":0.0,"choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"In a straight line.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Spatial Understanding","video_id":"459","visual_tokens":0,"wall_s":2.384185791015625e-07},{"answer_s":0.0,"choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":202.0,"question":"What is the new item introduced in the video?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Object Recognition","video_id":"459","visual_tokens":0,"wall_s":2.1792948246002197e-07},{"answer_s":0.0,"choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"correct":null,"feature_fingerprint":"b2a850d75ba1cc9e4ebe5836262f298d002fc5719174f72b23d1c896baf02b1e","fresh_state":true,"gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":227.0,"question":"What happened to the toy vehicle just now?","sample_id":202,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"459","visual_tokens":0,"wall_s":2.5331974029541016e-07},{"answer_s":0.0,"choices":["Eight.","Sixteen.","Four.","Twelve."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Eight.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Counting","video_id":"355","visual_tokens":0,"wall_s":2.2724270820617676e-07},{"answer_s":0.0,"choices":["Three.","Five.","Seven.","Four."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Four.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Counting","video_id":"355","visual_tokens":0,"wall_s":2.253800630569458e-07},{"answer_s":0.0,"choices":["Yellow.","Orange.","Green.","Blue."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":76.0,"question":"What is the color of the block that the hand is reaching for right now?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"355","visual_tokens":0,"wall_s":2.514570951461792e-07},{"answer_s":0.0,"choices":["Green.","Yellow.","Orange.","Red."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"355","visual_tokens":0,"wall_s":2.682209014892578e-07},{"answer_s":0.0,"choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"correct":null,"feature_fingerprint":"9d5fef6b1bfe5a09f4b00d7f86bb2255e8a7d672702cb2eae12232f91ddefe46","fresh_state":true,"gold_answer":"Green, yellow, orange.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":203,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"355","visual_tokens":0,"wall_s":2.1979212760925293e-07},{"answer_s":0.0,"choices":["Red.","Blue.","Yellow.","Purple."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Yellow.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":16.0,"question":"What color is the plastic block added on top of the green blocks?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"335","visual_tokens":0,"wall_s":2.7567148208618164e-07},{"answer_s":0.0,"choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"Red, green, and yellow.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"335","visual_tokens":0,"wall_s":3.6694109439849854e-07},{"answer_s":0.0,"choices":["A white block.","A green block.","An yellow block.","A blue block."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"An yellow block.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"335","visual_tokens":0,"wall_s":2.7008354663848877e-07},{"answer_s":0.0,"choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":141.0,"question":"What was the white text on the purple animated overlay read just now?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Text-Rich Understanding","video_id":"335","visual_tokens":0,"wall_s":2.5890767574310303e-07},{"answer_s":0.0,"choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"correct":null,"feature_fingerprint":"5a0a8ba3ba980776da4330eedc533a690fb166ba3236e5cbf3292071e1aeaaab","fresh_state":true,"gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","sample_id":204,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"335","visual_tokens":0,"wall_s":4.507601261138916e-07},{"answer_s":0.0,"choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Yellow, green.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":7.0,"question":"What are the colors of the crane arm?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Attribute Recognition","video_id":"448","visual_tokens":0,"wall_s":2.1606683731079102e-07},{"answer_s":0.0,"choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It lifts and swings gently.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":19.0,"question":"What happens to the orange block as the video progresses?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":2.4028122425079346e-07},{"answer_s":0.0,"choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":96.0,"question":"What was the sequence of actions performed with the blocks just now?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":2.0675361156463623e-07},{"answer_s":0.0,"choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"448","visual_tokens":0,"wall_s":2.421438694000244e-07},{"answer_s":0.0,"choices":["Ten.","Three.","Nine.","Eight."],"correct":null,"feature_fingerprint":"36d2b66967551a352d4ba3028b2cfb4ce5df6ed93ee232c7394a36052f87b7f7","fresh_state":true,"gold_answer":"Nine.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":265.0,"question":"How many green blocks are stacked now?","sample_id":208,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Counting","video_id":"448","visual_tokens":0,"wall_s":3.0919909477233887e-07},{"answer_s":0.0,"choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The hand adds another green block.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":25.0,"question":"What happens after the first green block is placed on the yellow block?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"118","visual_tokens":0,"wall_s":2.4028122425079346e-07},{"answer_s":0.0,"choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Pressing down the green blocks.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":118.0,"question":"What action do the hands perform just before moving away from the structure?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Action Recognition","video_id":"118","visual_tokens":0,"wall_s":2.9616057872772217e-07},{"answer_s":0.0,"choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Event Understanding","video_id":"118","visual_tokens":0,"wall_s":2.1047890186309814e-07},{"answer_s":0.0,"choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"In front of the original structure.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Spatial Understanding","video_id":"118","visual_tokens":0,"wall_s":2.905726432800293e-07},{"answer_s":0.0,"choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"correct":null,"feature_fingerprint":"b718c4930d20d7cb8b3e1431a38d463742c862557a77737b095e6bd5d601a671","fresh_state":true,"gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":315.0,"question":"What did the person do with the yellow block just now?","sample_id":209,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Action Recognition","video_id":"118","visual_tokens":0,"wall_s":2.2724270820617676e-07},{"answer_s":0.0,"choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add mixed numbers.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":20.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":2.849847078323364e-07},{"answer_s":0.0,"choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":130.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":4.023313522338867e-07},{"answer_s":0.0,"choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two mixed numbers.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":248.0,"question":"What might the speaker do next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":2.3283064365386963e-07},{"answer_s":0.0,"choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"How to add those two fractions numbers.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":396.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":2.905726432800293e-07},{"answer_s":0.0,"choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"correct":null,"feature_fingerprint":"1c48a952f5b7350d2420d53292bd92ab1493583057480c9542ae382a49a6c428","fresh_state":true,"gold_answer":"Adding the fractions together.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":478.0,"question":"What might the speaker explain next?","sample_id":212,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"125","visual_tokens":0,"wall_s":2.4586915969848633e-07},{"answer_s":0.0,"choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":11.0,"question":"What topic might the teacher explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":2.7939677238464355e-07},{"answer_s":0.0,"choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":190.0,"question":"What might the teacher explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":3.203749656677246e-07},{"answer_s":0.0,"choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":331.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":2.7567148208618164e-07},{"answer_s":0.0,"choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":500.0,"question":"What concept will the speaker likely explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":2.738088369369507e-07},{"answer_s":0.0,"choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"correct":null,"feature_fingerprint":"bd493dfe0141439b0119d5a77fcc167fed9176ea74a791116cd8aa18b9289992","fresh_state":true,"gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":646.0,"question":"What might the speaker explain next?","sample_id":218,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"492","visual_tokens":0,"wall_s":2.6263296604156494e-07},{"answer_s":0.0,"choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Multiplying Fractions.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":0,"query_time_s":10.0,"question":"What mathematical topic might the speaker focus on next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":2.849847078323364e-07},{"answer_s":0.0,"choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":1,"query_time_s":55.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":3.501772880554199e-07},{"answer_s":0.0,"choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":2,"query_time_s":149.0,"question":"What might the speaker do next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":1.7695128917694092e-07},{"answer_s":0.0,"choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"Another explanation to fractions.","gold_choice":"D","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":3,"query_time_s":231.0,"question":"What mathematical concept might the speaker move on to explain next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":2.5890767574310303e-07},{"answer_s":0.0,"choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"correct":null,"feature_fingerprint":"aed13a0f6d7cd67a69e50d5c3495f0e1d8f1a596a4a6da516623f199252c039d","fresh_state":true,"gold_answer":"How to multiply the denominators.","gold_choice":"C","ingest_s":0.0,"method":"support_oracle","no_compression":true,"peak_allocated_gib":0.0,"pred_answer":"","pred_choice":"","qa_index":4,"query_time_s":310.0,"question":"What might the speaker discuss next?","sample_id":219,"sampled_indices":[],"sampled_start_seconds":[],"skip_reason":"not_requested","source_frame_indices":[],"status":"skipped","task":"Prospective Reasoning","video_id":"456","visual_tokens":0,"wall_s":1.955777406692505e-07}],"long_reference_choice":"A","predictions":[{"answer_s":"0.18843233957886696","choices":"[\'How to add mixed numbers.\', \'The subtraction of mixed numbers.\', \'The multiplication of fractions.\', \'The properties of whole numbers.\']","correct":"1","gold_answer":"How to add mixed numbers.","gold_choice":"A","method":"causal","pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":"0","query_time_s":"20.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.24134743586182594","choices":"[\'How to add those two mixed numbers.\', \'The process of converting mixed numbers to improper fractions.\', \'Subtracting mixed numbers.\', \'The definition of mixed numbers.\']","correct":"0","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"causal","pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":"1","query_time_s":"130.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.20104572363197803","choices":"[\'How to add those two mixed numbers.\', \'Subtract the fractions.\', \'Multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"causal","pred_answer":"A) How to add those two mixed numbers.","pred_choice":"A","qa_index":"2","query_time_s":"248.0","question":"What might the speaker do next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.1998403836041689","choices":"[\'How to add mixed numbers.\', \'How to convert fractions to decimals.\', \'The process of subtracting fractions.\', \'How to add those two fractions numbers.\']","correct":"1","gold_answer":"How to add those two fractions numbers.","gold_choice":"D","method":"causal","pred_answer":"D) How to add those two fractions numbers.","pred_choice":"D","qa_index":"3","query_time_s":"396.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.15117954276502132","choices":"[\'Adding the fractions together.\', \'Converting the results back to whole numbers.\', \'Finding a different common denominator.\', \'Subtracting the fractions.\']","correct":"1","gold_answer":"Adding the fractions together.","gold_choice":"A","method":"causal","pred_answer":"A) Adding the fractions together.","pred_choice":"A","qa_index":"4","query_time_s":"478.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.16750607639551163","choices":"[\'How to add mixed numbers.\', \'The subtraction of mixed numbers.\', \'The multiplication of fractions.\', \'The properties of whole numbers.\']","correct":"1","gold_answer":"How to add mixed numbers.","gold_choice":"A","method":"fifo","pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":"0","query_time_s":"20.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.23644215986132622","choices":"[\'How to add those two mixed numbers.\', \'The process of converting mixed numbers to improper fractions.\', \'Subtracting mixed numbers.\', \'The definition of mixed numbers.\']","correct":"0","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"fifo","pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":"1","query_time_s":"130.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.19606392458081245","choices":"[\'How to add those two mixed numbers.\', \'Subtract the fractions.\', \'Multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"fifo","pred_answer":"A) How to add those two mixed numbers.","pred_choice":"A","qa_index":"2","query_time_s":"248.0","question":"What might the speaker do next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.18284097500145435","choices":"[\'How to add mixed numbers.\', \'How to convert fractions to decimals.\', \'The process of subtracting fractions.\', \'How to add those two fractions numbers.\']","correct":"0","gold_answer":"How to add those two fractions numbers.","gold_choice":"D","method":"fifo","pred_answer":"C) The process of subtracting fractions.","pred_choice":"C","qa_index":"3","query_time_s":"396.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.1415876541286707","choices":"[\'Adding the fractions together.\', \'Converting the results back to whole numbers.\', \'Finding a different common denominator.\', \'Subtracting the fractions.\']","correct":"1","gold_answer":"Adding the fractions together.","gold_choice":"A","method":"fifo","pred_answer":"A) Adding the fractions together.","pred_choice":"A","qa_index":"4","query_time_s":"478.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.17293584160506725","choices":"[\'How to add mixed numbers.\', \'The subtraction of mixed numbers.\', \'The multiplication of fractions.\', \'The properties of whole numbers.\']","correct":"1","gold_answer":"How to add mixed numbers.","gold_choice":"A","method":"hermes","pred_answer":"A) How to add mixed numbers.","pred_choice":"A","qa_index":"0","query_time_s":"20.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.23942515067756176","choices":"[\'How to add those two mixed numbers.\', \'The process of converting mixed numbers to improper fractions.\', \'Subtracting mixed numbers.\', \'The definition of mixed numbers.\']","correct":"0","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"hermes","pred_answer":"B) The process of converting mixed numbers to improper fractions.","pred_choice":"B","qa_index":"1","query_time_s":"130.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.1951775848865509","choices":"[\'How to add those two mixed numbers.\', \'Subtract the fractions.\', \'Multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"How to add those two mixed numbers.","gold_choice":"A","method":"hermes","pred_answer":"A) How to add those two mixed numbers.","pred_choice":"A","qa_index":"2","query_time_s":"248.0","question":"What might the speaker do next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.19778382405638695","choices":"[\'How to add mixed numbers.\', \'How to convert fractions to decimals.\', \'The process of subtracting fractions.\', \'How to add those two fractions numbers.\']","correct":"1","gold_answer":"How to add those two fractions numbers.","gold_choice":"D","method":"hermes","pred_answer":"D) How to add those two fractions numbers.","pred_choice":"D","qa_index":"3","query_time_s":"396.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.14115870371460915","choices":"[\'Adding the fractions together.\', \'Converting the results back to whole numbers.\', \'Finding a different common denominator.\', \'Subtracting the fractions.\']","correct":"1","gold_answer":"Adding the fractions together.","gold_choice":"A","method":"hermes","pred_answer":"A) Adding the fractions together.","pred_choice":"A","qa_index":"4","query_time_s":"478.0","question":"What might the speaker explain next?","sample_id":"212","task":"Prospective Reasoning","video_id":"125"},{"answer_s":"0.09205632843077183","choices":"[\'Red.\', \'Blue.\', \'Yellow.\', \'Purple.\']","correct":"0","gold_answer":"Yellow.","gold_choice":"C","method":"hermes","pred_answer":"A) Red.","pred_choice":"A","qa_index":"0","query_time_s":"16.0","question":"What color is the plastic block added on top of the green blocks?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.18797917291522026","choices":"[\'Blue, green, and yellow.\', \'Red, green, and yellow.\', \'Red, blue, and yellow.\', \'Green, blue, and orange.\']","correct":"1","gold_answer":"Red, green, and yellow.","gold_choice":"B","method":"hermes","pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":"1","query_time_s":"62.0","question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.12615688890218735","choices":"[\'A white block.\', \'A green block.\', \'An yellow block.\', \'A blue block.\']","correct":"0","gold_answer":"An yellow block.","gold_choice":"C","method":"hermes","pred_answer":"B) A green block.","pred_choice":"B","qa_index":"2","query_time_s":"85.0","question":"Which block was picked up and attached to the top of red block first just now?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.19608167745172977","choices":"[\'\\"Alphabet A transform to Box\\".\', \'\\"Alphabet B transform to Box\\".\', \'\\"Alphabet C transform to Box\\".\', \'\\"Alphabet D transform to Box\\".\']","correct":"0","gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","method":"hermes","pred_answer":"A) \\"Alphabet A transform to Box\\".","pred_choice":"A","qa_index":"3","query_time_s":"141.0","question":"What was the white text on the purple animated overlay read just now?","sample_id":"204","task":"Text-Rich Understanding","video_id":"335"},{"answer_s":"0.2551815826445818","choices":"[\'The blocks are arranged in a new pattern.\', \'The blocks are scattered and then gathered into a pile.\', \'The blocks are thrown away.\', \'The blocks are sorted by color.\']","correct":"1","gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","method":"hermes","pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":"4","query_time_s":"198.0","question":"What happens to the blocks after they are disassembled and mixed?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.09649430774152279","choices":"[\'Red.\', \'Blue.\', \'Yellow.\', \'Purple.\']","correct":"0","gold_answer":"Yellow.","gold_choice":"C","method":"fifo","pred_answer":"A) Red.","pred_choice":"A","qa_index":"0","query_time_s":"16.0","question":"What color is the plastic block added on top of the green blocks?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.19248576648533344","choices":"[\'Blue, green, and yellow.\', \'Red, green, and yellow.\', \'Red, blue, and yellow.\', \'Green, blue, and orange.\']","correct":"1","gold_answer":"Red, green, and yellow.","gold_choice":"B","method":"fifo","pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":"1","query_time_s":"62.0","question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.12397253513336182","choices":"[\'A white block.\', \'A green block.\', \'An yellow block.\', \'A blue block.\']","correct":"0","gold_answer":"An yellow block.","gold_choice":"C","method":"fifo","pred_answer":"B) A green block.","pred_choice":"B","qa_index":"2","query_time_s":"85.0","question":"Which block was picked up and attached to the top of red block first just now?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.19158986397087574","choices":"[\'\\"Alphabet A transform to Box\\".\', \'\\"Alphabet B transform to Box\\".\', \'\\"Alphabet C transform to Box\\".\', \'\\"Alphabet D transform to Box\\".\']","correct":"1","gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","method":"fifo","pred_answer":"B) \\"Alphabet B transform to Box\\".","pred_choice":"B","qa_index":"3","query_time_s":"141.0","question":"What was the white text on the purple animated overlay read just now?","sample_id":"204","task":"Text-Rich Understanding","video_id":"335"},{"answer_s":"0.24329162202775478","choices":"[\'The blocks are arranged in a new pattern.\', \'The blocks are scattered and then gathered into a pile.\', \'The blocks are thrown away.\', \'The blocks are sorted by color.\']","correct":"1","gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","method":"fifo","pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":"4","query_time_s":"198.0","question":"What happens to the blocks after they are disassembled and mixed?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.08776123635470867","choices":"[\'Red.\', \'Blue.\', \'Yellow.\', \'Purple.\']","correct":"0","gold_answer":"Yellow.","gold_choice":"C","method":"causal","pred_answer":"A) Red.","pred_choice":"A","qa_index":"0","query_time_s":"16.0","question":"What color is the plastic block added on top of the green blocks?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.17980062402784824","choices":"[\'Blue, green, and yellow.\', \'Red, green, and yellow.\', \'Red, blue, and yellow.\', \'Green, blue, and orange.\']","correct":"1","gold_answer":"Red, green, and yellow.","gold_choice":"B","method":"causal","pred_answer":"B) Red, green, and yellow.","pred_choice":"B","qa_index":"1","query_time_s":"62.0","question":"What are the primary colors of the plastic building blocks shown right now?","sample_id":"204","task":"Attribute Recognition","video_id":"335"},{"answer_s":"0.12179889902472496","choices":"[\'A white block.\', \'A green block.\', \'An yellow block.\', \'A blue block.\']","correct":"0","gold_answer":"An yellow block.","gold_choice":"C","method":"causal","pred_answer":"B) A green block.","pred_choice":"B","qa_index":"2","query_time_s":"85.0","question":"Which block was picked up and attached to the top of red block first just now?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.19785994850099087","choices":"[\'\\"Alphabet A transform to Box\\".\', \'\\"Alphabet B transform to Box\\".\', \'\\"Alphabet C transform to Box\\".\', \'\\"Alphabet D transform to Box\\".\']","correct":"0","gold_answer":"\\"Alphabet B transform to Box\\".","gold_choice":"B","method":"causal","pred_answer":"A) \\"Alphabet A transform to Box\\".","pred_choice":"A","qa_index":"3","query_time_s":"141.0","question":"What was the white text on the purple animated overlay read just now?","sample_id":"204","task":"Text-Rich Understanding","video_id":"335"},{"answer_s":"0.24552772752940655","choices":"[\'The blocks are arranged in a new pattern.\', \'The blocks are scattered and then gathered into a pile.\', \'The blocks are thrown away.\', \'The blocks are sorted by color.\']","correct":"1","gold_answer":"The blocks are scattered and then gathered into a pile.","gold_choice":"B","method":"causal","pred_answer":"B) The blocks are scattered and then gathered into a pile.","pred_choice":"B","qa_index":"4","query_time_s":"198.0","question":"What happens to the blocks after they are disassembled and mixed?","sample_id":"204","task":"Event Understanding","video_id":"335"},{"answer_s":"0.18892225623130798","choices":"[\'The hand removes the green block.\', \'The hand moves away and does nothing.\', \'The hand removes the yellow block.\', \'The hand adds another green block.\']","correct":"1","gold_answer":"The hand adds another green block.","gold_choice":"D","method":"causal","pred_answer":"D) The hand adds another green block.","pred_choice":"D","qa_index":"0","query_time_s":"25.0","question":"What happens after the first green block is placed on the yellow block?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.1951809711754322","choices":"[\'Adding a yellow block.\', \'Adding a blue block.\', \'Adjusting the blue block.\', \'Pressing down the green blocks.\']","correct":"1","gold_answer":"Pressing down the green blocks.","gold_choice":"D","method":"causal","pred_answer":"D) Pressing down the green blocks.","pred_choice":"D","qa_index":"1","query_time_s":"118.0","question":"What action do the hands perform just before moving away from the structure?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.3195121958851814","choices":"[\'The green block is placed first, followed by the yellow block.\', \'The yellow block is placed first, followed by the green block.\', \'The green block is removed, then the yellow block is placed.\', \'The yellow block is adjusted, then the green block is placed.\']","correct":"0","gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","method":"causal","pred_answer":"D) The yellow block is adjusted, then the green block is placed.","pred_choice":"D","qa_index":"2","query_time_s":"210.0","question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.16535842046141624","choices":"[\'Behind the original structure.\', \'Next to the original structure.\', \'On top of the original structure.\', \'In front of the original structure.\']","correct":"0","gold_answer":"In front of the original structure.","gold_choice":"D","method":"causal","pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":"3","query_time_s":"212.0","question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":"209","task":"Spatial Understanding","video_id":"118"},{"answer_s":"0.27233758196234703","choices":"[\'Removed them from the structure.\', \'Arranged them on the blue block and adjusted for alignment.\', \'Stacked them on top of the blue blocks.\', \'Placed them above the green block and another yellow block.\']","correct":"0","gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","method":"causal","pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":"4","query_time_s":"315.0","question":"What did the person do with the yellow block just now?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.18453293479979038","choices":"[\'The hand removes the green block.\', \'The hand moves away and does nothing.\', \'The hand removes the yellow block.\', \'The hand adds another green block.\']","correct":"1","gold_answer":"The hand adds another green block.","gold_choice":"D","method":"hermes","pred_answer":"D) The hand adds another green block.","pred_choice":"D","qa_index":"0","query_time_s":"25.0","question":"What happens after the first green block is placed on the yellow block?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.1943981647491455","choices":"[\'Adding a yellow block.\', \'Adding a blue block.\', \'Adjusting the blue block.\', \'Pressing down the green blocks.\']","correct":"1","gold_answer":"Pressing down the green blocks.","gold_choice":"D","method":"hermes","pred_answer":"D) Pressing down the green blocks.","pred_choice":"D","qa_index":"1","query_time_s":"118.0","question":"What action do the hands perform just before moving away from the structure?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.2917861081659794","choices":"[\'The green block is placed first, followed by the yellow block.\', \'The yellow block is placed first, followed by the green block.\', \'The green block is removed, then the yellow block is placed.\', \'The yellow block is adjusted, then the green block is placed.\']","correct":"0","gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","method":"hermes","pred_answer":"B) The yellow block is placed first, followed by the green block.","pred_choice":"B","qa_index":"2","query_time_s":"210.0","question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.16158512979745865","choices":"[\'Behind the original structure.\', \'Next to the original structure.\', \'On top of the original structure.\', \'In front of the original structure.\']","correct":"0","gold_answer":"In front of the original structure.","gold_choice":"D","method":"hermes","pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":"3","query_time_s":"212.0","question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":"209","task":"Spatial Understanding","video_id":"118"},{"answer_s":"0.26530105248093605","choices":"[\'Removed them from the structure.\', \'Arranged them on the blue block and adjusted for alignment.\', \'Stacked them on top of the blue blocks.\', \'Placed them above the green block and another yellow block.\']","correct":"0","gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","method":"hermes","pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":"4","query_time_s":"315.0","question":"What did the person do with the yellow block just now?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.19200599193572998","choices":"[\'The hand removes the green block.\', \'The hand moves away and does nothing.\', \'The hand removes the yellow block.\', \'The hand adds another green block.\']","correct":"1","gold_answer":"The hand adds another green block.","gold_choice":"D","method":"fifo","pred_answer":"D) The hand adds another green block.","pred_choice":"D","qa_index":"0","query_time_s":"25.0","question":"What happens after the first green block is placed on the yellow block?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.1902574636042118","choices":"[\'Adding a yellow block.\', \'Adding a blue block.\', \'Adjusting the blue block.\', \'Pressing down the green blocks.\']","correct":"1","gold_answer":"Pressing down the green blocks.","gold_choice":"D","method":"fifo","pred_answer":"D) Pressing down the green blocks.","pred_choice":"D","qa_index":"1","query_time_s":"118.0","question":"What action do the hands perform just before moving away from the structure?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.2800937108695507","choices":"[\'The green block is placed first, followed by the yellow block.\', \'The yellow block is placed first, followed by the green block.\', \'The green block is removed, then the yellow block is placed.\', \'The yellow block is adjusted, then the green block is placed.\']","correct":"0","gold_answer":"The green block is placed first, followed by the yellow block.","gold_choice":"A","method":"fifo","pred_answer":"D) The yellow block is adjusted, then the green block is placed.","pred_choice":"D","qa_index":"2","query_time_s":"210.0","question":"What sequence of actions occured with the green and yellow blocks just now?","sample_id":"209","task":"Event Understanding","video_id":"118"},{"answer_s":"0.1597305592149496","choices":"[\'Behind the original structure.\', \'Next to the original structure.\', \'On top of the original structure.\', \'In front of the original structure.\']","correct":"0","gold_answer":"In front of the original structure.","gold_choice":"D","method":"fifo","pred_answer":"B) Next to the original structure.","pred_choice":"B","qa_index":"3","query_time_s":"212.0","question":"Where is the new yellow and green block placed relative to the original structure?","sample_id":"209","task":"Spatial Understanding","video_id":"118"},{"answer_s":"0.2720623891800642","choices":"[\'Removed them from the structure.\', \'Arranged them on the blue block and adjusted for alignment.\', \'Stacked them on top of the blue blocks.\', \'Placed them above the green block and another yellow block.\']","correct":"0","gold_answer":"Placed them above the green block and another yellow block.","gold_choice":"D","method":"fifo","pred_answer":"B) Arranged them on the blue block and adjusted for alignment.","pred_choice":"B","qa_index":"4","query_time_s":"315.0","question":"What did the person do with the yellow block just now?","sample_id":"209","task":"Action Recognition","video_id":"118"},{"answer_s":"0.0916890911757946","choices":"[\'Eight.\', \'Sixteen.\', \'Four.\', \'Twelve.\']","correct":"1","gold_answer":"Eight.","gold_choice":"A","method":"causal","pred_answer":"A) Eight.","pred_choice":"A","qa_index":"0","query_time_s":"15.0","question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.08938242495059967","choices":"[\'Three.\', \'Five.\', \'Seven.\', \'Four.\']","correct":"1","gold_answer":"Four.","gold_choice":"D","method":"causal","pred_answer":"D) Four.","pred_choice":"D","qa_index":"1","query_time_s":"19.0","question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.0949619971215725","choices":"[\'Yellow.\', \'Orange.\', \'Green.\', \'Blue.\']","correct":"0","gold_answer":"Green.","gold_choice":"C","method":"causal","pred_answer":"A) Yellow.","pred_choice":"A","qa_index":"2","query_time_s":"76.0","question":"What is the color of the block that the hand is reaching for right now?","sample_id":"203","task":"Attribute Recognition","video_id":"355"},{"answer_s":"0.0900826994329691","choices":"[\'Green.\', \'Yellow.\', \'Orange.\', \'Red.\']","correct":"1","gold_answer":"Green.","gold_choice":"A","method":"causal","pred_answer":"A) Green.","pred_choice":"A","qa_index":"3","query_time_s":"134.0","question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.15957321971654892","choices":"[\'Green, yellow, orange.\', \'Yellow, green, orange.\', \'Orange, yellow, green.\', \'Green, orange, yellow.\']","correct":"0","gold_answer":"Green, yellow, orange.","gold_choice":"A","method":"causal","pred_answer":"B) Yellow, green, orange.","pred_choice":"B","qa_index":"4","query_time_s":"141.0","question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.09202076308429241","choices":"[\'Eight.\', \'Sixteen.\', \'Four.\', \'Twelve.\']","correct":"1","gold_answer":"Eight.","gold_choice":"A","method":"hermes","pred_answer":"A) Eight.","pred_choice":"A","qa_index":"0","query_time_s":"15.0","question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.09029870107769966","choices":"[\'Three.\', \'Five.\', \'Seven.\', \'Four.\']","correct":"1","gold_answer":"Four.","gold_choice":"D","method":"hermes","pred_answer":"D) Four.","pred_choice":"D","qa_index":"1","query_time_s":"19.0","question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.10152016207575798","choices":"[\'Yellow.\', \'Orange.\', \'Green.\', \'Blue.\']","correct":"0","gold_answer":"Green.","gold_choice":"C","method":"hermes","pred_answer":"A) Yellow.","pred_choice":"A","qa_index":"2","query_time_s":"76.0","question":"What is the color of the block that the hand is reaching for right now?","sample_id":"203","task":"Attribute Recognition","video_id":"355"},{"answer_s":"0.09015322476625443","choices":"[\'Green.\', \'Yellow.\', \'Orange.\', \'Red.\']","correct":"1","gold_answer":"Green.","gold_choice":"A","method":"hermes","pred_answer":"A) Green.","pred_choice":"A","qa_index":"3","query_time_s":"134.0","question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.1610526405274868","choices":"[\'Green, yellow, orange.\', \'Yellow, green, orange.\', \'Orange, yellow, green.\', \'Green, orange, yellow.\']","correct":"0","gold_answer":"Green, yellow, orange.","gold_choice":"A","method":"hermes","pred_answer":"B) Yellow, green, orange.","pred_choice":"B","qa_index":"4","query_time_s":"141.0","question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.092883026227355","choices":"[\'Eight.\', \'Sixteen.\', \'Four.\', \'Twelve.\']","correct":"1","gold_answer":"Eight.","gold_choice":"A","method":"fifo","pred_answer":"A) Eight.","pred_choice":"A","qa_index":"0","query_time_s":"15.0","question":"How many studs does each yellow interlocking block placed by the hand have?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.08998685888946056","choices":"[\'Three.\', \'Five.\', \'Seven.\', \'Four.\']","correct":"1","gold_answer":"Four.","gold_choice":"D","method":"fifo","pred_answer":"D) Four.","pred_choice":"D","qa_index":"1","query_time_s":"19.0","question":"How many interlocking blocks are positioned in a straight line by the hand?","sample_id":"203","task":"Counting","video_id":"355"},{"answer_s":"0.09487965330481529","choices":"[\'Yellow.\', \'Orange.\', \'Green.\', \'Blue.\']","correct":"0","gold_answer":"Green.","gold_choice":"C","method":"fifo","pred_answer":"A) Yellow.","pred_choice":"A","qa_index":"2","query_time_s":"76.0","question":"What is the color of the block that the hand is reaching for right now?","sample_id":"203","task":"Attribute Recognition","video_id":"355"},{"answer_s":"0.08945267833769321","choices":"[\'Green.\', \'Yellow.\', \'Orange.\', \'Red.\']","correct":"1","gold_answer":"Green.","gold_choice":"A","method":"fifo","pred_answer":"A) Green.","pred_choice":"A","qa_index":"3","query_time_s":"134.0","question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.15908676385879517","choices":"[\'Green, yellow, orange.\', \'Yellow, green, orange.\', \'Orange, yellow, green.\', \'Green, orange, yellow.\']","correct":"0","gold_answer":"Green, yellow, orange.","gold_choice":"A","method":"fifo","pred_answer":"B) Yellow, green, orange.","pred_choice":"B","qa_index":"4","query_time_s":"141.0","question":"What is the sequence of bricks added to the structure after the blue bricks?","sample_id":"203","task":"Event Understanding","video_id":"355"},{"answer_s":"0.15589812397956848","choices":"[\'mixed numbers and improper fractions.\', \'Add mixed numbers.\', \'Subtract improper fractions.\', \'Multiply fractions.\']","correct":"1","gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","method":"causal","pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":"0","query_time_s":"11.0","question":"What topic might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.2819257453083992","choices":"[\'Compare the fractions using decimal values.\', \'Explain how improper fractions is connected to proper fractions and whole fractions.\', \'Demonstrate how to multiply two fractions.\', \'Discuss the concept of mixed numbers.\']","correct":"1","gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","method":"causal","pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":"1","query_time_s":"190.0","question":"What might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.22746005840599537","choices":"[\'How to convert mixed numbers to improper fractions.\', \'The method for adding mixed numbers and improper fractions.\', \'Simplifying improper fractions.\', \'Dividing fractions by whole numbers.\']","correct":"0","gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","method":"causal","pred_answer":"B) The method for adding mixed numbers and improper fractions.","pred_choice":"B","qa_index":"2","query_time_s":"331.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.19382302276790142","choices":"[\'Multiplying mixed fractions by a whole number.\', \'Converting improper fractions to mixed numbers.\', \'Adding improper fractions together.\', \'Simplifying mixed fractions.\']","correct":"1","gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","method":"causal","pred_answer":"B) Converting improper fractions to mixed numbers.","pred_choice":"B","qa_index":"3","query_time_s":"500.0","question":"What concept will the speaker likely explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.29418681003153324","choices":"[\'Add the remainder to the quotient.\', \'Subtract 6 from 1.\', \'Divide 7 by 3.\', \'How to use dividing to convert improper fractions to a mixed number form.\']","correct":"1","gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","method":"causal","pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":"4","query_time_s":"646.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.15792346000671387","choices":"[\'mixed numbers and improper fractions.\', \'Add mixed numbers.\', \'Subtract improper fractions.\', \'Multiply fractions.\']","correct":"1","gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","method":"fifo","pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":"0","query_time_s":"11.0","question":"What topic might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.2823762558400631","choices":"[\'Compare the fractions using decimal values.\', \'Explain how improper fractions is connected to proper fractions and whole fractions.\', \'Demonstrate how to multiply two fractions.\', \'Discuss the concept of mixed numbers.\']","correct":"1","gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","method":"fifo","pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":"1","query_time_s":"190.0","question":"What might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.23033195920288563","choices":"[\'How to convert mixed numbers to improper fractions.\', \'The method for adding mixed numbers and improper fractions.\', \'Simplifying improper fractions.\', \'Dividing fractions by whole numbers.\']","correct":"0","gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","method":"fifo","pred_answer":"B) The method for adding mixed numbers and improper fractions.","pred_choice":"B","qa_index":"2","query_time_s":"331.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.13926327973604202","choices":"[\'Multiplying mixed fractions by a whole number.\', \'Converting improper fractions to mixed numbers.\', \'Adding improper fractions together.\', \'Simplifying mixed fractions.\']","correct":"0","gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","method":"fifo","pred_answer":"D) Simplifying mixed fractions.","pred_choice":"D","qa_index":"3","query_time_s":"500.0","question":"What concept will the speaker likely explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.29281827434897423","choices":"[\'Add the remainder to the quotient.\', \'Subtract 6 from 1.\', \'Divide 7 by 3.\', \'How to use dividing to convert improper fractions to a mixed number form.\']","correct":"1","gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","method":"fifo","pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":"4","query_time_s":"646.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.15730533003807068","choices":"[\'mixed numbers and improper fractions.\', \'Add mixed numbers.\', \'Subtract improper fractions.\', \'Multiply fractions.\']","correct":"1","gold_answer":"mixed numbers and improper fractions.","gold_choice":"A","method":"hermes","pred_answer":"A) mixed numbers and improper fractions.","pred_choice":"A","qa_index":"0","query_time_s":"11.0","question":"What topic might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.2928575463593006","choices":"[\'Compare the fractions using decimal values.\', \'Explain how improper fractions is connected to proper fractions and whole fractions.\', \'Demonstrate how to multiply two fractions.\', \'Discuss the concept of mixed numbers.\']","correct":"1","gold_answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","gold_choice":"B","method":"hermes","pred_answer":"B) Explain how improper fractions is connected to proper fractions and whole fractions.","pred_choice":"B","qa_index":"1","query_time_s":"190.0","question":"What might the teacher explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.22914502583444118","choices":"[\'How to convert mixed numbers to improper fractions.\', \'The method for adding mixed numbers and improper fractions.\', \'Simplifying improper fractions.\', \'Dividing fractions by whole numbers.\']","correct":"0","gold_answer":"How to convert mixed numbers to improper fractions.","gold_choice":"A","method":"hermes","pred_answer":"B) The method for adding mixed numbers and improper fractions.","pred_choice":"B","qa_index":"2","query_time_s":"331.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.19095884263515472","choices":"[\'Multiplying mixed fractions by a whole number.\', \'Converting improper fractions to mixed numbers.\', \'Adding improper fractions together.\', \'Simplifying mixed fractions.\']","correct":"1","gold_answer":"Converting improper fractions to mixed numbers.","gold_choice":"B","method":"hermes","pred_answer":"B) Converting improper fractions to mixed numbers.","pred_choice":"B","qa_index":"3","query_time_s":"500.0","question":"What concept will the speaker likely explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.2950042560696602","choices":"[\'Add the remainder to the quotient.\', \'Subtract 6 from 1.\', \'Divide 7 by 3.\', \'How to use dividing to convert improper fractions to a mixed number form.\']","correct":"1","gold_answer":"How to use dividing to convert improper fractions to a mixed number form.","gold_choice":"D","method":"hermes","pred_answer":"D) How to use dividing to convert improper fractions to a mixed number form.","pred_choice":"D","qa_index":"4","query_time_s":"646.0","question":"What might the speaker explain next?","sample_id":"218","task":"Prospective Reasoning","video_id":"492"},{"answer_s":"0.15728948451578617","choices":"[\'Adding Fractions.\', \'Dividing Fractions.\', \'Simplifying Fractions.\', \'Multiplying Fractions.\']","correct":"1","gold_answer":"Multiplying Fractions.","gold_choice":"D","method":"fifo","pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":"0","query_time_s":"10.0","question":"What mathematical topic might the speaker focus on next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.20548183284699917","choices":"[\'Explain how to convert the fractions.\', \'Show the steps to add the fractions.\', \'Show the steps to multiply the fractions.\', \'Discuss the importance of common denominators.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"fifo","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"1","query_time_s":"55.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.19723164476454258","choices":"[\'Add the fractions together.\', \'Subtract the fractions.\', \'Show the steps to multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"fifo","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"2","query_time_s":"149.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.1436773370951414","choices":"[\'Addition of fractions.\', \'Subtraction of fractions.\', \'Simplifying fractions.\', \'Another explanation to fractions.\']","correct":"1","gold_answer":"Another explanation to fractions.","gold_choice":"D","method":"fifo","pred_answer":"D) Another explanation to fractions.","pred_choice":"D","qa_index":"3","query_time_s":"231.0","question":"What mathematical concept might the speaker move on to explain next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.14306623674929142","choices":"[\'How to add fractions.\', \'How to divide fractions.\', \'How to multiply the denominators.\', \'How to subtract fractions.\']","correct":"0","gold_answer":"How to multiply the denominators.","gold_choice":"C","method":"fifo","pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":"4","query_time_s":"310.0","question":"What might the speaker discuss next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.14644737914204597","choices":"[\'Adding Fractions.\', \'Dividing Fractions.\', \'Simplifying Fractions.\', \'Multiplying Fractions.\']","correct":"1","gold_answer":"Multiplying Fractions.","gold_choice":"D","method":"hermes","pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":"0","query_time_s":"10.0","question":"What mathematical topic might the speaker focus on next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.20559757761657238","choices":"[\'Explain how to convert the fractions.\', \'Show the steps to add the fractions.\', \'Show the steps to multiply the fractions.\', \'Discuss the importance of common denominators.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"hermes","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"1","query_time_s":"55.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.20466398261487484","choices":"[\'Add the fractions together.\', \'Subtract the fractions.\', \'Show the steps to multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"hermes","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"2","query_time_s":"149.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.1263598408550024","choices":"[\'Addition of fractions.\', \'Subtraction of fractions.\', \'Simplifying fractions.\', \'Another explanation to fractions.\']","correct":"0","gold_answer":"Another explanation to fractions.","gold_choice":"D","method":"hermes","pred_answer":"C) Simplifying fractions.","pred_choice":"C","qa_index":"3","query_time_s":"231.0","question":"What mathematical concept might the speaker move on to explain next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.1434071771800518","choices":"[\'How to add fractions.\', \'How to divide fractions.\', \'How to multiply the denominators.\', \'How to subtract fractions.\']","correct":"0","gold_answer":"How to multiply the denominators.","gold_choice":"C","method":"hermes","pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":"4","query_time_s":"310.0","question":"What might the speaker discuss next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.14364814944565296","choices":"[\'Adding Fractions.\', \'Dividing Fractions.\', \'Simplifying Fractions.\', \'Multiplying Fractions.\']","correct":"1","gold_answer":"Multiplying Fractions.","gold_choice":"D","method":"causal","pred_answer":"D) Multiplying Fractions.","pred_choice":"D","qa_index":"0","query_time_s":"10.0","question":"What mathematical topic might the speaker focus on next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.20567441172897816","choices":"[\'Explain how to convert the fractions.\', \'Show the steps to add the fractions.\', \'Show the steps to multiply the fractions.\', \'Discuss the importance of common denominators.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"causal","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"1","query_time_s":"55.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.19790850020945072","choices":"[\'Add the fractions together.\', \'Subtract the fractions.\', \'Show the steps to multiply the fractions.\', \'Divide the fractions.\']","correct":"1","gold_answer":"Show the steps to multiply the fractions.","gold_choice":"C","method":"causal","pred_answer":"C) Show the steps to multiply the fractions.","pred_choice":"C","qa_index":"2","query_time_s":"149.0","question":"What might the speaker do next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.12642996944487095","choices":"[\'Addition of fractions.\', \'Subtraction of fractions.\', \'Simplifying fractions.\', \'Another explanation to fractions.\']","correct":"0","gold_answer":"Another explanation to fractions.","gold_choice":"D","method":"causal","pred_answer":"C) Simplifying fractions.","pred_choice":"C","qa_index":"3","query_time_s":"231.0","question":"What mathematical concept might the speaker move on to explain next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.14410200901329517","choices":"[\'How to add fractions.\', \'How to divide fractions.\', \'How to multiply the denominators.\', \'How to subtract fractions.\']","correct":"0","gold_answer":"How to multiply the denominators.","gold_choice":"C","method":"causal","pred_answer":"B) How to divide fractions.","pred_choice":"B","qa_index":"4","query_time_s":"310.0","question":"What might the speaker discuss next?","sample_id":"219","task":"Prospective Reasoning","video_id":"456"},{"answer_s":"0.13553927652537823","choices":"[\'Green and blue.\', \'Red and blue.\', \'Yellow and Red.\', \'Brown.\']","correct":"0","gold_answer":"Yellow and Red.","gold_choice":"C","method":"fifo","pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":"0","query_time_s":"12.0","question":"What colors are the wheels of the race car?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.13774332031607628","choices":"[\'A sad face.\', \'A happy face.\', \'A star.\', \'A heart.\']","correct":"1","gold_answer":"A happy face.","gold_choice":"B","method":"fifo","pred_answer":"B) A happy face.","pred_choice":"B","qa_index":"1","query_time_s":"72.0","question":"What feature is present on the front of the toy?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.14838625118136406","choices":"[\'In a circle.\', \'In a straight line.\', \'In a random pattern.\', \'In a triangular shape.\']","correct":"1","gold_answer":"In a straight line.","gold_choice":"B","method":"fifo","pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":"2","query_time_s":"128.0","question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":"202","task":"Spatial Understanding","video_id":"459"},{"answer_s":"0.28389716520905495","choices":"[\'A small structure made of blue and green blocks.\', \'A small structure made of yellow and red blocks with wheels.\', \'A toy vehicle with blue and yellow wheels.\', \'An orange and green interlocking block structure.\']","correct":"1","gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","method":"fifo","pred_answer":"B) A small structure made of yellow and red blocks with wheels.","pred_choice":"B","qa_index":"3","query_time_s":"202.0","question":"What is the new item introduced in the video?","sample_id":"202","task":"Object Recognition","video_id":"459"},{"answer_s":"0.22114179283380508","choices":"[\'The toy vehicle is disassembled.\', \'The toy vehicle moves to the right.\', \'The toy vehicle start circling around the table.\', \'The toy vehicle is lifted off the surface.\']","correct":"0","gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","method":"fifo","pred_answer":"D) The toy vehicle is lifted off the surface.","pred_choice":"D","qa_index":"4","query_time_s":"227.0","question":"What happened to the toy vehicle just now?","sample_id":"202","task":"Event Understanding","video_id":"459"},{"answer_s":"0.13250285387039185","choices":"[\'Green and blue.\', \'Red and blue.\', \'Yellow and Red.\', \'Brown.\']","correct":"0","gold_answer":"Yellow and Red.","gold_choice":"C","method":"causal","pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":"0","query_time_s":"12.0","question":"What colors are the wheels of the race car?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.13598902709782124","choices":"[\'A sad face.\', \'A happy face.\', \'A star.\', \'A heart.\']","correct":"1","gold_answer":"A happy face.","gold_choice":"B","method":"causal","pred_answer":"B) A happy face.","pred_choice":"B","qa_index":"1","query_time_s":"72.0","question":"What feature is present on the front of the toy?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.1508791297674179","choices":"[\'In a circle.\', \'In a straight line.\', \'In a random pattern.\', \'In a triangular shape.\']","correct":"1","gold_answer":"In a straight line.","gold_choice":"B","method":"causal","pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":"2","query_time_s":"128.0","question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":"202","task":"Spatial Understanding","video_id":"459"},{"answer_s":"0.2747975308448076","choices":"[\'A small structure made of blue and green blocks.\', \'A small structure made of yellow and red blocks with wheels.\', \'A toy vehicle with blue and yellow wheels.\', \'An orange and green interlocking block structure.\']","correct":"1","gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","method":"causal","pred_answer":"B) A small structure made of yellow and red blocks with wheels.","pred_choice":"B","qa_index":"3","query_time_s":"202.0","question":"What is the new item introduced in the video?","sample_id":"202","task":"Object Recognition","video_id":"459"},{"answer_s":"0.22018844820559025","choices":"[\'The toy vehicle is disassembled.\', \'The toy vehicle moves to the right.\', \'The toy vehicle start circling around the table.\', \'The toy vehicle is lifted off the surface.\']","correct":"0","gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","method":"causal","pred_answer":"D) The toy vehicle is lifted off the surface.","pred_choice":"D","qa_index":"4","query_time_s":"227.0","question":"What happened to the toy vehicle just now?","sample_id":"202","task":"Event Understanding","video_id":"459"},{"answer_s":"0.12713826820254326","choices":"[\'Green and blue.\', \'Red and blue.\', \'Yellow and Red.\', \'Brown.\']","correct":"0","gold_answer":"Yellow and Red.","gold_choice":"C","method":"hermes","pred_answer":"A) Green and blue.","pred_choice":"A","qa_index":"0","query_time_s":"12.0","question":"What colors are the wheels of the race car?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.1387714147567749","choices":"[\'A sad face.\', \'A happy face.\', \'A star.\', \'A heart.\']","correct":"1","gold_answer":"A happy face.","gold_choice":"B","method":"hermes","pred_answer":"B) A happy face.","pred_choice":"B","qa_index":"1","query_time_s":"72.0","question":"What feature is present on the front of the toy?","sample_id":"202","task":"Attribute Recognition","video_id":"459"},{"answer_s":"0.14673993736505508","choices":"[\'In a circle.\', \'In a straight line.\', \'In a random pattern.\', \'In a triangular shape.\']","correct":"1","gold_answer":"In a straight line.","gold_choice":"B","method":"hermes","pred_answer":"B) In a straight line.","pred_choice":"B","qa_index":"2","query_time_s":"128.0","question":"How are the yellow building blocks arranged before the blue block is placed on top?","sample_id":"202","task":"Spatial Understanding","video_id":"459"},{"answer_s":"0.2738387733697891","choices":"[\'A small structure made of blue and green blocks.\', \'A small structure made of yellow and red blocks with wheels.\', \'A toy vehicle with blue and yellow wheels.\', \'An orange and green interlocking block structure.\']","correct":"1","gold_answer":"A small structure made of yellow and red blocks with wheels.","gold_choice":"B","method":"hermes","pred_answer":"B) A small structure made of yellow and red blocks with wheels.","pred_choice":"B","qa_index":"3","query_time_s":"202.0","question":"What is the new item introduced in the video?","sample_id":"202","task":"Object Recognition","video_id":"459"},{"answer_s":"0.21748771145939827","choices":"[\'The toy vehicle is disassembled.\', \'The toy vehicle moves to the right.\', \'The toy vehicle start circling around the table.\', \'The toy vehicle is lifted off the surface.\']","correct":"0","gold_answer":"The toy vehicle start circling around the table.","gold_choice":"C","method":"hermes","pred_answer":"D) The toy vehicle is lifted off the surface.","pred_choice":"D","qa_index":"4","query_time_s":"227.0","question":"What happened to the toy vehicle just now?","sample_id":"202","task":"Event Understanding","video_id":"459"},{"answer_s":"0.15792081132531166","choices":"[\'Yellow, green.\', \'Yellow, blue.\', \'Red, green, blue.\', \'Yellow, blue, green.\']","correct":"0","gold_answer":"Yellow, green.","gold_choice":"A","method":"hermes","pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":"0","query_time_s":"7.0","question":"What are the colors of the crane arm?","sample_id":"208","task":"Attribute Recognition","video_id":"448"},{"answer_s":"0.15842362865805626","choices":"[\'It falls to the ground.\', \'It remains stationary.\', \'It lifts and swings gently.\', \'It changes color.\']","correct":"1","gold_answer":"It lifts and swings gently.","gold_choice":"C","method":"hermes","pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":"1","query_time_s":"19.0","question":"What happens to the orange block as the video progresses?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.2961467355489731","choices":"[\'The hand first places a green block, then a yellow block, followed by a blue block.\', \'The hand builds a structure with red, green, and blue blocks.\', \'The hand stacks blocks randomly.\', \'The hand places a yellow block, moves it, then places a green block next to it.\']","correct":"1","gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","method":"hermes","pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":"2","query_time_s":"96.0","question":"What was the sequence of actions performed with the blocks just now?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.1626466028392315","choices":"[\'It is disassembled.\', \'It is placed on top of the larger structure.\', \'It is left in the foreground.\', \'It is placed back beside the larger structure.\']","correct":"0","gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","method":"hermes","pred_answer":"A) It is disassembled.","pred_choice":"A","qa_index":"3","query_time_s":"171.0","question":"What happens to the smaller structure after it is first moved out of view?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.09141966328024864","choices":"[\'Ten.\', \'Three.\', \'Nine.\', \'Eight.\']","correct":"1","gold_answer":"Nine.","gold_choice":"C","method":"hermes","pred_answer":"C) Nine.","pred_choice":"C","qa_index":"4","query_time_s":"265.0","question":"How many green blocks are stacked now?","sample_id":"208","task":"Counting","video_id":"448"},{"answer_s":"0.16316485591232777","choices":"[\'Yellow, green.\', \'Yellow, blue.\', \'Red, green, blue.\', \'Yellow, blue, green.\']","correct":"0","gold_answer":"Yellow, green.","gold_choice":"A","method":"fifo","pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":"0","query_time_s":"7.0","question":"What are the colors of the crane arm?","sample_id":"208","task":"Attribute Recognition","video_id":"448"},{"answer_s":"0.16387391835451126","choices":"[\'It falls to the ground.\', \'It remains stationary.\', \'It lifts and swings gently.\', \'It changes color.\']","correct":"1","gold_answer":"It lifts and swings gently.","gold_choice":"C","method":"fifo","pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":"1","query_time_s":"19.0","question":"What happens to the orange block as the video progresses?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.3028052356094122","choices":"[\'The hand first places a green block, then a yellow block, followed by a blue block.\', \'The hand builds a structure with red, green, and blue blocks.\', \'The hand stacks blocks randomly.\', \'The hand places a yellow block, moves it, then places a green block next to it.\']","correct":"1","gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","method":"fifo","pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":"2","query_time_s":"96.0","question":"What was the sequence of actions performed with the blocks just now?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.16217474080622196","choices":"[\'It is disassembled.\', \'It is placed on top of the larger structure.\', \'It is left in the foreground.\', \'It is placed back beside the larger structure.\']","correct":"0","gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","method":"fifo","pred_answer":"A) It is disassembled.","pred_choice":"A","qa_index":"3","query_time_s":"171.0","question":"What happens to the smaller structure after it is first moved out of view?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.08687982149422169","choices":"[\'Ten.\', \'Three.\', \'Nine.\', \'Eight.\']","correct":"1","gold_answer":"Nine.","gold_choice":"C","method":"fifo","pred_answer":"C) Nine.","pred_choice":"C","qa_index":"4","query_time_s":"265.0","question":"How many green blocks are stacked now?","sample_id":"208","task":"Counting","video_id":"448"},{"answer_s":"0.15902036800980568","choices":"[\'Yellow, green.\', \'Yellow, blue.\', \'Red, green, blue.\', \'Yellow, blue, green.\']","correct":"0","gold_answer":"Yellow, green.","gold_choice":"A","method":"causal","pred_answer":"D) Yellow, blue, green.","pred_choice":"D","qa_index":"0","query_time_s":"7.0","question":"What are the colors of the crane arm?","sample_id":"208","task":"Attribute Recognition","video_id":"448"},{"answer_s":"0.1584992278367281","choices":"[\'It falls to the ground.\', \'It remains stationary.\', \'It lifts and swings gently.\', \'It changes color.\']","correct":"1","gold_answer":"It lifts and swings gently.","gold_choice":"C","method":"causal","pred_answer":"C) It lifts and swings gently.","pred_choice":"C","qa_index":"1","query_time_s":"19.0","question":"What happens to the orange block as the video progresses?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.30494783632457256","choices":"[\'The hand first places a green block, then a yellow block, followed by a blue block.\', \'The hand builds a structure with red, green, and blue blocks.\', \'The hand stacks blocks randomly.\', \'The hand places a yellow block, moves it, then places a green block next to it.\']","correct":"1","gold_answer":"The hand places a yellow block, moves it, then places a green block next to it.","gold_choice":"D","method":"causal","pred_answer":"D) The hand places a yellow block, moves it, then places a green","pred_choice":"D","qa_index":"2","query_time_s":"96.0","question":"What was the sequence of actions performed with the blocks just now?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.16320319846272469","choices":"[\'It is disassembled.\', \'It is placed on top of the larger structure.\', \'It is left in the foreground.\', \'It is placed back beside the larger structure.\']","correct":"0","gold_answer":"It is placed back beside the larger structure.","gold_choice":"D","method":"causal","pred_answer":"A) It is disassembled.","pred_choice":"A","qa_index":"3","query_time_s":"171.0","question":"What happens to the smaller structure after it is first moved out of view?","sample_id":"208","task":"Event Understanding","video_id":"448"},{"answer_s":"0.09594877250492573","choices":"[\'Ten.\', \'Three.\', \'Nine.\', \'Eight.\']","correct":"1","gold_answer":"Nine.","gold_choice":"C","method":"causal","pred_answer":"C) Nine.","pred_choice":"C","qa_index":"4","query_time_s":"265.0","question":"How many green blocks are stacked now?","sample_id":"208","task":"Counting","video_id":"448"}],"source_sha256":{"inference/abstract_hermes.py":"cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0","inference/llavaov_hermes.py":"45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0","inference/reindex_1d.py":"22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0","scripts/build_failure_pilot.py":"c81681e7f148702d07d1a2cc193a1ec32cf4f89778a911d92283088050ed7835","video_qa/base.py":"795cf98e8d1125ea934dc2c9744ba1bbd3b29701f17aae2d935f382c7343b2d5","video_qa/hermes_vqa.py":"f25bac6cd19db7148e7fa39cc9d2e96fb40d1f14709778f2490023a65dddc95b"},"videos":[{"conversations":[{"answer":"How to add mixed numbers.","choices":["How to add mixed numbers.","The subtraction of mixed numbers.","The multiplication of fractions.","The properties of whole numbers."],"end_time":20.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:00:20"},{"answer":"How to add those two mixed numbers.","choices":["How to add those two mixed numbers.","The process of converting mixed numbers to improper fractions.","Subtracting mixed numbers.","The definition of mixed numbers."],"end_time":130.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:02:10"},{"answer":"How to add those two mixed numbers.","choices":["How to add those two mixed numbers.","Subtract the fractions.","Multiply the fractions.","Divide the fractions."],"end_time":248.0,"question":"What might the speaker do next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:04:08"},{"answer":"How to add those two fractions numbers.","choices":["How to add mixed numbers.","How to convert fractions to decimals.","The process of subtracting fractions.","How to add those two fractions numbers."],"end_time":396.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:06:36"},{"answer":"Adding the fractions together.","choices":["Adding the fractions together.","Converting the results back to whole numbers.","Finding a different common denominator.","Subtracting the fractions."],"end_time":478.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:07:58"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"d6d7e364e74c3c0ea754334d02aeeb96f44615090737f85c764aa4191bc1ac6e","start":0,"stop":10,"tokens":1960},{"file":"chunk_0001.pt","qa_index":1,"sha256":"6080fd0364676ef310752376b79a02355ba70325579ebacff1fc05ce481f00f3","start":10,"stop":26,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"09e4b40dfc029b9a62c04a7d0a39a9f65dd046a9c762b0f508cb565eb8c700fc","start":26,"stop":42,"tokens":3136},{"file":"chunk_0003.pt","qa_index":1,"sha256":"2089565699b9a739878b2fd43b5946cd6f3e1a470fd00a77f965fd0faa2f7721","start":42,"stop":58,"tokens":3136},{"file":"chunk_0004.pt","qa_index":1,"sha256":"7cadf9b625100709c356f900937b95ae9ff60ac4b556554e774fa4a97292395c","start":58,"stop":65,"tokens":1372},{"file":"chunk_0005.pt","qa_index":2,"sha256":"349eabc38231a834d3dbfa10dfb0b0a5f9910dc3cf520236a6133620391baa6d","start":65,"stop":81,"tokens":3136},{"file":"chunk_0006.pt","qa_index":2,"sha256":"567e533c35369e4c81aaecbb2b554ccc19391a3f7e99b1f09781819dbec0baa1","start":81,"stop":97,"tokens":3136},{"file":"chunk_0007.pt","qa_index":2,"sha256":"f3513c2caac614f02204824ca68d4d4cb2d4a911b81f38c076d23f69388cc3a9","start":97,"stop":113,"tokens":3136},{"file":"chunk_0008.pt","qa_index":2,"sha256":"b7470df1b3f24c8c59d915217cdd5c52edc4f34e389bede801f30579df365835","start":113,"stop":124,"tokens":2156},{"file":"chunk_0009.pt","qa_index":3,"sha256":"1d54a1ef2c91574c55ac4737ba0b5c29733db4ae93f80df255f896bfcb3d7d00","start":124,"stop":140,"tokens":3136},{"file":"chunk_0010.pt","qa_index":3,"sha256":"8389050253fee3fbec4d4c3cf11b8d78b9ef1c37b5543e3475c5853036d05bb0","start":140,"stop":156,"tokens":3136},{"file":"chunk_0011.pt","qa_index":3,"sha256":"5ef37cdcd8cf7fee26d72cc9105ef3cfcfca6eb07d45b90ea755f1a6a9142f44","start":156,"stop":172,"tokens":3136},{"file":"chunk_0012.pt","qa_index":3,"sha256":"0d65228d69c955f6eb3be4c4cf46fd31a47dc30e8c8774fff4b417a553cab1d1","start":172,"stop":188,"tokens":3136},{"file":"chunk_0013.pt","qa_index":3,"sha256":"7d405de7385508a14d2b70894ef0360043355f7208e7fe3c93504f44f3c6b4b0","start":188,"stop":198,"tokens":1960},{"file":"chunk_0014.pt","qa_index":4,"sha256":"b56048c77c365f05275a0335507e98622ba7c52d28a43293b2a474e92ede8571","start":198,"stop":214,"tokens":3136},{"file":"chunk_0015.pt","qa_index":4,"sha256":"88cc971c693727d57fed893723acc8fde9b20e8736c4b2959c393f0241e0f4da","start":214,"stop":230,"tokens":3136},{"file":"chunk_0016.pt","qa_index":4,"sha256":"47a7b0caa6d79d8fe72b2f4e2c7e6ba5ad1c03d9c84943cc2b64ddbda897f5ec","start":230,"stop":239,"tokens":1764}],"identity":"5f85f034d8a294f1ed2f3bee2906c371edc994f3a507e33246603ef0554bd84b","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":20.736943997442722,"sample_id":212,"sampled_frames_available":272,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"0dfc9943bf380053745c25d0c6448698a5ee764feb8f99c631fcb145b23768bb"},"sample_id":212,"timestamps":[{"end_s":"0.03336666524410248","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0353667736053467","sampled_index":"1","source_frame":"60","start_s":"2.002000093460083"},{"end_s":"4.03736686706543","sampled_index":"2","source_frame":"120","start_s":"4.004000186920166"},{"end_s":"6.039366722106934","sampled_index":"3","source_frame":"180","start_s":"6.00600004196167"},{"end_s":"8.041366577148438","sampled_index":"4","source_frame":"240","start_s":"8.008000373840332"},{"end_s":"10.043366432189941","sampled_index":"5","source_frame":"300","start_s":"10.010000228881836"},{"end_s":"12.045366287231445","sampled_index":"6","source_frame":"360","start_s":"12.01200008392334"},{"end_s":"14.047367095947266","sampled_index":"7","source_frame":"420","start_s":"14.013999938964844"},{"end_s":"16.049365997314453","sampled_index":"8","source_frame":"480","start_s":"16.016000747680664"},{"end_s":"18.051366806030273","sampled_index":"9","source_frame":"540","start_s":"18.01799964904785"},{"end_s":"20.053367614746094","sampled_index":"10","source_frame":"600","start_s":"20.020000457763672"},{"end_s":"22.05536651611328","sampled_index":"11","source_frame":"660","start_s":"22.02199935913086"},{"end_s":"24.0573673248291","sampled_index":"12","source_frame":"720","start_s":"24.02400016784668"},{"end_s":"26.05936622619629","sampled_index":"13","source_frame":"780","start_s":"26.025999069213867"},{"end_s":"28.06136703491211","sampled_index":"14","source_frame":"840","start_s":"28.027999877929688"},{"end_s":"30.063365936279297","sampled_index":"15","source_frame":"900","start_s":"30.030000686645508"},{"end_s":"32.065364837646484","sampled_index":"16","source_frame":"960","start_s":"32.03200149536133"},{"end_s":"34.06736755371094","sampled_index":"17","source_frame":"1020","start_s":"34.034000396728516"},{"end_s":"36.069366455078125","sampled_index":"18","source_frame":"1080","start_s":"36.0359992980957"},{"end_s":"38.07136535644531","sampled_index":"19","source_frame":"1140","start_s":"38.03799819946289"},{"end_s":"40.073368072509766","sampled_index":"20","source_frame":"1200","start_s":"40.040000915527344"},{"end_s":"42.07536697387695","sampled_index":"21","source_frame":"1260","start_s":"42.04199981689453"},{"end_s":"44.07736587524414","sampled_index":"22","source_frame":"1320","start_s":"44.04399871826172"},{"end_s":"46.07936477661133","sampled_index":"23","source_frame":"1380","start_s":"46.04600143432617"},{"end_s":"48.08136749267578","sampled_index":"24","source_frame":"1440","start_s":"48.04800033569336"},{"end_s":"50.08336639404297","sampled_index":"25","source_frame":"1500","start_s":"50.04999923706055"},{"end_s":"52.085365295410156","sampled_index":"26","source_frame":"1560","start_s":"52.051998138427734"},{"end_s":"54.08736801147461","sampled_index":"27","source_frame":"1620","start_s":"54.05400085449219"},{"end_s":"56.0893669128418","sampled_index":"28","source_frame":"1680","start_s":"56.055999755859375"},{"end_s":"58.091365814208984","sampled_index":"29","source_frame":"1740","start_s":"58.05799865722656"},{"end_s":"60.09336853027344","sampled_index":"30","source_frame":"1800","start_s":"60.060001373291016"},{"end_s":"62.095367431640625","sampled_index":"31","source_frame":"1860","start_s":"62.0620002746582"},{"end_s":"64.09736633300781","sampled_index":"32","source_frame":"1920","start_s":"64.06400299072266"},{"end_s":"66.099365234375","sampled_index":"33","source_frame":"1980","start_s":"66.06600189208984"},{"end_s":"68.10136413574219","sampled_index":"34","source_frame":"2040","start_s":"68.06800079345703"},{"end_s":"70.10336303710938","sampled_index":"35","source_frame":"2100","start_s":"70.06999969482422"},{"end_s":"72.1053695678711","sampled_index":"36","source_frame":"2160","start_s":"72.0719985961914"},{"end_s":"74.10736846923828","sampled_index":"37","source_frame":"2220","start_s":"74.0739974975586"},{"end_s":"76.10936737060547","sampled_index":"38","source_frame":"2280","start_s":"76.07599639892578"},{"end_s":"78.11136627197266","sampled_index":"39","source_frame":"2340","start_s":"78.0780029296875"},{"end_s":"80.11336517333984","sampled_index":"40","source_frame":"2400","start_s":"80.08000183105469"},{"end_s":"82.11536407470703","sampled_index":"41","source_frame":"2460","start_s":"82.08200073242188"},{"end_s":"84.11736297607422","sampled_index":"42","source_frame":"2520","start_s":"84.08399963378906"},{"end_s":"86.11936950683594","sampled_index":"43","source_frame":"2580","start_s":"86.08599853515625"},{"end_s":"88.12136840820312","sampled_index":"44","source_frame":"2640","start_s":"88.08799743652344"},{"end_s":"90.12336730957031","sampled_index":"45","source_frame":"2700","start_s":"90.08999633789062"},{"end_s":"92.1253662109375","sampled_index":"46","source_frame":"2760","start_s":"92.09200286865234"},{"end_s":"94.12736511230469","sampled_index":"47","source_frame":"2820","start_s":"94.09400177001953"},{"end_s":"96.12936401367188","sampled_index":"48","source_frame":"2880","start_s":"96.09600067138672"},{"end_s":"98.13136291503906","sampled_index":"49","source_frame":"2940","start_s":"98.0979995727539"},{"end_s":"100.13336944580078","sampled_index":"50","source_frame":"3000","start_s":"100.0999984741211"},{"end_s":"102.13536834716797","sampled_index":"51","source_frame":"3060","start_s":"102.10199737548828"},{"end_s":"104.13736724853516","sampled_index":"52","source_frame":"3120","start_s":"104.10399627685547"},{"end_s":"106.13936614990234","sampled_index":"53","source_frame":"3180","start_s":"106.10600280761719"},{"end_s":"108.14136505126953","sampled_index":"54","source_frame":"3240","start_s":"108.10800170898438"},{"end_s":"110.14336395263672","sampled_index":"55","source_frame":"3300","start_s":"110.11000061035156"},{"end_s":"112.1453628540039","sampled_index":"56","source_frame":"3360","start_s":"112.11199951171875"},{"end_s":"114.14736938476562","sampled_index":"57","source_frame":"3420","start_s":"114.11399841308594"},{"end_s":"116.14936828613281","sampled_index":"58","source_frame":"3480","start_s":"116.11599731445312"},{"end_s":"118.1513671875","sampled_index":"59","source_frame":"3540","start_s":"118.11799621582031"},{"end_s":"120.15336608886719","sampled_index":"60","source_frame":"3600","start_s":"120.12000274658203"},{"end_s":"122.15536499023438","sampled_index":"61","source_frame":"3660","start_s":"122.12200164794922"},{"end_s":"124.15736389160156","sampled_index":"62","source_frame":"3720","start_s":"124.1240005493164"},{"end_s":"126.15937042236328","sampled_index":"63","source_frame":"3780","start_s":"126.1259994506836"},{"end_s":"128.16136169433594","sampled_index":"64","source_frame":"3840","start_s":"128.1280059814453"},{"end_s":"130.16336059570312","sampled_index":"65","source_frame":"3900","start_s":"130.1300048828125"},{"end_s":"132.1653594970703","sampled_index":"66","source_frame":"3960","start_s":"132.1320037841797"},{"end_s":"134.16737365722656","sampled_index":"67","source_frame":"4020","start_s":"134.13400268554688"},{"end_s":"136.16937255859375","sampled_index":"68","source_frame":"4080","start_s":"136.13600158691406"},{"end_s":"138.17137145996094","sampled_index":"69","source_frame":"4140","start_s":"138.13800048828125"},{"end_s":"140.17337036132812","sampled_index":"70","source_frame":"4200","start_s":"140.13999938964844"},{"end_s":"142.1753692626953","sampled_index":"71","source_frame":"4260","start_s":"142.14199829101562"},{"end_s":"144.1773681640625","sampled_index":"72","source_frame":"4320","start_s":"144.1439971923828"},{"end_s":"146.1793670654297","sampled_index":"73","source_frame":"4380","start_s":"146.14599609375"},{"end_s":"148.18136596679688","sampled_index":"74","source_frame":"4440","start_s":"148.1479949951172"},{"end_s":"150.18336486816406","sampled_index":"75","source_frame":"4500","start_s":"150.14999389648438"},{"end_s":"152.18536376953125","sampled_index":"76","source_frame":"4560","start_s":"152.15199279785156"},{"end_s":"154.18736267089844","sampled_index":"77","source_frame":"4620","start_s":"154.1540069580078"},{"end_s":"156.18936157226562","sampled_index":"78","source_frame":"4680","start_s":"156.156005859375"},{"end_s":"158.1913604736328","sampled_index":"79","source_frame":"4740","start_s":"158.1580047607422"},{"end_s":"160.193359375","sampled_index":"80","source_frame":"4800","start_s":"160.16000366210938"},{"end_s":"162.19537353515625","sampled_index":"81","source_frame":"4860","start_s":"162.16200256347656"},{"end_s":"164.19737243652344","sampled_index":"82","source_frame":"4920","start_s":"164.16400146484375"},{"end_s":"166.19937133789062","sampled_index":"83","source_frame":"4980","start_s":"166.16600036621094"},{"end_s":"168.2013702392578","sampled_index":"84","source_frame":"5040","start_s":"168.16799926757812"},{"end_s":"170.203369140625","sampled_index":"85","source_frame":"5100","start_s":"170.1699981689453"},{"end_s":"172.2053680419922","sampled_index":"86","source_frame":"5160","start_s":"172.1719970703125"},{"end_s":"174.20736694335938","sampled_index":"87","source_frame":"5220","start_s":"174.1739959716797"},{"end_s":"176.20936584472656","sampled_index":"88","source_frame":"5280","start_s":"176.17599487304688"},{"end_s":"178.21136474609375","sampled_index":"89","source_frame":"5340","start_s":"178.17799377441406"},{"end_s":"180.21336364746094","sampled_index":"90","source_frame":"5400","start_s":"180.17999267578125"},{"end_s":"182.21536254882812","sampled_index":"91","source_frame":"5460","start_s":"182.1820068359375"},{"end_s":"184.2173614501953","sampled_index":"92","source_frame":"5520","start_s":"184.1840057373047"},{"end_s":"186.2193603515625","sampled_index":"93","source_frame":"5580","start_s":"186.18600463867188"},{"end_s":"188.2213592529297","sampled_index":"94","source_frame":"5640","start_s":"188.18800354003906"},{"end_s":"190.22337341308594","sampled_index":"95","source_frame":"5700","start_s":"190.19000244140625"},{"end_s":"192.22537231445312","sampled_index":"96","source_frame":"5760","start_s":"192.19200134277344"},{"end_s":"194.2273712158203","sampled_index":"97","source_frame":"5820","start_s":"194.19400024414062"},{"end_s":"196.2293701171875","sampled_index":"98","source_frame":"5880","start_s":"196.1959991455078"},{"end_s":"198.2313690185547","sampled_index":"99","source_frame":"5940","start_s":"198.197998046875"},{"end_s":"200.23336791992188","sampled_index":"100","source_frame":"6000","start_s":"200.1999969482422"},{"end_s":"202.23536682128906","sampled_index":"101","source_frame":"6060","start_s":"202.20199584960938"},{"end_s":"204.23736572265625","sampled_index":"102","source_frame":"6120","start_s":"204.20399475097656"},{"end_s":"206.23936462402344","sampled_index":"103","source_frame":"6180","start_s":"206.20599365234375"},{"end_s":"208.24136352539062","sampled_index":"104","source_frame":"6240","start_s":"208.20799255371094"},{"end_s":"210.2433624267578","sampled_index":"105","source_frame":"6300","start_s":"210.2100067138672"},{"end_s":"212.245361328125","sampled_index":"106","source_frame":"6360","start_s":"212.21200561523438"},{"end_s":"214.2473602294922","sampled_index":"107","source_frame":"6420","start_s":"214.21400451660156"},{"end_s":"216.24935913085938","sampled_index":"108","source_frame":"6480","start_s":"216.21600341796875"},{"end_s":"218.25137329101562","sampled_index":"109","source_frame":"6540","start_s":"218.21800231933594"},{"end_s":"220.2533721923828","sampled_index":"110","source_frame":"6600","start_s":"220.22000122070312"},{"end_s":"222.25537109375","sampled_index":"111","source_frame":"6660","start_s":"222.2220001220703"},{"end_s":"224.2573699951172","sampled_index":"112","source_frame":"6720","start_s":"224.2239990234375"},{"end_s":"226.25936889648438","sampled_index":"113","source_frame":"6780","start_s":"226.2259979248047"},{"end_s":"228.26136779785156","sampled_index":"114","source_frame":"6840","start_s":"228.22799682617188"},{"end_s":"230.26336669921875","sampled_index":"115","source_frame":"6900","start_s":"230.22999572753906"},{"end_s":"232.26536560058594","sampled_index":"116","source_frame":"6960","start_s":"232.23199462890625"},{"end_s":"234.26736450195312","sampled_index":"117","source_frame":"7020","start_s":"234.23399353027344"},{"end_s":"236.2693634033203","sampled_index":"118","source_frame":"7080","start_s":"236.23599243164062"},{"end_s":"238.2713623046875","sampled_index":"119","source_frame":"7140","start_s":"238.23800659179688"},{"end_s":"240.2733612060547","sampled_index":"120","source_frame":"7200","start_s":"240.24000549316406"},{"end_s":"242.27536010742188","sampled_index":"121","source_frame":"7260","start_s":"242.24200439453125"},{"end_s":"244.27737426757812","sampled_index":"122","source_frame":"7320","start_s":"244.24400329589844"},{"end_s":"246.2793731689453","sampled_index":"123","source_frame":"7380","start_s":"246.24600219726562"},{"end_s":"248.2813720703125","sampled_index":"124","source_frame":"7440","start_s":"248.2480010986328"},{"end_s":"250.2833709716797","sampled_index":"125","source_frame":"7500","start_s":"250.25"},{"end_s":"252.28536987304688","sampled_index":"126","source_frame":"7560","start_s":"252.2519989013672"},{"end_s":"254.28736877441406","sampled_index":"127","source_frame":"7620","start_s":"254.25399780273438"},{"end_s":"256.28936767578125","sampled_index":"128","source_frame":"7680","start_s":"256.2560119628906"},{"end_s":"258.2913818359375","sampled_index":"129","source_frame":"7740","start_s":"258.25799560546875"},{"end_s":"260.2933654785156","sampled_index":"130","source_frame":"7800","start_s":"260.260009765625"},{"end_s":"262.2953796386719","sampled_index":"131","source_frame":"7860","start_s":"262.2619934082031"},{"end_s":"264.29736328125","sampled_index":"132","source_frame":"7920","start_s":"264.2640075683594"},{"end_s":"266.29937744140625","sampled_index":"133","source_frame":"7980","start_s":"266.2659912109375"},{"end_s":"268.3013610839844","sampled_index":"134","source_frame":"8040","start_s":"268.26800537109375"},{"end_s":"270.3033752441406","sampled_index":"135","source_frame":"8100","start_s":"270.2699890136719"},{"end_s":"272.30535888671875","sampled_index":"136","source_frame":"8160","start_s":"272.2720031738281"},{"end_s":"274.307373046875","sampled_index":"137","source_frame":"8220","start_s":"274.27398681640625"},{"end_s":"276.3093566894531","sampled_index":"138","source_frame":"8280","start_s":"276.2760009765625"},{"end_s":"278.3113708496094","sampled_index":"139","source_frame":"8340","start_s":"278.27801513671875"},{"end_s":"280.3133544921875","sampled_index":"140","source_frame":"8400","start_s":"280.2799987792969"},{"end_s":"282.31536865234375","sampled_index":"141","source_frame":"8460","start_s":"282.2820129394531"},{"end_s":"284.3173522949219","sampled_index":"142","source_frame":"8520","start_s":"284.28399658203125"},{"end_s":"286.3193664550781","sampled_index":"143","source_frame":"8580","start_s":"286.2860107421875"},{"end_s":"288.3213806152344","sampled_index":"144","source_frame":"8640","start_s":"288.2879943847656"},{"end_s":"290.3233642578125","sampled_index":"145","source_frame":"8700","start_s":"290.2900085449219"},{"end_s":"292.32537841796875","sampled_index":"146","source_frame":"8760","start_s":"292.2919921875"},{"end_s":"294.3273620605469","sampled_index":"147","source_frame":"8820","start_s":"294.29400634765625"},{"end_s":"296.3293762207031","sampled_index":"148","source_frame":"8880","start_s":"296.2959899902344"},{"end_s":"298.33135986328125","sampled_index":"149","source_frame":"8940","start_s":"298.2980041503906"},{"end_s":"300.3333740234375","sampled_index":"150","source_frame":"9000","start_s":"300.29998779296875"},{"end_s":"302.3353576660156","sampled_index":"151","source_frame":"9060","start_s":"302.302001953125"},{"end_s":"304.3373718261719","sampled_index":"152","source_frame":"9120","start_s":"304.3039855957031"},{"end_s":"306.33935546875","sampled_index":"153","source_frame":"9180","start_s":"306.3059997558594"},{"end_s":"308.34136962890625","sampled_index":"154","source_frame":"9240","start_s":"308.3080139160156"},{"end_s":"310.3433532714844","sampled_index":"155","source_frame":"9300","start_s":"310.30999755859375"},{"end_s":"312.3453674316406","sampled_index":"156","source_frame":"9360","start_s":"312.31201171875"},{"end_s":"314.3473815917969","sampled_index":"157","source_frame":"9420","start_s":"314.3139953613281"},{"end_s":"316.349365234375","sampled_index":"158","source_frame":"9480","start_s":"316.3160095214844"},{"end_s":"318.35137939453125","sampled_index":"159","source_frame":"9540","start_s":"318.3179931640625"},{"end_s":"320.3533630371094","sampled_index":"160","source_frame":"9600","start_s":"320.32000732421875"},{"end_s":"322.3553771972656","sampled_index":"161","source_frame":"9660","start_s":"322.3219909667969"},{"end_s":"324.35736083984375","sampled_index":"162","source_frame":"9720","start_s":"324.3240051269531"},{"end_s":"326.359375","sampled_index":"163","source_frame":"9780","start_s":"326.32598876953125"},{"end_s":"328.3613586425781","sampled_index":"164","source_frame":"9840","start_s":"328.3280029296875"},{"end_s":"330.3633728027344","sampled_index":"165","source_frame":"9900","start_s":"330.3299865722656"},{"end_s":"332.3653564453125","sampled_index":"166","source_frame":"9960","start_s":"332.3320007324219"},{"end_s":"334.36737060546875","sampled_index":"167","source_frame":"10020","start_s":"334.3340148925781"},{"end_s":"336.3693542480469","sampled_index":"168","source_frame":"10080","start_s":"336.33599853515625"},{"end_s":"338.3713684082031","sampled_index":"169","source_frame":"10140","start_s":"338.3380126953125"},{"end_s":"340.37335205078125","sampled_index":"170","source_frame":"10200","start_s":"340.3399963378906"},{"end_s":"342.3753662109375","sampled_index":"171","source_frame":"10260","start_s":"342.3420104980469"},{"end_s":"344.37738037109375","sampled_index":"172","source_frame":"10320","start_s":"344.343994140625"},{"end_s":"346.3793640136719","sampled_index":"173","source_frame":"10380","start_s":"346.34600830078125"},{"end_s":"348.3813781738281","sampled_index":"174","source_frame":"10440","start_s":"348.3479919433594"},{"end_s":"350.38336181640625","sampled_index":"175","source_frame":"10500","start_s":"350.3500061035156"},{"end_s":"352.3853759765625","sampled_index":"176","source_frame":"10560","start_s":"352.35198974609375"},{"end_s":"354.3873596191406","sampled_index":"177","source_frame":"10620","start_s":"354.35400390625"},{"end_s":"356.3893737792969","sampled_index":"178","source_frame":"10680","start_s":"356.3559875488281"},{"end_s":"358.391357421875","sampled_index":"179","source_frame":"10740","start_s":"358.3580017089844"},{"end_s":"360.39337158203125","sampled_index":"180","source_frame":"10800","start_s":"360.3599853515625"},{"end_s":"362.3953552246094","sampled_index":"181","source_frame":"10860","start_s":"362.36199951171875"},{"end_s":"364.3973693847656","sampled_index":"182","source_frame":"10920","start_s":"364.364013671875"},{"end_s":"366.39935302734375","sampled_index":"183","source_frame":"10980","start_s":"366.3659973144531"},{"end_s":"368.4013671875","sampled_index":"184","source_frame":"11040","start_s":"368.3680114746094"},{"end_s":"370.40338134765625","sampled_index":"185","source_frame":"11100","start_s":"370.3699951171875"},{"end_s":"372.4053649902344","sampled_index":"186","source_frame":"11160","start_s":"372.37200927734375"},{"end_s":"374.4073791503906","sampled_index":"187","source_frame":"11220","start_s":"374.3739929199219"},{"end_s":"376.40936279296875","sampled_index":"188","source_frame":"11280","start_s":"376.3760070800781"},{"end_s":"378.411376953125","sampled_index":"189","source_frame":"11340","start_s":"378.37799072265625"},{"end_s":"380.4133605957031","sampled_index":"190","source_frame":"11400","start_s":"380.3800048828125"},{"end_s":"382.4153747558594","sampled_index":"191","source_frame":"11460","start_s":"382.3819885253906"},{"end_s":"384.4173583984375","sampled_index":"192","source_frame":"11520","start_s":"384.3840026855469"},{"end_s":"386.41937255859375","sampled_index":"193","source_frame":"11580","start_s":"386.385986328125"},{"end_s":"388.4213562011719","sampled_index":"194","source_frame":"11640","start_s":"388.38800048828125"},{"end_s":"390.4233703613281","sampled_index":"195","source_frame":"11700","start_s":"390.3900146484375"},{"end_s":"392.42535400390625","sampled_index":"196","source_frame":"11760","start_s":"392.3919982910156"},{"end_s":"394.4273681640625","sampled_index":"197","source_frame":"11820","start_s":"394.3940124511719"},{"end_s":"396.4293518066406","sampled_index":"198","source_frame":"11880","start_s":"396.39599609375"},{"end_s":"398.4313659667969","sampled_index":"199","source_frame":"11940","start_s":"398.39801025390625"},{"end_s":"400.4333801269531","sampled_index":"200","source_frame":"12000","start_s":"400.3999938964844"},{"end_s":"402.43536376953125","sampled_index":"201","source_frame":"12060","start_s":"402.4020080566406"},{"end_s":"404.4373779296875","sampled_index":"202","source_frame":"12120","start_s":"404.40399169921875"},{"end_s":"406.4393615722656","sampled_index":"203","source_frame":"12180","start_s":"406.406005859375"},{"end_s":"408.4413757324219","sampled_index":"204","source_frame":"12240","start_s":"408.4079895019531"},{"end_s":"410.443359375","sampled_index":"205","source_frame":"12300","start_s":"410.4100036621094"},{"end_s":"412.44537353515625","sampled_index":"206","source_frame":"12360","start_s":"412.4119873046875"},{"end_s":"414.4473571777344","sampled_index":"207","source_frame":"12420","start_s":"414.41400146484375"},{"end_s":"416.4493713378906","sampled_index":"208","source_frame":"12480","start_s":"416.4159851074219"},{"end_s":"418.45135498046875","sampled_index":"209","source_frame":"12540","start_s":"418.4179992675781"},{"end_s":"420.453369140625","sampled_index":"210","source_frame":"12600","start_s":"420.4200134277344"},{"end_s":"422.4553527832031","sampled_index":"211","source_frame":"12660","start_s":"422.4219970703125"},{"end_s":"424.4573669433594","sampled_index":"212","source_frame":"12720","start_s":"424.42401123046875"},{"end_s":"426.4593811035156","sampled_index":"213","source_frame":"12780","start_s":"426.4259948730469"},{"end_s":"428.46136474609375","sampled_index":"214","source_frame":"12840","start_s":"428.4280090332031"},{"end_s":"430.46337890625","sampled_index":"215","source_frame":"12900","start_s":"430.42999267578125"},{"end_s":"432.4653625488281","sampled_index":"216","source_frame":"12960","start_s":"432.4320068359375"},{"end_s":"434.4673767089844","sampled_index":"217","source_frame":"13020","start_s":"434.4339904785156"},{"end_s":"436.4693603515625","sampled_index":"218","source_frame":"13080","start_s":"436.4360046386719"},{"end_s":"438.47137451171875","sampled_index":"219","source_frame":"13140","start_s":"438.43798828125"},{"end_s":"440.4733581542969","sampled_index":"220","source_frame":"13200","start_s":"440.44000244140625"},{"end_s":"442.4753723144531","sampled_index":"221","source_frame":"13260","start_s":"442.4419860839844"},{"end_s":"444.47735595703125","sampled_index":"222","source_frame":"13320","start_s":"444.4440002441406"},{"end_s":"446.4793701171875","sampled_index":"223","source_frame":"13380","start_s":"446.4460144042969"},{"end_s":"448.4813537597656","sampled_index":"224","source_frame":"13440","start_s":"448.447998046875"},{"end_s":"450.4833679199219","sampled_index":"225","source_frame":"13500","start_s":"450.45001220703125"},{"end_s":"452.4853515625","sampled_index":"226","source_frame":"13560","start_s":"452.4519958496094"},{"end_s":"454.48736572265625","sampled_index":"227","source_frame":"13620","start_s":"454.4540100097656"},{"end_s":"456.4893798828125","sampled_index":"228","source_frame":"13680","start_s":"456.45599365234375"},{"end_s":"458.4913635253906","sampled_index":"229","source_frame":"13740","start_s":"458.4580078125"},{"end_s":"460.4933776855469","sampled_index":"230","source_frame":"13800","start_s":"460.4599914550781"},{"end_s":"462.495361328125","sampled_index":"231","source_frame":"13860","start_s":"462.4620056152344"},{"end_s":"464.49737548828125","sampled_index":"232","source_frame":"13920","start_s":"464.4639892578125"},{"end_s":"466.4993591308594","sampled_index":"233","source_frame":"13980","start_s":"466.46600341796875"},{"end_s":"468.5013732910156","sampled_index":"234","source_frame":"14040","start_s":"468.4679870605469"},{"end_s":"470.50335693359375","sampled_index":"235","source_frame":"14100","start_s":"470.4700012207031"},{"end_s":"472.50537109375","sampled_index":"236","source_frame":"14160","start_s":"472.47198486328125"},{"end_s":"474.5073547363281","sampled_index":"237","source_frame":"14220","start_s":"474.4739990234375"},{"end_s":"476.5093688964844","sampled_index":"238","source_frame":"14280","start_s":"476.47601318359375"},{"end_s":"478.5113525390625","sampled_index":"239","source_frame":"14340","start_s":"478.4779968261719"},{"end_s":"480.51336669921875","sampled_index":"240","source_frame":"14400","start_s":"480.4800109863281"},{"end_s":"482.515380859375","sampled_index":"241","source_frame":"14460","start_s":"482.48199462890625"},{"end_s":"484.5173645019531","sampled_index":"242","source_frame":"14520","start_s":"484.4840087890625"},{"end_s":"486.5193786621094","sampled_index":"243","source_frame":"14580","start_s":"486.4859924316406"},{"end_s":"488.5213623046875","sampled_index":"244","source_frame":"14640","start_s":"488.4880065917969"},{"end_s":"490.52337646484375","sampled_index":"245","source_frame":"14700","start_s":"490.489990234375"},{"end_s":"492.5253601074219","sampled_index":"246","source_frame":"14760","start_s":"492.49200439453125"},{"end_s":"494.5273742675781","sampled_index":"247","source_frame":"14820","start_s":"494.4939880371094"},{"end_s":"496.52935791015625","sampled_index":"248","source_frame":"14880","start_s":"496.4960021972656"},{"end_s":"498.5313720703125","sampled_index":"249","source_frame":"14940","start_s":"498.49798583984375"},{"end_s":"500.5333557128906","sampled_index":"250","source_frame":"15000","start_s":"500.5"},{"end_s":"502.5353698730469","sampled_index":"251","source_frame":"15060","start_s":"502.50201416015625"},{"end_s":"504.537353515625","sampled_index":"252","source_frame":"15120","start_s":"504.5039978027344"},{"end_s":"506.53936767578125","sampled_index":"253","source_frame":"15180","start_s":"506.5060119628906"},{"end_s":"508.5413818359375","sampled_index":"254","source_frame":"15240","start_s":"508.50799560546875"},{"end_s":"510.5433654785156","sampled_index":"255","source_frame":"15300","start_s":"510.510009765625"},{"end_s":"512.5453491210938","sampled_index":"256","source_frame":"15360","start_s":"512.5120239257812"},{"end_s":"514.54736328125","sampled_index":"257","source_frame":"15420","start_s":"514.5139770507812"},{"end_s":"516.5493774414062","sampled_index":"258","source_frame":"15480","start_s":"516.5159912109375"},{"end_s":"518.5513916015625","sampled_index":"259","source_frame":"15540","start_s":"518.5180053710938"},{"end_s":"520.5533447265625","sampled_index":"260","source_frame":"15600","start_s":"520.52001953125"},{"end_s":"522.5553588867188","sampled_index":"261","source_frame":"15660","start_s":"522.52197265625"},{"end_s":"524.557373046875","sampled_index":"262","source_frame":"15720","start_s":"524.5239868164062"},{"end_s":"526.5593872070312","sampled_index":"263","source_frame":"15780","start_s":"526.5260009765625"},{"end_s":"528.5613403320312","sampled_index":"264","source_frame":"15840","start_s":"528.5280151367188"},{"end_s":"530.5633544921875","sampled_index":"265","source_frame":"15900","start_s":"530.530029296875"},{"end_s":"532.5653686523438","sampled_index":"266","source_frame":"15960","start_s":"532.531982421875"},{"end_s":"534.5673828125","sampled_index":"267","source_frame":"16020","start_s":"534.5339965820312"},{"end_s":"536.5693969726562","sampled_index":"268","source_frame":"16080","start_s":"536.5360107421875"},{"end_s":"538.5713500976562","sampled_index":"269","source_frame":"16140","start_s":"538.5380249023438"},{"end_s":"540.5733642578125","sampled_index":"270","source_frame":"16200","start_s":"540.5399780273438"},{"end_s":"542.5753784179688","sampled_index":"271","source_frame":"16260","start_s":"542.5419921875"}],"video_id":125,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_212_real.mp4","video_sha256":"0dfc9943bf380053745c25d0c6448698a5ee764feb8f99c631fcb145b23768bb"},{"conversations":[{"answer":"Yellow.","choices":["Red.","Blue.","Yellow.","Purple."],"end_time":16.0,"question":"What color is the plastic block added on top of the green blocks?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"0:00:16"},{"answer":"Red, green, and yellow.","choices":["Blue, green, and yellow.","Red, green, and yellow.","Red, blue, and yellow.","Green, blue, and orange."],"end_time":62.0,"question":"What are the primary colors of the plastic building blocks shown right now?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"0:01:02"},{"answer":"An yellow block.","choices":["A white block.","A green block.","An yellow block.","A blue block."],"end_time":85.0,"question":"Which block was picked up and attached to the top of red block first just now?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"0:01:25"},{"answer":"\\"Alphabet B transform to Box\\".","choices":["\\"Alphabet A transform to Box\\".","\\"Alphabet B transform to Box\\".","\\"Alphabet C transform to Box\\".","\\"Alphabet D transform to Box\\"."],"end_time":141.0,"question":"What was the white text on the purple animated overlay read just now?","required_ability":"episodic memory","task":"Text-Rich Understanding","time_stamp":"0:02:21"},{"answer":"The blocks are scattered and then gathered into a pile.","choices":["The blocks are arranged in a new pattern.","The blocks are scattered and then gathered into a pile.","The blocks are thrown away.","The blocks are sorted by color."],"end_time":198.0,"question":"What happens to the blocks after they are disassembled and mixed?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:03:18"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"38a412df71adc5ceaf38abb7aa4f26a1b070240c8ea061f147c23d3b5f19781b","start":0,"stop":8,"tokens":1568},{"file":"chunk_0001.pt","qa_index":1,"sha256":"87faaaaee2d601b34f5bb5644a44ee09fb67728e77cd230600fcdc64c2facacc","start":8,"stop":24,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"87c0759ee97f322b4769f393565fdb53d691f0819f17eb7e8cc30afaf1638c0a","start":24,"stop":31,"tokens":1372},{"file":"chunk_0003.pt","qa_index":2,"sha256":"a0a6a1ef5d97109eddd18240987c7d0ac113120cc6f679c96e452208ff25a1b2","start":31,"stop":43,"tokens":2352},{"file":"chunk_0004.pt","qa_index":3,"sha256":"ea1d14049ff00ecda9daec7648d97509594fafc27dd0ca5eadc8a600b636fa90","start":43,"stop":59,"tokens":3136},{"file":"chunk_0005.pt","qa_index":3,"sha256":"09a87e0e3b64f7ece2fac589c3de4e5724adef298dfc6159d22073aa8d87299e","start":59,"stop":71,"tokens":2352},{"file":"chunk_0006.pt","qa_index":4,"sha256":"fe8302e3e7c5f7435f66a9549a9b8606344123323e0b121a0b0c114f05d6fac5","start":71,"stop":87,"tokens":3136},{"file":"chunk_0007.pt","qa_index":4,"sha256":"718c47eafad3dea6ba11b501821ae261d6d2ba6469e0018e1d4b3203a3dc2515","start":87,"stop":99,"tokens":2352}],"identity":"6bc722e0b6b79db11c6245073e45407c065a2ab50dc33575fcf78b4ae9023e82","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":20.287435855716467,"sample_id":204,"sampled_frames_available":130,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"c25b10db8a1c94160da55179ae7445a7a101df458ed7c15abd257a16736f9c36"},"sample_id":204,"timestamps":[{"end_s":"0.03333333507180214","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0333333015441895","sampled_index":"1","source_frame":"60","start_s":"2.0"},{"end_s":"4.0333333015441895","sampled_index":"2","source_frame":"120","start_s":"4.0"},{"end_s":"6.0333333015441895","sampled_index":"3","source_frame":"180","start_s":"6.0"},{"end_s":"8.033333778381348","sampled_index":"4","source_frame":"240","start_s":"8.0"},{"end_s":"10.033333778381348","sampled_index":"5","source_frame":"300","start_s":"10.0"},{"end_s":"12.033333778381348","sampled_index":"6","source_frame":"360","start_s":"12.0"},{"end_s":"14.033333778381348","sampled_index":"7","source_frame":"420","start_s":"14.0"},{"end_s":"16.03333282470703","sampled_index":"8","source_frame":"480","start_s":"16.0"},{"end_s":"18.03333282470703","sampled_index":"9","source_frame":"540","start_s":"18.0"},{"end_s":"20.03333282470703","sampled_index":"10","source_frame":"600","start_s":"20.0"},{"end_s":"22.03333282470703","sampled_index":"11","source_frame":"660","start_s":"22.0"},{"end_s":"24.03333282470703","sampled_index":"12","source_frame":"720","start_s":"24.0"},{"end_s":"26.03333282470703","sampled_index":"13","source_frame":"780","start_s":"26.0"},{"end_s":"28.03333282470703","sampled_index":"14","source_frame":"840","start_s":"28.0"},{"end_s":"30.03333282470703","sampled_index":"15","source_frame":"900","start_s":"30.0"},{"end_s":"32.03333282470703","sampled_index":"16","source_frame":"960","start_s":"32.0"},{"end_s":"34.03333282470703","sampled_index":"17","source_frame":"1020","start_s":"34.0"},{"end_s":"36.03333282470703","sampled_index":"18","source_frame":"1080","start_s":"36.0"},{"end_s":"38.03333282470703","sampled_index":"19","source_frame":"1140","start_s":"38.0"},{"end_s":"40.03333282470703","sampled_index":"20","source_frame":"1200","start_s":"40.0"},{"end_s":"42.03333282470703","sampled_index":"21","source_frame":"1260","start_s":"42.0"},{"end_s":"44.03333282470703","sampled_index":"22","source_frame":"1320","start_s":"44.0"},{"end_s":"46.03333282470703","sampled_index":"23","source_frame":"1380","start_s":"46.0"},{"end_s":"48.03333282470703","sampled_index":"24","source_frame":"1440","start_s":"48.0"},{"end_s":"50.03333282470703","sampled_index":"25","source_frame":"1500","start_s":"50.0"},{"end_s":"52.03333282470703","sampled_index":"26","source_frame":"1560","start_s":"52.0"},{"end_s":"54.03333282470703","sampled_index":"27","source_frame":"1620","start_s":"54.0"},{"end_s":"56.03333282470703","sampled_index":"28","source_frame":"1680","start_s":"56.0"},{"end_s":"58.03333282470703","sampled_index":"29","source_frame":"1740","start_s":"58.0"},{"end_s":"60.03333282470703","sampled_index":"30","source_frame":"1800","start_s":"60.0"},{"end_s":"62.03333282470703","sampled_index":"31","source_frame":"1860","start_s":"62.0"},{"end_s":"64.03333282470703","sampled_index":"32","source_frame":"1920","start_s":"64.0"},{"end_s":"66.03333282470703","sampled_index":"33","source_frame":"1980","start_s":"66.0"},{"end_s":"68.03333282470703","sampled_index":"34","source_frame":"2040","start_s":"68.0"},{"end_s":"70.03333282470703","sampled_index":"35","source_frame":"2100","start_s":"70.0"},{"end_s":"72.03333282470703","sampled_index":"36","source_frame":"2160","start_s":"72.0"},{"end_s":"74.03333282470703","sampled_index":"37","source_frame":"2220","start_s":"74.0"},{"end_s":"76.03333282470703","sampled_index":"38","source_frame":"2280","start_s":"76.0"},{"end_s":"78.03333282470703","sampled_index":"39","source_frame":"2340","start_s":"78.0"},{"end_s":"80.03333282470703","sampled_index":"40","source_frame":"2400","start_s":"80.0"},{"end_s":"82.03333282470703","sampled_index":"41","source_frame":"2460","start_s":"82.0"},{"end_s":"84.03333282470703","sampled_index":"42","source_frame":"2520","start_s":"84.0"},{"end_s":"86.03333282470703","sampled_index":"43","source_frame":"2580","start_s":"86.0"},{"end_s":"88.03333282470703","sampled_index":"44","source_frame":"2640","start_s":"88.0"},{"end_s":"90.03333282470703","sampled_index":"45","source_frame":"2700","start_s":"90.0"},{"end_s":"92.03333282470703","sampled_index":"46","source_frame":"2760","start_s":"92.0"},{"end_s":"94.03333282470703","sampled_index":"47","source_frame":"2820","start_s":"94.0"},{"end_s":"96.03333282470703","sampled_index":"48","source_frame":"2880","start_s":"96.0"},{"end_s":"98.03333282470703","sampled_index":"49","source_frame":"2940","start_s":"98.0"},{"end_s":"100.03333282470703","sampled_index":"50","source_frame":"3000","start_s":"100.0"},{"end_s":"102.03333282470703","sampled_index":"51","source_frame":"3060","start_s":"102.0"},{"end_s":"104.03333282470703","sampled_index":"52","source_frame":"3120","start_s":"104.0"},{"end_s":"106.03333282470703","sampled_index":"53","source_frame":"3180","start_s":"106.0"},{"end_s":"108.03333282470703","sampled_index":"54","source_frame":"3240","start_s":"108.0"},{"end_s":"110.03333282470703","sampled_index":"55","source_frame":"3300","start_s":"110.0"},{"end_s":"112.03333282470703","sampled_index":"56","source_frame":"3360","start_s":"112.0"},{"end_s":"114.03333282470703","sampled_index":"57","source_frame":"3420","start_s":"114.0"},{"end_s":"116.03333282470703","sampled_index":"58","source_frame":"3480","start_s":"116.0"},{"end_s":"118.03333282470703","sampled_index":"59","source_frame":"3540","start_s":"118.0"},{"end_s":"120.03333282470703","sampled_index":"60","source_frame":"3600","start_s":"120.0"},{"end_s":"122.03333282470703","sampled_index":"61","source_frame":"3660","start_s":"122.0"},{"end_s":"124.03333282470703","sampled_index":"62","source_frame":"3720","start_s":"124.0"},{"end_s":"126.03333282470703","sampled_index":"63","source_frame":"3780","start_s":"126.0"},{"end_s":"128.03334045410156","sampled_index":"64","source_frame":"3840","start_s":"128.0"},{"end_s":"130.03334045410156","sampled_index":"65","source_frame":"3900","start_s":"130.0"},{"end_s":"132.03334045410156","sampled_index":"66","source_frame":"3960","start_s":"132.0"},{"end_s":"134.03334045410156","sampled_index":"67","source_frame":"4020","start_s":"134.0"},{"end_s":"136.03334045410156","sampled_index":"68","source_frame":"4080","start_s":"136.0"},{"end_s":"138.03334045410156","sampled_index":"69","source_frame":"4140","start_s":"138.0"},{"end_s":"140.03334045410156","sampled_index":"70","source_frame":"4200","start_s":"140.0"},{"end_s":"142.03334045410156","sampled_index":"71","source_frame":"4260","start_s":"142.0"},{"end_s":"144.03334045410156","sampled_index":"72","source_frame":"4320","start_s":"144.0"},{"end_s":"146.03334045410156","sampled_index":"73","source_frame":"4380","start_s":"146.0"},{"end_s":"148.03334045410156","sampled_index":"74","source_frame":"4440","start_s":"148.0"},{"end_s":"150.03334045410156","sampled_index":"75","source_frame":"4500","start_s":"150.0"},{"end_s":"152.03334045410156","sampled_index":"76","source_frame":"4560","start_s":"152.0"},{"end_s":"154.03334045410156","sampled_index":"77","source_frame":"4620","start_s":"154.0"},{"end_s":"156.03334045410156","sampled_index":"78","source_frame":"4680","start_s":"156.0"},{"end_s":"158.03334045410156","sampled_index":"79","source_frame":"4740","start_s":"158.0"},{"end_s":"160.03334045410156","sampled_index":"80","source_frame":"4800","start_s":"160.0"},{"end_s":"162.03334045410156","sampled_index":"81","source_frame":"4860","start_s":"162.0"},{"end_s":"164.03334045410156","sampled_index":"82","source_frame":"4920","start_s":"164.0"},{"end_s":"166.03334045410156","sampled_index":"83","source_frame":"4980","start_s":"166.0"},{"end_s":"168.03334045410156","sampled_index":"84","source_frame":"5040","start_s":"168.0"},{"end_s":"170.03334045410156","sampled_index":"85","source_frame":"5100","start_s":"170.0"},{"end_s":"172.03334045410156","sampled_index":"86","source_frame":"5160","start_s":"172.0"},{"end_s":"174.03334045410156","sampled_index":"87","source_frame":"5220","start_s":"174.0"},{"end_s":"176.03334045410156","sampled_index":"88","source_frame":"5280","start_s":"176.0"},{"end_s":"178.03334045410156","sampled_index":"89","source_frame":"5340","start_s":"178.0"},{"end_s":"180.03334045410156","sampled_index":"90","source_frame":"5400","start_s":"180.0"},{"end_s":"182.03334045410156","sampled_index":"91","source_frame":"5460","start_s":"182.0"},{"end_s":"184.03334045410156","sampled_index":"92","source_frame":"5520","start_s":"184.0"},{"end_s":"186.03334045410156","sampled_index":"93","source_frame":"5580","start_s":"186.0"},{"end_s":"188.03334045410156","sampled_index":"94","source_frame":"5640","start_s":"188.0"},{"end_s":"190.03334045410156","sampled_index":"95","source_frame":"5700","start_s":"190.0"},{"end_s":"192.03334045410156","sampled_index":"96","source_frame":"5760","start_s":"192.0"},{"end_s":"194.03334045410156","sampled_index":"97","source_frame":"5820","start_s":"194.0"},{"end_s":"196.03334045410156","sampled_index":"98","source_frame":"5880","start_s":"196.0"},{"end_s":"198.03334045410156","sampled_index":"99","source_frame":"5940","start_s":"198.0"},{"end_s":"200.03334045410156","sampled_index":"100","source_frame":"6000","start_s":"200.0"},{"end_s":"202.03334045410156","sampled_index":"101","source_frame":"6060","start_s":"202.0"},{"end_s":"204.03334045410156","sampled_index":"102","source_frame":"6120","start_s":"204.0"},{"end_s":"206.03334045410156","sampled_index":"103","source_frame":"6180","start_s":"206.0"},{"end_s":"208.03334045410156","sampled_index":"104","source_frame":"6240","start_s":"208.0"},{"end_s":"210.03334045410156","sampled_index":"105","source_frame":"6300","start_s":"210.0"},{"end_s":"212.03334045410156","sampled_index":"106","source_frame":"6360","start_s":"212.0"},{"end_s":"214.03334045410156","sampled_index":"107","source_frame":"6420","start_s":"214.0"},{"end_s":"216.03334045410156","sampled_index":"108","source_frame":"6480","start_s":"216.0"},{"end_s":"218.03334045410156","sampled_index":"109","source_frame":"6540","start_s":"218.0"},{"end_s":"220.03334045410156","sampled_index":"110","source_frame":"6600","start_s":"220.0"},{"end_s":"222.03334045410156","sampled_index":"111","source_frame":"6660","start_s":"222.0"},{"end_s":"224.03334045410156","sampled_index":"112","source_frame":"6720","start_s":"224.0"},{"end_s":"226.03334045410156","sampled_index":"113","source_frame":"6780","start_s":"226.0"},{"end_s":"228.03334045410156","sampled_index":"114","source_frame":"6840","start_s":"228.0"},{"end_s":"230.03334045410156","sampled_index":"115","source_frame":"6900","start_s":"230.0"},{"end_s":"232.03334045410156","sampled_index":"116","source_frame":"6960","start_s":"232.0"},{"end_s":"234.03334045410156","sampled_index":"117","source_frame":"7020","start_s":"234.0"},{"end_s":"236.03334045410156","sampled_index":"118","source_frame":"7080","start_s":"236.0"},{"end_s":"238.03334045410156","sampled_index":"119","source_frame":"7140","start_s":"238.0"},{"end_s":"240.03334045410156","sampled_index":"120","source_frame":"7200","start_s":"240.0"},{"end_s":"242.03334045410156","sampled_index":"121","source_frame":"7260","start_s":"242.0"},{"end_s":"244.03334045410156","sampled_index":"122","source_frame":"7320","start_s":"244.0"},{"end_s":"246.03334045410156","sampled_index":"123","source_frame":"7380","start_s":"246.0"},{"end_s":"248.03334045410156","sampled_index":"124","source_frame":"7440","start_s":"248.0"},{"end_s":"250.03334045410156","sampled_index":"125","source_frame":"7500","start_s":"250.0"},{"end_s":"252.03334045410156","sampled_index":"126","source_frame":"7560","start_s":"252.0"},{"end_s":"254.03334045410156","sampled_index":"127","source_frame":"7620","start_s":"254.0"},{"end_s":"256.0333251953125","sampled_index":"128","source_frame":"7680","start_s":"256.0"},{"end_s":"258.0333251953125","sampled_index":"129","source_frame":"7740","start_s":"258.0"}],"video_id":335,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_204_real.mp4","video_sha256":"c25b10db8a1c94160da55179ae7445a7a101df458ed7c15abd257a16736f9c36"},{"conversations":[{"answer":"The hand adds another green block.","choices":["The hand removes the green block.","The hand moves away and does nothing.","The hand removes the yellow block.","The hand adds another green block."],"end_time":25.0,"question":"What happens after the first green block is placed on the yellow block?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:00:25"},{"answer":"Pressing down the green blocks.","choices":["Adding a yellow block.","Adding a blue block.","Adjusting the blue block.","Pressing down the green blocks."],"end_time":118.0,"question":"What action do the hands perform just before moving away from the structure?","required_ability":"episodic memory","task":"Action Recognition","time_stamp":"0:01:58"},{"answer":"The green block is placed first, followed by the yellow block.","choices":["The green block is placed first, followed by the yellow block.","The yellow block is placed first, followed by the green block.","The green block is removed, then the yellow block is placed.","The yellow block is adjusted, then the green block is placed."],"end_time":210.0,"question":"What sequence of actions occured with the green and yellow blocks just now?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:03:30"},{"answer":"In front of the original structure.","choices":["Behind the original structure.","Next to the original structure.","On top of the original structure.","In front of the original structure."],"end_time":212.0,"question":"Where is the new yellow and green block placed relative to the original structure?","required_ability":"working memory","task":"Spatial Understanding","time_stamp":"00:03:32"},{"answer":"Placed them above the green block and another yellow block.","choices":["Removed them from the structure.","Arranged them on the blue block and adjusted for alignment.","Stacked them on top of the blue blocks.","Placed them above the green block and another yellow block."],"end_time":315.0,"question":"What did the person do with the yellow block just now?","required_ability":"episodic memory","task":"Action Recognition","time_stamp":"0:05:15"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"a9dbdb25ce5d39186f1c40d5295cd45d9971b079204ea2462f0a1e8ec9170c92","start":0,"stop":13,"tokens":2548},{"file":"chunk_0001.pt","qa_index":1,"sha256":"43fe3f25d41c7789e5714dff4eeea2379848472677d999a1ea28d98c238ccf47","start":13,"stop":29,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"e07fda33fc86531b713e88a065be2c9e876d2b3423a7247b05f7ebe091d69cb1","start":29,"stop":45,"tokens":3136},{"file":"chunk_0003.pt","qa_index":1,"sha256":"e5188163d2981a97fecb1066fc34b0d517c876c91a1d2999d2a578cc75bb075c","start":45,"stop":59,"tokens":2744},{"file":"chunk_0004.pt","qa_index":2,"sha256":"3a84884b63e8b39fced9102b54f3af62678352962f1db733d37ea8ffdf42d520","start":59,"stop":75,"tokens":3136},{"file":"chunk_0005.pt","qa_index":2,"sha256":"8ba26d68e83a7a55d9a2356cdfec749e06ec3751fe47c6946127a5e0dff489b4","start":75,"stop":91,"tokens":3136},{"file":"chunk_0006.pt","qa_index":2,"sha256":"acb31f9c9dfb40d8a945c13d937c1f6ca72bab373c61bf2e080ab4d06f01a28d","start":91,"stop":105,"tokens":2744},{"file":"chunk_0007.pt","qa_index":3,"sha256":"5807282ec5aa84667dea97147fbf2f83ec4005d8b49a6425bf953a8c53df86a6","start":105,"stop":106,"tokens":196},{"file":"chunk_0008.pt","qa_index":4,"sha256":"1d1f617bb484ae55c37f4554a15b25432beb834fc84a898b301c606aaa32b32b","start":106,"stop":122,"tokens":3136},{"file":"chunk_0009.pt","qa_index":4,"sha256":"928f87c77f3e4e8d47f4b3b5ca5a231bd99bfb6d8e7faa7d6c6e9ae3b7b3963e","start":122,"stop":138,"tokens":3136},{"file":"chunk_0010.pt","qa_index":4,"sha256":"b17ef5366de51913e882412f08fd86f00b3fce589a2960b65f3bd1fd58d3459d","start":138,"stop":154,"tokens":3136},{"file":"chunk_0011.pt","qa_index":4,"sha256":"c166802170291c018742f1c45f9af64af7fbaaee3385a1e8834d3a484a8ee6fe","start":154,"stop":158,"tokens":784}],"identity":"7be10708f7fce429661df0f0d320349463aca94eccae03a9bb83b424a098791f","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":31.23981400579214,"sample_id":209,"sampled_frames_available":188,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"d50664402feff2ac1fdf008eec52fc01d9b6645303132339410b8ffec353ed8a"},"sample_id":209,"timestamps":[{"end_s":"0.03333333507180214","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0333333015441895","sampled_index":"1","source_frame":"60","start_s":"2.0"},{"end_s":"4.0333333015441895","sampled_index":"2","source_frame":"120","start_s":"4.0"},{"end_s":"6.0333333015441895","sampled_index":"3","source_frame":"180","start_s":"6.0"},{"end_s":"8.033333778381348","sampled_index":"4","source_frame":"240","start_s":"8.0"},{"end_s":"10.033333778381348","sampled_index":"5","source_frame":"300","start_s":"10.0"},{"end_s":"12.033333778381348","sampled_index":"6","source_frame":"360","start_s":"12.0"},{"end_s":"14.033333778381348","sampled_index":"7","source_frame":"420","start_s":"14.0"},{"end_s":"16.03333282470703","sampled_index":"8","source_frame":"480","start_s":"16.0"},{"end_s":"18.03333282470703","sampled_index":"9","source_frame":"540","start_s":"18.0"},{"end_s":"20.03333282470703","sampled_index":"10","source_frame":"600","start_s":"20.0"},{"end_s":"22.03333282470703","sampled_index":"11","source_frame":"660","start_s":"22.0"},{"end_s":"24.03333282470703","sampled_index":"12","source_frame":"720","start_s":"24.0"},{"end_s":"26.03333282470703","sampled_index":"13","source_frame":"780","start_s":"26.0"},{"end_s":"28.03333282470703","sampled_index":"14","source_frame":"840","start_s":"28.0"},{"end_s":"30.03333282470703","sampled_index":"15","source_frame":"900","start_s":"30.0"},{"end_s":"32.03333282470703","sampled_index":"16","source_frame":"960","start_s":"32.0"},{"end_s":"34.03333282470703","sampled_index":"17","source_frame":"1020","start_s":"34.0"},{"end_s":"36.03333282470703","sampled_index":"18","source_frame":"1080","start_s":"36.0"},{"end_s":"38.03333282470703","sampled_index":"19","source_frame":"1140","start_s":"38.0"},{"end_s":"40.03333282470703","sampled_index":"20","source_frame":"1200","start_s":"40.0"},{"end_s":"42.03333282470703","sampled_index":"21","source_frame":"1260","start_s":"42.0"},{"end_s":"44.03333282470703","sampled_index":"22","source_frame":"1320","start_s":"44.0"},{"end_s":"46.03333282470703","sampled_index":"23","source_frame":"1380","start_s":"46.0"},{"end_s":"48.03333282470703","sampled_index":"24","source_frame":"1440","start_s":"48.0"},{"end_s":"50.03333282470703","sampled_index":"25","source_frame":"1500","start_s":"50.0"},{"end_s":"52.03333282470703","sampled_index":"26","source_frame":"1560","start_s":"52.0"},{"end_s":"54.03333282470703","sampled_index":"27","source_frame":"1620","start_s":"54.0"},{"end_s":"56.03333282470703","sampled_index":"28","source_frame":"1680","start_s":"56.0"},{"end_s":"58.03333282470703","sampled_index":"29","source_frame":"1740","start_s":"58.0"},{"end_s":"60.03333282470703","sampled_index":"30","source_frame":"1800","start_s":"60.0"},{"end_s":"62.03333282470703","sampled_index":"31","source_frame":"1860","start_s":"62.0"},{"end_s":"64.03333282470703","sampled_index":"32","source_frame":"1920","start_s":"64.0"},{"end_s":"66.03333282470703","sampled_index":"33","source_frame":"1980","start_s":"66.0"},{"end_s":"68.03333282470703","sampled_index":"34","source_frame":"2040","start_s":"68.0"},{"end_s":"70.03333282470703","sampled_index":"35","source_frame":"2100","start_s":"70.0"},{"end_s":"72.03333282470703","sampled_index":"36","source_frame":"2160","start_s":"72.0"},{"end_s":"74.03333282470703","sampled_index":"37","source_frame":"2220","start_s":"74.0"},{"end_s":"76.03333282470703","sampled_index":"38","source_frame":"2280","start_s":"76.0"},{"end_s":"78.03333282470703","sampled_index":"39","source_frame":"2340","start_s":"78.0"},{"end_s":"80.03333282470703","sampled_index":"40","source_frame":"2400","start_s":"80.0"},{"end_s":"82.03333282470703","sampled_index":"41","source_frame":"2460","start_s":"82.0"},{"end_s":"84.03333282470703","sampled_index":"42","source_frame":"2520","start_s":"84.0"},{"end_s":"86.03333282470703","sampled_index":"43","source_frame":"2580","start_s":"86.0"},{"end_s":"88.03333282470703","sampled_index":"44","source_frame":"2640","start_s":"88.0"},{"end_s":"90.03333282470703","sampled_index":"45","source_frame":"2700","start_s":"90.0"},{"end_s":"92.03333282470703","sampled_index":"46","source_frame":"2760","start_s":"92.0"},{"end_s":"94.03333282470703","sampled_index":"47","source_frame":"2820","start_s":"94.0"},{"end_s":"96.03333282470703","sampled_index":"48","source_frame":"2880","start_s":"96.0"},{"end_s":"98.03333282470703","sampled_index":"49","source_frame":"2940","start_s":"98.0"},{"end_s":"100.03333282470703","sampled_index":"50","source_frame":"3000","start_s":"100.0"},{"end_s":"102.03333282470703","sampled_index":"51","source_frame":"3060","start_s":"102.0"},{"end_s":"104.03333282470703","sampled_index":"52","source_frame":"3120","start_s":"104.0"},{"end_s":"106.03333282470703","sampled_index":"53","source_frame":"3180","start_s":"106.0"},{"end_s":"108.03333282470703","sampled_index":"54","source_frame":"3240","start_s":"108.0"},{"end_s":"110.03333282470703","sampled_index":"55","source_frame":"3300","start_s":"110.0"},{"end_s":"112.03333282470703","sampled_index":"56","source_frame":"3360","start_s":"112.0"},{"end_s":"114.03333282470703","sampled_index":"57","source_frame":"3420","start_s":"114.0"},{"end_s":"116.03333282470703","sampled_index":"58","source_frame":"3480","start_s":"116.0"},{"end_s":"118.03333282470703","sampled_index":"59","source_frame":"3540","start_s":"118.0"},{"end_s":"120.03333282470703","sampled_index":"60","source_frame":"3600","start_s":"120.0"},{"end_s":"122.03333282470703","sampled_index":"61","source_frame":"3660","start_s":"122.0"},{"end_s":"124.03333282470703","sampled_index":"62","source_frame":"3720","start_s":"124.0"},{"end_s":"126.03333282470703","sampled_index":"63","source_frame":"3780","start_s":"126.0"},{"end_s":"128.03334045410156","sampled_index":"64","source_frame":"3840","start_s":"128.0"},{"end_s":"130.03334045410156","sampled_index":"65","source_frame":"3900","start_s":"130.0"},{"end_s":"132.03334045410156","sampled_index":"66","source_frame":"3960","start_s":"132.0"},{"end_s":"134.03334045410156","sampled_index":"67","source_frame":"4020","start_s":"134.0"},{"end_s":"136.03334045410156","sampled_index":"68","source_frame":"4080","start_s":"136.0"},{"end_s":"138.03334045410156","sampled_index":"69","source_frame":"4140","start_s":"138.0"},{"end_s":"140.03334045410156","sampled_index":"70","source_frame":"4200","start_s":"140.0"},{"end_s":"142.03334045410156","sampled_index":"71","source_frame":"4260","start_s":"142.0"},{"end_s":"144.03334045410156","sampled_index":"72","source_frame":"4320","start_s":"144.0"},{"end_s":"146.03334045410156","sampled_index":"73","source_frame":"4380","start_s":"146.0"},{"end_s":"148.03334045410156","sampled_index":"74","source_frame":"4440","start_s":"148.0"},{"end_s":"150.03334045410156","sampled_index":"75","source_frame":"4500","start_s":"150.0"},{"end_s":"152.03334045410156","sampled_index":"76","source_frame":"4560","start_s":"152.0"},{"end_s":"154.03334045410156","sampled_index":"77","source_frame":"4620","start_s":"154.0"},{"end_s":"156.03334045410156","sampled_index":"78","source_frame":"4680","start_s":"156.0"},{"end_s":"158.03334045410156","sampled_index":"79","source_frame":"4740","start_s":"158.0"},{"end_s":"160.03334045410156","sampled_index":"80","source_frame":"4800","start_s":"160.0"},{"end_s":"162.03334045410156","sampled_index":"81","source_frame":"4860","start_s":"162.0"},{"end_s":"164.03334045410156","sampled_index":"82","source_frame":"4920","start_s":"164.0"},{"end_s":"166.03334045410156","sampled_index":"83","source_frame":"4980","start_s":"166.0"},{"end_s":"168.03334045410156","sampled_index":"84","source_frame":"5040","start_s":"168.0"},{"end_s":"170.03334045410156","sampled_index":"85","source_frame":"5100","start_s":"170.0"},{"end_s":"172.03334045410156","sampled_index":"86","source_frame":"5160","start_s":"172.0"},{"end_s":"174.03334045410156","sampled_index":"87","source_frame":"5220","start_s":"174.0"},{"end_s":"176.03334045410156","sampled_index":"88","source_frame":"5280","start_s":"176.0"},{"end_s":"178.03334045410156","sampled_index":"89","source_frame":"5340","start_s":"178.0"},{"end_s":"180.03334045410156","sampled_index":"90","source_frame":"5400","start_s":"180.0"},{"end_s":"182.03334045410156","sampled_index":"91","source_frame":"5460","start_s":"182.0"},{"end_s":"184.03334045410156","sampled_index":"92","source_frame":"5520","start_s":"184.0"},{"end_s":"186.03334045410156","sampled_index":"93","source_frame":"5580","start_s":"186.0"},{"end_s":"188.03334045410156","sampled_index":"94","source_frame":"5640","start_s":"188.0"},{"end_s":"190.03334045410156","sampled_index":"95","source_frame":"5700","start_s":"190.0"},{"end_s":"192.03334045410156","sampled_index":"96","source_frame":"5760","start_s":"192.0"},{"end_s":"194.03334045410156","sampled_index":"97","source_frame":"5820","start_s":"194.0"},{"end_s":"196.03334045410156","sampled_index":"98","source_frame":"5880","start_s":"196.0"},{"end_s":"198.03334045410156","sampled_index":"99","source_frame":"5940","start_s":"198.0"},{"end_s":"200.03334045410156","sampled_index":"100","source_frame":"6000","start_s":"200.0"},{"end_s":"202.03334045410156","sampled_index":"101","source_frame":"6060","start_s":"202.0"},{"end_s":"204.03334045410156","sampled_index":"102","source_frame":"6120","start_s":"204.0"},{"end_s":"206.03334045410156","sampled_index":"103","source_frame":"6180","start_s":"206.0"},{"end_s":"208.03334045410156","sampled_index":"104","source_frame":"6240","start_s":"208.0"},{"end_s":"210.03334045410156","sampled_index":"105","source_frame":"6300","start_s":"210.0"},{"end_s":"212.03334045410156","sampled_index":"106","source_frame":"6360","start_s":"212.0"},{"end_s":"214.03334045410156","sampled_index":"107","source_frame":"6420","start_s":"214.0"},{"end_s":"216.03334045410156","sampled_index":"108","source_frame":"6480","start_s":"216.0"},{"end_s":"218.03334045410156","sampled_index":"109","source_frame":"6540","start_s":"218.0"},{"end_s":"220.03334045410156","sampled_index":"110","source_frame":"6600","start_s":"220.0"},{"end_s":"222.03334045410156","sampled_index":"111","source_frame":"6660","start_s":"222.0"},{"end_s":"224.03334045410156","sampled_index":"112","source_frame":"6720","start_s":"224.0"},{"end_s":"226.03334045410156","sampled_index":"113","source_frame":"6780","start_s":"226.0"},{"end_s":"228.03334045410156","sampled_index":"114","source_frame":"6840","start_s":"228.0"},{"end_s":"230.03334045410156","sampled_index":"115","source_frame":"6900","start_s":"230.0"},{"end_s":"232.03334045410156","sampled_index":"116","source_frame":"6960","start_s":"232.0"},{"end_s":"234.03334045410156","sampled_index":"117","source_frame":"7020","start_s":"234.0"},{"end_s":"236.03334045410156","sampled_index":"118","source_frame":"7080","start_s":"236.0"},{"end_s":"238.03334045410156","sampled_index":"119","source_frame":"7140","start_s":"238.0"},{"end_s":"240.03334045410156","sampled_index":"120","source_frame":"7200","start_s":"240.0"},{"end_s":"242.03334045410156","sampled_index":"121","source_frame":"7260","start_s":"242.0"},{"end_s":"244.03334045410156","sampled_index":"122","source_frame":"7320","start_s":"244.0"},{"end_s":"246.03334045410156","sampled_index":"123","source_frame":"7380","start_s":"246.0"},{"end_s":"248.03334045410156","sampled_index":"124","source_frame":"7440","start_s":"248.0"},{"end_s":"250.03334045410156","sampled_index":"125","source_frame":"7500","start_s":"250.0"},{"end_s":"252.03334045410156","sampled_index":"126","source_frame":"7560","start_s":"252.0"},{"end_s":"254.03334045410156","sampled_index":"127","source_frame":"7620","start_s":"254.0"},{"end_s":"256.0333251953125","sampled_index":"128","source_frame":"7680","start_s":"256.0"},{"end_s":"258.0333251953125","sampled_index":"129","source_frame":"7740","start_s":"258.0"},{"end_s":"260.0333251953125","sampled_index":"130","source_frame":"7800","start_s":"260.0"},{"end_s":"262.0333251953125","sampled_index":"131","source_frame":"7860","start_s":"262.0"},{"end_s":"264.0333251953125","sampled_index":"132","source_frame":"7920","start_s":"264.0"},{"end_s":"266.0333251953125","sampled_index":"133","source_frame":"7980","start_s":"266.0"},{"end_s":"268.0333251953125","sampled_index":"134","source_frame":"8040","start_s":"268.0"},{"end_s":"270.0333251953125","sampled_index":"135","source_frame":"8100","start_s":"270.0"},{"end_s":"272.0333251953125","sampled_index":"136","source_frame":"8160","start_s":"272.0"},{"end_s":"274.0333251953125","sampled_index":"137","source_frame":"8220","start_s":"274.0"},{"end_s":"276.0333251953125","sampled_index":"138","source_frame":"8280","start_s":"276.0"},{"end_s":"278.0333251953125","sampled_index":"139","source_frame":"8340","start_s":"278.0"},{"end_s":"280.0333251953125","sampled_index":"140","source_frame":"8400","start_s":"280.0"},{"end_s":"282.0333251953125","sampled_index":"141","source_frame":"8460","start_s":"282.0"},{"end_s":"284.0333251953125","sampled_index":"142","source_frame":"8520","start_s":"284.0"},{"end_s":"286.0333251953125","sampled_index":"143","source_frame":"8580","start_s":"286.0"},{"end_s":"288.0333251953125","sampled_index":"144","source_frame":"8640","start_s":"288.0"},{"end_s":"290.0333251953125","sampled_index":"145","source_frame":"8700","start_s":"290.0"},{"end_s":"292.0333251953125","sampled_index":"146","source_frame":"8760","start_s":"292.0"},{"end_s":"294.0333251953125","sampled_index":"147","source_frame":"8820","start_s":"294.0"},{"end_s":"296.0333251953125","sampled_index":"148","source_frame":"8880","start_s":"296.0"},{"end_s":"298.0333251953125","sampled_index":"149","source_frame":"8940","start_s":"298.0"},{"end_s":"300.0333251953125","sampled_index":"150","source_frame":"9000","start_s":"300.0"},{"end_s":"302.0333251953125","sampled_index":"151","source_frame":"9060","start_s":"302.0"},{"end_s":"304.0333251953125","sampled_index":"152","source_frame":"9120","start_s":"304.0"},{"end_s":"306.0333251953125","sampled_index":"153","source_frame":"9180","start_s":"306.0"},{"end_s":"308.0333251953125","sampled_index":"154","source_frame":"9240","start_s":"308.0"},{"end_s":"310.0333251953125","sampled_index":"155","source_frame":"9300","start_s":"310.0"},{"end_s":"312.0333251953125","sampled_index":"156","source_frame":"9360","start_s":"312.0"},{"end_s":"314.0333251953125","sampled_index":"157","source_frame":"9420","start_s":"314.0"},{"end_s":"316.0333251953125","sampled_index":"158","source_frame":"9480","start_s":"316.0"},{"end_s":"318.0333251953125","sampled_index":"159","source_frame":"9540","start_s":"318.0"},{"end_s":"320.0333251953125","sampled_index":"160","source_frame":"9600","start_s":"320.0"},{"end_s":"322.0333251953125","sampled_index":"161","source_frame":"9660","start_s":"322.0"},{"end_s":"324.0333251953125","sampled_index":"162","source_frame":"9720","start_s":"324.0"},{"end_s":"326.0333251953125","sampled_index":"163","source_frame":"9780","start_s":"326.0"},{"end_s":"328.0333251953125","sampled_index":"164","source_frame":"9840","start_s":"328.0"},{"end_s":"330.0333251953125","sampled_index":"165","source_frame":"9900","start_s":"330.0"},{"end_s":"332.0333251953125","sampled_index":"166","source_frame":"9960","start_s":"332.0"},{"end_s":"334.0333251953125","sampled_index":"167","source_frame":"10020","start_s":"334.0"},{"end_s":"336.0333251953125","sampled_index":"168","source_frame":"10080","start_s":"336.0"},{"end_s":"338.0333251953125","sampled_index":"169","source_frame":"10140","start_s":"338.0"},{"end_s":"340.0333251953125","sampled_index":"170","source_frame":"10200","start_s":"340.0"},{"end_s":"342.0333251953125","sampled_index":"171","source_frame":"10260","start_s":"342.0"},{"end_s":"344.0333251953125","sampled_index":"172","source_frame":"10320","start_s":"344.0"},{"end_s":"346.0333251953125","sampled_index":"173","source_frame":"10380","start_s":"346.0"},{"end_s":"348.0333251953125","sampled_index":"174","source_frame":"10440","start_s":"348.0"},{"end_s":"350.0333251953125","sampled_index":"175","source_frame":"10500","start_s":"350.0"},{"end_s":"352.0333251953125","sampled_index":"176","source_frame":"10560","start_s":"352.0"},{"end_s":"354.0333251953125","sampled_index":"177","source_frame":"10620","start_s":"354.0"},{"end_s":"356.0333251953125","sampled_index":"178","source_frame":"10680","start_s":"356.0"},{"end_s":"358.0333251953125","sampled_index":"179","source_frame":"10740","start_s":"358.0"},{"end_s":"360.0333251953125","sampled_index":"180","source_frame":"10800","start_s":"360.0"},{"end_s":"362.0333251953125","sampled_index":"181","source_frame":"10860","start_s":"362.0"},{"end_s":"364.0333251953125","sampled_index":"182","source_frame":"10920","start_s":"364.0"},{"end_s":"366.0333251953125","sampled_index":"183","source_frame":"10980","start_s":"366.0"},{"end_s":"368.0333251953125","sampled_index":"184","source_frame":"11040","start_s":"368.0"},{"end_s":"370.0333251953125","sampled_index":"185","source_frame":"11100","start_s":"370.0"},{"end_s":"372.0333251953125","sampled_index":"186","source_frame":"11160","start_s":"372.0"},{"end_s":"374.0333251953125","sampled_index":"187","source_frame":"11220","start_s":"374.0"}],"video_id":118,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_209_real.mp4","video_sha256":"d50664402feff2ac1fdf008eec52fc01d9b6645303132339410b8ffec353ed8a"},{"conversations":[{"answer":"Eight.","choices":["Eight.","Sixteen.","Four.","Twelve."],"end_time":15.0,"question":"How many studs does each yellow interlocking block placed by the hand have?","required_ability":"working memory","task":"Counting","time_stamp":"00:00:15"},{"answer":"Four.","choices":["Three.","Five.","Seven.","Four."],"end_time":19.0,"question":"How many interlocking blocks are positioned in a straight line by the hand?","required_ability":"working memory","task":"Counting","time_stamp":"00:00:19"},{"answer":"Green.","choices":["Yellow.","Orange.","Green.","Blue."],"end_time":76.0,"question":"What is the color of the block that the hand is reaching for right now?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"00:01:16"},{"answer":"Green.","choices":["Green.","Yellow.","Orange.","Red."],"end_time":134.0,"question":"What color is the brick first added after the initial blue bricks form a two-layered structure?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"0:02:14"},{"answer":"Green, yellow, orange.","choices":["Green, yellow, orange.","Yellow, green, orange.","Orange, yellow, green.","Green, orange, yellow."],"end_time":141.0,"question":"What is the sequence of bricks added to the structure after the blue bricks?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"0:02:21"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"120bfde2058eda7f83d7b3cb4a9922312ad57d64600f068994ee396f92fa82be","start":0,"stop":8,"tokens":1568},{"file":"chunk_0001.pt","qa_index":1,"sha256":"b97913f091de97aca961379548c2b1055eb42193762a5988047a0bfb46b0fdd8","start":8,"stop":10,"tokens":392},{"file":"chunk_0002.pt","qa_index":2,"sha256":"71e7ff6b674d1296f98c4e7e683450ac512e09a5517957f07ea28bb22751e355","start":10,"stop":26,"tokens":3136},{"file":"chunk_0003.pt","qa_index":2,"sha256":"4a6dd8676689d1a97b1518be93f410b2a627422f4e8b87dd7ef929da9f9b90f9","start":26,"stop":38,"tokens":2352},{"file":"chunk_0004.pt","qa_index":3,"sha256":"e61883c3a95f6dac5e2dab7a7236d54c6e566e0d87804951a3548a6195b9d0e1","start":38,"stop":54,"tokens":3136},{"file":"chunk_0005.pt","qa_index":3,"sha256":"2be9b23bbb590a0a761f78f1b4659e1576a9b760e52d06c3405118b614526acd","start":54,"stop":67,"tokens":2548},{"file":"chunk_0006.pt","qa_index":4,"sha256":"e62ec33f65e8382d509116b0362677aabcbb4c6e3d38b51e8437a2449a6ab995","start":67,"stop":71,"tokens":784}],"identity":"8dc78f59bb7858140da988973a06609be3fb825a4e0295770a8169882e2d3550","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":13.500172331929207,"sample_id":203,"sampled_frames_available":101,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"d63f30dbc07081c58b68f1fe95ae2ce130e723cfdf6fedf068a2547662609ed0"},"sample_id":203,"timestamps":[{"end_s":"0.03333333507180214","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0333333015441895","sampled_index":"1","source_frame":"60","start_s":"2.0"},{"end_s":"4.0333333015441895","sampled_index":"2","source_frame":"120","start_s":"4.0"},{"end_s":"6.0333333015441895","sampled_index":"3","source_frame":"180","start_s":"6.0"},{"end_s":"8.033333778381348","sampled_index":"4","source_frame":"240","start_s":"8.0"},{"end_s":"10.033333778381348","sampled_index":"5","source_frame":"300","start_s":"10.0"},{"end_s":"12.033333778381348","sampled_index":"6","source_frame":"360","start_s":"12.0"},{"end_s":"14.033333778381348","sampled_index":"7","source_frame":"420","start_s":"14.0"},{"end_s":"16.03333282470703","sampled_index":"8","source_frame":"480","start_s":"16.0"},{"end_s":"18.03333282470703","sampled_index":"9","source_frame":"540","start_s":"18.0"},{"end_s":"20.03333282470703","sampled_index":"10","source_frame":"600","start_s":"20.0"},{"end_s":"22.03333282470703","sampled_index":"11","source_frame":"660","start_s":"22.0"},{"end_s":"24.03333282470703","sampled_index":"12","source_frame":"720","start_s":"24.0"},{"end_s":"26.03333282470703","sampled_index":"13","source_frame":"780","start_s":"26.0"},{"end_s":"28.03333282470703","sampled_index":"14","source_frame":"840","start_s":"28.0"},{"end_s":"30.03333282470703","sampled_index":"15","source_frame":"900","start_s":"30.0"},{"end_s":"32.03333282470703","sampled_index":"16","source_frame":"960","start_s":"32.0"},{"end_s":"34.03333282470703","sampled_index":"17","source_frame":"1020","start_s":"34.0"},{"end_s":"36.03333282470703","sampled_index":"18","source_frame":"1080","start_s":"36.0"},{"end_s":"38.03333282470703","sampled_index":"19","source_frame":"1140","start_s":"38.0"},{"end_s":"40.03333282470703","sampled_index":"20","source_frame":"1200","start_s":"40.0"},{"end_s":"42.03333282470703","sampled_index":"21","source_frame":"1260","start_s":"42.0"},{"end_s":"44.03333282470703","sampled_index":"22","source_frame":"1320","start_s":"44.0"},{"end_s":"46.03333282470703","sampled_index":"23","source_frame":"1380","start_s":"46.0"},{"end_s":"48.03333282470703","sampled_index":"24","source_frame":"1440","start_s":"48.0"},{"end_s":"50.03333282470703","sampled_index":"25","source_frame":"1500","start_s":"50.0"},{"end_s":"52.03333282470703","sampled_index":"26","source_frame":"1560","start_s":"52.0"},{"end_s":"54.03333282470703","sampled_index":"27","source_frame":"1620","start_s":"54.0"},{"end_s":"56.03333282470703","sampled_index":"28","source_frame":"1680","start_s":"56.0"},{"end_s":"58.03333282470703","sampled_index":"29","source_frame":"1740","start_s":"58.0"},{"end_s":"60.03333282470703","sampled_index":"30","source_frame":"1800","start_s":"60.0"},{"end_s":"62.03333282470703","sampled_index":"31","source_frame":"1860","start_s":"62.0"},{"end_s":"64.03333282470703","sampled_index":"32","source_frame":"1920","start_s":"64.0"},{"end_s":"66.03333282470703","sampled_index":"33","source_frame":"1980","start_s":"66.0"},{"end_s":"68.03333282470703","sampled_index":"34","source_frame":"2040","start_s":"68.0"},{"end_s":"70.03333282470703","sampled_index":"35","source_frame":"2100","start_s":"70.0"},{"end_s":"72.03333282470703","sampled_index":"36","source_frame":"2160","start_s":"72.0"},{"end_s":"74.03333282470703","sampled_index":"37","source_frame":"2220","start_s":"74.0"},{"end_s":"76.03333282470703","sampled_index":"38","source_frame":"2280","start_s":"76.0"},{"end_s":"78.03333282470703","sampled_index":"39","source_frame":"2340","start_s":"78.0"},{"end_s":"80.03333282470703","sampled_index":"40","source_frame":"2400","start_s":"80.0"},{"end_s":"82.03333282470703","sampled_index":"41","source_frame":"2460","start_s":"82.0"},{"end_s":"84.03333282470703","sampled_index":"42","source_frame":"2520","start_s":"84.0"},{"end_s":"86.03333282470703","sampled_index":"43","source_frame":"2580","start_s":"86.0"},{"end_s":"88.03333282470703","sampled_index":"44","source_frame":"2640","start_s":"88.0"},{"end_s":"90.03333282470703","sampled_index":"45","source_frame":"2700","start_s":"90.0"},{"end_s":"92.03333282470703","sampled_index":"46","source_frame":"2760","start_s":"92.0"},{"end_s":"94.03333282470703","sampled_index":"47","source_frame":"2820","start_s":"94.0"},{"end_s":"96.03333282470703","sampled_index":"48","source_frame":"2880","start_s":"96.0"},{"end_s":"98.03333282470703","sampled_index":"49","source_frame":"2940","start_s":"98.0"},{"end_s":"100.03333282470703","sampled_index":"50","source_frame":"3000","start_s":"100.0"},{"end_s":"102.03333282470703","sampled_index":"51","source_frame":"3060","start_s":"102.0"},{"end_s":"104.03333282470703","sampled_index":"52","source_frame":"3120","start_s":"104.0"},{"end_s":"106.03333282470703","sampled_index":"53","source_frame":"3180","start_s":"106.0"},{"end_s":"108.03333282470703","sampled_index":"54","source_frame":"3240","start_s":"108.0"},{"end_s":"110.03333282470703","sampled_index":"55","source_frame":"3300","start_s":"110.0"},{"end_s":"112.03333282470703","sampled_index":"56","source_frame":"3360","start_s":"112.0"},{"end_s":"114.03333282470703","sampled_index":"57","source_frame":"3420","start_s":"114.0"},{"end_s":"116.03333282470703","sampled_index":"58","source_frame":"3480","start_s":"116.0"},{"end_s":"118.03333282470703","sampled_index":"59","source_frame":"3540","start_s":"118.0"},{"end_s":"120.03333282470703","sampled_index":"60","source_frame":"3600","start_s":"120.0"},{"end_s":"122.03333282470703","sampled_index":"61","source_frame":"3660","start_s":"122.0"},{"end_s":"124.03333282470703","sampled_index":"62","source_frame":"3720","start_s":"124.0"},{"end_s":"126.03333282470703","sampled_index":"63","source_frame":"3780","start_s":"126.0"},{"end_s":"128.03334045410156","sampled_index":"64","source_frame":"3840","start_s":"128.0"},{"end_s":"130.03334045410156","sampled_index":"65","source_frame":"3900","start_s":"130.0"},{"end_s":"132.03334045410156","sampled_index":"66","source_frame":"3960","start_s":"132.0"},{"end_s":"134.03334045410156","sampled_index":"67","source_frame":"4020","start_s":"134.0"},{"end_s":"136.03334045410156","sampled_index":"68","source_frame":"4080","start_s":"136.0"},{"end_s":"138.03334045410156","sampled_index":"69","source_frame":"4140","start_s":"138.0"},{"end_s":"140.03334045410156","sampled_index":"70","source_frame":"4200","start_s":"140.0"},{"end_s":"142.03334045410156","sampled_index":"71","source_frame":"4260","start_s":"142.0"},{"end_s":"144.03334045410156","sampled_index":"72","source_frame":"4320","start_s":"144.0"},{"end_s":"146.03334045410156","sampled_index":"73","source_frame":"4380","start_s":"146.0"},{"end_s":"148.03334045410156","sampled_index":"74","source_frame":"4440","start_s":"148.0"},{"end_s":"150.03334045410156","sampled_index":"75","source_frame":"4500","start_s":"150.0"},{"end_s":"152.03334045410156","sampled_index":"76","source_frame":"4560","start_s":"152.0"},{"end_s":"154.03334045410156","sampled_index":"77","source_frame":"4620","start_s":"154.0"},{"end_s":"156.03334045410156","sampled_index":"78","source_frame":"4680","start_s":"156.0"},{"end_s":"158.03334045410156","sampled_index":"79","source_frame":"4740","start_s":"158.0"},{"end_s":"160.03334045410156","sampled_index":"80","source_frame":"4800","start_s":"160.0"},{"end_s":"162.03334045410156","sampled_index":"81","source_frame":"4860","start_s":"162.0"},{"end_s":"164.03334045410156","sampled_index":"82","source_frame":"4920","start_s":"164.0"},{"end_s":"166.03334045410156","sampled_index":"83","source_frame":"4980","start_s":"166.0"},{"end_s":"168.03334045410156","sampled_index":"84","source_frame":"5040","start_s":"168.0"},{"end_s":"170.03334045410156","sampled_index":"85","source_frame":"5100","start_s":"170.0"},{"end_s":"172.03334045410156","sampled_index":"86","source_frame":"5160","start_s":"172.0"},{"end_s":"174.03334045410156","sampled_index":"87","source_frame":"5220","start_s":"174.0"},{"end_s":"176.03334045410156","sampled_index":"88","source_frame":"5280","start_s":"176.0"},{"end_s":"178.03334045410156","sampled_index":"89","source_frame":"5340","start_s":"178.0"},{"end_s":"180.03334045410156","sampled_index":"90","source_frame":"5400","start_s":"180.0"},{"end_s":"182.03334045410156","sampled_index":"91","source_frame":"5460","start_s":"182.0"},{"end_s":"184.03334045410156","sampled_index":"92","source_frame":"5520","start_s":"184.0"},{"end_s":"186.03334045410156","sampled_index":"93","source_frame":"5580","start_s":"186.0"},{"end_s":"188.03334045410156","sampled_index":"94","source_frame":"5640","start_s":"188.0"},{"end_s":"190.03334045410156","sampled_index":"95","source_frame":"5700","start_s":"190.0"},{"end_s":"192.03334045410156","sampled_index":"96","source_frame":"5760","start_s":"192.0"},{"end_s":"194.03334045410156","sampled_index":"97","source_frame":"5820","start_s":"194.0"},{"end_s":"196.03334045410156","sampled_index":"98","source_frame":"5880","start_s":"196.0"},{"end_s":"198.03334045410156","sampled_index":"99","source_frame":"5940","start_s":"198.0"},{"end_s":"200.03334045410156","sampled_index":"100","source_frame":"6000","start_s":"200.0"}],"video_id":355,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_203_real.mp4","video_sha256":"d63f30dbc07081c58b68f1fe95ae2ce130e723cfdf6fedf068a2547662609ed0"},{"conversations":[{"answer":"mixed numbers and improper fractions.","choices":["mixed numbers and improper fractions.","Add mixed numbers.","Subtract improper fractions.","Multiply fractions."],"end_time":11.0,"question":"What topic might the teacher explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:00:11"},{"answer":"Explain how improper fractions is connected to proper fractions and whole fractions.","choices":["Compare the fractions using decimal values.","Explain how improper fractions is connected to proper fractions and whole fractions.","Demonstrate how to multiply two fractions.","Discuss the concept of mixed numbers."],"end_time":190.0,"question":"What might the teacher explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:03:10"},{"answer":"How to convert mixed numbers to improper fractions.","choices":["How to convert mixed numbers to improper fractions.","The method for adding mixed numbers and improper fractions.","Simplifying improper fractions.","Dividing fractions by whole numbers."],"end_time":331.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"0:05:31"},{"answer":"Converting improper fractions to mixed numbers.","choices":["Multiplying mixed fractions by a whole number.","Converting improper fractions to mixed numbers.","Adding improper fractions together.","Simplifying mixed fractions."],"end_time":500.0,"question":"What concept will the speaker likely explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:08:20"},{"answer":"How to use dividing to convert improper fractions to a mixed number form.","choices":["Add the remainder to the quotient.","Subtract 6 from 1.","Divide 7 by 3.","How to use dividing to convert improper fractions to a mixed number form."],"end_time":646.0,"question":"What might the speaker explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:10:46"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"38242d80ec49dba468a624a35d24ee83c20079607ac2c495affddc757f8b8cb0","start":0,"stop":6,"tokens":1176},{"file":"chunk_0001.pt","qa_index":1,"sha256":"8fecf3c9b0b554b52f3c5f733c899bbc8b9e193be443b1ba22b6d553716797ef","start":6,"stop":22,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"6e59df7ec6c3e56e80eae23e2597e43e17aa059ef06aa12deaa2fc02625db952","start":22,"stop":38,"tokens":3136},{"file":"chunk_0003.pt","qa_index":1,"sha256":"44066991fb612597d439af05de1cd7c2d08772473038bcb1337be348a7c02378","start":38,"stop":54,"tokens":3136},{"file":"chunk_0004.pt","qa_index":1,"sha256":"ad85600114edff2c2b23e7a4b8e808e5393926d5c0123516cff021f7dbe07403","start":54,"stop":70,"tokens":3136},{"file":"chunk_0005.pt","qa_index":1,"sha256":"dbec3969d4279667f8f79a1c1f41bc95e5e67271f3c38915132d75edbde525cc","start":70,"stop":86,"tokens":3136},{"file":"chunk_0006.pt","qa_index":1,"sha256":"c9103c0793725141192127a00bfb4950668cecca8fd5ac7ff9042c19df9e154e","start":86,"stop":95,"tokens":1764},{"file":"chunk_0007.pt","qa_index":2,"sha256":"a1b2dd97b067d4fa3269effc42186d9621f7c2e6f513432c5aec674d1870d9d0","start":95,"stop":111,"tokens":3136},{"file":"chunk_0008.pt","qa_index":2,"sha256":"ab6ab687694c2d4f602c1bd03dd575b320b9e5719f3aa38600eb5204cff5a35d","start":111,"stop":127,"tokens":3136},{"file":"chunk_0009.pt","qa_index":2,"sha256":"2723602bd21184bd57204c143b3d7e82aa6da394cd0e1f6c7711afee754a40f9","start":127,"stop":143,"tokens":3136},{"file":"chunk_0010.pt","qa_index":2,"sha256":"fecbc36538217b03c12f889141dfe5e456c106d61e7f91cf9e716f80990546c3","start":143,"stop":159,"tokens":3136},{"file":"chunk_0011.pt","qa_index":2,"sha256":"0526127f62e18b93193291f5f8eef59ce15fef468c9ded046b4f5cfb9eadc6a8","start":159,"stop":166,"tokens":1372},{"file":"chunk_0012.pt","qa_index":3,"sha256":"d20579d67e8e38acc4e64cc592c83419d07e83cd77be2bc5bc960252d3d276c7","start":166,"stop":182,"tokens":3136},{"file":"chunk_0013.pt","qa_index":3,"sha256":"6d34a6508c31c06066f8d55c361f3f520d5059ca61e2b40e083cb2c596be1a3b","start":182,"stop":198,"tokens":3136},{"file":"chunk_0014.pt","qa_index":3,"sha256":"df4646131b1ecc26c1e5f1e97fa028b493dc204e629e27e43a5feff1c244b099","start":198,"stop":214,"tokens":3136},{"file":"chunk_0015.pt","qa_index":3,"sha256":"fc8dd05512e2780b30aa0cfb13df9daa2c8d8bb342b33c634987b7949e793d2d","start":214,"stop":230,"tokens":3136},{"file":"chunk_0016.pt","qa_index":3,"sha256":"19f352e019cbf0297fe92ae9e451bacb41eb9a22753d24d0ec10a60fd1ac755c","start":230,"stop":246,"tokens":3136},{"file":"chunk_0017.pt","qa_index":3,"sha256":"c149c964c90092f53447a94ddedf53eb3b0ada1f861d24471eaf189be752fe35","start":246,"stop":250,"tokens":784},{"file":"chunk_0018.pt","qa_index":4,"sha256":"45d401dc49200e4c49329d5631e43b2881b028a8cc05bc6dac71e5f91a78922c","start":250,"stop":266,"tokens":3136},{"file":"chunk_0019.pt","qa_index":4,"sha256":"4f34c1815911966df0988799da44c46af246350c3322f2bd6949a6186c4e66f6","start":266,"stop":282,"tokens":3136},{"file":"chunk_0020.pt","qa_index":4,"sha256":"fb18bdb73b77163c8ecdc48902226a11ea84bb001ffd9bf86d8e8f69150bb245","start":282,"stop":298,"tokens":3136},{"file":"chunk_0021.pt","qa_index":4,"sha256":"4edff3ed61c6ba3a7b5834faf0d76dfd31cb94b352aebe7291df4bbe36cedb59","start":298,"stop":314,"tokens":3136},{"file":"chunk_0022.pt","qa_index":4,"sha256":"16f0bfa09f91f4c085665c57a1347a5742f45a4b4359b7b97b23b0bac5a4de91","start":314,"stop":323,"tokens":1764}],"identity":"a1091d4e58852bef9afb6b98038032e6633feaa21b6bc5cb3d537d8498c3b3ad","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":28.963512014597654,"sample_id":218,"sampled_frames_available":383,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"058e1ef70b012470786687bc0f97e6b8f226981a2bc2b123a9e3a1e94dae8367"},"sample_id":218,"timestamps":[{"end_s":"0.03336666524410248","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0353667736053467","sampled_index":"1","source_frame":"60","start_s":"2.002000093460083"},{"end_s":"4.03736686706543","sampled_index":"2","source_frame":"120","start_s":"4.004000186920166"},{"end_s":"6.039366722106934","sampled_index":"3","source_frame":"180","start_s":"6.00600004196167"},{"end_s":"8.041366577148438","sampled_index":"4","source_frame":"240","start_s":"8.008000373840332"},{"end_s":"10.043366432189941","sampled_index":"5","source_frame":"300","start_s":"10.010000228881836"},{"end_s":"12.045366287231445","sampled_index":"6","source_frame":"360","start_s":"12.01200008392334"},{"end_s":"14.047367095947266","sampled_index":"7","source_frame":"420","start_s":"14.013999938964844"},{"end_s":"16.049365997314453","sampled_index":"8","source_frame":"480","start_s":"16.016000747680664"},{"end_s":"18.051366806030273","sampled_index":"9","source_frame":"540","start_s":"18.01799964904785"},{"end_s":"20.053367614746094","sampled_index":"10","source_frame":"600","start_s":"20.020000457763672"},{"end_s":"22.05536651611328","sampled_index":"11","source_frame":"660","start_s":"22.02199935913086"},{"end_s":"24.0573673248291","sampled_index":"12","source_frame":"720","start_s":"24.02400016784668"},{"end_s":"26.05936622619629","sampled_index":"13","source_frame":"780","start_s":"26.025999069213867"},{"end_s":"28.06136703491211","sampled_index":"14","source_frame":"840","start_s":"28.027999877929688"},{"end_s":"30.063365936279297","sampled_index":"15","source_frame":"900","start_s":"30.030000686645508"},{"end_s":"32.065364837646484","sampled_index":"16","source_frame":"960","start_s":"32.03200149536133"},{"end_s":"34.06736755371094","sampled_index":"17","source_frame":"1020","start_s":"34.034000396728516"},{"end_s":"36.069366455078125","sampled_index":"18","source_frame":"1080","start_s":"36.0359992980957"},{"end_s":"38.07136535644531","sampled_index":"19","source_frame":"1140","start_s":"38.03799819946289"},{"end_s":"40.073368072509766","sampled_index":"20","source_frame":"1200","start_s":"40.040000915527344"},{"end_s":"42.07536697387695","sampled_index":"21","source_frame":"1260","start_s":"42.04199981689453"},{"end_s":"44.07736587524414","sampled_index":"22","source_frame":"1320","start_s":"44.04399871826172"},{"end_s":"46.07936477661133","sampled_index":"23","source_frame":"1380","start_s":"46.04600143432617"},{"end_s":"48.08136749267578","sampled_index":"24","source_frame":"1440","start_s":"48.04800033569336"},{"end_s":"50.08336639404297","sampled_index":"25","source_frame":"1500","start_s":"50.04999923706055"},{"end_s":"52.085365295410156","sampled_index":"26","source_frame":"1560","start_s":"52.051998138427734"},{"end_s":"54.08736801147461","sampled_index":"27","source_frame":"1620","start_s":"54.05400085449219"},{"end_s":"56.0893669128418","sampled_index":"28","source_frame":"1680","start_s":"56.055999755859375"},{"end_s":"58.091365814208984","sampled_index":"29","source_frame":"1740","start_s":"58.05799865722656"},{"end_s":"60.09336853027344","sampled_index":"30","source_frame":"1800","start_s":"60.060001373291016"},{"end_s":"62.095367431640625","sampled_index":"31","source_frame":"1860","start_s":"62.0620002746582"},{"end_s":"64.09736633300781","sampled_index":"32","source_frame":"1920","start_s":"64.06400299072266"},{"end_s":"66.099365234375","sampled_index":"33","source_frame":"1980","start_s":"66.06600189208984"},{"end_s":"68.10136413574219","sampled_index":"34","source_frame":"2040","start_s":"68.06800079345703"},{"end_s":"70.10336303710938","sampled_index":"35","source_frame":"2100","start_s":"70.06999969482422"},{"end_s":"72.1053695678711","sampled_index":"36","source_frame":"2160","start_s":"72.0719985961914"},{"end_s":"74.10736846923828","sampled_index":"37","source_frame":"2220","start_s":"74.0739974975586"},{"end_s":"76.10936737060547","sampled_index":"38","source_frame":"2280","start_s":"76.07599639892578"},{"end_s":"78.11136627197266","sampled_index":"39","source_frame":"2340","start_s":"78.0780029296875"},{"end_s":"80.11336517333984","sampled_index":"40","source_frame":"2400","start_s":"80.08000183105469"},{"end_s":"82.11536407470703","sampled_index":"41","source_frame":"2460","start_s":"82.08200073242188"},{"end_s":"84.11736297607422","sampled_index":"42","source_frame":"2520","start_s":"84.08399963378906"},{"end_s":"86.11936950683594","sampled_index":"43","source_frame":"2580","start_s":"86.08599853515625"},{"end_s":"88.12136840820312","sampled_index":"44","source_frame":"2640","start_s":"88.08799743652344"},{"end_s":"90.12336730957031","sampled_index":"45","source_frame":"2700","start_s":"90.08999633789062"},{"end_s":"92.1253662109375","sampled_index":"46","source_frame":"2760","start_s":"92.09200286865234"},{"end_s":"94.12736511230469","sampled_index":"47","source_frame":"2820","start_s":"94.09400177001953"},{"end_s":"96.12936401367188","sampled_index":"48","source_frame":"2880","start_s":"96.09600067138672"},{"end_s":"98.13136291503906","sampled_index":"49","source_frame":"2940","start_s":"98.0979995727539"},{"end_s":"100.13336944580078","sampled_index":"50","source_frame":"3000","start_s":"100.0999984741211"},{"end_s":"102.13536834716797","sampled_index":"51","source_frame":"3060","start_s":"102.10199737548828"},{"end_s":"104.13736724853516","sampled_index":"52","source_frame":"3120","start_s":"104.10399627685547"},{"end_s":"106.13936614990234","sampled_index":"53","source_frame":"3180","start_s":"106.10600280761719"},{"end_s":"108.14136505126953","sampled_index":"54","source_frame":"3240","start_s":"108.10800170898438"},{"end_s":"110.14336395263672","sampled_index":"55","source_frame":"3300","start_s":"110.11000061035156"},{"end_s":"112.1453628540039","sampled_index":"56","source_frame":"3360","start_s":"112.11199951171875"},{"end_s":"114.14736938476562","sampled_index":"57","source_frame":"3420","start_s":"114.11399841308594"},{"end_s":"116.14936828613281","sampled_index":"58","source_frame":"3480","start_s":"116.11599731445312"},{"end_s":"118.1513671875","sampled_index":"59","source_frame":"3540","start_s":"118.11799621582031"},{"end_s":"120.15336608886719","sampled_index":"60","source_frame":"3600","start_s":"120.12000274658203"},{"end_s":"122.15536499023438","sampled_index":"61","source_frame":"3660","start_s":"122.12200164794922"},{"end_s":"124.15736389160156","sampled_index":"62","source_frame":"3720","start_s":"124.1240005493164"},{"end_s":"126.15937042236328","sampled_index":"63","source_frame":"3780","start_s":"126.1259994506836"},{"end_s":"128.16136169433594","sampled_index":"64","source_frame":"3840","start_s":"128.1280059814453"},{"end_s":"130.16336059570312","sampled_index":"65","source_frame":"3900","start_s":"130.1300048828125"},{"end_s":"132.1653594970703","sampled_index":"66","source_frame":"3960","start_s":"132.1320037841797"},{"end_s":"134.16737365722656","sampled_index":"67","source_frame":"4020","start_s":"134.13400268554688"},{"end_s":"136.16937255859375","sampled_index":"68","source_frame":"4080","start_s":"136.13600158691406"},{"end_s":"138.17137145996094","sampled_index":"69","source_frame":"4140","start_s":"138.13800048828125"},{"end_s":"140.17337036132812","sampled_index":"70","source_frame":"4200","start_s":"140.13999938964844"},{"end_s":"142.1753692626953","sampled_index":"71","source_frame":"4260","start_s":"142.14199829101562"},{"end_s":"144.1773681640625","sampled_index":"72","source_frame":"4320","start_s":"144.1439971923828"},{"end_s":"146.1793670654297","sampled_index":"73","source_frame":"4380","start_s":"146.14599609375"},{"end_s":"148.18136596679688","sampled_index":"74","source_frame":"4440","start_s":"148.1479949951172"},{"end_s":"150.18336486816406","sampled_index":"75","source_frame":"4500","start_s":"150.14999389648438"},{"end_s":"152.18536376953125","sampled_index":"76","source_frame":"4560","start_s":"152.15199279785156"},{"end_s":"154.18736267089844","sampled_index":"77","source_frame":"4620","start_s":"154.1540069580078"},{"end_s":"156.18936157226562","sampled_index":"78","source_frame":"4680","start_s":"156.156005859375"},{"end_s":"158.1913604736328","sampled_index":"79","source_frame":"4740","start_s":"158.1580047607422"},{"end_s":"160.193359375","sampled_index":"80","source_frame":"4800","start_s":"160.16000366210938"},{"end_s":"162.19537353515625","sampled_index":"81","source_frame":"4860","start_s":"162.16200256347656"},{"end_s":"164.19737243652344","sampled_index":"82","source_frame":"4920","start_s":"164.16400146484375"},{"end_s":"166.19937133789062","sampled_index":"83","source_frame":"4980","start_s":"166.16600036621094"},{"end_s":"168.2013702392578","sampled_index":"84","source_frame":"5040","start_s":"168.16799926757812"},{"end_s":"170.203369140625","sampled_index":"85","source_frame":"5100","start_s":"170.1699981689453"},{"end_s":"172.2053680419922","sampled_index":"86","source_frame":"5160","start_s":"172.1719970703125"},{"end_s":"174.20736694335938","sampled_index":"87","source_frame":"5220","start_s":"174.1739959716797"},{"end_s":"176.20936584472656","sampled_index":"88","source_frame":"5280","start_s":"176.17599487304688"},{"end_s":"178.21136474609375","sampled_index":"89","source_frame":"5340","start_s":"178.17799377441406"},{"end_s":"180.21336364746094","sampled_index":"90","source_frame":"5400","start_s":"180.17999267578125"},{"end_s":"182.21536254882812","sampled_index":"91","source_frame":"5460","start_s":"182.1820068359375"},{"end_s":"184.2173614501953","sampled_index":"92","source_frame":"5520","start_s":"184.1840057373047"},{"end_s":"186.2193603515625","sampled_index":"93","source_frame":"5580","start_s":"186.18600463867188"},{"end_s":"188.2213592529297","sampled_index":"94","source_frame":"5640","start_s":"188.18800354003906"},{"end_s":"190.22337341308594","sampled_index":"95","source_frame":"5700","start_s":"190.19000244140625"},{"end_s":"192.22537231445312","sampled_index":"96","source_frame":"5760","start_s":"192.19200134277344"},{"end_s":"194.2273712158203","sampled_index":"97","source_frame":"5820","start_s":"194.19400024414062"},{"end_s":"196.2293701171875","sampled_index":"98","source_frame":"5880","start_s":"196.1959991455078"},{"end_s":"198.2313690185547","sampled_index":"99","source_frame":"5940","start_s":"198.197998046875"},{"end_s":"200.23336791992188","sampled_index":"100","source_frame":"6000","start_s":"200.1999969482422"},{"end_s":"202.23536682128906","sampled_index":"101","source_frame":"6060","start_s":"202.20199584960938"},{"end_s":"204.23736572265625","sampled_index":"102","source_frame":"6120","start_s":"204.20399475097656"},{"end_s":"206.23936462402344","sampled_index":"103","source_frame":"6180","start_s":"206.20599365234375"},{"end_s":"208.24136352539062","sampled_index":"104","source_frame":"6240","start_s":"208.20799255371094"},{"end_s":"210.2433624267578","sampled_index":"105","source_frame":"6300","start_s":"210.2100067138672"},{"end_s":"212.245361328125","sampled_index":"106","source_frame":"6360","start_s":"212.21200561523438"},{"end_s":"214.2473602294922","sampled_index":"107","source_frame":"6420","start_s":"214.21400451660156"},{"end_s":"216.24935913085938","sampled_index":"108","source_frame":"6480","start_s":"216.21600341796875"},{"end_s":"218.25137329101562","sampled_index":"109","source_frame":"6540","start_s":"218.21800231933594"},{"end_s":"220.2533721923828","sampled_index":"110","source_frame":"6600","start_s":"220.22000122070312"},{"end_s":"222.25537109375","sampled_index":"111","source_frame":"6660","start_s":"222.2220001220703"},{"end_s":"224.2573699951172","sampled_index":"112","source_frame":"6720","start_s":"224.2239990234375"},{"end_s":"226.25936889648438","sampled_index":"113","source_frame":"6780","start_s":"226.2259979248047"},{"end_s":"228.26136779785156","sampled_index":"114","source_frame":"6840","start_s":"228.22799682617188"},{"end_s":"230.26336669921875","sampled_index":"115","source_frame":"6900","start_s":"230.22999572753906"},{"end_s":"232.26536560058594","sampled_index":"116","source_frame":"6960","start_s":"232.23199462890625"},{"end_s":"234.26736450195312","sampled_index":"117","source_frame":"7020","start_s":"234.23399353027344"},{"end_s":"236.2693634033203","sampled_index":"118","source_frame":"7080","start_s":"236.23599243164062"},{"end_s":"238.2713623046875","sampled_index":"119","source_frame":"7140","start_s":"238.23800659179688"},{"end_s":"240.2733612060547","sampled_index":"120","source_frame":"7200","start_s":"240.24000549316406"},{"end_s":"242.27536010742188","sampled_index":"121","source_frame":"7260","start_s":"242.24200439453125"},{"end_s":"244.27737426757812","sampled_index":"122","source_frame":"7320","start_s":"244.24400329589844"},{"end_s":"246.2793731689453","sampled_index":"123","source_frame":"7380","start_s":"246.24600219726562"},{"end_s":"248.2813720703125","sampled_index":"124","source_frame":"7440","start_s":"248.2480010986328"},{"end_s":"250.2833709716797","sampled_index":"125","source_frame":"7500","start_s":"250.25"},{"end_s":"252.28536987304688","sampled_index":"126","source_frame":"7560","start_s":"252.2519989013672"},{"end_s":"254.28736877441406","sampled_index":"127","source_frame":"7620","start_s":"254.25399780273438"},{"end_s":"256.28936767578125","sampled_index":"128","source_frame":"7680","start_s":"256.2560119628906"},{"end_s":"258.2913818359375","sampled_index":"129","source_frame":"7740","start_s":"258.25799560546875"},{"end_s":"260.2933654785156","sampled_index":"130","source_frame":"7800","start_s":"260.260009765625"},{"end_s":"262.2953796386719","sampled_index":"131","source_frame":"7860","start_s":"262.2619934082031"},{"end_s":"264.29736328125","sampled_index":"132","source_frame":"7920","start_s":"264.2640075683594"},{"end_s":"266.29937744140625","sampled_index":"133","source_frame":"7980","start_s":"266.2659912109375"},{"end_s":"268.3013610839844","sampled_index":"134","source_frame":"8040","start_s":"268.26800537109375"},{"end_s":"270.3033752441406","sampled_index":"135","source_frame":"8100","start_s":"270.2699890136719"},{"end_s":"272.30535888671875","sampled_index":"136","source_frame":"8160","start_s":"272.2720031738281"},{"end_s":"274.307373046875","sampled_index":"137","source_frame":"8220","start_s":"274.27398681640625"},{"end_s":"276.3093566894531","sampled_index":"138","source_frame":"8280","start_s":"276.2760009765625"},{"end_s":"278.3113708496094","sampled_index":"139","source_frame":"8340","start_s":"278.27801513671875"},{"end_s":"280.3133544921875","sampled_index":"140","source_frame":"8400","start_s":"280.2799987792969"},{"end_s":"282.31536865234375","sampled_index":"141","source_frame":"8460","start_s":"282.2820129394531"},{"end_s":"284.3173522949219","sampled_index":"142","source_frame":"8520","start_s":"284.28399658203125"},{"end_s":"286.3193664550781","sampled_index":"143","source_frame":"8580","start_s":"286.2860107421875"},{"end_s":"288.3213806152344","sampled_index":"144","source_frame":"8640","start_s":"288.2879943847656"},{"end_s":"290.3233642578125","sampled_index":"145","source_frame":"8700","start_s":"290.2900085449219"},{"end_s":"292.32537841796875","sampled_index":"146","source_frame":"8760","start_s":"292.2919921875"},{"end_s":"294.3273620605469","sampled_index":"147","source_frame":"8820","start_s":"294.29400634765625"},{"end_s":"296.3293762207031","sampled_index":"148","source_frame":"8880","start_s":"296.2959899902344"},{"end_s":"298.33135986328125","sampled_index":"149","source_frame":"8940","start_s":"298.2980041503906"},{"end_s":"300.3333740234375","sampled_index":"150","source_frame":"9000","start_s":"300.29998779296875"},{"end_s":"302.3353576660156","sampled_index":"151","source_frame":"9060","start_s":"302.302001953125"},{"end_s":"304.3373718261719","sampled_index":"152","source_frame":"9120","start_s":"304.3039855957031"},{"end_s":"306.33935546875","sampled_index":"153","source_frame":"9180","start_s":"306.3059997558594"},{"end_s":"308.34136962890625","sampled_index":"154","source_frame":"9240","start_s":"308.3080139160156"},{"end_s":"310.3433532714844","sampled_index":"155","source_frame":"9300","start_s":"310.30999755859375"},{"end_s":"312.3453674316406","sampled_index":"156","source_frame":"9360","start_s":"312.31201171875"},{"end_s":"314.3473815917969","sampled_index":"157","source_frame":"9420","start_s":"314.3139953613281"},{"end_s":"316.349365234375","sampled_index":"158","source_frame":"9480","start_s":"316.3160095214844"},{"end_s":"318.35137939453125","sampled_index":"159","source_frame":"9540","start_s":"318.3179931640625"},{"end_s":"320.3533630371094","sampled_index":"160","source_frame":"9600","start_s":"320.32000732421875"},{"end_s":"322.3553771972656","sampled_index":"161","source_frame":"9660","start_s":"322.3219909667969"},{"end_s":"324.35736083984375","sampled_index":"162","source_frame":"9720","start_s":"324.3240051269531"},{"end_s":"326.359375","sampled_index":"163","source_frame":"9780","start_s":"326.32598876953125"},{"end_s":"328.3613586425781","sampled_index":"164","source_frame":"9840","start_s":"328.3280029296875"},{"end_s":"330.3633728027344","sampled_index":"165","source_frame":"9900","start_s":"330.3299865722656"},{"end_s":"332.3653564453125","sampled_index":"166","source_frame":"9960","start_s":"332.3320007324219"},{"end_s":"334.36737060546875","sampled_index":"167","source_frame":"10020","start_s":"334.3340148925781"},{"end_s":"336.3693542480469","sampled_index":"168","source_frame":"10080","start_s":"336.33599853515625"},{"end_s":"338.3713684082031","sampled_index":"169","source_frame":"10140","start_s":"338.3380126953125"},{"end_s":"340.37335205078125","sampled_index":"170","source_frame":"10200","start_s":"340.3399963378906"},{"end_s":"342.3753662109375","sampled_index":"171","source_frame":"10260","start_s":"342.3420104980469"},{"end_s":"344.37738037109375","sampled_index":"172","source_frame":"10320","start_s":"344.343994140625"},{"end_s":"346.3793640136719","sampled_index":"173","source_frame":"10380","start_s":"346.34600830078125"},{"end_s":"348.3813781738281","sampled_index":"174","source_frame":"10440","start_s":"348.3479919433594"},{"end_s":"350.38336181640625","sampled_index":"175","source_frame":"10500","start_s":"350.3500061035156"},{"end_s":"352.3853759765625","sampled_index":"176","source_frame":"10560","start_s":"352.35198974609375"},{"end_s":"354.3873596191406","sampled_index":"177","source_frame":"10620","start_s":"354.35400390625"},{"end_s":"356.3893737792969","sampled_index":"178","source_frame":"10680","start_s":"356.3559875488281"},{"end_s":"358.391357421875","sampled_index":"179","source_frame":"10740","start_s":"358.3580017089844"},{"end_s":"360.39337158203125","sampled_index":"180","source_frame":"10800","start_s":"360.3599853515625"},{"end_s":"362.3953552246094","sampled_index":"181","source_frame":"10860","start_s":"362.36199951171875"},{"end_s":"364.3973693847656","sampled_index":"182","source_frame":"10920","start_s":"364.364013671875"},{"end_s":"366.39935302734375","sampled_index":"183","source_frame":"10980","start_s":"366.3659973144531"},{"end_s":"368.4013671875","sampled_index":"184","source_frame":"11040","start_s":"368.3680114746094"},{"end_s":"370.40338134765625","sampled_index":"185","source_frame":"11100","start_s":"370.3699951171875"},{"end_s":"372.4053649902344","sampled_index":"186","source_frame":"11160","start_s":"372.37200927734375"},{"end_s":"374.4073791503906","sampled_index":"187","source_frame":"11220","start_s":"374.3739929199219"},{"end_s":"376.40936279296875","sampled_index":"188","source_frame":"11280","start_s":"376.3760070800781"},{"end_s":"378.411376953125","sampled_index":"189","source_frame":"11340","start_s":"378.37799072265625"},{"end_s":"380.4133605957031","sampled_index":"190","source_frame":"11400","start_s":"380.3800048828125"},{"end_s":"382.4153747558594","sampled_index":"191","source_frame":"11460","start_s":"382.3819885253906"},{"end_s":"384.4173583984375","sampled_index":"192","source_frame":"11520","start_s":"384.3840026855469"},{"end_s":"386.41937255859375","sampled_index":"193","source_frame":"11580","start_s":"386.385986328125"},{"end_s":"388.4213562011719","sampled_index":"194","source_frame":"11640","start_s":"388.38800048828125"},{"end_s":"390.4233703613281","sampled_index":"195","source_frame":"11700","start_s":"390.3900146484375"},{"end_s":"392.42535400390625","sampled_index":"196","source_frame":"11760","start_s":"392.3919982910156"},{"end_s":"394.4273681640625","sampled_index":"197","source_frame":"11820","start_s":"394.3940124511719"},{"end_s":"396.4293518066406","sampled_index":"198","source_frame":"11880","start_s":"396.39599609375"},{"end_s":"398.4313659667969","sampled_index":"199","source_frame":"11940","start_s":"398.39801025390625"},{"end_s":"400.4333801269531","sampled_index":"200","source_frame":"12000","start_s":"400.3999938964844"},{"end_s":"402.43536376953125","sampled_index":"201","source_frame":"12060","start_s":"402.4020080566406"},{"end_s":"404.4373779296875","sampled_index":"202","source_frame":"12120","start_s":"404.40399169921875"},{"end_s":"406.4393615722656","sampled_index":"203","source_frame":"12180","start_s":"406.406005859375"},{"end_s":"408.4413757324219","sampled_index":"204","source_frame":"12240","start_s":"408.4079895019531"},{"end_s":"410.443359375","sampled_index":"205","source_frame":"12300","start_s":"410.4100036621094"},{"end_s":"412.44537353515625","sampled_index":"206","source_frame":"12360","start_s":"412.4119873046875"},{"end_s":"414.4473571777344","sampled_index":"207","source_frame":"12420","start_s":"414.41400146484375"},{"end_s":"416.4493713378906","sampled_index":"208","source_frame":"12480","start_s":"416.4159851074219"},{"end_s":"418.45135498046875","sampled_index":"209","source_frame":"12540","start_s":"418.4179992675781"},{"end_s":"420.453369140625","sampled_index":"210","source_frame":"12600","start_s":"420.4200134277344"},{"end_s":"422.4553527832031","sampled_index":"211","source_frame":"12660","start_s":"422.4219970703125"},{"end_s":"424.4573669433594","sampled_index":"212","source_frame":"12720","start_s":"424.42401123046875"},{"end_s":"426.4593811035156","sampled_index":"213","source_frame":"12780","start_s":"426.4259948730469"},{"end_s":"428.46136474609375","sampled_index":"214","source_frame":"12840","start_s":"428.4280090332031"},{"end_s":"430.46337890625","sampled_index":"215","source_frame":"12900","start_s":"430.42999267578125"},{"end_s":"432.4653625488281","sampled_index":"216","source_frame":"12960","start_s":"432.4320068359375"},{"end_s":"434.4673767089844","sampled_index":"217","source_frame":"13020","start_s":"434.4339904785156"},{"end_s":"436.4693603515625","sampled_index":"218","source_frame":"13080","start_s":"436.4360046386719"},{"end_s":"438.47137451171875","sampled_index":"219","source_frame":"13140","start_s":"438.43798828125"},{"end_s":"440.4733581542969","sampled_index":"220","source_frame":"13200","start_s":"440.44000244140625"},{"end_s":"442.4753723144531","sampled_index":"221","source_frame":"13260","start_s":"442.4419860839844"},{"end_s":"444.47735595703125","sampled_index":"222","source_frame":"13320","start_s":"444.4440002441406"},{"end_s":"446.4793701171875","sampled_index":"223","source_frame":"13380","start_s":"446.4460144042969"},{"end_s":"448.4813537597656","sampled_index":"224","source_frame":"13440","start_s":"448.447998046875"},{"end_s":"450.4833679199219","sampled_index":"225","source_frame":"13500","start_s":"450.45001220703125"},{"end_s":"452.4853515625","sampled_index":"226","source_frame":"13560","start_s":"452.4519958496094"},{"end_s":"454.48736572265625","sampled_index":"227","source_frame":"13620","start_s":"454.4540100097656"},{"end_s":"456.4893798828125","sampled_index":"228","source_frame":"13680","start_s":"456.45599365234375"},{"end_s":"458.4913635253906","sampled_index":"229","source_frame":"13740","start_s":"458.4580078125"},{"end_s":"460.4933776855469","sampled_index":"230","source_frame":"13800","start_s":"460.4599914550781"},{"end_s":"462.495361328125","sampled_index":"231","source_frame":"13860","start_s":"462.4620056152344"},{"end_s":"464.49737548828125","sampled_index":"232","source_frame":"13920","start_s":"464.4639892578125"},{"end_s":"466.4993591308594","sampled_index":"233","source_frame":"13980","start_s":"466.46600341796875"},{"end_s":"468.5013732910156","sampled_index":"234","source_frame":"14040","start_s":"468.4679870605469"},{"end_s":"470.50335693359375","sampled_index":"235","source_frame":"14100","start_s":"470.4700012207031"},{"end_s":"472.50537109375","sampled_index":"236","source_frame":"14160","start_s":"472.47198486328125"},{"end_s":"474.5073547363281","sampled_index":"237","source_frame":"14220","start_s":"474.4739990234375"},{"end_s":"476.5093688964844","sampled_index":"238","source_frame":"14280","start_s":"476.47601318359375"},{"end_s":"478.5113525390625","sampled_index":"239","source_frame":"14340","start_s":"478.4779968261719"},{"end_s":"480.51336669921875","sampled_index":"240","source_frame":"14400","start_s":"480.4800109863281"},{"end_s":"482.515380859375","sampled_index":"241","source_frame":"14460","start_s":"482.48199462890625"},{"end_s":"484.5173645019531","sampled_index":"242","source_frame":"14520","start_s":"484.4840087890625"},{"end_s":"486.5193786621094","sampled_index":"243","source_frame":"14580","start_s":"486.4859924316406"},{"end_s":"488.5213623046875","sampled_index":"244","source_frame":"14640","start_s":"488.4880065917969"},{"end_s":"490.52337646484375","sampled_index":"245","source_frame":"14700","start_s":"490.489990234375"},{"end_s":"492.5253601074219","sampled_index":"246","source_frame":"14760","start_s":"492.49200439453125"},{"end_s":"494.5273742675781","sampled_index":"247","source_frame":"14820","start_s":"494.4939880371094"},{"end_s":"496.52935791015625","sampled_index":"248","source_frame":"14880","start_s":"496.4960021972656"},{"end_s":"498.5313720703125","sampled_index":"249","source_frame":"14940","start_s":"498.49798583984375"},{"end_s":"500.5333557128906","sampled_index":"250","source_frame":"15000","start_s":"500.5"},{"end_s":"502.5353698730469","sampled_index":"251","source_frame":"15060","start_s":"502.50201416015625"},{"end_s":"504.537353515625","sampled_index":"252","source_frame":"15120","start_s":"504.5039978027344"},{"end_s":"506.53936767578125","sampled_index":"253","source_frame":"15180","start_s":"506.5060119628906"},{"end_s":"508.5413818359375","sampled_index":"254","source_frame":"15240","start_s":"508.50799560546875"},{"end_s":"510.5433654785156","sampled_index":"255","source_frame":"15300","start_s":"510.510009765625"},{"end_s":"512.5453491210938","sampled_index":"256","source_frame":"15360","start_s":"512.5120239257812"},{"end_s":"514.54736328125","sampled_index":"257","source_frame":"15420","start_s":"514.5139770507812"},{"end_s":"516.5493774414062","sampled_index":"258","source_frame":"15480","start_s":"516.5159912109375"},{"end_s":"518.5513916015625","sampled_index":"259","source_frame":"15540","start_s":"518.5180053710938"},{"end_s":"520.5533447265625","sampled_index":"260","source_frame":"15600","start_s":"520.52001953125"},{"end_s":"522.5553588867188","sampled_index":"261","source_frame":"15660","start_s":"522.52197265625"},{"end_s":"524.557373046875","sampled_index":"262","source_frame":"15720","start_s":"524.5239868164062"},{"end_s":"526.5593872070312","sampled_index":"263","source_frame":"15780","start_s":"526.5260009765625"},{"end_s":"528.5613403320312","sampled_index":"264","source_frame":"15840","start_s":"528.5280151367188"},{"end_s":"530.5633544921875","sampled_index":"265","source_frame":"15900","start_s":"530.530029296875"},{"end_s":"532.5653686523438","sampled_index":"266","source_frame":"15960","start_s":"532.531982421875"},{"end_s":"534.5673828125","sampled_index":"267","source_frame":"16020","start_s":"534.5339965820312"},{"end_s":"536.5693969726562","sampled_index":"268","source_frame":"16080","start_s":"536.5360107421875"},{"end_s":"538.5713500976562","sampled_index":"269","source_frame":"16140","start_s":"538.5380249023438"},{"end_s":"540.5733642578125","sampled_index":"270","source_frame":"16200","start_s":"540.5399780273438"},{"end_s":"542.5753784179688","sampled_index":"271","source_frame":"16260","start_s":"542.5419921875"},{"end_s":"544.577392578125","sampled_index":"272","source_frame":"16320","start_s":"544.5440063476562"},{"end_s":"546.579345703125","sampled_index":"273","source_frame":"16380","start_s":"546.5460205078125"},{"end_s":"548.5813598632812","sampled_index":"274","source_frame":"16440","start_s":"548.5479736328125"},{"end_s":"550.5833740234375","sampled_index":"275","source_frame":"16500","start_s":"550.5499877929688"},{"end_s":"552.5853881835938","sampled_index":"276","source_frame":"16560","start_s":"552.552001953125"},{"end_s":"554.5873413085938","sampled_index":"277","source_frame":"16620","start_s":"554.5540161132812"},{"end_s":"556.58935546875","sampled_index":"278","source_frame":"16680","start_s":"556.5560302734375"},{"end_s":"558.5913696289062","sampled_index":"279","source_frame":"16740","start_s":"558.5579833984375"},{"end_s":"560.5933227539062","sampled_index":"280","source_frame":"16800","start_s":"560.5599975585938"},{"end_s":"562.5953369140625","sampled_index":"281","source_frame":"16860","start_s":"562.56201171875"},{"end_s":"564.5973510742188","sampled_index":"282","source_frame":"16920","start_s":"564.5640258789062"},{"end_s":"566.5993041992188","sampled_index":"283","source_frame":"16980","start_s":"566.5659790039062"},{"end_s":"568.601318359375","sampled_index":"284","source_frame":"17040","start_s":"568.5679931640625"},{"end_s":"570.6033325195312","sampled_index":"285","source_frame":"17100","start_s":"570.5700073242188"},{"end_s":"572.6053466796875","sampled_index":"286","source_frame":"17160","start_s":"572.572021484375"},{"end_s":"574.6073608398438","sampled_index":"287","source_frame":"17220","start_s":"574.573974609375"},{"end_s":"576.6093139648438","sampled_index":"288","source_frame":"17280","start_s":"576.5759887695312"},{"end_s":"578.611328125","sampled_index":"289","source_frame":"17340","start_s":"578.5780029296875"},{"end_s":"580.6133422851562","sampled_index":"290","source_frame":"17400","start_s":"580.5800170898438"},{"end_s":"582.6153564453125","sampled_index":"291","source_frame":"17460","start_s":"582.5819702148438"},{"end_s":"584.6173095703125","sampled_index":"292","source_frame":"17520","start_s":"584.583984375"},{"end_s":"586.6193237304688","sampled_index":"293","source_frame":"17580","start_s":"586.5859985351562"},{"end_s":"588.621337890625","sampled_index":"294","source_frame":"17640","start_s":"588.5880126953125"},{"end_s":"590.6233520507812","sampled_index":"295","source_frame":"17700","start_s":"590.5900268554688"},{"end_s":"592.6253051757812","sampled_index":"296","source_frame":"17760","start_s":"592.5919799804688"},{"end_s":"594.6273193359375","sampled_index":"297","source_frame":"17820","start_s":"594.593994140625"},{"end_s":"596.6293334960938","sampled_index":"298","source_frame":"17880","start_s":"596.5960083007812"},{"end_s":"598.63134765625","sampled_index":"299","source_frame":"17940","start_s":"598.5980224609375"},{"end_s":"600.6333618164062","sampled_index":"300","source_frame":"18000","start_s":"600.5999755859375"},{"end_s":"602.6353149414062","sampled_index":"301","source_frame":"18060","start_s":"602.6019897460938"},{"end_s":"604.6373291015625","sampled_index":"302","source_frame":"18120","start_s":"604.60400390625"},{"end_s":"606.6393432617188","sampled_index":"303","source_frame":"18180","start_s":"606.6060180664062"},{"end_s":"608.641357421875","sampled_index":"304","source_frame":"18240","start_s":"608.6079711914062"},{"end_s":"610.643310546875","sampled_index":"305","source_frame":"18300","start_s":"610.6099853515625"},{"end_s":"612.6453247070312","sampled_index":"306","source_frame":"18360","start_s":"612.6119995117188"},{"end_s":"614.6473388671875","sampled_index":"307","source_frame":"18420","start_s":"614.614013671875"},{"end_s":"616.6493530273438","sampled_index":"308","source_frame":"18480","start_s":"616.6160278320312"},{"end_s":"618.6513061523438","sampled_index":"309","source_frame":"18540","start_s":"618.6179809570312"},{"end_s":"620.6533203125","sampled_index":"310","source_frame":"18600","start_s":"620.6199951171875"},{"end_s":"622.6553344726562","sampled_index":"311","source_frame":"18660","start_s":"622.6220092773438"},{"end_s":"624.6573486328125","sampled_index":"312","source_frame":"18720","start_s":"624.6240234375"},{"end_s":"626.6593627929688","sampled_index":"313","source_frame":"18780","start_s":"626.6259765625"},{"end_s":"628.6613159179688","sampled_index":"314","source_frame":"18840","start_s":"628.6279907226562"},{"end_s":"630.663330078125","sampled_index":"315","source_frame":"18900","start_s":"630.6300048828125"},{"end_s":"632.6653442382812","sampled_index":"316","source_frame":"18960","start_s":"632.6320190429688"},{"end_s":"634.6673583984375","sampled_index":"317","source_frame":"19020","start_s":"634.6339721679688"},{"end_s":"636.6693115234375","sampled_index":"318","source_frame":"19080","start_s":"636.635986328125"},{"end_s":"638.6713256835938","sampled_index":"319","source_frame":"19140","start_s":"638.6380004882812"},{"end_s":"640.67333984375","sampled_index":"320","source_frame":"19200","start_s":"640.6400146484375"},{"end_s":"642.6753540039062","sampled_index":"321","source_frame":"19260","start_s":"642.6420288085938"},{"end_s":"644.6773071289062","sampled_index":"322","source_frame":"19320","start_s":"644.6439819335938"},{"end_s":"646.6793212890625","sampled_index":"323","source_frame":"19380","start_s":"646.64599609375"},{"end_s":"648.6813354492188","sampled_index":"324","source_frame":"19440","start_s":"648.6480102539062"},{"end_s":"650.683349609375","sampled_index":"325","source_frame":"19500","start_s":"650.6500244140625"},{"end_s":"652.6853637695312","sampled_index":"326","source_frame":"19560","start_s":"652.6519775390625"},{"end_s":"654.6873168945312","sampled_index":"327","source_frame":"19620","start_s":"654.6539916992188"},{"end_s":"656.6893310546875","sampled_index":"328","source_frame":"19680","start_s":"656.656005859375"},{"end_s":"658.6913452148438","sampled_index":"329","source_frame":"19740","start_s":"658.6580200195312"},{"end_s":"660.693359375","sampled_index":"330","source_frame":"19800","start_s":"660.6599731445312"},{"end_s":"662.6953125","sampled_index":"331","source_frame":"19860","start_s":"662.6619873046875"},{"end_s":"664.6973266601562","sampled_index":"332","source_frame":"19920","start_s":"664.6640014648438"},{"end_s":"666.6993408203125","sampled_index":"333","source_frame":"19980","start_s":"666.666015625"},{"end_s":"668.7013549804688","sampled_index":"334","source_frame":"20040","start_s":"668.6680297851562"},{"end_s":"670.7033081054688","sampled_index":"335","source_frame":"20100","start_s":"670.6699829101562"},{"end_s":"672.705322265625","sampled_index":"336","source_frame":"20160","start_s":"672.6719970703125"},{"end_s":"674.7073364257812","sampled_index":"337","source_frame":"20220","start_s":"674.6740112304688"},{"end_s":"676.7093505859375","sampled_index":"338","source_frame":"20280","start_s":"676.676025390625"},{"end_s":"678.7113037109375","sampled_index":"339","source_frame":"20340","start_s":"678.677978515625"},{"end_s":"680.7133178710938","sampled_index":"340","source_frame":"20400","start_s":"680.6799926757812"},{"end_s":"682.71533203125","sampled_index":"341","source_frame":"20460","start_s":"682.6820068359375"},{"end_s":"684.7173461914062","sampled_index":"342","source_frame":"20520","start_s":"684.6840209960938"},{"end_s":"686.7193603515625","sampled_index":"343","source_frame":"20580","start_s":"686.6859741210938"},{"end_s":"688.7213134765625","sampled_index":"344","source_frame":"20640","start_s":"688.68798828125"},{"end_s":"690.7233276367188","sampled_index":"345","source_frame":"20700","start_s":"690.6900024414062"},{"end_s":"692.725341796875","sampled_index":"346","source_frame":"20760","start_s":"692.6920166015625"},{"end_s":"694.7273559570312","sampled_index":"347","source_frame":"20820","start_s":"694.6939697265625"},{"end_s":"696.7293090820312","sampled_index":"348","source_frame":"20880","start_s":"696.6959838867188"},{"end_s":"698.7313232421875","sampled_index":"349","source_frame":"20940","start_s":"698.697998046875"},{"end_s":"700.7333374023438","sampled_index":"350","source_frame":"21000","start_s":"700.7000122070312"},{"end_s":"702.7353515625","sampled_index":"351","source_frame":"21060","start_s":"702.7020263671875"},{"end_s":"704.7373046875","sampled_index":"352","source_frame":"21120","start_s":"704.7039794921875"},{"end_s":"706.7393188476562","sampled_index":"353","source_frame":"21180","start_s":"706.7059936523438"},{"end_s":"708.7413330078125","sampled_index":"354","source_frame":"21240","start_s":"708.7080078125"},{"end_s":"710.7433471679688","sampled_index":"355","source_frame":"21300","start_s":"710.7100219726562"},{"end_s":"712.745361328125","sampled_index":"356","source_frame":"21360","start_s":"712.7119750976562"},{"end_s":"714.747314453125","sampled_index":"357","source_frame":"21420","start_s":"714.7139892578125"},{"end_s":"716.7493286132812","sampled_index":"358","source_frame":"21480","start_s":"716.7160034179688"},{"end_s":"718.7513427734375","sampled_index":"359","source_frame":"21540","start_s":"718.718017578125"},{"end_s":"720.7533569335938","sampled_index":"360","source_frame":"21600","start_s":"720.719970703125"},{"end_s":"722.7553100585938","sampled_index":"361","source_frame":"21660","start_s":"722.7219848632812"},{"end_s":"724.75732421875","sampled_index":"362","source_frame":"21720","start_s":"724.7239990234375"},{"end_s":"726.7593383789062","sampled_index":"363","source_frame":"21780","start_s":"726.7260131835938"},{"end_s":"728.7613525390625","sampled_index":"364","source_frame":"21840","start_s":"728.72802734375"},{"end_s":"730.7633056640625","sampled_index":"365","source_frame":"21900","start_s":"730.72998046875"},{"end_s":"732.7653198242188","sampled_index":"366","source_frame":"21960","start_s":"732.7319946289062"},{"end_s":"734.767333984375","sampled_index":"367","source_frame":"22020","start_s":"734.7340087890625"},{"end_s":"736.7693481445312","sampled_index":"368","source_frame":"22080","start_s":"736.7360229492188"},{"end_s":"738.7713623046875","sampled_index":"369","source_frame":"22140","start_s":"738.7379760742188"},{"end_s":"740.7733154296875","sampled_index":"370","source_frame":"22200","start_s":"740.739990234375"},{"end_s":"742.7753295898438","sampled_index":"371","source_frame":"22260","start_s":"742.7420043945312"},{"end_s":"744.77734375","sampled_index":"372","source_frame":"22320","start_s":"744.7440185546875"},{"end_s":"746.7793579101562","sampled_index":"373","source_frame":"22380","start_s":"746.7459716796875"},{"end_s":"748.7813110351562","sampled_index":"374","source_frame":"22440","start_s":"748.7479858398438"},{"end_s":"750.7833251953125","sampled_index":"375","source_frame":"22500","start_s":"750.75"},{"end_s":"752.7853393554688","sampled_index":"376","source_frame":"22560","start_s":"752.7520141601562"},{"end_s":"754.787353515625","sampled_index":"377","source_frame":"22620","start_s":"754.7540283203125"},{"end_s":"756.789306640625","sampled_index":"378","source_frame":"22680","start_s":"756.7559814453125"},{"end_s":"758.7913208007812","sampled_index":"379","source_frame":"22740","start_s":"758.7579956054688"},{"end_s":"760.7933349609375","sampled_index":"380","source_frame":"22800","start_s":"760.760009765625"},{"end_s":"762.7953491210938","sampled_index":"381","source_frame":"22860","start_s":"762.7620239257812"},{"end_s":"764.79736328125","sampled_index":"382","source_frame":"22920","start_s":"764.7639770507812"}],"video_id":492,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_218_real.mp4","video_sha256":"058e1ef70b012470786687bc0f97e6b8f226981a2bc2b123a9e3a1e94dae8367"},{"conversations":[{"answer":"Multiplying Fractions.","choices":["Adding Fractions.","Dividing Fractions.","Simplifying Fractions.","Multiplying Fractions."],"end_time":10.0,"question":"What mathematical topic might the speaker focus on next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:00:10"},{"answer":"Show the steps to multiply the fractions.","choices":["Explain how to convert the fractions.","Show the steps to add the fractions.","Show the steps to multiply the fractions.","Discuss the importance of common denominators."],"end_time":55.0,"question":"What might the speaker do next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:00:55"},{"answer":"Show the steps to multiply the fractions.","choices":["Add the fractions together.","Subtract the fractions.","Show the steps to multiply the fractions.","Divide the fractions."],"end_time":149.0,"question":"What might the speaker do next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:02:29"},{"answer":"Another explanation to fractions.","choices":["Addition of fractions.","Subtraction of fractions.","Simplifying fractions.","Another explanation to fractions."],"end_time":231.0,"question":"What mathematical concept might the speaker move on to explain next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:03:51"},{"answer":"How to multiply the denominators.","choices":["How to add fractions.","How to divide fractions.","How to multiply the denominators.","How to subtract fractions."],"end_time":310.0,"question":"What might the speaker discuss next?","required_ability":"episodic memory","task":"Prospective Reasoning","time_stamp":"00:05:10"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"d9cb4147fc0307906c3043f19c3651b5eef240c9f8bd433c9ef15e2511d4acde","start":0,"stop":5,"tokens":980},{"file":"chunk_0001.pt","qa_index":1,"sha256":"effdfce1bc72f29bf356e523353540882c0edc54849fcdaa7abdd20ae23fedb0","start":5,"stop":21,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"9aeff23e4140492817b5cd17100f16bc0fb7130678a0f6eb7edf101f77c6adfa","start":21,"stop":28,"tokens":1372},{"file":"chunk_0003.pt","qa_index":2,"sha256":"15fac86212a3cebaefdc6535a07e66f1f8058aa87ed8125e23ab96e3773ba466","start":28,"stop":44,"tokens":3136},{"file":"chunk_0004.pt","qa_index":2,"sha256":"27e2d293bfd24d12bd4b43ebc043cb55c5fb5b965beaf7559dea1194ee9de258","start":44,"stop":60,"tokens":3136},{"file":"chunk_0005.pt","qa_index":2,"sha256":"c10d66058b3e01ef7d650b7307dafd869ffe3bfcc9d4a216b00e6094a53f4865","start":60,"stop":75,"tokens":2940},{"file":"chunk_0006.pt","qa_index":3,"sha256":"69c19851af14e7e5873c98c0f8e4a89292025c65ee4ec00247e677eee122b826","start":75,"stop":91,"tokens":3136},{"file":"chunk_0007.pt","qa_index":3,"sha256":"595e59f25fceb5a12d922e7de4140a849f2d8819d5914575a63f5af9f5ad2bee","start":91,"stop":107,"tokens":3136},{"file":"chunk_0008.pt","qa_index":3,"sha256":"a5f97b9dfee4446dfc99be9aae9c185d2ca4778964375cf03e1f009d7e16ad75","start":107,"stop":116,"tokens":1764},{"file":"chunk_0009.pt","qa_index":4,"sha256":"e645bae63ad215ab2f6aaafc34dab29c4c29489bf3fc285b0f9a24cde17cd579","start":116,"stop":132,"tokens":3136},{"file":"chunk_0010.pt","qa_index":4,"sha256":"1a850cf48b6fae3d77e001ad2094ef65a6baf9c4b6d7abbb2a79a77150e24a44","start":132,"stop":148,"tokens":3136},{"file":"chunk_0011.pt","qa_index":4,"sha256":"95611732b343fb61c5015e8562a0aa409ad5a16c20baee90d7bb17ae0846bc21","start":148,"stop":155,"tokens":1372}],"identity":"de02260103752f791bea7150b6aa98e102e88cbac9f4130256dad7eccb56ed9b","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":13.906434461474419,"sample_id":219,"sampled_frames_available":171,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"abcfb2648dc865cf030d04f75f7f5f0b778fe8620ed6619a0003b56e9c68c05f"},"sample_id":219,"timestamps":[{"end_s":"0.03336666524410248","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0353667736053467","sampled_index":"1","source_frame":"60","start_s":"2.002000093460083"},{"end_s":"4.03736686706543","sampled_index":"2","source_frame":"120","start_s":"4.004000186920166"},{"end_s":"6.039366722106934","sampled_index":"3","source_frame":"180","start_s":"6.00600004196167"},{"end_s":"8.041366577148438","sampled_index":"4","source_frame":"240","start_s":"8.008000373840332"},{"end_s":"10.043366432189941","sampled_index":"5","source_frame":"300","start_s":"10.010000228881836"},{"end_s":"12.045366287231445","sampled_index":"6","source_frame":"360","start_s":"12.01200008392334"},{"end_s":"14.047367095947266","sampled_index":"7","source_frame":"420","start_s":"14.013999938964844"},{"end_s":"16.049365997314453","sampled_index":"8","source_frame":"480","start_s":"16.016000747680664"},{"end_s":"18.051366806030273","sampled_index":"9","source_frame":"540","start_s":"18.01799964904785"},{"end_s":"20.053367614746094","sampled_index":"10","source_frame":"600","start_s":"20.020000457763672"},{"end_s":"22.05536651611328","sampled_index":"11","source_frame":"660","start_s":"22.02199935913086"},{"end_s":"24.0573673248291","sampled_index":"12","source_frame":"720","start_s":"24.02400016784668"},{"end_s":"26.05936622619629","sampled_index":"13","source_frame":"780","start_s":"26.025999069213867"},{"end_s":"28.06136703491211","sampled_index":"14","source_frame":"840","start_s":"28.027999877929688"},{"end_s":"30.063365936279297","sampled_index":"15","source_frame":"900","start_s":"30.030000686645508"},{"end_s":"32.065364837646484","sampled_index":"16","source_frame":"960","start_s":"32.03200149536133"},{"end_s":"34.06736755371094","sampled_index":"17","source_frame":"1020","start_s":"34.034000396728516"},{"end_s":"36.069366455078125","sampled_index":"18","source_frame":"1080","start_s":"36.0359992980957"},{"end_s":"38.07136535644531","sampled_index":"19","source_frame":"1140","start_s":"38.03799819946289"},{"end_s":"40.073368072509766","sampled_index":"20","source_frame":"1200","start_s":"40.040000915527344"},{"end_s":"42.07536697387695","sampled_index":"21","source_frame":"1260","start_s":"42.04199981689453"},{"end_s":"44.07736587524414","sampled_index":"22","source_frame":"1320","start_s":"44.04399871826172"},{"end_s":"46.07936477661133","sampled_index":"23","source_frame":"1380","start_s":"46.04600143432617"},{"end_s":"48.08136749267578","sampled_index":"24","source_frame":"1440","start_s":"48.04800033569336"},{"end_s":"50.08336639404297","sampled_index":"25","source_frame":"1500","start_s":"50.04999923706055"},{"end_s":"52.085365295410156","sampled_index":"26","source_frame":"1560","start_s":"52.051998138427734"},{"end_s":"54.08736801147461","sampled_index":"27","source_frame":"1620","start_s":"54.05400085449219"},{"end_s":"56.0893669128418","sampled_index":"28","source_frame":"1680","start_s":"56.055999755859375"},{"end_s":"58.091365814208984","sampled_index":"29","source_frame":"1740","start_s":"58.05799865722656"},{"end_s":"60.09336853027344","sampled_index":"30","source_frame":"1800","start_s":"60.060001373291016"},{"end_s":"62.095367431640625","sampled_index":"31","source_frame":"1860","start_s":"62.0620002746582"},{"end_s":"64.09736633300781","sampled_index":"32","source_frame":"1920","start_s":"64.06400299072266"},{"end_s":"66.099365234375","sampled_index":"33","source_frame":"1980","start_s":"66.06600189208984"},{"end_s":"68.10136413574219","sampled_index":"34","source_frame":"2040","start_s":"68.06800079345703"},{"end_s":"70.10336303710938","sampled_index":"35","source_frame":"2100","start_s":"70.06999969482422"},{"end_s":"72.1053695678711","sampled_index":"36","source_frame":"2160","start_s":"72.0719985961914"},{"end_s":"74.10736846923828","sampled_index":"37","source_frame":"2220","start_s":"74.0739974975586"},{"end_s":"76.10936737060547","sampled_index":"38","source_frame":"2280","start_s":"76.07599639892578"},{"end_s":"78.11136627197266","sampled_index":"39","source_frame":"2340","start_s":"78.0780029296875"},{"end_s":"80.11336517333984","sampled_index":"40","source_frame":"2400","start_s":"80.08000183105469"},{"end_s":"82.11536407470703","sampled_index":"41","source_frame":"2460","start_s":"82.08200073242188"},{"end_s":"84.11736297607422","sampled_index":"42","source_frame":"2520","start_s":"84.08399963378906"},{"end_s":"86.11936950683594","sampled_index":"43","source_frame":"2580","start_s":"86.08599853515625"},{"end_s":"88.12136840820312","sampled_index":"44","source_frame":"2640","start_s":"88.08799743652344"},{"end_s":"90.12336730957031","sampled_index":"45","source_frame":"2700","start_s":"90.08999633789062"},{"end_s":"92.1253662109375","sampled_index":"46","source_frame":"2760","start_s":"92.09200286865234"},{"end_s":"94.12736511230469","sampled_index":"47","source_frame":"2820","start_s":"94.09400177001953"},{"end_s":"96.12936401367188","sampled_index":"48","source_frame":"2880","start_s":"96.09600067138672"},{"end_s":"98.13136291503906","sampled_index":"49","source_frame":"2940","start_s":"98.0979995727539"},{"end_s":"100.13336944580078","sampled_index":"50","source_frame":"3000","start_s":"100.0999984741211"},{"end_s":"102.13536834716797","sampled_index":"51","source_frame":"3060","start_s":"102.10199737548828"},{"end_s":"104.13736724853516","sampled_index":"52","source_frame":"3120","start_s":"104.10399627685547"},{"end_s":"106.13936614990234","sampled_index":"53","source_frame":"3180","start_s":"106.10600280761719"},{"end_s":"108.14136505126953","sampled_index":"54","source_frame":"3240","start_s":"108.10800170898438"},{"end_s":"110.14336395263672","sampled_index":"55","source_frame":"3300","start_s":"110.11000061035156"},{"end_s":"112.1453628540039","sampled_index":"56","source_frame":"3360","start_s":"112.11199951171875"},{"end_s":"114.14736938476562","sampled_index":"57","source_frame":"3420","start_s":"114.11399841308594"},{"end_s":"116.14936828613281","sampled_index":"58","source_frame":"3480","start_s":"116.11599731445312"},{"end_s":"118.1513671875","sampled_index":"59","source_frame":"3540","start_s":"118.11799621582031"},{"end_s":"120.15336608886719","sampled_index":"60","source_frame":"3600","start_s":"120.12000274658203"},{"end_s":"122.15536499023438","sampled_index":"61","source_frame":"3660","start_s":"122.12200164794922"},{"end_s":"124.15736389160156","sampled_index":"62","source_frame":"3720","start_s":"124.1240005493164"},{"end_s":"126.15937042236328","sampled_index":"63","source_frame":"3780","start_s":"126.1259994506836"},{"end_s":"128.16136169433594","sampled_index":"64","source_frame":"3840","start_s":"128.1280059814453"},{"end_s":"130.16336059570312","sampled_index":"65","source_frame":"3900","start_s":"130.1300048828125"},{"end_s":"132.1653594970703","sampled_index":"66","source_frame":"3960","start_s":"132.1320037841797"},{"end_s":"134.16737365722656","sampled_index":"67","source_frame":"4020","start_s":"134.13400268554688"},{"end_s":"136.16937255859375","sampled_index":"68","source_frame":"4080","start_s":"136.13600158691406"},{"end_s":"138.17137145996094","sampled_index":"69","source_frame":"4140","start_s":"138.13800048828125"},{"end_s":"140.17337036132812","sampled_index":"70","source_frame":"4200","start_s":"140.13999938964844"},{"end_s":"142.1753692626953","sampled_index":"71","source_frame":"4260","start_s":"142.14199829101562"},{"end_s":"144.1773681640625","sampled_index":"72","source_frame":"4320","start_s":"144.1439971923828"},{"end_s":"146.1793670654297","sampled_index":"73","source_frame":"4380","start_s":"146.14599609375"},{"end_s":"148.18136596679688","sampled_index":"74","source_frame":"4440","start_s":"148.1479949951172"},{"end_s":"150.18336486816406","sampled_index":"75","source_frame":"4500","start_s":"150.14999389648438"},{"end_s":"152.18536376953125","sampled_index":"76","source_frame":"4560","start_s":"152.15199279785156"},{"end_s":"154.18736267089844","sampled_index":"77","source_frame":"4620","start_s":"154.1540069580078"},{"end_s":"156.18936157226562","sampled_index":"78","source_frame":"4680","start_s":"156.156005859375"},{"end_s":"158.1913604736328","sampled_index":"79","source_frame":"4740","start_s":"158.1580047607422"},{"end_s":"160.193359375","sampled_index":"80","source_frame":"4800","start_s":"160.16000366210938"},{"end_s":"162.19537353515625","sampled_index":"81","source_frame":"4860","start_s":"162.16200256347656"},{"end_s":"164.19737243652344","sampled_index":"82","source_frame":"4920","start_s":"164.16400146484375"},{"end_s":"166.19937133789062","sampled_index":"83","source_frame":"4980","start_s":"166.16600036621094"},{"end_s":"168.2013702392578","sampled_index":"84","source_frame":"5040","start_s":"168.16799926757812"},{"end_s":"170.203369140625","sampled_index":"85","source_frame":"5100","start_s":"170.1699981689453"},{"end_s":"172.2053680419922","sampled_index":"86","source_frame":"5160","start_s":"172.1719970703125"},{"end_s":"174.20736694335938","sampled_index":"87","source_frame":"5220","start_s":"174.1739959716797"},{"end_s":"176.20936584472656","sampled_index":"88","source_frame":"5280","start_s":"176.17599487304688"},{"end_s":"178.21136474609375","sampled_index":"89","source_frame":"5340","start_s":"178.17799377441406"},{"end_s":"180.21336364746094","sampled_index":"90","source_frame":"5400","start_s":"180.17999267578125"},{"end_s":"182.21536254882812","sampled_index":"91","source_frame":"5460","start_s":"182.1820068359375"},{"end_s":"184.2173614501953","sampled_index":"92","source_frame":"5520","start_s":"184.1840057373047"},{"end_s":"186.2193603515625","sampled_index":"93","source_frame":"5580","start_s":"186.18600463867188"},{"end_s":"188.22137451171875","sampled_index":"94","source_frame":"5640","start_s":"188.18800354003906"},{"end_s":"190.22337341308594","sampled_index":"95","source_frame":"5700","start_s":"190.19000244140625"},{"end_s":"192.22537231445312","sampled_index":"96","source_frame":"5760","start_s":"192.19200134277344"},{"end_s":"194.2273712158203","sampled_index":"97","source_frame":"5820","start_s":"194.19400024414062"},{"end_s":"196.22938537597656","sampled_index":"98","source_frame":"5880","start_s":"196.1959991455078"},{"end_s":"198.23138427734375","sampled_index":"99","source_frame":"5940","start_s":"198.197998046875"},{"end_s":"200.23338317871094","sampled_index":"100","source_frame":"6000","start_s":"200.1999969482422"},{"end_s":"202.23538208007812","sampled_index":"101","source_frame":"6060","start_s":"202.20199584960938"},{"end_s":"204.2373809814453","sampled_index":"102","source_frame":"6120","start_s":"204.20399475097656"},{"end_s":"206.2393798828125","sampled_index":"103","source_frame":"6180","start_s":"206.20599365234375"},{"end_s":"208.2413787841797","sampled_index":"104","source_frame":"6240","start_s":"208.20799255371094"},{"end_s":"210.24337768554688","sampled_index":"105","source_frame":"6300","start_s":"210.2100067138672"},{"end_s":"212.24537658691406","sampled_index":"106","source_frame":"6360","start_s":"212.21200561523438"},{"end_s":"214.24737548828125","sampled_index":"107","source_frame":"6420","start_s":"214.21400451660156"},{"end_s":"216.24937438964844","sampled_index":"108","source_frame":"6480","start_s":"216.21600341796875"},{"end_s":"218.25137329101562","sampled_index":"109","source_frame":"6540","start_s":"218.21800231933594"},{"end_s":"220.2533721923828","sampled_index":"110","source_frame":"6600","start_s":"220.22000122070312"},{"end_s":"222.25537109375","sampled_index":"111","source_frame":"6660","start_s":"222.2220001220703"},{"end_s":"224.25738525390625","sampled_index":"112","source_frame":"6720","start_s":"224.2239990234375"},{"end_s":"226.25938415527344","sampled_index":"113","source_frame":"6780","start_s":"226.2259979248047"},{"end_s":"228.26138305664062","sampled_index":"114","source_frame":"6840","start_s":"228.22799682617188"},{"end_s":"230.2633819580078","sampled_index":"115","source_frame":"6900","start_s":"230.22999572753906"},{"end_s":"232.265380859375","sampled_index":"116","source_frame":"6960","start_s":"232.23199462890625"},{"end_s":"234.2673797607422","sampled_index":"117","source_frame":"7020","start_s":"234.23399353027344"},{"end_s":"236.26937866210938","sampled_index":"118","source_frame":"7080","start_s":"236.23599243164062"},{"end_s":"238.27137756347656","sampled_index":"119","source_frame":"7140","start_s":"238.23800659179688"},{"end_s":"240.27337646484375","sampled_index":"120","source_frame":"7200","start_s":"240.24000549316406"},{"end_s":"242.27537536621094","sampled_index":"121","source_frame":"7260","start_s":"242.24200439453125"},{"end_s":"244.27737426757812","sampled_index":"122","source_frame":"7320","start_s":"244.24400329589844"},{"end_s":"246.2793731689453","sampled_index":"123","source_frame":"7380","start_s":"246.24600219726562"},{"end_s":"248.2813720703125","sampled_index":"124","source_frame":"7440","start_s":"248.2480010986328"},{"end_s":"250.2833709716797","sampled_index":"125","source_frame":"7500","start_s":"250.25"},{"end_s":"252.28538513183594","sampled_index":"126","source_frame":"7560","start_s":"252.2519989013672"},{"end_s":"254.28738403320312","sampled_index":"127","source_frame":"7620","start_s":"254.25399780273438"},{"end_s":"256.28936767578125","sampled_index":"128","source_frame":"7680","start_s":"256.2560119628906"},{"end_s":"258.2913818359375","sampled_index":"129","source_frame":"7740","start_s":"258.25799560546875"},{"end_s":"260.2933654785156","sampled_index":"130","source_frame":"7800","start_s":"260.260009765625"},{"end_s":"262.2953796386719","sampled_index":"131","source_frame":"7860","start_s":"262.2619934082031"},{"end_s":"264.29736328125","sampled_index":"132","source_frame":"7920","start_s":"264.2640075683594"},{"end_s":"266.29937744140625","sampled_index":"133","source_frame":"7980","start_s":"266.2659912109375"},{"end_s":"268.3013916015625","sampled_index":"134","source_frame":"8040","start_s":"268.26800537109375"},{"end_s":"270.3033752441406","sampled_index":"135","source_frame":"8100","start_s":"270.2699890136719"},{"end_s":"272.3053894042969","sampled_index":"136","source_frame":"8160","start_s":"272.2720031738281"},{"end_s":"274.307373046875","sampled_index":"137","source_frame":"8220","start_s":"274.27398681640625"},{"end_s":"276.30938720703125","sampled_index":"138","source_frame":"8280","start_s":"276.2760009765625"},{"end_s":"278.3113708496094","sampled_index":"139","source_frame":"8340","start_s":"278.27801513671875"},{"end_s":"280.3133850097656","sampled_index":"140","source_frame":"8400","start_s":"280.2799987792969"},{"end_s":"282.31536865234375","sampled_index":"141","source_frame":"8460","start_s":"282.2820129394531"},{"end_s":"284.3173828125","sampled_index":"142","source_frame":"8520","start_s":"284.28399658203125"},{"end_s":"286.3193664550781","sampled_index":"143","source_frame":"8580","start_s":"286.2860107421875"},{"end_s":"288.3213806152344","sampled_index":"144","source_frame":"8640","start_s":"288.2879943847656"},{"end_s":"290.3233642578125","sampled_index":"145","source_frame":"8700","start_s":"290.2900085449219"},{"end_s":"292.32537841796875","sampled_index":"146","source_frame":"8760","start_s":"292.2919921875"},{"end_s":"294.327392578125","sampled_index":"147","source_frame":"8820","start_s":"294.29400634765625"},{"end_s":"296.3293762207031","sampled_index":"148","source_frame":"8880","start_s":"296.2959899902344"},{"end_s":"298.3313903808594","sampled_index":"149","source_frame":"8940","start_s":"298.2980041503906"},{"end_s":"300.3333740234375","sampled_index":"150","source_frame":"9000","start_s":"300.29998779296875"},{"end_s":"302.33538818359375","sampled_index":"151","source_frame":"9060","start_s":"302.302001953125"},{"end_s":"304.3373718261719","sampled_index":"152","source_frame":"9120","start_s":"304.3039855957031"},{"end_s":"306.3393859863281","sampled_index":"153","source_frame":"9180","start_s":"306.3059997558594"},{"end_s":"308.34136962890625","sampled_index":"154","source_frame":"9240","start_s":"308.3080139160156"},{"end_s":"310.3433837890625","sampled_index":"155","source_frame":"9300","start_s":"310.30999755859375"},{"end_s":"312.3453674316406","sampled_index":"156","source_frame":"9360","start_s":"312.31201171875"},{"end_s":"314.3473815917969","sampled_index":"157","source_frame":"9420","start_s":"314.3139953613281"},{"end_s":"316.349365234375","sampled_index":"158","source_frame":"9480","start_s":"316.3160095214844"},{"end_s":"318.35137939453125","sampled_index":"159","source_frame":"9540","start_s":"318.3179931640625"},{"end_s":"320.3533630371094","sampled_index":"160","source_frame":"9600","start_s":"320.32000732421875"},{"end_s":"322.3553771972656","sampled_index":"161","source_frame":"9660","start_s":"322.3219909667969"},{"end_s":"324.3573913574219","sampled_index":"162","source_frame":"9720","start_s":"324.3240051269531"},{"end_s":"326.359375","sampled_index":"163","source_frame":"9780","start_s":"326.32598876953125"},{"end_s":"328.36138916015625","sampled_index":"164","source_frame":"9840","start_s":"328.3280029296875"},{"end_s":"330.3633728027344","sampled_index":"165","source_frame":"9900","start_s":"330.3299865722656"},{"end_s":"332.3653869628906","sampled_index":"166","source_frame":"9960","start_s":"332.3320007324219"},{"end_s":"334.36737060546875","sampled_index":"167","source_frame":"10020","start_s":"334.3340148925781"},{"end_s":"336.369384765625","sampled_index":"168","source_frame":"10080","start_s":"336.33599853515625"},{"end_s":"338.3713684082031","sampled_index":"169","source_frame":"10140","start_s":"338.3380126953125"},{"end_s":"340.3733825683594","sampled_index":"170","source_frame":"10200","start_s":"340.3399963378906"}],"video_id":456,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_219_real.mp4","video_sha256":"abcfb2648dc865cf030d04f75f7f5f0b778fe8620ed6619a0003b56e9c68c05f"},{"conversations":[{"answer":"Yellow and Red.","choices":["Green and blue.","Red and blue.","Yellow and Red.","Brown."],"end_time":12.0,"question":"What colors are the wheels of the race car?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"00:00:12"},{"answer":"A happy face.","choices":["A sad face.","A happy face.","A star.","A heart."],"end_time":72.0,"question":"What feature is present on the front of the toy?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"00:01:12"},{"answer":"In a straight line.","choices":["In a circle.","In a straight line.","In a random pattern.","In a triangular shape."],"end_time":128.0,"question":"How are the yellow building blocks arranged before the blue block is placed on top?","required_ability":"working memory","task":"Spatial Understanding","time_stamp":"0:02:08"},{"answer":"A small structure made of yellow and red blocks with wheels.","choices":["A small structure made of blue and green blocks.","A small structure made of yellow and red blocks with wheels.","A toy vehicle with blue and yellow wheels.","An orange and green interlocking block structure."],"end_time":202.0,"question":"What is the new item introduced in the video?","required_ability":"working memory","task":"Object Recognition","time_stamp":"00:03:22"},{"answer":"The toy vehicle start circling around the table.","choices":["The toy vehicle is disassembled.","The toy vehicle moves to the right.","The toy vehicle start circling around the table.","The toy vehicle is lifted off the surface."],"end_time":227.0,"question":"What happened to the toy vehicle just now?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:03:47"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"2f4946a0412eb0352f2245ae654716d5ce2f622e4194e4034c06158666a6cea7","start":0,"stop":6,"tokens":1176},{"file":"chunk_0001.pt","qa_index":1,"sha256":"4beab4a7498033874ba54e0653c9380d532da98bb0de43e4f6e97e805986307a","start":6,"stop":22,"tokens":3136},{"file":"chunk_0002.pt","qa_index":1,"sha256":"7a46e093bc4ab5e515f3ccf75bf5198ba174c472a63f89a171b0aed01d347334","start":22,"stop":36,"tokens":2744},{"file":"chunk_0003.pt","qa_index":2,"sha256":"0a6c2563b1d6852965eb5c7cf69220c60c2fd39e183cbc1236615fb502ef74f6","start":36,"stop":52,"tokens":3136},{"file":"chunk_0004.pt","qa_index":2,"sha256":"cdeb6fa6e4782481a8ebf52c8ba9cc2b2edbd0efd704e3e8769f121c6bcd7555","start":52,"stop":64,"tokens":2352},{"file":"chunk_0005.pt","qa_index":3,"sha256":"21fa22c04c8ce5b4bbdf86032fc1035a0b7275871ccdfd773a026e9bf3349e2c","start":64,"stop":80,"tokens":3136},{"file":"chunk_0006.pt","qa_index":3,"sha256":"5cedbd1a07ab02956c7affd13dd6bf671d6c62f490ce6322d02c01be1976b18a","start":80,"stop":96,"tokens":3136},{"file":"chunk_0007.pt","qa_index":3,"sha256":"b03dd3d7bb99febb2f95ce2904008e4016942995adb83c4ea7484a17797c6d14","start":96,"stop":101,"tokens":980},{"file":"chunk_0008.pt","qa_index":4,"sha256":"79cc9aad80644c8527ddee3a011e0f31adbbc356e5c3705d5ce3f7adc0c53fd1","start":101,"stop":114,"tokens":2548}],"identity":"688b7668eedd70aa07f16f890d178864aaa10df904e886f9b0eb070405e21e51","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":21.939352111890912,"sample_id":202,"sampled_frames_available":121,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"cef4d37f6bc2fe533d8c3264d1f00cc47dccdf3fe92b06113150966d8acfa668"},"sample_id":202,"timestamps":[{"end_s":"0.03333333507180214","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0333333015441895","sampled_index":"1","source_frame":"60","start_s":"2.0"},{"end_s":"4.0333333015441895","sampled_index":"2","source_frame":"120","start_s":"4.0"},{"end_s":"6.0333333015441895","sampled_index":"3","source_frame":"180","start_s":"6.0"},{"end_s":"8.033333778381348","sampled_index":"4","source_frame":"240","start_s":"8.0"},{"end_s":"10.033333778381348","sampled_index":"5","source_frame":"300","start_s":"10.0"},{"end_s":"12.033333778381348","sampled_index":"6","source_frame":"360","start_s":"12.0"},{"end_s":"14.033333778381348","sampled_index":"7","source_frame":"420","start_s":"14.0"},{"end_s":"16.03333282470703","sampled_index":"8","source_frame":"480","start_s":"16.0"},{"end_s":"18.03333282470703","sampled_index":"9","source_frame":"540","start_s":"18.0"},{"end_s":"20.03333282470703","sampled_index":"10","source_frame":"600","start_s":"20.0"},{"end_s":"22.03333282470703","sampled_index":"11","source_frame":"660","start_s":"22.0"},{"end_s":"24.03333282470703","sampled_index":"12","source_frame":"720","start_s":"24.0"},{"end_s":"26.03333282470703","sampled_index":"13","source_frame":"780","start_s":"26.0"},{"end_s":"28.03333282470703","sampled_index":"14","source_frame":"840","start_s":"28.0"},{"end_s":"30.03333282470703","sampled_index":"15","source_frame":"900","start_s":"30.0"},{"end_s":"32.03333282470703","sampled_index":"16","source_frame":"960","start_s":"32.0"},{"end_s":"34.03333282470703","sampled_index":"17","source_frame":"1020","start_s":"34.0"},{"end_s":"36.03333282470703","sampled_index":"18","source_frame":"1080","start_s":"36.0"},{"end_s":"38.03333282470703","sampled_index":"19","source_frame":"1140","start_s":"38.0"},{"end_s":"40.03333282470703","sampled_index":"20","source_frame":"1200","start_s":"40.0"},{"end_s":"42.03333282470703","sampled_index":"21","source_frame":"1260","start_s":"42.0"},{"end_s":"44.03333282470703","sampled_index":"22","source_frame":"1320","start_s":"44.0"},{"end_s":"46.03333282470703","sampled_index":"23","source_frame":"1380","start_s":"46.0"},{"end_s":"48.03333282470703","sampled_index":"24","source_frame":"1440","start_s":"48.0"},{"end_s":"50.03333282470703","sampled_index":"25","source_frame":"1500","start_s":"50.0"},{"end_s":"52.03333282470703","sampled_index":"26","source_frame":"1560","start_s":"52.0"},{"end_s":"54.03333282470703","sampled_index":"27","source_frame":"1620","start_s":"54.0"},{"end_s":"56.03333282470703","sampled_index":"28","source_frame":"1680","start_s":"56.0"},{"end_s":"58.03333282470703","sampled_index":"29","source_frame":"1740","start_s":"58.0"},{"end_s":"60.03333282470703","sampled_index":"30","source_frame":"1800","start_s":"60.0"},{"end_s":"62.03333282470703","sampled_index":"31","source_frame":"1860","start_s":"62.0"},{"end_s":"64.03333282470703","sampled_index":"32","source_frame":"1920","start_s":"64.0"},{"end_s":"66.03333282470703","sampled_index":"33","source_frame":"1980","start_s":"66.0"},{"end_s":"68.03333282470703","sampled_index":"34","source_frame":"2040","start_s":"68.0"},{"end_s":"70.03333282470703","sampled_index":"35","source_frame":"2100","start_s":"70.0"},{"end_s":"72.03333282470703","sampled_index":"36","source_frame":"2160","start_s":"72.0"},{"end_s":"74.03333282470703","sampled_index":"37","source_frame":"2220","start_s":"74.0"},{"end_s":"76.03333282470703","sampled_index":"38","source_frame":"2280","start_s":"76.0"},{"end_s":"78.03333282470703","sampled_index":"39","source_frame":"2340","start_s":"78.0"},{"end_s":"80.03333282470703","sampled_index":"40","source_frame":"2400","start_s":"80.0"},{"end_s":"82.03333282470703","sampled_index":"41","source_frame":"2460","start_s":"82.0"},{"end_s":"84.03333282470703","sampled_index":"42","source_frame":"2520","start_s":"84.0"},{"end_s":"86.03333282470703","sampled_index":"43","source_frame":"2580","start_s":"86.0"},{"end_s":"88.03333282470703","sampled_index":"44","source_frame":"2640","start_s":"88.0"},{"end_s":"90.03333282470703","sampled_index":"45","source_frame":"2700","start_s":"90.0"},{"end_s":"92.03333282470703","sampled_index":"46","source_frame":"2760","start_s":"92.0"},{"end_s":"94.03333282470703","sampled_index":"47","source_frame":"2820","start_s":"94.0"},{"end_s":"96.03333282470703","sampled_index":"48","source_frame":"2880","start_s":"96.0"},{"end_s":"98.03333282470703","sampled_index":"49","source_frame":"2940","start_s":"98.0"},{"end_s":"100.03333282470703","sampled_index":"50","source_frame":"3000","start_s":"100.0"},{"end_s":"102.03333282470703","sampled_index":"51","source_frame":"3060","start_s":"102.0"},{"end_s":"104.03333282470703","sampled_index":"52","source_frame":"3120","start_s":"104.0"},{"end_s":"106.03333282470703","sampled_index":"53","source_frame":"3180","start_s":"106.0"},{"end_s":"108.03333282470703","sampled_index":"54","source_frame":"3240","start_s":"108.0"},{"end_s":"110.03333282470703","sampled_index":"55","source_frame":"3300","start_s":"110.0"},{"end_s":"112.03333282470703","sampled_index":"56","source_frame":"3360","start_s":"112.0"},{"end_s":"114.03333282470703","sampled_index":"57","source_frame":"3420","start_s":"114.0"},{"end_s":"116.03333282470703","sampled_index":"58","source_frame":"3480","start_s":"116.0"},{"end_s":"118.03333282470703","sampled_index":"59","source_frame":"3540","start_s":"118.0"},{"end_s":"120.03333282470703","sampled_index":"60","source_frame":"3600","start_s":"120.0"},{"end_s":"122.03333282470703","sampled_index":"61","source_frame":"3660","start_s":"122.0"},{"end_s":"124.03333282470703","sampled_index":"62","source_frame":"3720","start_s":"124.0"},{"end_s":"126.03333282470703","sampled_index":"63","source_frame":"3780","start_s":"126.0"},{"end_s":"128.03334045410156","sampled_index":"64","source_frame":"3840","start_s":"128.0"},{"end_s":"130.03334045410156","sampled_index":"65","source_frame":"3900","start_s":"130.0"},{"end_s":"132.03334045410156","sampled_index":"66","source_frame":"3960","start_s":"132.0"},{"end_s":"134.03334045410156","sampled_index":"67","source_frame":"4020","start_s":"134.0"},{"end_s":"136.03334045410156","sampled_index":"68","source_frame":"4080","start_s":"136.0"},{"end_s":"138.03334045410156","sampled_index":"69","source_frame":"4140","start_s":"138.0"},{"end_s":"140.03334045410156","sampled_index":"70","source_frame":"4200","start_s":"140.0"},{"end_s":"142.03334045410156","sampled_index":"71","source_frame":"4260","start_s":"142.0"},{"end_s":"144.03334045410156","sampled_index":"72","source_frame":"4320","start_s":"144.0"},{"end_s":"146.03334045410156","sampled_index":"73","source_frame":"4380","start_s":"146.0"},{"end_s":"148.03334045410156","sampled_index":"74","source_frame":"4440","start_s":"148.0"},{"end_s":"150.03334045410156","sampled_index":"75","source_frame":"4500","start_s":"150.0"},{"end_s":"152.03334045410156","sampled_index":"76","source_frame":"4560","start_s":"152.0"},{"end_s":"154.03334045410156","sampled_index":"77","source_frame":"4620","start_s":"154.0"},{"end_s":"156.03334045410156","sampled_index":"78","source_frame":"4680","start_s":"156.0"},{"end_s":"158.03334045410156","sampled_index":"79","source_frame":"4740","start_s":"158.0"},{"end_s":"160.03334045410156","sampled_index":"80","source_frame":"4800","start_s":"160.0"},{"end_s":"162.03334045410156","sampled_index":"81","source_frame":"4860","start_s":"162.0"},{"end_s":"164.03334045410156","sampled_index":"82","source_frame":"4920","start_s":"164.0"},{"end_s":"166.03334045410156","sampled_index":"83","source_frame":"4980","start_s":"166.0"},{"end_s":"168.03334045410156","sampled_index":"84","source_frame":"5040","start_s":"168.0"},{"end_s":"170.03334045410156","sampled_index":"85","source_frame":"5100","start_s":"170.0"},{"end_s":"172.03334045410156","sampled_index":"86","source_frame":"5160","start_s":"172.0"},{"end_s":"174.03334045410156","sampled_index":"87","source_frame":"5220","start_s":"174.0"},{"end_s":"176.03334045410156","sampled_index":"88","source_frame":"5280","start_s":"176.0"},{"end_s":"178.03334045410156","sampled_index":"89","source_frame":"5340","start_s":"178.0"},{"end_s":"180.03334045410156","sampled_index":"90","source_frame":"5400","start_s":"180.0"},{"end_s":"182.03334045410156","sampled_index":"91","source_frame":"5460","start_s":"182.0"},{"end_s":"184.03334045410156","sampled_index":"92","source_frame":"5520","start_s":"184.0"},{"end_s":"186.03334045410156","sampled_index":"93","source_frame":"5580","start_s":"186.0"},{"end_s":"188.03334045410156","sampled_index":"94","source_frame":"5640","start_s":"188.0"},{"end_s":"190.03334045410156","sampled_index":"95","source_frame":"5700","start_s":"190.0"},{"end_s":"192.03334045410156","sampled_index":"96","source_frame":"5760","start_s":"192.0"},{"end_s":"194.03334045410156","sampled_index":"97","source_frame":"5820","start_s":"194.0"},{"end_s":"196.03334045410156","sampled_index":"98","source_frame":"5880","start_s":"196.0"},{"end_s":"198.03334045410156","sampled_index":"99","source_frame":"5940","start_s":"198.0"},{"end_s":"200.03334045410156","sampled_index":"100","source_frame":"6000","start_s":"200.0"},{"end_s":"202.03334045410156","sampled_index":"101","source_frame":"6060","start_s":"202.0"},{"end_s":"204.03334045410156","sampled_index":"102","source_frame":"6120","start_s":"204.0"},{"end_s":"206.03334045410156","sampled_index":"103","source_frame":"6180","start_s":"206.0"},{"end_s":"208.03334045410156","sampled_index":"104","source_frame":"6240","start_s":"208.0"},{"end_s":"210.03334045410156","sampled_index":"105","source_frame":"6300","start_s":"210.0"},{"end_s":"212.03334045410156","sampled_index":"106","source_frame":"6360","start_s":"212.0"},{"end_s":"214.03334045410156","sampled_index":"107","source_frame":"6420","start_s":"214.0"},{"end_s":"216.03334045410156","sampled_index":"108","source_frame":"6480","start_s":"216.0"},{"end_s":"218.03334045410156","sampled_index":"109","source_frame":"6540","start_s":"218.0"},{"end_s":"220.03334045410156","sampled_index":"110","source_frame":"6600","start_s":"220.0"},{"end_s":"222.03334045410156","sampled_index":"111","source_frame":"6660","start_s":"222.0"},{"end_s":"224.03334045410156","sampled_index":"112","source_frame":"6720","start_s":"224.0"},{"end_s":"226.03334045410156","sampled_index":"113","source_frame":"6780","start_s":"226.0"},{"end_s":"228.03334045410156","sampled_index":"114","source_frame":"6840","start_s":"228.0"},{"end_s":"230.03334045410156","sampled_index":"115","source_frame":"6900","start_s":"230.0"},{"end_s":"232.03334045410156","sampled_index":"116","source_frame":"6960","start_s":"232.0"},{"end_s":"234.03334045410156","sampled_index":"117","source_frame":"7020","start_s":"234.0"},{"end_s":"236.03334045410156","sampled_index":"118","source_frame":"7080","start_s":"236.0"},{"end_s":"238.03334045410156","sampled_index":"119","source_frame":"7140","start_s":"238.0"},{"end_s":"240.03334045410156","sampled_index":"120","source_frame":"7200","start_s":"240.0"}],"video_id":459,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_202_real.mp4","video_sha256":"cef4d37f6bc2fe533d8c3264d1f00cc47dccdf3fe92b06113150966d8acfa668"},{"conversations":[{"answer":"Yellow, green.","choices":["Yellow, green.","Yellow, blue.","Red, green, blue.","Yellow, blue, green."],"end_time":7.0,"question":"What are the colors of the crane arm?","required_ability":"working memory","task":"Attribute Recognition","time_stamp":"0:00:07"},{"answer":"It lifts and swings gently.","choices":["It falls to the ground.","It remains stationary.","It lifts and swings gently.","It changes color."],"end_time":19.0,"question":"What happens to the orange block as the video progresses?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"0:00:19"},{"answer":"The hand places a yellow block, moves it, then places a green block next to it.","choices":["The hand first places a green block, then a yellow block, followed by a blue block.","The hand builds a structure with red, green, and blue blocks.","The hand stacks blocks randomly.","The hand places a yellow block, moves it, then places a green block next to it."],"end_time":96.0,"question":"What was the sequence of actions performed with the blocks just now?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:01:36"},{"answer":"It is placed back beside the larger structure.","choices":["It is disassembled.","It is placed on top of the larger structure.","It is left in the foreground.","It is placed back beside the larger structure."],"end_time":171.0,"question":"What happens to the smaller structure after it is first moved out of view?","required_ability":"episodic memory","task":"Event Understanding","time_stamp":"00:02:51"},{"answer":"Nine.","choices":["Ten.","Three.","Nine.","Eight."],"end_time":265.0,"question":"How many green blocks are stacked now?","required_ability":"working memory","task":"Counting","time_stamp":"00:04:25"}],"feature_meta":{"chunks":[{"file":"chunk_0000.pt","qa_index":0,"sha256":"751e198871b7765fc1489aca3bf1a3f7757c9ec5af79e449577412559361c60f","start":0,"stop":4,"tokens":784},{"file":"chunk_0001.pt","qa_index":1,"sha256":"efb61fab4dcc9fc51ba1e1899dcce23d6678559b4f0dd09e964b6edb5606d11a","start":4,"stop":10,"tokens":1176},{"file":"chunk_0002.pt","qa_index":2,"sha256":"c9900793009eec76a11d6cabdff2de08bf5782351e96a028934f71a5ecc83708","start":10,"stop":26,"tokens":3136},{"file":"chunk_0003.pt","qa_index":2,"sha256":"75abbdcb9200b7ea358c75db4a8c8a6a89b7eaf69d5133b50af9ea70f03b5f13","start":26,"stop":42,"tokens":3136},{"file":"chunk_0004.pt","qa_index":2,"sha256":"a5e5b003a821ed7f5dc4fb8102dd66ede4c9ad85d47f61ca2b84202ee51d4bed","start":42,"stop":48,"tokens":1176},{"file":"chunk_0005.pt","qa_index":3,"sha256":"220e3a81945c04bfc70d16bfecf7313411e5469afbf754181da7911fb8b742e6","start":48,"stop":64,"tokens":3136},{"file":"chunk_0006.pt","qa_index":3,"sha256":"cd0bb1cc23ad39c77810411e488b1c0619b6b58accdd36f51acffc160d5e42c3","start":64,"stop":80,"tokens":3136},{"file":"chunk_0007.pt","qa_index":3,"sha256":"f4e67633c57ecaa9b40f60413857b98fb497bf79c865ce4e5214e89d604fa458","start":80,"stop":86,"tokens":1176},{"file":"chunk_0008.pt","qa_index":4,"sha256":"31d845dd3c08349351958edea8d5b9cc84c1354df43f3b8318f58a40119a1759","start":86,"stop":102,"tokens":3136},{"file":"chunk_0009.pt","qa_index":4,"sha256":"e78de9f928528f5a2bf21a24fe120600cebd9817cb3c752f0ceaeda367fa1c1d","start":102,"stop":118,"tokens":3136},{"file":"chunk_0010.pt","qa_index":4,"sha256":"134f84dbdd2cbb7af136952340ef1b8fa2a0b8e8738d0b2af0d52cc49d9bcee0","start":118,"stop":133,"tokens":2940}],"identity":"ea2e3e4e54c0cf208d0307f0dd06df55a3e9802cbf91f502b967d0b510ac9240","note":"Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization","preparation_s":25.82067745923996,"sample_id":208,"sampled_frames_available":164,"sampling_step":60,"source_fps_rounded":30,"source_video_sha256":"59d2c4b8fb7e55bb933fd2fc681736af5c0299a0aafc3890147cbd175947cf77"},"sample_id":208,"timestamps":[{"end_s":"0.03333333507180214","sampled_index":"0","source_frame":"0","start_s":"0.0"},{"end_s":"2.0333333015441895","sampled_index":"1","source_frame":"60","start_s":"2.0"},{"end_s":"4.0333333015441895","sampled_index":"2","source_frame":"120","start_s":"4.0"},{"end_s":"6.0333333015441895","sampled_index":"3","source_frame":"180","start_s":"6.0"},{"end_s":"8.033333778381348","sampled_index":"4","source_frame":"240","start_s":"8.0"},{"end_s":"10.033333778381348","sampled_index":"5","source_frame":"300","start_s":"10.0"},{"end_s":"12.033333778381348","sampled_index":"6","source_frame":"360","start_s":"12.0"},{"end_s":"14.033333778381348","sampled_index":"7","source_frame":"420","start_s":"14.0"},{"end_s":"16.03333282470703","sampled_index":"8","source_frame":"480","start_s":"16.0"},{"end_s":"18.03333282470703","sampled_index":"9","source_frame":"540","start_s":"18.0"},{"end_s":"20.03333282470703","sampled_index":"10","source_frame":"600","start_s":"20.0"},{"end_s":"22.03333282470703","sampled_index":"11","source_frame":"660","start_s":"22.0"},{"end_s":"24.03333282470703","sampled_index":"12","source_frame":"720","start_s":"24.0"},{"end_s":"26.03333282470703","sampled_index":"13","source_frame":"780","start_s":"26.0"},{"end_s":"28.03333282470703","sampled_index":"14","source_frame":"840","start_s":"28.0"},{"end_s":"30.03333282470703","sampled_index":"15","source_frame":"900","start_s":"30.0"},{"end_s":"32.03333282470703","sampled_index":"16","source_frame":"960","start_s":"32.0"},{"end_s":"34.03333282470703","sampled_index":"17","source_frame":"1020","start_s":"34.0"},{"end_s":"36.03333282470703","sampled_index":"18","source_frame":"1080","start_s":"36.0"},{"end_s":"38.03333282470703","sampled_index":"19","source_frame":"1140","start_s":"38.0"},{"end_s":"40.03333282470703","sampled_index":"20","source_frame":"1200","start_s":"40.0"},{"end_s":"42.03333282470703","sampled_index":"21","source_frame":"1260","start_s":"42.0"},{"end_s":"44.03333282470703","sampled_index":"22","source_frame":"1320","start_s":"44.0"},{"end_s":"46.03333282470703","sampled_index":"23","source_frame":"1380","start_s":"46.0"},{"end_s":"48.03333282470703","sampled_index":"24","source_frame":"1440","start_s":"48.0"},{"end_s":"50.03333282470703","sampled_index":"25","source_frame":"1500","start_s":"50.0"},{"end_s":"52.03333282470703","sampled_index":"26","source_frame":"1560","start_s":"52.0"},{"end_s":"54.03333282470703","sampled_index":"27","source_frame":"1620","start_s":"54.0"},{"end_s":"56.03333282470703","sampled_index":"28","source_frame":"1680","start_s":"56.0"},{"end_s":"58.03333282470703","sampled_index":"29","source_frame":"1740","start_s":"58.0"},{"end_s":"60.03333282470703","sampled_index":"30","source_frame":"1800","start_s":"60.0"},{"end_s":"62.03333282470703","sampled_index":"31","source_frame":"1860","start_s":"62.0"},{"end_s":"64.03333282470703","sampled_index":"32","source_frame":"1920","start_s":"64.0"},{"end_s":"66.03333282470703","sampled_index":"33","source_frame":"1980","start_s":"66.0"},{"end_s":"68.03333282470703","sampled_index":"34","source_frame":"2040","start_s":"68.0"},{"end_s":"70.03333282470703","sampled_index":"35","source_frame":"2100","start_s":"70.0"},{"end_s":"72.03333282470703","sampled_index":"36","source_frame":"2160","start_s":"72.0"},{"end_s":"74.03333282470703","sampled_index":"37","source_frame":"2220","start_s":"74.0"},{"end_s":"76.03333282470703","sampled_index":"38","source_frame":"2280","start_s":"76.0"},{"end_s":"78.03333282470703","sampled_index":"39","source_frame":"2340","start_s":"78.0"},{"end_s":"80.03333282470703","sampled_index":"40","source_frame":"2400","start_s":"80.0"},{"end_s":"82.03333282470703","sampled_index":"41","source_frame":"2460","start_s":"82.0"},{"end_s":"84.03333282470703","sampled_index":"42","source_frame":"2520","start_s":"84.0"},{"end_s":"86.03333282470703","sampled_index":"43","source_frame":"2580","start_s":"86.0"},{"end_s":"88.03333282470703","sampled_index":"44","source_frame":"2640","start_s":"88.0"},{"end_s":"90.03333282470703","sampled_index":"45","source_frame":"2700","start_s":"90.0"},{"end_s":"92.03333282470703","sampled_index":"46","source_frame":"2760","start_s":"92.0"},{"end_s":"94.03333282470703","sampled_index":"47","source_frame":"2820","start_s":"94.0"},{"end_s":"96.03333282470703","sampled_index":"48","source_frame":"2880","start_s":"96.0"},{"end_s":"98.03333282470703","sampled_index":"49","source_frame":"2940","start_s":"98.0"},{"end_s":"100.03333282470703","sampled_index":"50","source_frame":"3000","start_s":"100.0"},{"end_s":"102.03333282470703","sampled_index":"51","source_frame":"3060","start_s":"102.0"},{"end_s":"104.03333282470703","sampled_index":"52","source_frame":"3120","start_s":"104.0"},{"end_s":"106.03333282470703","sampled_index":"53","source_frame":"3180","start_s":"106.0"},{"end_s":"108.03333282470703","sampled_index":"54","source_frame":"3240","start_s":"108.0"},{"end_s":"110.03333282470703","sampled_index":"55","source_frame":"3300","start_s":"110.0"},{"end_s":"112.03333282470703","sampled_index":"56","source_frame":"3360","start_s":"112.0"},{"end_s":"114.03333282470703","sampled_index":"57","source_frame":"3420","start_s":"114.0"},{"end_s":"116.03333282470703","sampled_index":"58","source_frame":"3480","start_s":"116.0"},{"end_s":"118.03333282470703","sampled_index":"59","source_frame":"3540","start_s":"118.0"},{"end_s":"120.03333282470703","sampled_index":"60","source_frame":"3600","start_s":"120.0"},{"end_s":"122.03333282470703","sampled_index":"61","source_frame":"3660","start_s":"122.0"},{"end_s":"124.03333282470703","sampled_index":"62","source_frame":"3720","start_s":"124.0"},{"end_s":"126.03333282470703","sampled_index":"63","source_frame":"3780","start_s":"126.0"},{"end_s":"128.03334045410156","sampled_index":"64","source_frame":"3840","start_s":"128.0"},{"end_s":"130.03334045410156","sampled_index":"65","source_frame":"3900","start_s":"130.0"},{"end_s":"132.03334045410156","sampled_index":"66","source_frame":"3960","start_s":"132.0"},{"end_s":"134.03334045410156","sampled_index":"67","source_frame":"4020","start_s":"134.0"},{"end_s":"136.03334045410156","sampled_index":"68","source_frame":"4080","start_s":"136.0"},{"end_s":"138.03334045410156","sampled_index":"69","source_frame":"4140","start_s":"138.0"},{"end_s":"140.03334045410156","sampled_index":"70","source_frame":"4200","start_s":"140.0"},{"end_s":"142.03334045410156","sampled_index":"71","source_frame":"4260","start_s":"142.0"},{"end_s":"144.03334045410156","sampled_index":"72","source_frame":"4320","start_s":"144.0"},{"end_s":"146.03334045410156","sampled_index":"73","source_frame":"4380","start_s":"146.0"},{"end_s":"148.03334045410156","sampled_index":"74","source_frame":"4440","start_s":"148.0"},{"end_s":"150.03334045410156","sampled_index":"75","source_frame":"4500","start_s":"150.0"},{"end_s":"152.03334045410156","sampled_index":"76","source_frame":"4560","start_s":"152.0"},{"end_s":"154.03334045410156","sampled_index":"77","source_frame":"4620","start_s":"154.0"},{"end_s":"156.03334045410156","sampled_index":"78","source_frame":"4680","start_s":"156.0"},{"end_s":"158.03334045410156","sampled_index":"79","source_frame":"4740","start_s":"158.0"},{"end_s":"160.03334045410156","sampled_index":"80","source_frame":"4800","start_s":"160.0"},{"end_s":"162.03334045410156","sampled_index":"81","source_frame":"4860","start_s":"162.0"},{"end_s":"164.03334045410156","sampled_index":"82","source_frame":"4920","start_s":"164.0"},{"end_s":"166.03334045410156","sampled_index":"83","source_frame":"4980","start_s":"166.0"},{"end_s":"168.03334045410156","sampled_index":"84","source_frame":"5040","start_s":"168.0"},{"end_s":"170.03334045410156","sampled_index":"85","source_frame":"5100","start_s":"170.0"},{"end_s":"172.03334045410156","sampled_index":"86","source_frame":"5160","start_s":"172.0"},{"end_s":"174.03334045410156","sampled_index":"87","source_frame":"5220","start_s":"174.0"},{"end_s":"176.03334045410156","sampled_index":"88","source_frame":"5280","start_s":"176.0"},{"end_s":"178.03334045410156","sampled_index":"89","source_frame":"5340","start_s":"178.0"},{"end_s":"180.03334045410156","sampled_index":"90","source_frame":"5400","start_s":"180.0"},{"end_s":"182.03334045410156","sampled_index":"91","source_frame":"5460","start_s":"182.0"},{"end_s":"184.03334045410156","sampled_index":"92","source_frame":"5520","start_s":"184.0"},{"end_s":"186.03334045410156","sampled_index":"93","source_frame":"5580","start_s":"186.0"},{"end_s":"188.03334045410156","sampled_index":"94","source_frame":"5640","start_s":"188.0"},{"end_s":"190.03334045410156","sampled_index":"95","source_frame":"5700","start_s":"190.0"},{"end_s":"192.03334045410156","sampled_index":"96","source_frame":"5760","start_s":"192.0"},{"end_s":"194.03334045410156","sampled_index":"97","source_frame":"5820","start_s":"194.0"},{"end_s":"196.03334045410156","sampled_index":"98","source_frame":"5880","start_s":"196.0"},{"end_s":"198.03334045410156","sampled_index":"99","source_frame":"5940","start_s":"198.0"},{"end_s":"200.03334045410156","sampled_index":"100","source_frame":"6000","start_s":"200.0"},{"end_s":"202.03334045410156","sampled_index":"101","source_frame":"6060","start_s":"202.0"},{"end_s":"204.03334045410156","sampled_index":"102","source_frame":"6120","start_s":"204.0"},{"end_s":"206.03334045410156","sampled_index":"103","source_frame":"6180","start_s":"206.0"},{"end_s":"208.03334045410156","sampled_index":"104","source_frame":"6240","start_s":"208.0"},{"end_s":"210.03334045410156","sampled_index":"105","source_frame":"6300","start_s":"210.0"},{"end_s":"212.03334045410156","sampled_index":"106","source_frame":"6360","start_s":"212.0"},{"end_s":"214.03334045410156","sampled_index":"107","source_frame":"6420","start_s":"214.0"},{"end_s":"216.03334045410156","sampled_index":"108","source_frame":"6480","start_s":"216.0"},{"end_s":"218.03334045410156","sampled_index":"109","source_frame":"6540","start_s":"218.0"},{"end_s":"220.03334045410156","sampled_index":"110","source_frame":"6600","start_s":"220.0"},{"end_s":"222.03334045410156","sampled_index":"111","source_frame":"6660","start_s":"222.0"},{"end_s":"224.03334045410156","sampled_index":"112","source_frame":"6720","start_s":"224.0"},{"end_s":"226.03334045410156","sampled_index":"113","source_frame":"6780","start_s":"226.0"},{"end_s":"228.03334045410156","sampled_index":"114","source_frame":"6840","start_s":"228.0"},{"end_s":"230.03334045410156","sampled_index":"115","source_frame":"6900","start_s":"230.0"},{"end_s":"232.03334045410156","sampled_index":"116","source_frame":"6960","start_s":"232.0"},{"end_s":"234.03334045410156","sampled_index":"117","source_frame":"7020","start_s":"234.0"},{"end_s":"236.03334045410156","sampled_index":"118","source_frame":"7080","start_s":"236.0"},{"end_s":"238.03334045410156","sampled_index":"119","source_frame":"7140","start_s":"238.0"},{"end_s":"240.03334045410156","sampled_index":"120","source_frame":"7200","start_s":"240.0"},{"end_s":"242.03334045410156","sampled_index":"121","source_frame":"7260","start_s":"242.0"},{"end_s":"244.03334045410156","sampled_index":"122","source_frame":"7320","start_s":"244.0"},{"end_s":"246.03334045410156","sampled_index":"123","source_frame":"7380","start_s":"246.0"},{"end_s":"248.03334045410156","sampled_index":"124","source_frame":"7440","start_s":"248.0"},{"end_s":"250.03334045410156","sampled_index":"125","source_frame":"7500","start_s":"250.0"},{"end_s":"252.03334045410156","sampled_index":"126","source_frame":"7560","start_s":"252.0"},{"end_s":"254.03334045410156","sampled_index":"127","source_frame":"7620","start_s":"254.0"},{"end_s":"256.0333251953125","sampled_index":"128","source_frame":"7680","start_s":"256.0"},{"end_s":"258.0333251953125","sampled_index":"129","source_frame":"7740","start_s":"258.0"},{"end_s":"260.0333251953125","sampled_index":"130","source_frame":"7800","start_s":"260.0"},{"end_s":"262.0333251953125","sampled_index":"131","source_frame":"7860","start_s":"262.0"},{"end_s":"264.0333251953125","sampled_index":"132","source_frame":"7920","start_s":"264.0"},{"end_s":"266.0333251953125","sampled_index":"133","source_frame":"7980","start_s":"266.0"},{"end_s":"268.0333251953125","sampled_index":"134","source_frame":"8040","start_s":"268.0"},{"end_s":"270.0333251953125","sampled_index":"135","source_frame":"8100","start_s":"270.0"},{"end_s":"272.0333251953125","sampled_index":"136","source_frame":"8160","start_s":"272.0"},{"end_s":"274.0333251953125","sampled_index":"137","source_frame":"8220","start_s":"274.0"},{"end_s":"276.0333251953125","sampled_index":"138","source_frame":"8280","start_s":"276.0"},{"end_s":"278.0333251953125","sampled_index":"139","source_frame":"8340","start_s":"278.0"},{"end_s":"280.0333251953125","sampled_index":"140","source_frame":"8400","start_s":"280.0"},{"end_s":"282.0333251953125","sampled_index":"141","source_frame":"8460","start_s":"282.0"},{"end_s":"284.0333251953125","sampled_index":"142","source_frame":"8520","start_s":"284.0"},{"end_s":"286.0333251953125","sampled_index":"143","source_frame":"8580","start_s":"286.0"},{"end_s":"288.0333251953125","sampled_index":"144","source_frame":"8640","start_s":"288.0"},{"end_s":"290.0333251953125","sampled_index":"145","source_frame":"8700","start_s":"290.0"},{"end_s":"292.0333251953125","sampled_index":"146","source_frame":"8760","start_s":"292.0"},{"end_s":"294.0333251953125","sampled_index":"147","source_frame":"8820","start_s":"294.0"},{"end_s":"296.0333251953125","sampled_index":"148","source_frame":"8880","start_s":"296.0"},{"end_s":"298.0333251953125","sampled_index":"149","source_frame":"8940","start_s":"298.0"},{"end_s":"300.0333251953125","sampled_index":"150","source_frame":"9000","start_s":"300.0"},{"end_s":"302.0333251953125","sampled_index":"151","source_frame":"9060","start_s":"302.0"},{"end_s":"304.0333251953125","sampled_index":"152","source_frame":"9120","start_s":"304.0"},{"end_s":"306.0333251953125","sampled_index":"153","source_frame":"9180","start_s":"306.0"},{"end_s":"308.0333251953125","sampled_index":"154","source_frame":"9240","start_s":"308.0"},{"end_s":"310.0333251953125","sampled_index":"155","source_frame":"9300","start_s":"310.0"},{"end_s":"312.0333251953125","sampled_index":"156","source_frame":"9360","start_s":"312.0"},{"end_s":"314.0333251953125","sampled_index":"157","source_frame":"9420","start_s":"314.0"},{"end_s":"316.0333251953125","sampled_index":"158","source_frame":"9480","start_s":"316.0"},{"end_s":"318.0333251953125","sampled_index":"159","source_frame":"9540","start_s":"318.0"},{"end_s":"320.0333251953125","sampled_index":"160","source_frame":"9600","start_s":"320.0"},{"end_s":"322.0333251953125","sampled_index":"161","source_frame":"9660","start_s":"322.0"},{"end_s":"324.0333251953125","sampled_index":"162","source_frame":"9720","start_s":"324.0"},{"end_s":"326.0333251953125","sampled_index":"163","source_frame":"9780","start_s":"326.0"}],"video_id":448,"video_path":"/root/autodl-tmp/hermes_causal_pilot/videos/sample_208_real.mp4","video_sha256":"59d2c4b8fb7e55bb933fd2fc681736af5c0299a0aafc3890147cbd175947cf77"}]}'
assert hashlib.sha256(REFERENCE_JSON.encode()).hexdigest() == '5a3939a2b7e6925539d093d2adecbaf5b0c1e9993364703225cd0020f916e3ce'
if not VALIDATE_ONLY:
    import fcntl
    session_lock=(WORK/'.notebook.lock').open('a')
    try:
        fcntl.flock(session_lock,fcntl.LOCK_EX | fcntl.LOCK_NB)
        for name,source in [('stage_runner.py',RUNNER_SOURCE),('stage_context_utils.py',CONTEXT_SOURCE),('context_input_utils.py',INPUT_SOURCE),('diagnostic_pilot_utils.py',UTILS_SOURCE)]:
            target=WORK/name
            if target.exists() and target.read_text(encoding='utf-8')!=source:
                raise RuntimeError('Embedded source changed: use a new WORK directory')
            target.write_text(source,encoding='utf-8',newline='\n')
        target=WORK/'reference.json'
        if target.exists() and target.read_text(encoding='utf-8')!=REFERENCE_JSON:
            raise RuntimeError('Reference changed: use a new WORK directory')
        target.write_text(REFERENCE_JSON,encoding='utf-8',newline='\n')
    except Exception:
        session_lock.close()
        raise
print('EMBEDDED_STAGE_SOURCES_PASS')


EMBEDDED_STAGE_SOURCES_PASS


## 3. GPU gates, 200 main answers and optional separate long case
Masks cover all layers and every decode call. Stop on a failed preflight or historical replay mismatch. Liveness is not proof of successful progress.


In [8]:
if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        repository_revision=BASE_REVISION,runner_sha256=hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest(),
        reuse_candidate=str(REUSE),max_hours=MAX_HOURS,long_case=RUN_LONG_CASE,gpu_driver=smi.stdout.strip()),indent=2),encoding='utf-8')
    try:
        logged([PYTHON,'-u',RUNNER,'--repo',REPO,'--work',WORK,'--model',MODEL_PATH,'--reuse',REUSE,
                '--long-case',str(int(RUN_LONG_CASE)),'--max-hours',str(MAX_HOURS)],'stage.log',ENV,REPO)
    finally:
        try:
            if (WORK/'exports/completion.json').is_file():
                repack="import sys,json; from pathlib import Path; sys.path.insert(0,sys.argv[1]); import stage_runner as s; w=Path(sys.argv[1]); s.package(w,json.loads((w/'exports/completion.json').read_text())['status'])"
                result=probe([PYTHON,'-c',repack,WORK],ENV,REPO)
                print(result.stdout,flush=True)
                assert result.returncode==0,result.stderr[-2000:]
        finally:
            fcntl.flock(session_lock,fcntl.LOCK_UN)
            session_lock.close()
else:
    print('VALIDATION_PASS: every stage notebook cell executed; network/install/GPU and filesystem mutations skipped')


SETUP_STAGE stage.log
STAGE {"detail":"Selecting diagnostic videos","elapsed_s":0.0,"heartbeat_unix_time":1795197286.1954148,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"starting"}
STAGE {"elapsed_s":2.6,"heartbeat_unix_time":1795197288.7758186,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"loading_stage_model"}
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[I 261121 01:54:51 llavaov_hermes:930] n_init: 13
[I 261121 01:54:51 llavaov_hermes:931] kv_size: 4000
FEATURE_REUSE_VERIFIED sample=204
STAGE {"elapsed_s":5.1,"heartbeat_unix_time":1795197291.2904358,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","sample_id":204,"seconds_since_stage_update":0.0,"stage":"gpu_preflight"}
We detected that you are passing `past_key_values` as a t

RuntimeError: Command failed (1); see /root/autodl-tmp/hermes_stage_attribution_v2/logs/stage.log. Failure evidence, if created, stays in WORK.

In [9]:
import json
from pathlib import Path

path = Path("/root/autodl-tmp/hermes_stage_attribution_v2/exports/stage_preflight_diagnostics.json")
report = json.loads(path.read_text())

print("FP32 result:", report["fp32"].get("result", {}).get("status", "missing"))

for row in report["production"]["construction_by_layer"]:
    if row["exceeds_active_bound"]:
        print(
            f"Layer {row['layer']} {row['kind']}: "
            f"{row['exceeds_active_bound']}/{row['elements']} outside tolerance, "
            f"max abs={row['max_abs']:.6f}, RMS={row['rms']:.6f}"
        )

FP32 result: pass
Layer 9 K: 3/25088 outside tolerance, max abs=0.023438, RMS=0.002134
Layer 11 K: 1/25088 outside tolerance, max abs=0.015625, RMS=0.002249
Layer 13 K: 2/25088 outside tolerance, max abs=0.021484, RMS=0.001970
Layer 13 V: 2/25088 outside tolerance, max abs=0.017578, RMS=0.001399
Layer 14 K: 4/25088 outside tolerance, max abs=0.024414, RMS=0.002093
Layer 14 V: 1/25088 outside tolerance, max abs=0.014893, RMS=0.001405
Layer 15 K: 3/25088 outside tolerance, max abs=0.027344, RMS=0.002042
Layer 16 K: 4/25088 outside tolerance, max abs=0.017578, RMS=0.002088
Layer 16 V: 2/25088 outside tolerance, max abs=0.019531, RMS=0.001666
Layer 17 K: 3/25088 outside tolerance, max abs=0.020508, RMS=0.002130
Layer 18 K: 4/25088 outside tolerance, max abs=0.027344, RMS=0.002106
Layer 19 K: 1/25088 outside tolerance, max abs=0.016602, RMS=0.001940
Layer 20 K: 3/25088 outside tolerance, max abs=0.041016, RMS=0.002087
Layer 21 K: 9/25088 outside tolerance, max abs=0.034912, RMS=0.002215
Lay

In [12]:
import shutil
import subprocess
import tempfile
from pathlib import Path

trial = Path(tempfile.mkdtemp(
    prefix="hermes_stage_same_shape_", dir=WORK.parent
))

for name in (
    "stage_context_utils.py",
    "context_input_utils.py",
    "diagnostic_pilot_utils.py",
    "reference.json",
):
    shutil.copy2(WORK / name, trial / name)

source = (WORK / "stage_runner.py").read_text(encoding="utf-8")

# Preserve the FP32 physical-removal reference. In FP16, change only
# the hidden history values and require exactly identical recent K/V.
original = """    restore(model,sink)
    ingest(model,bank,[1],meta,logical_start=stop)
    independent=snapshot(model)
"""

replacement = """    reference_start = s
    if model.dtype == torch.float16:
        changed = prefix_state(states["B"], stop)
        for keys, values in changed["kv"]:
            keys[:, :, s:stop].neg_()
            values[:, :, s:stop].neg_()
        restore(model, changed)
        with HistoryMask(model, s, stop):
            ingest(model, bank, [1], meta)
        reference_start = stop
        kv_atol = kv_rtol = 0.0
    else:
        restore(model, sink)
        ingest(model, bank, [1], meta, logical_start=stop)
    independent = snapshot(model)
"""

assert source.count(original) == 1
assert source.count("ik[:,:,s:]") == source.count("iv[:,:,s:]") == 2
source = source.replace(original, replacement)
source = source.replace("ik[:,:,s:]", "ik[:,:,reference_start:]")
source = source.replace("iv[:,:,s:]", "iv[:,:,reference_start:]")

# Keep the FP32 logit assertion. Record the FP16 full-logit differences.
# The runner's existing top-token assertions still run in both precisions.
old_check = "            torch.testing.assert_close(a,b,atol=atol,rtol=rtol)"
new_check = """            if model.dtype == torch.float16:
                stats = error_statistics(a, b, atol, rtol)
                stats["argmax_equal"] = bool(torch.equal(
                    a[:, -1].argmax(-1), b[:, -1].argmax(-1)
                ))
                diagnostics.setdefault("fp16_logit_comparisons", []).append(stats)
                print("FP16_LOGITS", stats, flush=True)
                if save:
                    save()
            else:
                torch.testing.assert_close(a, b, atol=atol, rtol=rtol)"""

assert source.count(old_check) == 2
source = source.replace(old_check, new_check)

# Require exact FP16 readout agreement when only masked history changes.
readout = "        masked,_,events=probe(model,state,ids,s,stop)"
assert source.count(readout) == 1
source = source.replace(readout, readout + """
        if model.dtype == torch.float16:
            changed = prefix_state(state, cache_length(state["kv"]))
            for keys, values in changed["kv"]:
                keys[:, :, s:stop].neg_()
                values[:, :, s:stop].neg_()
            changed_logits, _, _ = probe(model, changed, ids, s, stop)
            for original_logits, perturbed_logits in zip(masked, changed_logits):
                torch.testing.assert_close(
                    original_logits, perturbed_logits, atol=0, rtol=0,
                    msg="Hidden history changed FP16 readout",
                )
""")

stop = "                preflight_done=True"
assert source.count(stop) == 1
source = source.replace(
    stop,
    '                print("SAME_SHAPE_PREFLIGHT_PASS", flush=True)\n'
    '                return',
)

runner = trial / "stage_runner.py"
compile(source, str(runner), "exec")
runner.write_text(source, encoding="utf-8")

print("Diagnostic folder:", trial, flush=True)

command = [
    str(PYTHON), "-u", str(runner),
    "--repo", str(REPO),
    "--work", str(trial),
    "--model", str(MODEL_PATH),
    "--reuse", str(REUSE),
    "--long-case", "0",
]

with (trial / "diagnostic.log").open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command, env=ENV, cwd=REPO,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as proc:
        for line in proc.stdout:
            log.write(line)
            print(line, end="", flush=True)
        print("Exit code:", proc.wait())

Diagnostic folder: /root/autodl-tmp/hermes_stage_same_shape_kozhxkh8
STAGE {"detail":"Selecting diagnostic videos","elapsed_s":0.0,"heartbeat_unix_time":1795197733.528147,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"starting"}
STAGE {"elapsed_s":2.6,"heartbeat_unix_time":1795197736.1728225,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"loading_stage_model"}
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[I 261121 02:02:18 llavaov_hermes:930] n_init: 13
[I 261121 02:02:18 llavaov_hermes:931] kv_size: 4000
FEATURE_REUSE_VERIFIED sample=204
STAGE {"elapsed_s":5.4,"heartbeat_unix_time":1795197738.887191,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","sample_id":204,"seconds_since_stage_update":0.0,"stage":"gpu_preflight"}
We detected 

In [13]:
import json
import shutil
import subprocess
from IPython.display import FileLink, display

run_work = trial.with_name(trial.name + "_qa")
run_work.mkdir(exist_ok=True)

for name in (
    "stage_context_utils.py",
    "context_input_utils.py",
    "diagnostic_pilot_utils.py",
    "reference.json",
):
    shutil.copy2(trial / name, run_work / name)

# Continue to QA after the same validated preflight.
stop = '                print("SAME_SHAPE_PREFLIGHT_PASS", flush=True)\n                return'
assert source.count(stop) == 1
qa_source = source.replace(
    stop,
    '                print("SAME_SHAPE_PREFLIGHT_PASS", flush=True)\n'
    '                preflight_done=True',
).replace(
    'v2_strict_fp32_structure_then_fp16_production',
    'fp32_physical_reference_and_fp16_same_shape_invariance',
)

qa_runner = run_work / "stage_runner.py"
if qa_runner.exists():
    assert qa_runner.read_text(encoding="utf-8") == qa_source
else:
    qa_runner.write_text(qa_source, encoding="utf-8")

command = [
    str(PYTHON), "-u", str(qa_runner),
    "--repo", str(REPO),
    "--work", str(run_work),
    "--model", str(MODEL_PATH),
    "--reuse", str(REUSE),
    "--long-case", str(int(RUN_LONG_CASE)),
    "--max-hours", str(MAX_HOURS),
]

logs = run_work / "logs"
logs.mkdir(exist_ok=True)
print("Full run folder:", run_work, flush=True)

with (logs / "stage.log").open("a", encoding="utf-8") as log:
    with subprocess.Popen(
        command, env=ENV, cwd=REPO,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as proc:
        for line in proc.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        exit_code = proc.wait()

print("Exit code:", exit_code)
completion = run_work / "exports" / "completion.json"
if completion.exists():
    result = json.loads(completion.read_text(encoding="utf-8"))
    print("Status:", result["status"], "| Main answers:", result.get("main_answers"))

artifact = run_work / "HERMES_stage_attribution_evidence.zip"
if artifact.exists():
    download = Path.cwd() / f"{run_work.name}_evidence.zip"
    shutil.copy2(artifact, download)
    display(FileLink(download.name))

Full run folder: /root/autodl-tmp/hermes_stage_same_shape_kozhxkh8_qa
STAGE {"detail":"Selecting diagnostic videos","elapsed_s":0.0,"heartbeat_unix_time":1795197808.6409774,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"starting"}
STAGE {"elapsed_s":2.5,"heartbeat_unix_time":1795197811.1758268,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","seconds_since_stage_update":0.0,"stage":"loading_stage_model"}
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[I 261121 02:03:33 llavaov_hermes:930] n_init: 13
[I 261121 02:03:33 llavaov_hermes:931] kv_size: 4000
FEATURE_REUSE_VERIFIED sample=204
STAGE {"elapsed_s":5.1,"heartbeat_unix_time":1795197813.7436495,"note":"Heartbeat shows process liveness; it does not prove download or GPU progress","sample_id":204,"seconds_since_stage_update":0.0,"stage":"gpu_preflight"}
We detect

/root/autodl-tmp/HERMES-reproduction/hermes_stage_same_shape_kozhxkh8_qa_evidence.zip

## 4. Download evidence
After a runner error stops Run All, run this final cell manually if failure evidence exists. Earlier setup errors are in WORK/logs.


In [ ]:
if not VALIDATE_ONLY:
    from IPython.display import display, Markdown, FileLink
    completion = json.loads((WORK/'exports/completion.json').read_text(encoding='utf-8'))
    report = WORK/'exports/report.md'
    if report.exists():
        display(Markdown(report.read_text(encoding='utf-8')))
    print('Status:', completion['status'], '| Main answers:', completion.get('main_answers', 'see checkpoints'))
    if completion['status'] != 'complete':
        print('Partial results are preserved. A matching rerun resumes checkpoints; failed setup must be fixed first.')
    artifact = WORK/'HERMES_stage_attribution_evidence.zip'
    link = Path.cwd()/artifact.name
    if link.resolve() != artifact.resolve():
        if link.is_symlink():
            link.unlink()
        # Always refresh the small evidence copy, never link/copy videos or weights here.
        shutil.copy2(artifact, link)
    display(FileLink(artifact.name))
    print('Evidence:', artifact, '| MiB:', round(artifact.stat().st_size/1024**2, 2))
    print('Send this ZIP and the saved executed notebook back for analysis. Preserve asset caches for reuse.')
